# Solar Filament 2026 -- PQ Pipeline (Seg(YOLO26)->Embedding(DINOv3))

## tl;dr

This is a complete train -> grouped OOF -> test inference notebook:

1. group physical observations before splitting annotator records;
2. choose the medoid annotation record for instance training;
3. train a YOLO26 segmentation detector (Stage 1, "Seg");
4. train a DINOv3-backboned discriminative-embedding net conditioned on the detector's output (Stage 2, "Embedding");
5. evaluate the detector -> embedding-net -> greedy-cluster decode path with Panoptic Quality;
6. train one final model on ~95% of all labeled data and write `submission.csv`.

Stage 2 was an energy-regression net decoded via `cv2.watershed` until the "embedding-head instance-separation swap" (see PROGRESS.md): watershed is known to struggle on thin, elongated, touching shapes -- this competition's exact failure mode -- relative to discriminative-embedding separation (Cellpose-style). The energy path is kept, not deleted (`train_yolo_conditioned_energy_net`, `predict_yolo_energy_instances`).

By default only fold 0 trains/scores -- a quick, honest sanity check (a real ~80/20 split drawn
from the full 5-fold grouped split, `cfg.n_folds=5`), not a full 5-fold CV sweep.
`train_all_folds`/`run_parallel_training` are still available for a full CV run; see the
Training section.

The notebook uses only official train annotations and official H-alpha images. It does **not** use external test labels, MAGFiLO mirrors containing test annotations, or leaderboard exploits.


## August 7 metric update

The [competition evaluation page](https://www.kaggle.com/competitions/filament-segmentation-2026/overview/evaluation) changed the primary metric from Dice to **Panoptic Quality (PQ)** on 7 August 2026:

\[
PQ = \frac{\sum_{(y,\hat y)\in TP} IoU(y,\hat y)}{|TP| + 0.5|FP| + 0.5|FN|}.
\]

Following the original PQ paper cited by the organizers, a pair is a true positive only when `IoU > 0.5`. The notebook reports:

- **SQ**: mean IoU of matched instances;
- **RQ**: recognition quality, `TP / (TP + 0.5 FP + 0.5 FN)`;
- **PQ = SQ × RQ**.

This materially changes model selection: every missing and extra filament matters, and fragmented/merged predictions usually fail the `IoU > 0.5` match.

## Kaggle run contract

- Attach the official `filament-segmentation-2026` competition data.
- Enable a GPU; two T4s are supported for parallel fold training.
- Enable Internet for first-run package installation and pretrained weights.
- By default only fold 0 trains/scores (quick sanity check); the single final-fit model
  (trained on ~95% of all data) is what actually produces `submission.csv`. Checkpoints are
  reused when cells are rerun. Use `train_all_folds`/`run_parallel_training` for a full 5-fold
  CV score instead.
- Set `DEBUG=True` for a two-epoch smoke run on small subsets.

In [1]:
from pathlib import Path
import importlib.util, os, subprocess, sys

# Must be set before torch initializes the CUDA allocator (torch itself is imported later, in
# the core module cell) -- mitigates the fragmentation PyTorch's own OOM message pointed at
# ("1.90 GiB is reserved by PyTorch but unallocated" while a 1.16 GiB request still failed).
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

KAGGLE_DATA_AVAILABLE = Path('/kaggle/input').exists()
if KAGGLE_DATA_AVAILABLE:
    specs = {
        'ultralytics': 'ultralytics==8.4.103',
        'pycocotools': 'pycocotools==2.0.11',
        'transformers': 'transformers>=4.56.0',
        'tqdm': 'tqdm>=4.66.0',
        'timm': 'timm>=1.0.0',  # semantic challenger encoders (FilamentUNet)
    }
    missing = [pkg for pkg in specs if importlib.util.find_spec(pkg) is None]
    if missing:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *[specs[m] for m in missing]])
    try:
        from kaggle_secrets import UserSecretsClient
        from huggingface_hub import login
        login(token=UserSecretsClient().get_secret('HF_TOKEN'))
    except Exception as error:
        print('HF_TOKEN secret not found or login failed. DINOv3 checkpoints are gated: accept the '
              'license at huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m with your HF account, '
              'create an access token, and add it as a Kaggle notebook secret named HF_TOKEN.', repr(error))
print({'kaggle_data_available': KAGGLE_DATA_AVAILABLE})


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 43.3 MB/s eta 0:00:00
HF_TOKEN secret not found or login failed. DINOv3 checkpoints are gated: accept the license at huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m with your HF account, create an access token, and add it as a Kaggle notebook secret named HF_TOKEN. ConnectionError('Connection error trying to communicate with service.')
{'kaggle_data_available': True}


## Pipeline

### 1. Configuration, COCO indexing, medoid annotators, and grouped folds

In [2]:
%%writefile solar_filament_lib.py
"""Full leakage-safe Solar Filament train/OOF/test pipeline for Kaggle.

Primary metric: Panoptic Quality (competition update, 7 Aug 2026).
The module uses only official train annotations and H-alpha images.
"""

from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import re
import random
import shutil
import subprocess
import sys
import time
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Any, Mapping, Sequence

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# Belt-and-suspenders: a Hugging Face model (DINOv3, via GlobalViTRefiner) previously hit a
# 40+ GiB single-tensor CUDA OOM inside a torch._inductor-compiled backward kernel -- a shape
# bug, not a real memory shortfall. Disabling Dynamo/compile globally keeps every model in this
# notebook on the plain eager path, so nothing gets silently JIT-compiled underneath us.
torch._dynamo.config.disable = True

from pycocotools import mask as mask_utils
from scipy.optimize import linear_sum_assignment
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18
from tqdm.auto import tqdm


@dataclass
class CFG:
    seed: int = 20260729
    n_folds: int = 5
    image_size: int = 1024
    detector_model: str = "yolo26s-seg.pt"
    detector_epochs: int = 30
    detector_batch: int = 4
    detector_patience: int = 7
    detector_conf: float = 0.20
    detector_fallback_conf: float = 0.05
    detector_iou: float = 0.60
    # YOLO26-specific tuning, chosen for THIS competition's specific shape (many thin,
    # closely-packed/touching filament instances per image, mean ~7.3/image, single class,
    # strict IoU>0.5 metric) -- not generic "best practice" defaults. See PROGRESS.md.
    #
    # nms=False uses YOLO26's NMS-free one-to-one head (trained regardless of this choice, at
    # no extra cost -- see yolo_region_mask, cell 13). Standard box-IoU NMS risks discarding a
    # genuine second instance whose box overlaps a higher-confidence neighbor's -- exactly the
    # risk profile of closely-packed filaments. Docs note the NMS head "typically achieves
    # slightly higher accuracy" in general-purpose benchmarks; that's a generic-object-detection
    # finding, not evidence for our specific touching-instance failure mode, which is the actual
    # thing being diagnosed here.
    detector_nms: bool = False
    # overlap_mask=True (Ultralytics' own default; this codebase had it hardcoded to False
    # before this entry) merges overlapping instance masks into one per-pixel-argmax training
    # target instead of keeping every instance mask as an independent channel -- the
    # recommended setting specifically for scenes with many instances per image, which is
    # exactly our case (mean ~7.3/image) rather than the few-large-object case False better
    # suits.
    detector_overlap_mask: bool = True
    # Pastes instances from one training image onto another -- directly synthesizes MORE
    # densely-packed, touching-instance training examples from a relatively small dataset
    # (~1,593 images), manufacturing more of exactly the scenario (multiple close instances per
    # image) that's hardest for the model, rather than hoping the ~1,593 real images already
    # cover it well. Was not used at all before this entry.
    detector_copy_paste: float = 0.3
    # box/cls loss gains: our metric (PQ) is entirely IoU-based, and classification is nearly
    # free here (single_cls=True, one class) -- upweighting box relative to Ultralytics' own
    # defaults (box=7.5, cls=0.5) pushes training to prioritize the localization precision PQ
    # actually scores, over a classification task that's already essentially solved by
    # construction. dfl is deliberately left unset/unpassed: YOLO26 removes DFL entirely (see
    # PROGRESS.md/YOLO26 docs), so passing it risked an API mismatch against a loss term that
    # may no longer exist.
    detector_box_gain: float = 9.0
    detector_cls_gain: float = 0.3
    mask_ratio: int = 1  # was 2 -- lower preserves more mask detail for thin filament boundaries
    refiner_crop: int = 256
    refiner_context: float = 1.8
    refiner_min_crop: int = 96
    refiner_epochs: int = 25
    refiner_batch: int = 16
    refiner_lr: float = 3e-4
    refiner_threshold: float = 0.65
    refiner_backbone: str = "vit"  # "vit" (DINOv2/DINOv3 via transformers AutoBackbone) or "resnet18"
    # HF access to facebook/dinov3-* approved 2026-09-26. Fall back to "facebook/dinov2-with-registers-base"
    # (ungated, Apache-2.0) if this notebook is ever run before DINOv3 access is available again.
    vit_hub_id: str = "facebook/dinov3-vits16-pretrain-lvd1689m"
    vit_out_indices: tuple = (3, 6, 9, 12)
    vit_unfreeze_last_n: int = 4
    vit_backbone_lr_mult: float = 0.1
    # Global-context redesign: the backbone runs once per full image (not once per crop), so
    # instance decoding uses ROIAlign on that single feature map instead of re-encoding crops.
    vit_image_size: int = 1024
    vit_roi_size: int = 16
    vit_images_per_batch: int = 2
    minimum_area: int = 96
    pq_match_iou: float = 0.50
    ensemble_iou: float = 0.50
    minimum_fold_support: int = 2
    singleton_score: float = 0.90
    num_workers: int = 2
    use_tta: bool = True
    save_oof_masks: bool = False
    # Fraction of the full labeled set held out (via build_final_manifest, reusing
    # assign_grouped_folds at a higher n_folds) purely as a live early-stopping signal for the
    # single final submission model -- not used by the 5-fold CV path, which is evaluation-only.
    final_holdout_fraction: float = 0.05
    # Region+boundary dense segmenter (replaces the YOLO+ROI-refiner path when
    # refiner_backbone == "vit"; see RegionBoundarySegmenter). boundary_width is the ring
    # radius in pixels, each side, at vit_image_size resolution; boundary_loss_weight is
    # lambda in L_total = L_region + boundary_loss_weight * L_boundary.
    boundary_width: int = 3
    boundary_loss_weight: float = 1.0
    # Deep-Watershed-Transform-style energy head (third head, additive -- see
    # RegionBoundarySegmenter/build_semantic_targets): a per-pixel [0,1] regression target,
    # each instance's own distance-transform normalized by its own max so thin and thick
    # filaments both span the same [0,1] range, peaking at each instance's medial axis.
    # Existing region+boundary supervision is a hard classification signal that carries no
    # information about *where inside a blob* one instance's center is -- energy adds exactly
    # that, which separate_instances_watershed uses as a learned seed/landscape signal instead
    # of a geometric distance-transform of the (already-merged) semantic region mask. See
    # PROGRESS.md. energy_loss_weight is lambda in L_total's third term (MSE, not the
    # bce+dice blend the other two heads use -- energy is a continuous regression target, not
    # a binary mask). energy_seed_threshold gates which pixels count as high-confidence seeds
    # at decode time (separate_instances_watershed's `energy` argument).
    energy_loss_weight: float = 1.0
    energy_seed_threshold: float = 0.5
    # Decode-time only toggle, not part of any training fingerprint: when False,
    # predict_yolo_energy_instances skips the energy net entirely and returns YOLO's own
    # instance masks unrefined, to isolate the detector's standalone PQ contribution.
    use_energy_watershed: bool = False
    # ReduceLROnPlateau (monitoring val_crop_dice) + early stopping, replacing a fixed
    # CosineAnnealingLR(T_max=refiner_epochs) schedule that decayed LR to ~0 by epoch
    # refiner_epochs regardless of whether validation was still improving -- see PROGRESS.md.
    # refiner_epochs is now a ceiling, not a target: training stops early if val_crop_dice
    # hasn't improved for early_stopping_patience epochs.
    lr_plateau_factor: float = 0.5
    lr_plateau_patience: int = 2
    early_stopping_patience: int = 4
    # Discriminative instance-embedding head (replaces energy+watershed as the active
    # Stage 2 decode -- see PROGRESS.md "embedding-head instance-separation swap"): instead
    # of regressing a continuous energy ridge and relying on cv2.watershed to split touching
    # instances, the net predicts a per-pixel embedding_dim-D vector; pixels of the same
    # instance are pulled together (within embedding_delta_v of their instance mean) and
    # different instances' mean embeddings pushed apart (beyond 2*embedding_delta_d) via a
    # discriminative loss (De Brabandere et al. 2017). Chosen because watershed is known to
    # struggle on thin, elongated, touching shapes -- this competition's exact failure mode --
    # relative to flow/embedding-based separation (Cellpose-style). See PROGRESS.md.
    embedding_dim: int = 8
    embedding_delta_v: float = 0.5
    embedding_delta_d: float = 1.5
    embedding_reg_weight: float = 1e-3
    # Stage 2 trains on YOLO masks for YOLO's own training images, which are much cleaner
    # than the masks it gets at test time. This is the probability that a training sample's
    # region mask is degraded toward test-time quality (see corrupt_region_mask). 0 disables it.
    region_mask_corruption: float = 0.5
    # Decode-time only toggle, not part of any training fingerprint -- mirrors
    # use_energy_watershed's role for the (now dormant) energy decode: when False,
    # predict_yolo_embedding_instances skips the embedding net and returns YOLO's own instance
    # masks unrefined.
    use_embedding_decode: bool = True
    # Per-epoch checkpoint/early-stopping selection for the embedding nets runs the ACTUAL
    # decode (YOLO instance masks -> embedding clustering) and scores real PQ against ground
    # truth, instead of val_embedding_loss (a proxy -- lower embedding-space distance doesn't
    # provably mean better decoded masks once clustering is in the loop). Capped at this many
    # validation images per epoch (a FIXED prefix, not re-sampled each epoch, so PQ changes
    # reflect real model improvement, not sampling noise) -- full decode is cheap CPU-side
    # clustering, but still costs one extra YOLO+DINOv3 forward pass per image per epoch, so
    # this bounds that cost instead of running it over an entire (possibly ~300+ image) fold.
    # See PROGRESS.md.
    # None = the full validation fold (default). 64 images gave a PQ too noisy to pick an epoch
    # by (bootstrap_pq_ci makes that visible); the val-loss pass already runs the DINOv3 forward
    # over the full fold every epoch, so full-fold PQ only adds YOLO inference + CPU clustering
    # per image. Set an int to cap it again if epochs get too slow.
    embedding_pq_eval_cap: int | None = None
    # Whether an image whose decode finds nothing gets classical_fallback_seed's dummy blob.
    # Default False: under the official pooled metric an empty prediction on an image costs
    # nothing, while the blob is almost always a false positive; and a public entrant whose
    # scorer is checked against the organizer notebook submits "no dummy mask rows" for
    # zero-detection images (srivatsav-kannan/solar-seg). Not confirmed on the Kaggle page itself
    # (unreadable here) -- flip to True if a submission without a row for every image is ever
    # rejected. See PROGRESS.md.
    use_classical_fallback: bool = False
    debug: bool = False


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def discover_competition_paths() -> dict[str, Path]:
    roots = sorted(Path("/kaggle/input").glob("**/MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
    if len(roots) != 1:
        raise RuntimeError(f"Expected one official train JSON, found {roots}")
    annotation_path = roots[0]
    data_root = annotation_path.parent.parent
    train_dir = data_root / "train" / "train_images"
    test_dir = data_root / "test" / "test_images"
    if not train_dir.is_dir() or not test_dir.is_dir():
        raise FileNotFoundError(f"Invalid competition tree under {data_root}")
    return {"root": data_root, "json": annotation_path, "train": train_dir, "test": test_dir}


def load_coco(annotation_path: Path) -> dict[str, Any]:
    with annotation_path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def build_coco_indices(coco: Mapping[str, Any]) -> dict[str, Any]:
    images_by_id = {str(image["id"]): image for image in coco["images"]}
    annotations_by_image: dict[str, list[Mapping[str, Any]]] = {}
    for annotation in coco["annotations"]:
        annotations_by_image.setdefault(
            str(annotation["image_id"]), []
        ).append(annotation)
    image_ids_by_file: dict[str, list[str]] = {}
    for image_id, image in images_by_id.items():
        image_ids_by_file.setdefault(image["file_name"], []).append(image_id)
    for filename in image_ids_by_file:
        image_ids_by_file[filename].sort()
    return {
        "images_by_id": images_by_id,
        "annotations_by_image": annotations_by_image,
        "image_ids_by_file": image_ids_by_file,
    }


def annotation_to_rle(annotation: Mapping[str, Any], height: int, width: int) -> dict[str, Any]:
    segmentation = annotation.get("segmentation")
    if isinstance(segmentation, dict):
        rle = dict(segmentation)
        if isinstance(rle.get("counts"), list):
            rle = mask_utils.frPyObjects(rle, height, width)
        return rle
    polygons = segmentation or []
    if not polygons:
        return mask_utils.encode(np.asfortranarray(np.zeros((height, width), dtype=np.uint8)))
    rles = mask_utils.frPyObjects(polygons, height, width)
    return mask_utils.merge(rles)


def annotation_to_mask(annotation: Mapping[str, Any], height: int, width: int) -> np.ndarray:
    return mask_utils.decode(annotation_to_rle(annotation, height, width)).astype(bool)


def semantic_rle_for_record(
    image_id: int,
    images_by_id: Mapping[int, Mapping[str, Any]],
    annotations_by_image: Mapping[int, Sequence[Mapping[str, Any]]],
) -> dict[str, Any]:
    image = images_by_id[image_id]
    rles = [
        annotation_to_rle(annotation, int(image["height"]), int(image["width"]))
        for annotation in annotations_by_image.get(image_id, [])
        if not annotation.get("iscrowd", 0)
    ]
    if not rles:
        return mask_utils.encode(
            np.asfortranarray(np.zeros((int(image["height"]), int(image["width"])), dtype=np.uint8))
        )
    return mask_utils.merge(rles)


def choose_medoid_annotators(indices: Mapping[str, Any]) -> dict[str, int]:
    """Choose the annotator closest to the others, avoiding max-area bias."""
    selected: dict[str, int] = {}
    for filename, image_ids in indices["image_ids_by_file"].items():
        if len(image_ids) == 1:
            selected[filename] = image_ids[0]
            continue
        union_rles = [
            semantic_rle_for_record(
                image_id, indices["images_by_id"], indices["annotations_by_image"]
            )
            for image_id in image_ids
        ]
        pairwise_iou = np.asarray(mask_utils.iou(union_rles, union_rles, [0] * len(union_rles)))
        mean_agreement = (pairwise_iou.sum(axis=1) - 1.0) / max(len(image_ids) - 1, 1)
        selected[filename] = image_ids[int(np.argmax(mean_agreement))]
    return selected


def build_observation_table(indices: Mapping[str, Any]) -> pd.DataFrame:
    rows = []
    for filename, image_ids in sorted(indices["image_ids_by_file"].items()):
        counts = [len(indices["annotations_by_image"].get(image_id, [])) for image_id in image_ids]
        rows.append(
            {
                "filename": filename,
                "site": Path(filename).stem[-2:],
                "n_annotators": len(image_ids),
                "mean_instances": float(np.mean(counts)),
                "max_instances": max(counts, default=0),
            }
        )
    return pd.DataFrame(rows)


def assign_grouped_folds(observations: pd.DataFrame, cfg: CFG) -> pd.DataFrame:
    frame = observations.copy().sort_values("filename").reset_index(drop=True)
    count_bin = pd.qcut(
        frame["mean_instances"].rank(method="first"),
        q=min(5, len(frame)), labels=False, duplicates="drop",
    ).astype(int)
    frame["stratum"] = frame["site"] + "_q" + count_bin.astype(str)
    frame["fold"] = -1
    fold_sizes = np.zeros(cfg.n_folds, dtype=int)
    rng = np.random.default_rng(cfg.seed)
    strata = sorted(frame.groupby("stratum").groups.items(), key=lambda item: (-len(item[1]), item[0]))
    for _, row_indices in strata:
        row_indices = np.asarray(list(row_indices), dtype=int)
        rng.shuffle(row_indices)
        for row_index in row_indices:
            choices = np.flatnonzero(fold_sizes == fold_sizes.min())
            fold = int(rng.choice(choices))
            frame.loc[row_index, "fold"] = fold
            fold_sizes[fold] += 1
    assert frame["filename"].is_unique
    assert (frame["fold"] >= 0).all()
    return frame


def build_final_manifest(observations: pd.DataFrame, cfg: CFG) -> pd.DataFrame:
    """Splits the full labeled set into a ~(1 - final_holdout_fraction) train portion and a
    fresh, independent ~final_holdout_fraction holdout (fold == 0), reusing assign_grouped_folds's
    existing balanced/stratified assignment unchanged -- its qcut stratification bin count is
    hardcoded to min(5, len(frame)) regardless of cfg.n_folds, so bumping n_folds here just makes
    each bucket smaller (~5% each at n_folds=20) without needing any new splitting logic. Only
    fold == 0 (the holdout) is ever referenced by name; train_fold's existing fold != 0 mask
    correctly captures the rest regardless of how many other bucket values (1..n-1) they land in.
    """
    n_holdout_groups = max(round(1 / cfg.final_holdout_fraction), 2)
    return assign_grouped_folds(observations, replace(cfg, n_folds=n_holdout_groups))


Writing solar_filament_lib.py


In [3]:
%%writefile -a solar_filament_lib.py


# ---------------------------------------------------------------------------
# Resume-safety helpers: atomic checkpoint/metadata writes, and scoped config
# fingerprints (train_detector/train_refiner use these to tell "safe to skip
# or resume" apart from "must retrain", instead of the old file-existence-only
# check, which could silently trust a checkpoint saved after just one epoch).
# ---------------------------------------------------------------------------

def _atomic_write(write_fn, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    tmp = destination.with_name(destination.name + f".tmp.{os.getpid()}")
    try:
        write_fn(tmp)
        os.replace(tmp, destination)  # atomic overwrite on both POSIX and Windows
    finally:
        tmp.unlink(missing_ok=True)


def atomic_torch_save(obj: Any, destination: Path) -> None:
    _atomic_write(lambda tmp: torch.save(obj, tmp), destination)


def atomic_write_json(payload: Any, destination: Path) -> None:
    _atomic_write(lambda tmp: tmp.write_text(json.dumps(payload, indent=2), encoding="utf-8"), destination)


def atomic_copy_file(source: Path, destination: Path) -> None:
    _atomic_write(lambda tmp: shutil.copy2(source, tmp), destination)


def now_iso() -> str:
    from datetime import datetime, timezone
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def read_json_if_valid(path: Path) -> dict | None:
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        return None


# Only fields that actually change what gets *trained* -- deliberately excludes pure
# post-processing/inference knobs (detector_conf, ensemble_iou, minimum_fold_support,
# singleton_score, use_tta, ...) so the OOF-driven post-processing sweep (audit item #1)
# doesn't force a full retrain on every iteration. refiner_threshold is included for the
# refiner despite being an inference-style knob elsewhere: both refiner training loops use
# it inside the val loop to compute the dice that gates which epoch's weights are "best".
_DETECTOR_CFG_FIELDS = (
    "seed", "n_folds", "image_size", "detector_model", "detector_epochs",
    "detector_batch", "detector_patience", "mask_ratio", "detector_overlap_mask",
    "detector_copy_paste", "detector_box_gain", "detector_cls_gain", "debug",
)
_REFINER_CFG_FIELDS = (
    "seed", "n_folds", "refiner_crop", "refiner_context", "refiner_min_crop",
    "refiner_epochs", "refiner_batch", "refiner_lr", "refiner_threshold",
    "refiner_backbone", "vit_hub_id", "vit_out_indices", "vit_unfreeze_last_n",
    "vit_backbone_lr_mult", "vit_image_size", "vit_roi_size", "vit_images_per_batch",
    "boundary_width", "boundary_loss_weight", "energy_loss_weight", "energy_seed_threshold",
    "lr_plateau_factor", "lr_plateau_patience", "early_stopping_patience", "debug",
)


def _scoped_fingerprint(cfg: "CFG", fields: Sequence[str], extra: str = "") -> str:
    payload = json.dumps({field: getattr(cfg, field) for field in fields}, sort_keys=True) + "\n" + extra
    return hashlib.sha256(payload.encode()).hexdigest()


def detector_config_fingerprint(cfg: "CFG") -> str:
    # Does not fold in fold_manifest: train_detector doesn't currently receive it (only a
    # pre-built data_yaml), and fold membership is already (imperfectly) tracked by
    # prepare_yolo_fold's own .ready marker -- same pre-existing, already-flagged gap, not
    # introduced here. Out of scope to thread fold_manifest through for this pass.
    return _scoped_fingerprint(cfg, _DETECTOR_CFG_FIELDS)


def refiner_config_fingerprint(cfg: "CFG", fold_manifest: pd.DataFrame) -> str:
    manifest_digest = "\n".join(
        f"{row.filename},{row.fold}" for row in fold_manifest.sort_values("filename").itertuples()
    )
    # Also fold in a hash of this module's own currently-running source. CFG fields and the
    # fold manifest alone can't detect a pure code change (e.g. a model-forward bugfix) --
    # execution-confirmed risk: a real run left a status="complete" checkpoint behind from a
    # NaN-poisoned training pass (val dice ~0.0 still beats best_dice's -1.0 init), which a
    # naive rerun would silently reuse forever after any code-only fix. See PROGRESS.md.
    code_digest = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()
    return _scoped_fingerprint(cfg, _REFINER_CFG_FIELDS, manifest_digest + "\n" + code_digest)


# Seg(YOLO)->Energy pipeline (2026-09-30, see PROGRESS.md): Stage 2 (YoloConditionedEnergyNet)
# is trained against a FROZEN, already-trained YOLO detector's own output -- its cache must
# invalidate not only on its own CFG fields/code changes but also whenever the detector it
# depends on changes (a different fingerprint, e.g. a config sweep or a retrain), since Stage 2
# would otherwise be silently trained/scored against stale detector masks. detector_conf/
# detector_fallback_conf/detector_iou/detector_nms are included despite being inference-style
# knobs elsewhere: they directly change what yolo_region_mask actually produces for Stage 2 to
# condition on, unlike (say) refiner_threshold which is applied only at final decode time.
# detector_overlap_mask/detector_copy_paste/detector_box_gain/detector_cls_gain are NOT listed
# here directly -- they're training-time-only args passed to model.train(), so a change to any
# of them is already covered transitively: it changes detector_config_fingerprint(cfg), which
# is folded into this fingerprint via the detector_fingerprint argument below.
_YOLO_ENERGY_CFG_FIELDS = (
    "seed", "n_folds", "refiner_lr", "refiner_backbone", "vit_hub_id", "vit_out_indices",
    "vit_unfreeze_last_n", "vit_backbone_lr_mult", "vit_image_size", "vit_images_per_batch",
    "energy_loss_weight", "energy_seed_threshold", "lr_plateau_factor", "lr_plateau_patience",
    "early_stopping_patience", "detector_conf", "detector_fallback_conf", "detector_iou",
    "detector_nms", "image_size", "use_tta", "debug",
)


def yolo_energy_fingerprint(cfg: "CFG", fold_manifest: pd.DataFrame, detector_fingerprint: str) -> str:
    manifest_digest = "\n".join(
        f"{row.filename},{row.fold}" for row in fold_manifest.sort_values("filename").itertuples()
    )
    code_digest = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()
    return _scoped_fingerprint(
        cfg, _YOLO_ENERGY_CFG_FIELDS, manifest_digest + "\n" + detector_fingerprint + "\n" + code_digest
    )


# Mirrors _YOLO_ENERGY_CFG_FIELDS/yolo_energy_fingerprint exactly, for the embedding-head
# decode that replaces energy+watershed as the active Stage 2 (see PROGRESS.md). Same
# detector-dependency reasoning applies: Stage 2 conditions on the frozen detector's region
# mask, so this fingerprint must invalidate whenever the detector's own fingerprint changes.
_YOLO_EMBEDDING_CFG_FIELDS = (
    "seed", "n_folds", "refiner_lr", "refiner_backbone", "vit_hub_id", "vit_out_indices",
    "vit_unfreeze_last_n", "vit_backbone_lr_mult", "vit_image_size", "vit_images_per_batch",
    "embedding_dim", "embedding_delta_v", "embedding_delta_d", "embedding_reg_weight",
    "region_mask_corruption",
    "embedding_pq_eval_cap", "lr_plateau_factor", "lr_plateau_patience", "early_stopping_patience",
    "detector_conf", "detector_fallback_conf", "detector_iou", "detector_nms", "image_size",
    "use_tta", "minimum_area", "pq_match_iou", "use_classical_fallback", "debug",
)


def yolo_embedding_fingerprint(cfg: "CFG", fold_manifest: pd.DataFrame, detector_fingerprint: str) -> str:
    manifest_digest = "\n".join(
        f"{row.filename},{row.fold}" for row in fold_manifest.sort_values("filename").itertuples()
    )
    code_digest = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()
    return _scoped_fingerprint(
        cfg, _YOLO_EMBEDDING_CFG_FIELDS, manifest_digest + "\n" + detector_fingerprint + "\n" + code_digest
    )


def _record_refiner_epoch(
    meta_path: Path, fold: int, fingerprint: str, target_epochs: int, epoch: int,
    validation_dice: float, is_best: bool, best_dice: float,
    train_diag: dict | None = None, val_diag: dict | None = None,
) -> None:
    """Called every epoch (not just on a new best) so epochs_completed always reflects real
    progress -- this is what stops a crash at, say, epoch 5/8 from leaving behind a checkpoint
    that a later resume check would wrongly trust as fully trained."""
    meta = read_json_if_valid(meta_path) or {}
    meta.update(
        stage="refiner", status="in_progress", fold=fold, config_fingerprint=fingerprint,
        target_epochs=target_epochs, epochs_completed=epoch + 1, best_metric_value=best_dice,
        attempt_updated_at=now_iso(),
    )
    meta.setdefault("history", []).append({
        "epoch": epoch + 1, "val_crop_dice": validation_dice, "is_best": is_best,
        "timestamp": now_iso(), "train_diag": train_diag, "val_diag": val_diag,
    })
    atomic_write_json(meta, meta_path)


def _finalize_refiner_meta(meta_path: Path, fold: int, fingerprint: str, target_epochs: int) -> None:
    meta = read_json_if_valid(meta_path) or {}
    meta.update(status="complete", fold=fold, config_fingerprint=fingerprint,
                target_epochs=target_epochs, attempt_updated_at=now_iso())
    atomic_write_json(meta, meta_path)


Appending to solar_filament_lib.py


### 2. Single-class YOLO dataset and detector training

The train split also gets two on-disk photometric augmented copies per image (CLAHE contrast, mild Gaussian noise) written next to the originals -- see `photometric_augmentations`/`write_yolo_split` below.


In [4]:
%%writefile -a solar_filament_lib.py


def mask_to_yolo_polygon(mask: np.ndarray, width: int, height: int) -> str | None:
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    contour = max(contours, key=cv2.contourArea).reshape(-1, 2)
    if len(contour) < 3:
        return None
    contour = contour.astype(np.float32)
    contour[:, 0] = np.clip(contour[:, 0] / width, 0, 1)
    contour[:, 1] = np.clip(contour[:, 1] / height, 0, 1)
    return "0 " + " ".join(f"{value:.6f}" for value in contour.ravel())


def audit_multi_contour_instances(indices: Mapping[str, Any]) -> dict[str, float | int]:
    """Measurement only, no behavior change -- see PROGRESS.md audit item #3.
    mask_to_yolo_polygon (above) keeps only max(contours, key=cv2.contourArea): any OTHER
    contour of a multi-part/occluded instance mask (e.g. a filament split by an occluding
    feature, or two legitimately disconnected blobs sharing one annotation) silently never
    becomes training-label area. This tallies how often that actually happens and how much area
    it costs, across EVERY non-crowd annotation (all folds -- a data-quality question, not a
    per-fold model one), so the fix is prioritized by its real frequency rather than guessed at.
    Pure CPU, no GPU/trained model needed."""
    affected_instances = total_instances = 0
    area_loss_fractions: list[float] = []
    for image_id, info in indices["images_by_id"].items():
        height, width = int(info["height"]), int(info["width"])
        for annotation in indices["annotations_by_image"].get(image_id, []):
            if annotation.get("iscrowd", 0):
                continue
            total_instances += 1
            mask = annotation_to_mask(annotation, height, width)
            contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if len(contours) <= 1:
                continue
            areas = sorted((cv2.contourArea(contour) for contour in contours), reverse=True)
            total_area = sum(areas)
            if total_area <= 0:
                continue
            dropped_fraction = sum(areas[1:]) / total_area
            if dropped_fraction > 0:
                affected_instances += 1
                area_loss_fractions.append(dropped_fraction)
    return {
        "affected_instances": affected_instances,
        "total_instances": total_instances,
        "affected_fraction": affected_instances / total_instances if total_instances else 0.0,
        "mean_area_loss_fraction": float(np.mean(area_loss_fractions)) if area_loss_fractions else 0.0,
    }


def photometric_augmentations(image: np.ndarray, rng: np.random.Generator) -> list[tuple[str, np.ndarray]]:
    """On-disk photometric variants of a training image, pixel-only so the existing YOLO polygon
    labels apply unchanged (no geometry to rewarp). Deliberately NOT flips/rotations/scale/mosaic
    -- train_detector already gets those live from Ultralytics every epoch (see its hyperparameters),
    so duplicating them as fixed on-disk copies would add redundant, non-fresh variety for no
    benefit. hsv_h/hsv_s are pinned to 0 for this grayscale H-alpha data (see CFG), so what
    Ultralytics' online augmentation does NOT cover is local-contrast/exposure variation and
    sensor-noise robustness -- CLAHE and mild Gaussian noise, respectively."""
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(image)
    noisy = np.clip(image.astype(np.float32) + rng.normal(0, 6.0, image.shape), 0, 255).astype(np.uint8)
    return [("clahe", clahe), ("noise", noisy)]


def write_yolo_split(
    split_dir: Path,
    filenames: Sequence[str],
    train_image_dir: Path,
    selected_image_ids: Mapping[str, int],
    indices: Mapping[str, Any],
    augment: bool = False,
) -> None:
    image_output = split_dir / "images"
    label_output = split_dir / "labels"
    image_output.mkdir(parents=True, exist_ok=True)
    label_output.mkdir(parents=True, exist_ok=True)
    for filename in filenames:
        source = train_image_dir / filename
        destination = image_output / filename
        if not destination.exists():
            os.symlink(source, destination)
        image_id = selected_image_ids[filename]
        info = indices["images_by_id"][image_id]
        lines = []
        for annotation in indices["annotations_by_image"].get(image_id, []):
            if annotation.get("iscrowd", 0):
                continue
            mask = annotation_to_mask(annotation, int(info["height"]), int(info["width"]))
            line = mask_to_yolo_polygon(mask, int(info["width"]), int(info["height"]))
            if line is not None:
                lines.append(line)
        lines_text = "\n".join(lines)
        (label_output / f"{Path(filename).stem}.txt").write_text(lines_text, encoding="utf-8")

        if augment:
            # Train split only (the caller never passes augment=True for val) -- augmenting
            # validation images would just be duplicate pixel data of instances already seen,
            # not a fair held-out check. Labels are the SAME polygons as the original (pixel-only
            # transform), so lines_text is reused verbatim, no re-encoding.
            stem = Path(filename).stem
            ext = Path(filename).suffix
            rng = np.random.default_rng(abs(hash(filename)) % (2 ** 32))
            image = cv2.imread(str(source), cv2.IMREAD_GRAYSCALE)
            for tag, variant in photometric_augmentations(image, rng):
                cv2.imwrite(str(image_output / f"{stem}_{tag}{ext}"), variant)
                (label_output / f"{stem}_{tag}.txt").write_text(lines_text, encoding="utf-8")


def prepare_yolo_fold(
    fold: int,
    fold_manifest: pd.DataFrame,
    train_image_dir: Path,
    selected_image_ids: Mapping[str, int],
    indices: Mapping[str, Any],
    work_dir: Path,
) -> Path:
    fold_dir = work_dir / "yolo_data" / f"fold_{fold}"
    marker = fold_dir / ".ready"
    if marker.exists():
        return fold_dir / "data.yaml"
    train_files = fold_manifest.loc[fold_manifest["fold"] != fold, "filename"].tolist()
    val_files = fold_manifest.loc[fold_manifest["fold"] == fold, "filename"].tolist()
    # Output directory is fold_dir/train/images: originals (symlinked) plus CLAHE/noise
    # variants (materialized files) side by side, so train_detector trains on both at once --
    # nothing downstream needs to know the split is now larger, Ultralytics just reads the dir.
    write_yolo_split(fold_dir / "train", train_files, train_image_dir, selected_image_ids, indices, augment=True)
    write_yolo_split(fold_dir / "val", val_files, train_image_dir, selected_image_ids, indices)
    yaml_text = (
        f"path: {fold_dir}\n"
        "train: train/images\n"
        "val: val/images\n"
        "names:\n  0: filament\n"
    )
    yaml_path = fold_dir / "data.yaml"
    yaml_path.write_text(yaml_text, encoding="utf-8")
    marker.write_text("ready", encoding="utf-8")
    return yaml_path


def train_detector(fold: int, data_yaml: Path, cfg: CFG, work_dir: Path) -> Path:
    from ultralytics import YOLO

    output_dir = work_dir / "models" / f"fold_{fold}"
    output_dir.mkdir(parents=True, exist_ok=True)
    final_path = output_dir / "detector.pt"
    meta_path = output_dir / "detector_meta.json"
    fingerprint = detector_config_fingerprint(cfg)
    target_epochs = 2 if cfg.debug else cfg.detector_epochs
    run_dir = work_dir / "detector_runs" / f"fold_{fold}"
    last_checkpoint = run_dir / "weights" / "last.pt"
    meta = read_json_if_valid(meta_path)

    if (final_path.exists() and meta is not None and meta.get("status") == "complete"
            and meta.get("config_fingerprint") == fingerprint):
        return final_path

    # Resume-eligible only if a previous attempt under the SAME config left Ultralytics' own
    # last.pt in place. A cfg change (or no prior attempt) means the partial run dir is stale
    # or foreign -- wipe it rather than risk resuming under a mismatched config. Ultralytics'
    # own resume= only honors an overridden `data` on resume, not epochs/augmentation knobs
    # (verified against its BaseTrainer.check_resume source), so our fingerprint gate has to
    # run before calling it, not rely on it.
    resume_ok = (
        meta is not None and meta.get("config_fingerprint") == fingerprint
        and meta.get("status") == "in_progress" and last_checkpoint.exists()
    )
    if not resume_ok and run_dir.exists():
        shutil.rmtree(run_dir, ignore_errors=True)

    atomic_write_json({
        "stage": "detector", "status": "in_progress", "fold": fold,
        "attempt_updated_at": now_iso(), "config_fingerprint": fingerprint,
        "target_epochs": target_epochs, "epochs_completed": 0,
    }, meta_path)

    gpu_count = torch.cuda.device_count()
    device: int | str = ",".join(map(str, range(gpu_count))) if gpu_count > 1 else (0 if gpu_count == 1 else "cpu")
    common_kwargs = dict(
        data=str(data_yaml), imgsz=cfg.image_size, batch=cfg.detector_batch, device=device,
        patience=cfg.detector_patience, project=str(work_dir / "detector_runs"),
        name=f"fold_{fold}", exist_ok=True, verbose=True,
    )
    if resume_ok:
        model = YOLO(str(last_checkpoint))
        try:
            model.train(resume=str(last_checkpoint), **common_kwargs)
        except (AssertionError, subprocess.CalledProcessError) as error:
            # Single-GPU: Ultralytics raises this AssertionError directly, in-process, so its
            # text ("... nothing to resume.") is directly inspectable -- the original check.
            # Multi-GPU (device_count > 1): Ultralytics spawns training in a SEPARATE
            # subprocess via torch.distributed.run. The identical assertion still fires inside
            # that subprocess (execution-confirmed on a real 2-GPU Kaggle run -- both rank0 and
            # rank1 logged the exact same "training to N epochs is finished, nothing to
            # resume" text), but by the time it propagates back here it's wrapped in
            # subprocess.CalledProcessError with no captured stdout/stderr (Ultralytics' own
            # subprocess.run call doesn't set capture_output=True), so there is no text to
            # match against in that case. Corroborate via run_dir/weights/best.pt instead: that
            # file only exists if a full previous attempt already validated and saved a best
            # checkpoint, which a genuinely different subprocess crash (OOM, real code error)
            # wouldn't necessarily have left behind. See PROGRESS.md.
            if isinstance(error, AssertionError) and "nothing to resume" not in str(error):
                raise
            if isinstance(error, subprocess.CalledProcessError) and not (run_dir / "weights" / "best.pt").exists():
                raise
            # Ultralytics' own epoch counter shows this run already finished all its epochs;
            # nothing left to do but pick up best.pt below.
    else:
        model = YOLO(cfg.detector_model)
        # overlap_mask/copy_paste/box/cls tuned for THIS competition's shape (many thin,
        # closely-packed filament instances per image, single class, IoU-based metric) -- see
        # CFG's own field comments and PROGRESS.md, not generic defaults. Like mask_ratio/
        # hsv_*/mosaic above, these only take effect on a fresh (non-resumed) start -- Ultralytics'
        # own resume= reuses the original run's saved args, not these fresh kwargs (see the
        # resume_ok branch above / this function's earlier comment on why).
        model.train(
            epochs=target_epochs, seed=cfg.seed + fold, single_cls=True, mask_ratio=cfg.mask_ratio,
            overlap_mask=cfg.detector_overlap_mask, copy_paste=cfg.detector_copy_paste,
            box=cfg.detector_box_gain, cls=cfg.detector_cls_gain,
            hsv_h=0.0, hsv_s=0.0, hsv_v=0.25,
            fliplr=0.5, flipud=0.5, degrees=12, scale=0.30, translate=0.08,
            mosaic=0.25, close_mosaic=5, **common_kwargs,
        )

    trainer = getattr(model, "trainer", None)
    # getattr with a fallback, not a direct attribute access: trainer.epoch is documented as set
    # DURING Ultralytics' own training loop but doesn't reliably persist after .train() returns
    # across versions (execution-confirmed: raised AttributeError on 'SegmentationTrainer' with
    # the ultralytics version YOLO26 support pulled in). epochs_completed/best_path are pure
    # metadata (not read by this function's own resume-guard above, which only checks
    # status/config_fingerprint) -- version-fragile internal attributes aren't worth crashing
    # a whole training run over, same reasoning as the "nothing to resume" string-match check
    # elsewhere in this function (see PROGRESS.md's residual-risk notes on Ultralytics API
    # fragility). Falls back to target_epochs / the conventional weights path if either
    # attribute is missing.
    epochs_completed = (getattr(trainer, "epoch", target_epochs - 1) + 1) if trainer is not None else target_epochs
    best_path = str(getattr(trainer, "best", run_dir / "weights" / "best.pt")) if trainer is not None else str(run_dir / "weights" / "best.pt")
    atomic_copy_file(Path(best_path), final_path)
    atomic_write_json({
        "stage": "detector", "status": "complete", "fold": fold,
        "attempt_updated_at": now_iso(), "config_fingerprint": fingerprint,
        "target_epochs": target_epochs, "epochs_completed": epochs_completed,
    }, meta_path)
    # Ultralytics keeps the full Trainer (model + EMA model + optimizer state + val
    # dataloader, all GPU-resident) alive on model.trainer after .train() returns. Left
    # alone, that memory is still reserved by this process when train_refiner starts right
    # after in the same fold, and combined with allocator fragmentation it starved a later
    # refiner allocation of just over 1 GiB on a GPU that still had ~100 MiB reported free.
    # Drop the references and force the CUDA allocator to actually give the memory back.
    if trainer is not None:
        del model.trainer
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return final_path

Appending to solar_filament_lib.py


### 3. Seed-conditioned refiner training

### Region+boundary dense segmenter, detector-free -- dormant

`RegionBoundarySegmenter` (`cfg.refiner_backbone == "vit"`) was the active path from
2026-09-28 until the Seg(YOLO)->Energy pivot (section 6) superseded it. `train_fold` no longer
reads `refiner_backbone` at all -- it always runs Seg(YOLO)->Energy now, regardless of this
flag's value. This class, its dataset, and its detector-free decode path are kept in the
module, untouched and unused, in case this design needs reviving. See `PROGRESS.md` for the
full history (why the detector was dropped, then why it came back as YOLO26).

**Setup, still relevant to the active path too:** `facebook/dinov3-*` repos are gated; accept
the license, create an HF access token, and add it as a Kaggle secret named `HF_TOKEN`. Cell 3
logs in automatically if present.

In [5]:
%%writefile -a solar_filament_lib.py


def square_bounds(mask: np.ndarray, context: float, minimum: int) -> tuple[int, int, int, int]:
    height, width = mask.shape
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return 0, 0, width, height
    center_x = (xs.min() + xs.max() + 1) / 2
    center_y = (ys.min() + ys.max() + 1) / 2
    side = int(math.ceil(max(np.ptp(xs) + 1, np.ptp(ys) + 1) * context))
    side = min(max(side, minimum), min(height, width))
    x0, y0 = int(round(center_x - side / 2)), int(round(center_y - side / 2))
    x1, y1 = x0 + side, y0 + side
    if x0 < 0: x1 -= x0; x0 = 0
    if y0 < 0: y1 -= y0; y0 = 0
    if x1 > width: x0 -= x1 - width; x1 = width
    if y1 > height: y0 -= y1 - height; y1 = height
    return x0, y0, x1, y1


def jitter_seed_mask(mask: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    seed = mask.astype(np.uint8)
    radius = int(rng.integers(-5, 6))
    if radius != 0:
        kernel_size = 2 * abs(radius) + 1
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
        seed = cv2.dilate(seed, kernel) if radius > 0 else cv2.erode(seed, kernel)
    shift_x, shift_y = int(rng.integers(-8, 9)), int(rng.integers(-8, 9))
    matrix = np.float32([[1, 0, shift_x], [0, 1, shift_y]])
    seed = cv2.warpAffine(seed, matrix, (seed.shape[1], seed.shape[0]), flags=cv2.INTER_NEAREST)
    return seed.astype(bool) if seed.any() else mask.astype(bool)


class ConvBlock(nn.Sequential):
    def __init__(self, input_channels: int, output_channels: int) -> None:
        super().__init__(
            nn.Conv2d(input_channels, output_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(output_channels), nn.ReLU(inplace=True),
            nn.Conv2d(output_channels, output_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(output_channels), nn.ReLU(inplace=True),
        )


def soft_dice_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    # Cast to float32 before reducing -- under torch.amp.autocast, .sigmoid() stays fp16 and
    # plain .sum() is NOT on autocast's fp32-promotion list (unlike F.binary_cross_entropy_with_logits,
    # which is autocast-safe). A 256x256 crop is 65536 pixels; once a meaningful fraction of a
    # poorly-calibrated early decoder's outputs sit near 1.0, probabilities.sum() can reach or
    # exceed fp16's max (65504) -- a literal inf, which produces inf/nan gradients and drives
    # GradScaler to repeatedly halve its scale until it underflows to exactly 0.0, at which point
    # it can never recover (growth from 0.0 * 2 is still 0.0) and every future step becomes a
    # silent all-zero-gradient no-op. Execution-confirmed: real Kaggle runs showed
    # scaler_step_skips collapsing final_scale to exactly 0.0 in epoch 1 across all 5 folds, with
    # val_crop_dice pinned at exactly 0.00000 for all 8 epochs after -- the refiner never trained
    # at all. Validation was unaffected because it sums boolean tensors, which promote to int64
    # regardless of autocast, so it never overflowed -- it just kept measuring the frozen model.
    probabilities = logits.float().sigmoid()
    targets = targets.float()
    intersection = (probabilities * targets).sum(dim=(1, 2, 3))
    denominator = probabilities.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    return (1 - (2 * intersection + 1) / (denominator + 1)).mean()


def energy_regression_loss(energy_logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """MSE between sigmoid(energy_logits) and the per-pixel [0,1] energy target (see
    build_semantic_targets), same defensive fp32-before-reduction pattern as soft_dice_loss --
    cheap insurance against the exact AMP-overflow bug class already hit twice in this file,
    even though .mean() (unlike soft_dice_loss's old .sum()) is not overflow-prone at any image
    size. Unlike region/boundary, this is a continuous regression target, not a binary mask --
    no bce+dice blend, plain MSE."""
    predictions = energy_logits.float().sigmoid()
    return F.mse_loss(predictions, targets.float())


# ---------------------------------------------------------------------------
# resnet18 path: per-instance crop refiner (validated baseline, unchanged)
# ---------------------------------------------------------------------------

def discriminative_embedding_loss(embeddings: torch.Tensor, label_maps: torch.Tensor, cfg: "CFG") -> torch.Tensor:
    """Pull same-instance pixels together, push different-instance mean embeddings apart
    (De Brabandere et al. 2017) -- the training signal for YoloConditionedEmbeddingNet's
    embedding_dim-D output, replacing energy_regression_loss as the active Stage 2 loss (see
    PROGRESS.md "embedding-head instance-separation swap"). Instance count varies per image, so
    this loops over the batch in Python rather than vectorizing -- cfg.vit_images_per_batch is
    small (2 by default), so the loop cost is negligible next to the DINOv3 backbone forward
    pass it sits next to. Background pixels are not constrained at all (same scope as the
    decode step, which only ever clusters pixels already inside a YOLO instance mask)."""
    batch_size = embeddings.shape[0]
    total = embeddings.new_zeros(())
    counted = 0
    for index in range(batch_size):
        embedding = embeddings[index].permute(1, 2, 0)  # (H, W, D)
        labels = label_maps[index]
        instance_ids = [value for value in labels.unique().tolist() if value != 0]
        if not instance_ids:
            continue
        means = []
        pull = embedding.new_zeros(())
        for instance_id in instance_ids:
            pixels = embedding[labels == instance_id]
            mean = pixels.mean(dim=0)
            means.append(mean)
            pull = pull + F.relu(torch.norm(pixels - mean, dim=1) - cfg.embedding_delta_v).pow(2).mean()
        pull = pull / len(instance_ids)
        means_tensor = torch.stack(means)
        if len(instance_ids) > 1:
            distance = torch.cdist(means_tensor, means_tensor)
            pair_mask = ~torch.eye(len(instance_ids), dtype=torch.bool, device=distance.device)
            push = F.relu(2 * cfg.embedding_delta_d - distance[pair_mask]).pow(2).mean()
        else:
            push = embedding.new_zeros(())
        reg = torch.norm(means_tensor, dim=1).mean()
        total = total + pull + push + cfg.embedding_reg_weight * reg
        counted += 1
    return total / max(counted, 1)


class RefinerDataset(Dataset):
    def __init__(
        self,
        filenames: Sequence[str],
        image_dir: Path,
        selected_image_ids: Mapping[str, int],
        indices: Mapping[str, Any],
        cfg: CFG,
        augment: bool,
    ) -> None:
        self.image_dir, self.indices, self.cfg, self.augment = image_dir, indices, cfg, augment
        self.samples = []
        for filename in filenames:
            image_id = selected_image_ids[filename]
            for annotation in indices["annotations_by_image"].get(image_id, []):
                if not annotation.get("iscrowd", 0):
                    self.samples.append((filename, image_id, annotation))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        filename, image_id, annotation = self.samples[index]
        info = self.indices["images_by_id"][image_id]
        image = cv2.imread(str(self.image_dir / filename), cv2.IMREAD_GRAYSCALE)
        target = annotation_to_mask(annotation, int(info["height"]), int(info["width"]))
        rng = np.random.default_rng(self.cfg.seed + index + random.randint(0, 1_000_000))
        seed = jitter_seed_mask(target, rng) if self.augment else target
        x0, y0, x1, y1 = square_bounds(seed, self.cfg.refiner_context, self.cfg.refiner_min_crop)
        image_crop = cv2.resize(image[y0:y1, x0:x1], (self.cfg.refiner_crop,) * 2, interpolation=cv2.INTER_LINEAR)
        target_crop = cv2.resize(target[y0:y1, x0:x1].astype(np.uint8), (self.cfg.refiner_crop,) * 2, interpolation=cv2.INTER_NEAREST)
        seed_crop = cv2.resize(seed[y0:y1, x0:x1].astype(np.uint8), (self.cfg.refiner_crop,) * 2, interpolation=cv2.INTER_NEAREST)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(image_crop)
        features = np.stack([image_crop, clahe, seed_crop * 255], axis=0).astype(np.float32) / 255.0
        if self.augment:
            k = random.randrange(4)
            features = np.rot90(features, k, axes=(1, 2)).copy()
            target_crop = np.rot90(target_crop, k).copy()
            if random.random() < 0.5:
                features = features[:, :, ::-1].copy(); target_crop = target_crop[:, ::-1].copy()
            gain = random.uniform(0.9, 1.1)
            features[:2] = np.clip(features[:2] * gain, 0, 1)
        return torch.from_numpy(features), torch.from_numpy(target_crop[None].astype(np.float32))


class SeedRefiner(nn.Module):
    def __init__(self, pretrained: bool = True) -> None:
        super().__init__()
        encoder = resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)
        self.stem = nn.Sequential(encoder.conv1, encoder.bn1, encoder.relu)
        self.pool, self.layer1, self.layer2, self.layer3, self.layer4 = (
            encoder.maxpool, encoder.layer1, encoder.layer2, encoder.layer3, encoder.layer4
        )
        self.up4, self.dec4 = nn.ConvTranspose2d(512, 256, 2, 2), ConvBlock(512, 256)
        self.up3, self.dec3 = nn.ConvTranspose2d(256, 128, 2, 2), ConvBlock(256, 128)
        self.up2, self.dec2 = nn.ConvTranspose2d(128, 64, 2, 2), ConvBlock(128, 64)
        self.up1, self.dec1 = nn.ConvTranspose2d(64, 32, 2, 2), ConvBlock(96, 32)
        self.up0, self.head = nn.ConvTranspose2d(32, 16, 2, 2), nn.Conv2d(16, 1, 1)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, tensor: torch.Tensor) -> torch.Tensor:
        tensor = (tensor - self.mean) / self.std
        x0 = self.stem(tensor); x1 = self.layer1(self.pool(x0)); x2 = self.layer2(x1)
        x3 = self.layer3(x2); x4 = self.layer4(x3)
        x = self.dec4(torch.cat([self.up4(x4), x3], dim=1))
        x = self.dec3(torch.cat([self.up3(x), x2], dim=1))
        x = self.dec2(torch.cat([self.up2(x), x1], dim=1))
        x = self.dec1(torch.cat([self.up1(x), x0], dim=1))
        return self.head(self.up0(x))


def _train_crop_refiner(
    fold: int,
    fold_manifest: pd.DataFrame,
    image_dir: Path,
    selected_image_ids: Mapping[str, int],
    indices: Mapping[str, Any],
    cfg: CFG,
    work_dir: Path,
    checkpoint: Path,
    meta_path: Path,
    fingerprint: str,
    target_epochs: int,
) -> Path:
    train_files = fold_manifest.loc[fold_manifest["fold"] != fold, "filename"].tolist()
    val_files = fold_manifest.loc[fold_manifest["fold"] == fold, "filename"].tolist()
    if cfg.debug:
        train_files, val_files = train_files[:24], val_files[:8]
    train_dataset = RefinerDataset(train_files, image_dir, selected_image_ids, indices, cfg, True)
    val_dataset = RefinerDataset(val_files, image_dir, selected_image_ids, indices, cfg, False)
    train_loader = DataLoader(train_dataset, batch_size=cfg.refiner_batch, shuffle=True,
                              num_workers=cfg.num_workers, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=cfg.refiner_batch * 2, shuffle=False,
                            num_workers=cfg.num_workers, pin_memory=True)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = SeedRefiner(pretrained=True).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.refiner_lr, weight_decay=1e-4)
    epochs = target_epochs
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    best_dice = -1.0
    for epoch in range(epochs):
        model.train()
        for features, targets in tqdm(train_loader, desc=f"fold{fold} refiner train e{epoch + 1}/{epochs}", leave=False):
            features, targets = features.to(device), targets.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                logits = model(features)
                loss = 0.45 * F.binary_cross_entropy_with_logits(logits, targets) + 0.55 * soft_dice_loss(logits, targets)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        model.eval(); dice_total = sample_count = 0
        with torch.no_grad():
            for features, targets in tqdm(val_loader, desc=f"fold{fold} refiner val e{epoch + 1}/{epochs}", leave=False):
                features, targets = features.to(device), targets.to(device).bool()
                predictions = model(features).sigmoid() >= cfg.refiner_threshold
                intersection = (predictions & targets).sum(dim=(1, 2, 3)).float()
                denominator = predictions.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
                dice_total += ((2 * intersection + 1e-7) / (denominator + 1e-7)).sum().item()
                sample_count += len(features)
        validation_dice = dice_total / max(sample_count, 1)
        print(f"fold={fold} refiner_epoch={epoch + 1} val_crop_dice={validation_dice:.5f}")
        is_best = validation_dice > best_dice
        if is_best:
            best_dice = validation_dice
            atomic_torch_save(model.state_dict(), checkpoint)
        _record_refiner_epoch(meta_path, fold, fingerprint, target_epochs, epoch, validation_dice, is_best, best_dice)
        scheduler.step()
    _finalize_refiner_meta(meta_path, fold, fingerprint, target_epochs)
    return checkpoint


# ---------------------------------------------------------------------------
# vit path: global-context refiner. The ViT backbone runs once per full image
# (not once per crop) so its self-attention actually sees whole-disk context,
# which is what DINO generalization depends on. Instance masks are then
# decoded from ROIAlign crops of that single feature map. Provisional: this
# replaces the earlier per-crop ViTSeedRefiner; if OOF/LB do not hold up
# against the resnet18 baseline, revert per PROGRESS.md rather than keep
# iterating on this design.
# ---------------------------------------------------------------------------

class ImageRefinerDataset(Dataset):
    """One sample per physical image, holding every instance box/seed/target crop
    so a single backbone pass can serve all of that image's instances."""

    def __init__(
        self,
        filenames: Sequence[str],
        image_dir: Path,
        selected_image_ids: Mapping[str, int],
        indices: Mapping[str, Any],
        cfg: CFG,
        augment: bool,
    ) -> None:
        self.image_dir, self.indices, self.cfg, self.augment = image_dir, indices, cfg, augment
        self.filenames = filenames
        self.selected_image_ids = selected_image_ids

    def __len__(self) -> int:
        return len(self.filenames)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        cfg = self.cfg
        filename = self.filenames[index]
        image_id = self.selected_image_ids[filename]
        info = self.indices["images_by_id"][image_id]
        size = cfg.vit_image_size
        raw = cv2.imread(str(self.image_dir / filename), cv2.IMREAD_GRAYSCALE)
        image = cv2.resize(raw, (size, size), interpolation=cv2.INTER_LINEAR)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(image)
        image_tensor = np.stack([image, clahe, image], axis=0).astype(np.float32) / 255.0

        rng = np.random.default_rng(cfg.seed + index + random.randint(0, 1_000_000))
        boxes, seed_crops, target_crops = [], [], []
        for annotation in self.indices["annotations_by_image"].get(image_id, []):
            if annotation.get("iscrowd", 0):
                continue
            target_native = annotation_to_mask(annotation, int(info["height"]), int(info["width"]))
            target = cv2.resize(target_native.astype(np.uint8), (size, size), interpolation=cv2.INTER_NEAREST).astype(bool)
            seed = jitter_seed_mask(target, rng) if self.augment else target
            x0, y0, x1, y1 = square_bounds(seed, cfg.refiner_context, cfg.refiner_min_crop)
            seed_crop = cv2.resize(seed[y0:y1, x0:x1].astype(np.uint8), (cfg.refiner_crop,) * 2, interpolation=cv2.INTER_NEAREST)
            target_crop = cv2.resize(target[y0:y1, x0:x1].astype(np.uint8), (cfg.refiner_crop,) * 2, interpolation=cv2.INTER_NEAREST)
            boxes.append([x0, y0, x1, y1])
            seed_crops.append(seed_crop)
            target_crops.append(target_crop)

        if self.augment and boxes and random.random() < 0.5:
            image_tensor = image_tensor[:, :, ::-1].copy()
            seed_crops = [c[:, ::-1].copy() for c in seed_crops]
            target_crops = [c[:, ::-1].copy() for c in target_crops]
            boxes = [[size - x1, y0, size - x0, y1] for x0, y0, x1, y1 in boxes]

        if self.augment:
            image_tensor = np.clip(image_tensor + rng.normal(0, 0.01, image_tensor.shape), 0, 1).astype(np.float32)

        empty_crop = np.zeros((0, cfg.refiner_crop, cfg.refiner_crop), dtype=np.float32)
        return {
            "image": torch.from_numpy(image_tensor),
            "boxes": torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4)),
            "seed_crops": torch.from_numpy(np.stack(seed_crops).astype(np.float32))[:, None] if seed_crops else torch.from_numpy(empty_crop)[:, None],
            "target_crops": torch.from_numpy(np.stack(target_crops).astype(np.float32))[:, None] if target_crops else torch.from_numpy(empty_crop)[:, None],
        }


def _vit_refiner_collate(batch: Sequence[dict[str, torch.Tensor]]) -> dict[str, Any]:
    return {
        "images": torch.stack([sample["image"] for sample in batch]),
        "boxes": [sample["boxes"] for sample in batch],
        "seed_crops": [sample["seed_crops"] for sample in batch],
        "target_crops": [sample["target_crops"] for sample in batch],
    }


class GlobalViTRefiner(nn.Module):
    """Runs a pretrained self-supervised ViT (DINOv2 or DINOv3, via cfg.vit_hub_id) once over the
    full image, then decodes each detected instance from an ROIAlign crop of that single feature map
    instead of re-cropping raw pixels and re-encoding per instance. This is what lets the backbone
    self-attention actually integrate whole-disk context, which the per-crop design this replaces
    could not: a 256px crop only gave DINO a tiny, already-isolated window to attend over.

    Multi-scale fusion is still shallow-vs-deep semantics at one grid (ViT has no resolution
    pyramid), same as before. The seed-mask skip channel is still concatenated at full crop
    resolution before the head, since ROI-pooled features alone are still too coarse for PQ's
    boundary precision.
    """

    def __init__(self, cfg: "CFG") -> None:
        super().__init__()
        from transformers import AutoBackbone

        self.backbone = AutoBackbone.from_pretrained(
            cfg.vit_hub_id,
            out_indices=list(cfg.vit_out_indices),
            reshape_hidden_states=True,
            attn_implementation="eager",
        )
        # DINOv3's config ships pretraining-time position-embedding perturbations
        # (pos_embed_jitter/shift/rescale) that only activate in model.train() mode --
        # exactly the mode we use for fine-tuning. We are not trying to reproduce DINOv3's
        # own pretraining recipe, just use it as a feature extractor, so these must stay off.
        # A dynamic-shape interpolation in that path is also the leading suspect for the
        # CUDA OOM (a single-tensor 40+ GiB allocation request during backward is a shape
        # bug, not a legitimate batch-size memory shortfall).
        for disabled_field in ("pos_embed_jitter", "pos_embed_shift", "pos_embed_rescale"):
            if hasattr(self.backbone.config, disabled_field):
                setattr(self.backbone.config, disabled_field, None)
        self.patch_size = self.backbone.config.patch_size
        if cfg.vit_image_size % self.patch_size:
            print(f"GlobalViTRefiner: vit_image_size={cfg.vit_image_size} not divisible by "
                  f"patch_size={self.patch_size}; ROI alignment will be approximate.")
        hidden_size = self.backbone.config.hidden_size
        n_scales = len(cfg.vit_out_indices)
        fuse_channels = 256
        self.roi_size = cfg.vit_roi_size
        self.scale_proj = nn.ModuleList(nn.Conv2d(hidden_size, fuse_channels, 1) for _ in range(n_scales))
        self.fuse = ConvBlock(fuse_channels * n_scales, fuse_channels)
        self.pre_upsample = ConvBlock(fuse_channels, 128)
        self.post_upsample = ConvBlock(128 + 1, 32)  # +1: raw seed-mask skip channel at full resolution
        self.head = nn.Conv2d(32, 1, 1)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def encode_image(self, images: torch.Tensor) -> list[torch.Tensor]:
        normalized = (images - self.mean) / self.std
        return self.backbone(pixel_values=normalized).feature_maps

    def decode_instances(
        self,
        feature_maps: Sequence[torch.Tensor],
        boxes_per_image: Sequence[torch.Tensor],
        seed_crops: torch.Tensor,
    ) -> torch.Tensor:
        from torchvision.ops import roi_align

        spatial_scale = 1.0 / self.patch_size
        pooled = []
        # sampling_ratio is pinned (not left adaptive/-1): when the feature map requires grad
        # (fine-tuning the last backbone blocks), torchvision routes roi_align through a
        # pure-Python differentiable fallback that materializes a [K, C, PH, PW, sample_y,
        # sample_x] tensor. Adaptive sampling scales sample_y/x with ROI size in feature-map
        # units, and our seed boxes can span most of the image -- that combination is what
        # produced the CUDA OOM (confirmed by traceback: crash was inside roi_align's
        # masked_index, not the backbone). Pinning it bounds memory regardless of ROI size.
        for proj, feature_map in zip(self.scale_proj, feature_maps):
            aligned = roi_align(feature_map, list(boxes_per_image), output_size=self.roi_size,
                                 spatial_scale=spatial_scale, sampling_ratio=2, aligned=True)
            pooled.append(proj(aligned))
        if pooled[0].shape[0] == 0:
            return pooled[0].new_zeros((0, 1, seed_crops.shape[-2], seed_crops.shape[-1]))
        x = self.pre_upsample(self.fuse(torch.cat(pooled, dim=1)))
        x = F.interpolate(x, size=seed_crops.shape[-2:], mode="bilinear", align_corners=False)
        x = self.post_upsample(torch.cat([x, seed_crops], dim=1))
        return self.head(x)


def vit_param_groups(model: "GlobalViTRefiner", cfg: "CFG") -> list[dict[str, Any]]:
    """Freeze all but the last vit_unfreeze_last_n transformer blocks; fine-tune the rest of the
    backbone at a reduced LR and the decoder head at the full refiner_lr."""
    backbone_named = [(n, p) for n, p in model.named_parameters() if n.startswith("backbone.")]
    head_named = [(n, p) for n, p in model.named_parameters() if not n.startswith("backbone.")]
    block_ids = set()
    for name, _ in backbone_named:
        match = re.search(r"\.layer\.(\d+)\.", name)
        if match:
            block_ids.add(int(match.group(1)))
    threshold = (max(block_ids) - cfg.vit_unfreeze_last_n + 1) if block_ids else None
    trainable_backbone = []
    for name, p in backbone_named:
        match = re.search(r"\.layer\.(\d+)\.", name)
        idx = int(match.group(1)) if match else None
        unfreeze = threshold is not None and idx is not None and idx >= threshold
        p.requires_grad = unfreeze
        if unfreeze:
            trainable_backbone.append(p)
    if threshold is None:
        print("vit_param_groups: could not locate transformer block indices by name; "
              "backbone stays fully frozen, only the decoder head trains.")
    for _, p in head_named:
        p.requires_grad = True
    groups = [{"params": [p for _, p in head_named], "lr": cfg.refiner_lr}]
    if trainable_backbone:
        groups.append({"params": trainable_backbone, "lr": cfg.refiner_lr * cfg.vit_backbone_lr_mult})
    return groups


def _train_global_vit_refiner(
    fold: int,
    fold_manifest: pd.DataFrame,
    image_dir: Path,
    selected_image_ids: Mapping[str, int],
    indices: Mapping[str, Any],
    cfg: CFG,
    work_dir: Path,
    checkpoint: Path,
    meta_path: Path,
    fingerprint: str,
    target_epochs: int,
) -> Path:
    train_files = fold_manifest.loc[fold_manifest["fold"] != fold, "filename"].tolist()
    val_files = fold_manifest.loc[fold_manifest["fold"] == fold, "filename"].tolist()
    if cfg.debug:
        train_files, val_files = train_files[:24], val_files[:8]
    train_dataset = ImageRefinerDataset(train_files, image_dir, selected_image_ids, indices, cfg, True)
    val_dataset = ImageRefinerDataset(val_files, image_dir, selected_image_ids, indices, cfg, False)
    # Diagnostic: fold 0's refiner ran 8 straight epochs with val_crop_dice pinned at exactly
    # 0.00000 and a persistent "scheduler.step() before optimizer.step()" warning in a real
    # Kaggle run (confirmed again in a second run, fold 1). Root cause still open -- see
    # PROGRESS.md. This sanity print (GPU-free, runs before any training) and the per-epoch
    # train_diag/val_diag counters below exist to settle it on the next run: either the raw
    # annotation counts here are empty (a data-path bug specific to this dataset class, despite
    # the identical lookup working fine in write_yolo_split), or instances are non-zero but
    # GradScaler is silently skipping every optimizer step (persistent inf/nan gradients).
    sample_files = train_files[:5]
    sample_counts = [
        len(indices["annotations_by_image"].get(selected_image_ids[filename], []))
        for filename in sample_files
    ]
    print(f"fold={fold} refiner sanity check: train_images={len(train_dataset)} val_images={len(val_dataset)} "
          f"sample_annotation_counts={dict(zip(sample_files, sample_counts))}")
    train_loader = DataLoader(train_dataset, batch_size=cfg.vit_images_per_batch, shuffle=True,
                              num_workers=cfg.num_workers, collate_fn=_vit_refiner_collate, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=cfg.vit_images_per_batch, shuffle=False,
                            num_workers=cfg.num_workers, collate_fn=_vit_refiner_collate)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = GlobalViTRefiner(cfg).to(device)
    optimizer = torch.optim.AdamW(vit_param_groups(model, cfg), weight_decay=1e-4)
    epochs = target_epochs
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    best_dice = -1.0
    for epoch in range(epochs):
        model.train()
        train_batches = train_skipped = train_instances = train_scale_skips = 0
        for batch in tqdm(train_loader, desc=f"fold{fold} vit refiner train e{epoch + 1}/{epochs}", leave=False):
            train_batches += 1
            seed_crops = torch.cat(batch["seed_crops"]).to(device)
            if seed_crops.shape[0] == 0:
                train_skipped += 1
                continue
            train_instances += seed_crops.shape[0]
            images = batch["images"].to(device)
            boxes = [b.to(device) for b in batch["boxes"]]
            targets = torch.cat(batch["target_crops"]).to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                feature_maps = model.encode_image(images)
                logits = model.decode_instances(feature_maps, boxes, seed_crops)
                loss = 0.45 * F.binary_cross_entropy_with_logits(logits, targets) + 0.55 * soft_dice_loss(logits, targets)
            scale_before = scaler.get_scale()
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            if scaler.get_scale() < scale_before:
                train_scale_skips += 1
        train_diag = {"batches": train_batches, "skipped_empty": train_skipped,
                      "instances": train_instances, "scaler_step_skips": train_scale_skips,
                      "final_scale": scaler.get_scale()}
        print(f"fold={fold} refiner_epoch={epoch + 1} train_diag: batches={train_batches} "
              f"skipped_empty={train_skipped} instances={train_instances} "
              f"scaler_step_skips={train_scale_skips} final_scale={scaler.get_scale():.1f}")
        model.eval(); dice_total = sample_count = 0
        val_batches = val_skipped = 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"fold{fold} vit refiner val e{epoch + 1}/{epochs}", leave=False):
                val_batches += 1
                seed_crops = torch.cat(batch["seed_crops"]).to(device)
                if seed_crops.shape[0] == 0:
                    val_skipped += 1
                    continue
                images = batch["images"].to(device)
                boxes = [b.to(device) for b in batch["boxes"]]
                targets = torch.cat(batch["target_crops"]).to(device).bool()
                feature_maps = model.encode_image(images)
                predictions = model.decode_instances(feature_maps, boxes, seed_crops).sigmoid() >= cfg.refiner_threshold
                intersection = (predictions & targets).sum(dim=(1, 2, 3)).float()
                denominator = predictions.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
                dice_total += ((2 * intersection + 1e-7) / (denominator + 1e-7)).sum().item()
                sample_count += len(targets)
        val_diag = {"batches": val_batches, "skipped_empty": val_skipped, "instances": sample_count}
        validation_dice = dice_total / max(sample_count, 1)
        print(f"fold={fold} refiner_epoch={epoch + 1} val_crop_dice={validation_dice:.5f} "
              f"val_diag: batches={val_batches} skipped_empty={val_skipped} instances={sample_count}")
        is_best = validation_dice > best_dice
        if is_best:
            best_dice = validation_dice
            atomic_torch_save(model.state_dict(), checkpoint)
        _record_refiner_epoch(meta_path, fold, fingerprint, target_epochs, epoch, validation_dice,
                              is_best, best_dice, train_diag, val_diag)
        scheduler.step()
    _finalize_refiner_meta(meta_path, fold, fingerprint, target_epochs)
    return checkpoint


# ---------------------------------------------------------------------------
# Region+boundary dense segmenter: single DINOv3 pass over the full image produces a
# region map and a boundary map directly (DCAN-style: instances recovered by subtracting
# the boundary from the region and running connected components / watershed, not by a
# detector). Provisional, per explicit user request to drop the YOLO detector stage "for
# now" -- GlobalViTRefiner/ImageRefinerDataset/train_detector/prepare_yolo_fold above are
# kept, not deleted, in case this needs reverting (same convention every other provisional
# design change in this notebook has used -- see PROGRESS.md).
# ---------------------------------------------------------------------------

def boundary_ring_mask(mask: np.ndarray, width: int) -> np.ndarray:
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * width + 1, 2 * width + 1))
    mask_u8 = mask.astype(np.uint8)
    dilated = cv2.dilate(mask_u8, kernel).astype(bool)
    eroded = cv2.erode(mask_u8, kernel).astype(bool)
    return dilated & ~eroded


def build_semantic_targets(
    image_id: int,
    indices: Mapping[str, Any],
    size: int,
    boundary_width: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Unions every non-crowd instance's mask (region), boundary ring (boundary), and a
    Deep-Watershed-Transform-style energy map for one image, all at size x size. Region ground
    truth is the same semantic union semantic_rle_for_record already computes, just as a dense
    array instead of an RLE.

    Each instance is resized to size x size BEFORE its boundary ring/energy are computed (not
    after unioning at native resolution) so boundary_width means what CFG documents it as -- a
    ring radius in pixels at vit_image_size resolution -- regardless of native annotation
    resolution. Computing the ring natively and downsampling the union afterward silently
    shrunk it by the native/target resolution ratio (native imagery is ~2048px vs. size=1024,
    so the ring was roughly half the documented width) -- see PROGRESS.md. This must stay
    per-instance: resizing the union region first and ringing/energy-transforming that would
    erase the internal seam between two touching-but-separate instances, which
    separate_instances_watershed depends on to split them apart.

    energy: each instance's own cv2.distanceTransform, normalized by ITS OWN max distance (not
    a global constant) so a thin filament (small max distance) and a thick one both span the
    same [0,1] range and peak at 1.0 at their own medial axis -- using unnormalized raw
    distance would bias supervision almost entirely toward thick instances. Where instances
    don't overlap (the common case), the per-instance union is just each instance's own energy;
    where they do overlap, taking the elementwise max keeps whichever instance's ridge is
    locally more confident.
    """
    info = indices["images_by_id"][image_id]
    height, width = int(info["height"]), int(info["width"])
    region = np.zeros((size, size), dtype=bool)
    boundary = np.zeros((size, size), dtype=bool)
    energy = np.zeros((size, size), dtype=np.float32)
    for annotation in indices["annotations_by_image"].get(image_id, []):
        if annotation.get("iscrowd", 0):
            continue
        instance_mask = annotation_to_mask(annotation, height, width)
        instance_resized = cv2.resize(instance_mask.astype(np.uint8), (size, size),
                                       interpolation=cv2.INTER_NEAREST).astype(bool)
        region |= instance_resized
        boundary |= boundary_ring_mask(instance_resized, boundary_width)
        instance_u8 = instance_resized.astype(np.uint8)
        distance = cv2.distanceTransform(instance_u8, cv2.DIST_L2, 5)
        max_distance = float(distance.max())
        if max_distance > 0:
            energy = np.maximum(energy, (distance / max_distance).astype(np.float32))

    return region, boundary, energy


def build_instance_label_map(image_id: int, indices: Mapping[str, Any], size: int) -> np.ndarray:
    """Per-instance integer label map (0=background, 1..K) at size x size, for
    discriminative_embedding_loss -- unlike build_semantic_targets, which only needs the UNION
    of instances, the embedding loss needs each instance's own identity to compute per-instance
    means. Same per-instance-then-resize order as build_semantic_targets (resizing the union
    mask first would erase the seam between touching instances, which the embedding head's
    whole purpose is to reconstruct). Overlapping instances: the later annotation wins the
    overlap pixels (plain array assignment order) -- an accepted simplification, same as
    refine_instances_with_energy_watershed's overlap handling elsewhere in this file."""
    info = indices["images_by_id"][image_id]
    height, width = int(info["height"]), int(info["width"])
    labels = np.zeros((size, size), dtype=np.int32)
    next_label = 1
    for annotation in indices["annotations_by_image"].get(image_id, []):
        if annotation.get("iscrowd", 0):
            continue
        instance_mask = annotation_to_mask(annotation, height, width)
        instance_resized = cv2.resize(instance_mask.astype(np.uint8), (size, size),
                                       interpolation=cv2.INTER_NEAREST).astype(bool)
        if instance_resized.any():
            labels[instance_resized] = next_label
            next_label += 1
    return labels


class SegmentationDataset(Dataset):
    """One sample per physical image: dense region + boundary targets, no per-instance
    boxes/seeds -- the point of dropping the detector is that instances are recovered from
    these two dense maps by post-processing (separate_instances_watershed, cell 13), not
    proposed up front."""

    def __init__(
        self,
        filenames: Sequence[str],
        image_dir: Path,
        selected_image_ids: Mapping[str, int],
        indices: Mapping[str, Any],
        cfg: CFG,
        augment: bool,
    ) -> None:
        self.image_dir, self.indices, self.cfg, self.augment = image_dir, indices, cfg, augment
        self.filenames = filenames
        self.selected_image_ids = selected_image_ids

    def __len__(self) -> int:
        return len(self.filenames)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        cfg = self.cfg
        filename = self.filenames[index]
        image_id = self.selected_image_ids[filename]
        size = cfg.vit_image_size
        raw = cv2.imread(str(self.image_dir / filename), cv2.IMREAD_GRAYSCALE)
        image = cv2.resize(raw, (size, size), interpolation=cv2.INTER_LINEAR)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(image)
        image_tensor = np.stack([image, clahe, image], axis=0).astype(np.float32) / 255.0
        region, boundary, energy = build_semantic_targets(image_id, self.indices, size, cfg.boundary_width)

        if self.augment:
            rng = np.random.default_rng(cfg.seed + index + random.randint(0, 1_000_000))
            if random.random() < 0.5:
                image_tensor = image_tensor[:, :, ::-1].copy()
                region = region[:, ::-1].copy()
                boundary = boundary[:, ::-1].copy()
                energy = energy[:, ::-1].copy()
            if random.random() < 0.5:
                image_tensor = image_tensor[:, ::-1, :].copy()
                region = region[::-1, :].copy()
                boundary = boundary[::-1, :].copy()
                energy = energy[::-1, :].copy()
            image_tensor = np.clip(image_tensor + rng.normal(0, 0.01, image_tensor.shape), 0, 1).astype(np.float32)

        return {
            "image": torch.from_numpy(image_tensor),
            "region": torch.from_numpy(region[None].astype(np.float32)),
            "boundary": torch.from_numpy(boundary[None].astype(np.float32)),
            "energy": torch.from_numpy(energy[None].astype(np.float32)),
        }


class RegionBoundarySegmenter(nn.Module):
    """Single DINOv3 pass over the full image -> dense region + boundary maps, replacing the
    detector + per-instance ROIAlign-refiner path. Shares the exact backbone and multi-scale
    fuse/pre_upsample trunk GlobalViTRefiner already uses (same AutoBackbone setup, same
    pos_embed-jitter disabling -- see that class's docstring for why); only what happens
    after pre_upsample differs -- one bilinear jump straight to full image resolution (the
    same 16x ratio decode_instances already uses, ROI 16x16 -> crop 256x256), then a
    raw-image conv stem concatenated as a full-resolution skip. This replaces the seed-mask
    skip GlobalViTRefiner relied on for boundary precision -- there is no detector seed here,
    so the skip is fed by the input image itself instead of a proposed instance mask.
    """

    def __init__(self, cfg: "CFG") -> None:
        super().__init__()
        from transformers import AutoBackbone

        self.backbone = AutoBackbone.from_pretrained(
            cfg.vit_hub_id,
            out_indices=list(cfg.vit_out_indices),
            reshape_hidden_states=True,
            attn_implementation="eager",
        )
        for disabled_field in ("pos_embed_jitter", "pos_embed_shift", "pos_embed_rescale"):
            if hasattr(self.backbone.config, disabled_field):
                setattr(self.backbone.config, disabled_field, None)
        self.patch_size = self.backbone.config.patch_size
        hidden_size = self.backbone.config.hidden_size
        n_scales = len(cfg.vit_out_indices)
        fuse_channels = 256
        self.scale_proj = nn.ModuleList(nn.Conv2d(hidden_size, fuse_channels, 1) for _ in range(n_scales))
        self.fuse = ConvBlock(fuse_channels * n_scales, fuse_channels)
        self.pre_upsample = ConvBlock(fuse_channels, 128)
        self.image_stem = ConvBlock(3, 16)
        self.post_upsample = ConvBlock(128 + 16, 32)  # +16: raw-image skip at full resolution
        self.region_head = nn.Conv2d(32, 1, 1)
        self.boundary_head = nn.Conv2d(32, 1, 1)
        # Third head, additive -- see build_semantic_targets/PROGRESS.md. Predicts a per-pixel
        # [0,1] Deep-Watershed-Transform-style energy map (sigmoid at use, same as
        # region/boundary) that peaks at each instance's medial axis -- a learned signal
        # separate_instances_watershed can use for seeding/landscape instead of a purely
        # geometric distance-transform of the (already-merged) region mask.
        #
        # Bai & Urtasun's original DWT paper conditions its energy network explicitly on an
        # already-computed semantic segmentation mask (a separate, frozen upstream stage) --
        # not just shared backbone features. The first version of this head shared only the
        # bare post_upsample trunk with region/boundary, with no explicit region signal fed in
        # at all -- a real, accidental deviation from the paper, not a deliberate
        # simplification. energy_conditioning closes most of that gap cheaply, without a
        # second network/forward pass: it turns the region head's own current prediction into
        # extra input channels for energy_head, so energy has explicit access to "where region
        # currently believes foreground is" (mirroring the paper's mask-conditioning) rather
        # than having to implicitly re-derive that from shared trunk features alone.
        # .detach() below is what actually captures the paper's "frozen upstream stage" spirit
        # in a jointly-trained model: energy's loss must not perturb region_head's own
        # training through this path. See DeepWatershedEnergyNet, further below, for a fuller
        # two-stage variant (separate network, genuinely trained after region is frozen) kept
        # dormant for future comparison -- see PROGRESS.md.
        self.energy_conditioning = nn.Conv2d(1, 16, 3, padding=1)
        self.energy_head = nn.Conv2d(32 + 16, 1, 1)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, images: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        normalized = (images - self.mean) / self.std
        # Run the backbone's self-attention in fp32, not autocast's fp16. batch1_diag (added in
        # the previous debugging round) caught region_logits/boundary_logits already NaN on the
        # very first forward pass, before any optimizer step ever ran -- that rules out the
        # loss-reduction overflow theory (soft_dice_loss already reduces in fp32, untouched) and
        # the GradScaler-init_scale theory (a bad forward pass is NaN regardless of loss scale).
        # Full 1024x1024 images run attention over 4096 patch tokens -- fp16's ~65504 max and
        # narrow dynamic range is a well-documented source of attention-softmax NaN at this
        # scale, and every previous design that "worked" numerically only ever processed far
        # fewer tokens per attention pass. See PROGRESS.md.
        with torch.amp.autocast("cuda", enabled=False):
            feature_maps = self.backbone(pixel_values=normalized.float()).feature_maps
        pooled = [proj(feature_map) for proj, feature_map in zip(self.scale_proj, feature_maps)]
        x = self.pre_upsample(self.fuse(torch.cat(pooled, dim=1)))
        x = F.interpolate(x, size=images.shape[-2:], mode="bilinear", align_corners=False)
        x = self.post_upsample(torch.cat([x, self.image_stem(images)], dim=1))
        region_logits = self.region_head(x)
        boundary_logits = self.boundary_head(x)
        region_condition = self.energy_conditioning(region_logits.sigmoid().detach())
        energy_logits = self.energy_head(torch.cat([x, region_condition], dim=1))
        return region_logits, boundary_logits, energy_logits


# ---------------------------------------------------------------------------
# Paper-faithful two-stage energy net (Bai & Urtasun, "Deep Watershed Transform for Instance
# Segmentation", CVPR 2017) -- dormant, experimental, NOT called by any active driver cell.
# RegionBoundarySegmenter.energy_head (above) is a cheap, jointly-trained approximation of the
# paper's actual design (see its docstring); this class is the fuller version, kept here for a
# future try if the joint approximation turns out not to be accurate enough once real data is
# available. See PROGRESS.md.
# ---------------------------------------------------------------------------

class DeepWatershedEnergyNet(nn.Module):
    """A SEPARATE network, trained AFTER and CONDITIONED ON an already-trained (frozen)
    RegionBoundarySegmenter's own region-probability output -- not sharing its backbone/trunk
    at all, matching the paper's actual two-stage structure (a semantic segmentation stage,
    then a distinct energy stage that takes the raw image *and* that stage's mask as input).
    Deliberately a small, from-scratch conv stack, not another ViT pass -- the paper's own
    energy network is a comparatively lightweight second stage, not a repeat of the full
    semantic-segmentation backbone.

    Simplified relative to the paper in one specific, clearly-scoped way: predicts continuous
    [0,1] energy via regression (matching this codebase's existing energy target/decode --
    build_semantic_targets/separate_instances_watershed/energy_regression_loss, so results are
    comparable to RegionBoundarySegmenter.energy_head), not the paper's own discretized K-level
    energy classification plus a separate gradient-direction map and custom modified-watershed-
    cut decoding algorithm. The specific thing this class targets -- explicit two-stage
    conditioning on a frozen upstream mask, rather than joint multi-task training from shared
    features -- is faithful; that finer decode-algorithm detail is not reproduced.
    """

    def __init__(self, base_channels: int = 32) -> None:
        super().__init__()
        # 3 image channels (raw/CLAHE/raw, matching RegionBoundarySegmenter's own input
        # convention) + 1 region-probability channel from the frozen upstream stage.
        self.stem = nn.Sequential(
            ConvBlock(4, base_channels),
            ConvBlock(base_channels, base_channels * 2),
        )
        self.energy_head = nn.Conv2d(base_channels * 2, 1, 1)

    def forward(self, images: torch.Tensor, region_probability: torch.Tensor) -> torch.Tensor:
        x = self.stem(torch.cat([images, region_probability], dim=1))
        return self.energy_head(x)


def _train_two_stage_energy_net(
    fold: int,
    fold_manifest: pd.DataFrame,
    image_dir: Path,
    selected_image_ids: Mapping[str, int],
    indices: Mapping[str, Any],
    cfg: CFG,
    region_checkpoint: Path,
    output_checkpoint: Path,
    target_epochs: int,
) -> Path:
    """Stage-2 training for DeepWatershedEnergyNet: loads an already-trained
    RegionBoundarySegmenter checkpoint (region_checkpoint) and freezes it (no_grad, eval mode,
    never updated here) purely as a region-probability generator -- the actual two-stage
    procedure the paper describes, unlike RegionBoundarySegmenter's own joint energy_head
    which trains all three heads together from one shared backbone pass. Dormant/experimental:
    not called by any active driver cell yet. No AMP/GradScaler here deliberately -- this is a
    small, from-scratch conv net (no ViT backbone, no large-token attention pass), so the
    fp16-under-autocast NaN class that motivated AMP everywhere else in this file doesn't apply,
    and plain fp32 keeps a rarely-exercised, experimental path as simple as possible while this
    codebase's numerics keep getting shaken out elsewhere.
    """
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    region_model = RegionBoundarySegmenter(cfg).to(device)
    region_model.load_state_dict(torch.load(region_checkpoint, map_location=device))
    region_model.eval()
    for parameter in region_model.parameters():
        parameter.requires_grad = False

    train_files = fold_manifest.loc[fold_manifest["fold"] != fold, "filename"].tolist()
    val_files = fold_manifest.loc[fold_manifest["fold"] == fold, "filename"].tolist()
    if cfg.debug:
        train_files, val_files = train_files[:24], val_files[:8]
    train_dataset = SegmentationDataset(train_files, image_dir, selected_image_ids, indices, cfg, True)
    val_dataset = SegmentationDataset(val_files, image_dir, selected_image_ids, indices, cfg, False)
    # persistent_workers avoids respawning worker processes every single epoch (DataLoader's
    # default behavior otherwise) -- besides being wasteful, tearing down and respawning
    # workers repeatedly is what produces the harmless-but-noisy "Exception ignored in:
    # DataLoaderIter.__del__ ... AssertionError: can only test a child process" spam seen with
    # cfg.debug=True's very short epochs (a known PyTorch/multiprocessing teardown-timing quirk,
    # never actually raised/propagated -- training continues normally either way). Only valid
    # when num_workers > 0.
    train_loader = DataLoader(train_dataset, batch_size=cfg.vit_images_per_batch, shuffle=True,
                              num_workers=cfg.num_workers, drop_last=True,
                              persistent_workers=cfg.num_workers > 0)
    val_loader = DataLoader(val_dataset, batch_size=cfg.vit_images_per_batch, shuffle=False,
                            num_workers=cfg.num_workers, persistent_workers=cfg.num_workers > 0)

    energy_model = DeepWatershedEnergyNet().to(device)
    optimizer = torch.optim.AdamW(energy_model.parameters(), lr=cfg.refiner_lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=cfg.lr_plateau_factor, patience=cfg.lr_plateau_patience
    )
    best_mse = float("inf")
    epochs_since_improvement = 0

    for epoch in range(target_epochs):
        energy_model.train()
        for batch in tqdm(train_loader, desc=f"fold{fold} two-stage energy train e{epoch + 1}/{target_epochs}", leave=False):
            images = batch["image"].to(device)
            energy_targets = batch["energy"].to(device)
            with torch.no_grad():
                region_probability = region_model(images)[0].sigmoid()
            optimizer.zero_grad(set_to_none=True)
            energy_logits = energy_model(images, region_probability)
            loss = energy_regression_loss(energy_logits, energy_targets)
            loss.backward()
            optimizer.step()

        energy_model.eval(); mse_total = sample_count = 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"fold{fold} two-stage energy val e{epoch + 1}/{target_epochs}", leave=False):
                images = batch["image"].to(device)
                energy_targets = batch["energy"].to(device)
                region_probability = region_model(images)[0].sigmoid()
                energy_logits = energy_model(images, region_probability)
                mse_total += F.mse_loss(energy_logits.sigmoid(), energy_targets).item() * len(images)
                sample_count += len(images)
        validation_mse = mse_total / max(sample_count, 1)
        print(f"fold={fold} two-stage energy epoch={epoch + 1} val_energy_mse={validation_mse:.5f}")
        is_best = validation_mse < best_mse
        if is_best:
            best_mse = validation_mse
            epochs_since_improvement = 0
            atomic_torch_save(energy_model.state_dict(), output_checkpoint)
        else:
            epochs_since_improvement += 1
        scheduler.step(validation_mse)
        if epochs_since_improvement >= cfg.early_stopping_patience:
            print(f"fold={fold} two-stage energy early stopping at epoch {epoch + 1}/{target_epochs} "
                  f"(no val_energy_mse improvement for {cfg.early_stopping_patience} epochs)")
            break
    return output_checkpoint


# ---------------------------------------------------------------------------
# Seg(YOLO)->Energy pipeline (2026-09-30) -- the ACTIVE path as of this entry (see
# train_fold, cell 17, and PROGRESS.md). Stage 1 ("Seg") is YOLO26's own predicted instance
# masks (train_detector/prepare_yolo_fold, cell 8, revived -- unioned into one region map via
# yolo_region_mask, cell 13). Stage 2 ("Energy") is YoloConditionedEnergyNet below: a
# DINOv3-backboned network, conditioned on the frozen Stage-1 region map plus the raw image,
# predicting the same per-pixel [0,1] energy target used everywhere else in this file
# (build_semantic_targets). This is the actual two-stage structure Bai & Urtasun's Deep
# Watershed Transform paper describes -- Stage 1 a separate, frozen upstream semantic
# segmentation network, Stage 2 conditioned on its mask -- with YOLO26 as Stage 1 instead of a
# from-scratch semantic segmentation net. RegionBoundarySegmenter/_train_segmenter (detector-
# free, single DINOv3 pass, no detector) and DeepWatershedEnergyNet/_train_two_stage_energy_net
# (paper-faithful two-stage variant conditioned on a frozen RegionBoundarySegmenter instead of
# YOLO) are both kept, not deleted, no longer called by the active path.
# ---------------------------------------------------------------------------

class YoloConditionedEnergyDataset(Dataset):
    """Stage 2 dataset: same image/energy-target construction as SegmentationDataset, plus a
    precomputed YOLO region mask (see precompute_yolo_region_masks, called once before
    training since the detector is frozen and its masks never change across epochs) as the
    Stage-1 conditioning input for YoloConditionedEnergyNet."""

    def __init__(
        self,
        filenames: Sequence[str],
        image_dir: Path,
        selected_image_ids: Mapping[str, int],
        indices: Mapping[str, Any],
        region_masks: Mapping[str, np.ndarray],
        cfg: CFG,
        augment: bool,
    ) -> None:
        self.image_dir, self.indices, self.cfg, self.augment = image_dir, indices, cfg, augment
        self.filenames, self.selected_image_ids, self.region_masks = filenames, selected_image_ids, region_masks

    def __len__(self) -> int:
        return len(self.filenames)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        cfg = self.cfg
        filename = self.filenames[index]
        image_id = self.selected_image_ids[filename]
        size = cfg.vit_image_size
        raw = cv2.imread(str(self.image_dir / filename), cv2.IMREAD_GRAYSCALE)
        image = cv2.resize(raw, (size, size), interpolation=cv2.INTER_LINEAR)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(image)
        image_tensor = np.stack([image, clahe, image], axis=0).astype(np.float32) / 255.0
        # Region/boundary discarded here -- only the energy target is needed for Stage 2. Costs
        # a redundant region/boundary computation (build_semantic_targets always computes all
        # three together), accepted deliberately to reuse proven, tested code rather than fork
        # an energy-only variant for a training path that runs occasionally, not per-batch-hot.
        _, _, energy = build_semantic_targets(image_id, self.indices, size, cfg.boundary_width)
        region_mask = self.region_masks[filename].astype(np.float32)

        if self.augment:
            rng = np.random.default_rng(cfg.seed + index + random.randint(0, 1_000_000))
            if random.random() < 0.5:
                image_tensor = image_tensor[:, :, ::-1].copy()
                energy = energy[:, ::-1].copy()
                region_mask = region_mask[:, ::-1].copy()
            if random.random() < 0.5:
                image_tensor = image_tensor[:, ::-1, :].copy()
                energy = energy[::-1, :].copy()
                region_mask = region_mask[::-1, :].copy()
            image_tensor = np.clip(image_tensor + rng.normal(0, 0.01, image_tensor.shape), 0, 1).astype(np.float32)

        return {
            "image": torch.from_numpy(image_tensor),
            "region_mask": torch.from_numpy(region_mask[None]),
            "energy": torch.from_numpy(energy[None].astype(np.float32)),
        }


class YoloConditionedEnergyNet(nn.Module):
    """Stage 2 of the Seg(YOLO)->Energy pipeline: a DINOv3-backboned network predicting a
    per-pixel [0,1] energy map conditioned on the raw image AND a frozen YOLO detector's own
    region mask (see yolo_region_mask, cell 13) -- the actual two-stage structure the DWT paper
    describes, with YOLO26 supplying Stage 1 instead of a from-scratch semantic segmentation
    net, and DINOv3 supplying Stage 2's visual features. Shares the same backbone/multi-scale
    fuse/pre_upsample trunk and fp32-backbone-forward fix as RegionBoundarySegmenter (see that
    class's docstring for why the fp32 forward matters -- this is the exact fix that closed the
    NaN-on-batch-1 bug, and skipping it here would silently reopen it in a new class), just
    conditioned on the YOLO region mask via an extra skip channel instead of producing its own
    region head.
    """

    def __init__(self, cfg: "CFG") -> None:
        super().__init__()
        from transformers import AutoBackbone

        self.backbone = AutoBackbone.from_pretrained(
            cfg.vit_hub_id,
            out_indices=list(cfg.vit_out_indices),
            reshape_hidden_states=True,
            attn_implementation="eager",
        )
        for disabled_field in ("pos_embed_jitter", "pos_embed_shift", "pos_embed_rescale"):
            if hasattr(self.backbone.config, disabled_field):
                setattr(self.backbone.config, disabled_field, None)
        hidden_size = self.backbone.config.hidden_size
        n_scales = len(cfg.vit_out_indices)
        fuse_channels = 256
        self.scale_proj = nn.ModuleList(nn.Conv2d(hidden_size, fuse_channels, 1) for _ in range(n_scales))
        self.fuse = ConvBlock(fuse_channels * n_scales, fuse_channels)
        self.pre_upsample = ConvBlock(fuse_channels, 128)
        self.image_stem = ConvBlock(3, 16)
        self.region_stem = ConvBlock(1, 16)  # YOLO's own frozen region mask, the Stage-1 signal
        self.post_upsample = ConvBlock(128 + 16 + 16, 32)
        self.energy_head = nn.Conv2d(32, 1, 1)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, images: torch.Tensor, region_mask: torch.Tensor) -> torch.Tensor:
        normalized = (images - self.mean) / self.std
        # Same fp32-backbone-forward fix as RegionBoundarySegmenter -- see its docstring/
        # PROGRESS.md. Full 1024x1024 images run attention over 4096 patch tokens; fp16's
        # narrow dynamic range is a well-documented source of attention-softmax NaN at this
        # scale, execution-confirmed in this exact codebase once already.
        with torch.amp.autocast("cuda", enabled=False):
            feature_maps = self.backbone(pixel_values=normalized.float()).feature_maps
        pooled = [proj(feature_map) for proj, feature_map in zip(self.scale_proj, feature_maps)]
        x = self.pre_upsample(self.fuse(torch.cat(pooled, dim=1)))
        x = F.interpolate(x, size=images.shape[-2:], mode="bilinear", align_corners=False)
        x = self.post_upsample(torch.cat([x, self.image_stem(images), self.region_stem(region_mask)], dim=1))
        return self.energy_head(x)


def precompute_yolo_region_masks(
    detector: Any,
    filenames: Sequence[str],
    image_dir: Path,
    cfg: CFG,
) -> dict[str, np.ndarray]:
    """Runs the already-trained, frozen YOLO detector once per image and unions its predicted
    instance masks into one dense boolean region mask at cfg.vit_image_size resolution -- Stage
    1 ("Seg") of the Seg(YOLO)->Energy pipeline. Precomputed once, not per-epoch: the detector
    never updates during Stage 2 training, so its masks are identical every epoch -- recomputing
    them each epoch would be pure waste. See yolo_region_mask, cell 13, for the per-image logic
    (shared with inference)."""
    return {
        filename: yolo_region_mask(detector, image_dir / filename, cfg)
        for filename in tqdm(filenames, desc="precomputing YOLO region masks", leave=False)
    }


def _train_yolo_conditioned_energy_net(
    fold: int,
    fold_manifest: pd.DataFrame,
    image_dir: Path,
    selected_image_ids: Mapping[str, int],
    indices: Mapping[str, Any],
    cfg: CFG,
    detector_checkpoint: Path,
    checkpoint: Path,
    meta_path: Path,
    fingerprint: str,
    target_epochs: int,
) -> Path:
    from ultralytics import YOLO

    train_files = fold_manifest.loc[fold_manifest["fold"] != fold, "filename"].tolist()
    val_files = fold_manifest.loc[fold_manifest["fold"] == fold, "filename"].tolist()
    if cfg.debug:
        train_files, val_files = train_files[:24], val_files[:8]

    detector = YOLO(str(detector_checkpoint))
    train_region_masks = precompute_yolo_region_masks(detector, train_files, image_dir, cfg)
    val_region_masks = precompute_yolo_region_masks(detector, val_files, image_dir, cfg)
    del detector; gc.collect(); torch.cuda.empty_cache()

    train_dataset = YoloConditionedEnergyDataset(train_files, image_dir, selected_image_ids, indices,
                                                  train_region_masks, cfg, True)
    val_dataset = YoloConditionedEnergyDataset(val_files, image_dir, selected_image_ids, indices,
                                                val_region_masks, cfg, False)
    train_loader = DataLoader(train_dataset, batch_size=cfg.vit_images_per_batch, shuffle=True,
                              num_workers=cfg.num_workers, drop_last=True,
                              persistent_workers=cfg.num_workers > 0)
    val_loader = DataLoader(val_dataset, batch_size=cfg.vit_images_per_batch, shuffle=False,
                            num_workers=cfg.num_workers, persistent_workers=cfg.num_workers > 0)

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    energy_model = YoloConditionedEnergyNet(cfg).to(device)
    optimizer = torch.optim.AdamW(vit_param_groups(energy_model, cfg), weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=cfg.lr_plateau_factor, patience=cfg.lr_plateau_patience
    )
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available(), init_scale=2.0 ** 10)
    best_mse = float("inf")
    epochs_since_improvement = 0

    for epoch in range(target_epochs):
        energy_model.train()
        train_batches = train_scale_skips = 0
        for batch in tqdm(train_loader, desc=f"fold{fold} yolo-energy train e{epoch + 1}/{target_epochs}", leave=False):
            train_batches += 1
            images = batch["image"].to(device)
            region_mask = batch["region_mask"].to(device)
            energy_targets = batch["energy"].to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                energy_logits = energy_model(images, region_mask)
                loss = energy_regression_loss(energy_logits, energy_targets)
            scale_before = scaler.get_scale()
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            if scaler.get_scale() < scale_before:
                train_scale_skips += 1
        print(f"fold={fold} yolo-energy epoch={epoch + 1} train_diag: batches={train_batches} "
              f"scaler_step_skips={train_scale_skips} final_scale={scaler.get_scale():.1f}")

        energy_model.eval(); mse_total = sample_count = 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"fold{fold} yolo-energy val e{epoch + 1}/{target_epochs}", leave=False):
                images = batch["image"].to(device)
                region_mask = batch["region_mask"].to(device)
                energy_targets = batch["energy"].to(device)
                energy_logits = energy_model(images, region_mask)
                mse_total += F.mse_loss(energy_logits.sigmoid(), energy_targets).item() * len(images)
                sample_count += len(images)
        validation_mse = mse_total / max(sample_count, 1)
        print(f"fold={fold} yolo-energy epoch={epoch + 1} val_energy_mse={validation_mse:.5f} "
              f"lr={optimizer.param_groups[0]['lr']:.2e}")
        is_best = validation_mse < best_mse
        if is_best:
            best_mse = validation_mse
            epochs_since_improvement = 0
            atomic_torch_save(energy_model.state_dict(), checkpoint)
        else:
            epochs_since_improvement += 1
        # Reuses _record_refiner_epoch/its "val_crop_dice" field name as a generic per-epoch
        # metric recorder (it only stores whatever value it's given and doesn't interpret it) --
        # the JSON history will show val_energy_mse's value under that key. Accepted rather than
        # forking a near-identical helper just to rename one field.
        _record_refiner_epoch(meta_path, fold, fingerprint, target_epochs, epoch, validation_mse, is_best,
                              best_mse, {"batches": train_batches, "scaler_step_skips": train_scale_skips,
                                         "final_scale": scaler.get_scale()}, {"instances": sample_count})
        scheduler.step(validation_mse)
        if epochs_since_improvement >= cfg.early_stopping_patience:
            print(f"fold={fold} yolo-energy early stopping at epoch {epoch + 1}/{target_epochs} "
                  f"(no val_energy_mse improvement for {cfg.early_stopping_patience} epochs)")
            break
    _finalize_refiner_meta(meta_path, fold, fingerprint, target_epochs)
    return checkpoint


def train_yolo_conditioned_energy_net(
    fold: int,
    fold_manifest: pd.DataFrame,
    image_dir: Path,
    selected_image_ids: Mapping[str, int],
    indices: Mapping[str, Any],
    cfg: CFG,
    work_dir: Path,
    detector_checkpoint: Path,
) -> Path:
    output_dir = work_dir / "models" / f"fold_{fold}"
    output_dir.mkdir(parents=True, exist_ok=True)
    checkpoint = output_dir / "energy_net.pt"
    meta_path = output_dir / "energy_net_meta.json"
    fingerprint = yolo_energy_fingerprint(cfg, fold_manifest, detector_config_fingerprint(cfg))
    target_epochs = 2 if cfg.debug else cfg.refiner_epochs
    meta = read_json_if_valid(meta_path)
    if (checkpoint.exists() and meta is not None and meta.get("status") == "complete"
            and meta.get("config_fingerprint") == fingerprint):
        return checkpoint
    return _train_yolo_conditioned_energy_net(
        fold, fold_manifest, image_dir, selected_image_ids, indices, cfg, detector_checkpoint,
        checkpoint, meta_path, fingerprint, target_epochs,
    )


def corrupt_region_mask(mask: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Degrade a clean YOLO region mask toward what YOLO produces on unseen images: grow or
    shrink the edges, drop whole instances (misses), add stray blobs (false positives), and
    sometimes blank the channel. Applied to the network input only; labels are untouched."""
    if rng.random() < 0.1:
        return np.zeros_like(mask)
    out = mask.astype(np.uint8)
    radius = int(rng.integers(1, 5))
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * radius + 1, 2 * radius + 1))
    out = cv2.dilate(out, kernel) if rng.random() < 0.5 else cv2.erode(out, kernel)
    count, components = cv2.connectedComponents(out, connectivity=8)
    for label in range(1, count):
        if rng.random() < 0.15:
            out[components == label] = 0
    height, width = out.shape
    for _ in range(int(rng.integers(0, 3))):
        center = (int(rng.integers(0, width)), int(rng.integers(0, height)))
        axes = (int(rng.integers(3, 25)), int(rng.integers(2, 8)))
        cv2.ellipse(out, center, axes, float(rng.uniform(0, 180)), 0, 360, 1, -1)
    return out.astype(mask.dtype)


class YoloConditionedEmbeddingDataset(Dataset):
    """Stage 2 dataset for the embedding-head decode (replaces YoloConditionedEnergyDataset as
    the active Stage 2 target -- see PROGRESS.md "embedding-head instance-separation swap").
    Same image/region-mask construction, but the target is a per-instance integer label map
    (build_instance_label_map) instead of a continuous energy map -- the discriminative loss
    needs per-instance identity, not a single scalar ridge."""

    def __init__(
        self,
        filenames: Sequence[str],
        image_dir: Path,
        selected_image_ids: Mapping[str, int],
        indices: Mapping[str, Any],
        region_masks: Mapping[str, np.ndarray],
        cfg: CFG,
        augment: bool,
    ) -> None:
        self.image_dir, self.indices, self.cfg, self.augment = image_dir, indices, cfg, augment
        self.filenames, self.selected_image_ids, self.region_masks = filenames, selected_image_ids, region_masks

    def __len__(self) -> int:
        return len(self.filenames)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        cfg = self.cfg
        filename = self.filenames[index]
        image_id = self.selected_image_ids[filename]
        size = cfg.vit_image_size
        raw = cv2.imread(str(self.image_dir / filename), cv2.IMREAD_GRAYSCALE)
        image = cv2.resize(raw, (size, size), interpolation=cv2.INTER_LINEAR)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(image)
        image_tensor = np.stack([image, clahe, image], axis=0).astype(np.float32) / 255.0
        labels = build_instance_label_map(image_id, self.indices, size)
        region_mask = self.region_masks[filename].astype(np.float32)

        if self.augment:
            rng = np.random.default_rng(cfg.seed + index + random.randint(0, 1_000_000))
            if random.random() < 0.5:
                image_tensor = image_tensor[:, :, ::-1].copy()
                labels = labels[:, ::-1].copy()
                region_mask = region_mask[:, ::-1].copy()
            if random.random() < 0.5:
                image_tensor = image_tensor[:, ::-1, :].copy()
                labels = labels[::-1, :].copy()
                region_mask = region_mask[::-1, :].copy()
            image_tensor = np.clip(image_tensor + rng.normal(0, 0.01, image_tensor.shape), 0, 1).astype(np.float32)
            if rng.random() < cfg.region_mask_corruption:
                region_mask = corrupt_region_mask(region_mask, rng)

        return {
            "image": torch.from_numpy(image_tensor),
            "region_mask": torch.from_numpy(region_mask[None]),
            "labels": torch.from_numpy(labels.astype(np.int64)),
        }


class YoloConditionedEmbeddingNet(nn.Module):
    """Stage 2 of the Seg(YOLO)->Embedding pipeline (replaces YoloConditionedEnergyNet as the
    active Stage 2 -- see PROGRESS.md "embedding-head instance-separation swap"): same
    backbone/multi-scale-fuse/pre_upsample trunk and fp32-backbone-forward fix as
    YoloConditionedEnergyNet (see that class docstring for why the fp32 forward matters), but
    the head regresses a per-pixel embedding_dim-D vector instead of a scalar energy value --
    trained with discriminative_embedding_loss to pull same-instance pixels together and push
    different instances mean embeddings apart. Chosen over energy+watershed because watershed
    is known to struggle on thin, elongated, touching shapes (this competition exact failure
    mode), where flow/embedding-based separation (Cellpose-style) generally does better. See
    PROGRESS.md.
    """

    def __init__(self, cfg: "CFG") -> None:
        super().__init__()
        from transformers import AutoBackbone

        self.backbone = AutoBackbone.from_pretrained(
            cfg.vit_hub_id,
            out_indices=list(cfg.vit_out_indices),
            reshape_hidden_states=True,
            attn_implementation="eager",
        )
        for disabled_field in ("pos_embed_jitter", "pos_embed_shift", "pos_embed_rescale"):
            if hasattr(self.backbone.config, disabled_field):
                setattr(self.backbone.config, disabled_field, None)
        hidden_size = self.backbone.config.hidden_size
        n_scales = len(cfg.vit_out_indices)
        fuse_channels = 256
        self.scale_proj = nn.ModuleList(nn.Conv2d(hidden_size, fuse_channels, 1) for _ in range(n_scales))
        self.fuse = ConvBlock(fuse_channels * n_scales, fuse_channels)
        self.pre_upsample = ConvBlock(fuse_channels, 128)
        self.image_stem = ConvBlock(3, 16)
        self.region_stem = ConvBlock(1, 16)  # YOLO own frozen region mask, the Stage-1 signal
        self.post_upsample = ConvBlock(128 + 16 + 16, 32)
        self.embedding_head = nn.Conv2d(32, cfg.embedding_dim, 1)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, images: torch.Tensor, region_mask: torch.Tensor) -> torch.Tensor:
        normalized = (images - self.mean) / self.std
        with torch.amp.autocast("cuda", enabled=False):
            feature_maps = self.backbone(pixel_values=normalized.float()).feature_maps
        pooled = [proj(feature_map) for proj, feature_map in zip(self.scale_proj, feature_maps)]
        x = self.pre_upsample(self.fuse(torch.cat(pooled, dim=1)))
        x = F.interpolate(x, size=images.shape[-2:], mode="bilinear", align_corners=False)
        x = self.post_upsample(torch.cat([x, self.image_stem(images), self.region_stem(region_mask)], dim=1))
        return self.embedding_head(x)


class YoloConditionedEmbeddingUNet(nn.Module):
    """Alternative Stage 2 architecture for the embedding decode -- same DINOv3 backbone as
    YoloConditionedEmbeddingNet, but a genuine progressive, skip-connected decoder instead of
    one bilinear jump from the backbone's patch-grid resolution straight to full image
    resolution. See PROGRESS.md "U-Net-style decoder" to-do: competitors on this exact problem
    use real U-Nets (multi-scale skip connections), which are known to preserve fine spatial
    detail -- relevant since filaments/barbs can be only a few pixels wide.

    DINOv3's own intermediate layers (vit_out_indices) are NOT a spatial feature pyramid -- a
    plain ViT has no built-in downsampling between blocks, so every requested layer shares the
    identical patch-grid resolution (depth varies, not scale). A real encoder-side
    multi-resolution skip connection therefore is not available from this backbone. This
    decoder instead takes the skip role from the INPUT side: the raw image + YOLO region mask
    are downsampled to each intermediate decoder resolution and concatenated in at every
    upsampling stage -- the same pattern ViT-encoder/CNN-decoder hybrid segmentation designs
    (e.g. SETR, TransUNet) use for exactly this reason, since there is no encoder-side feature
    map at those resolutions to borrow from instead. Four upsampling stages progressively
    refine (dinov3-vits16 at vit_image_size=1024 means a 64x64 patch grid -> 4 doublings reaches
    1024 exactly), which is the actual mechanism expected to help, not just a deeper network --
    one bilinear jump has no opportunity to re-examine fine detail at intermediate resolutions,
    a progressive decoder does at every stage.
    """

    def __init__(self, cfg: "CFG") -> None:
        super().__init__()
        from transformers import AutoBackbone

        self.backbone = AutoBackbone.from_pretrained(
            cfg.vit_hub_id,
            out_indices=list(cfg.vit_out_indices),
            reshape_hidden_states=True,
            attn_implementation="eager",
        )
        for disabled_field in ("pos_embed_jitter", "pos_embed_shift", "pos_embed_rescale"):
            if hasattr(self.backbone.config, disabled_field):
                setattr(self.backbone.config, disabled_field, None)
        hidden_size = self.backbone.config.hidden_size
        n_scales = len(cfg.vit_out_indices)
        fuse_channels = 256
        self.scale_proj = nn.ModuleList(nn.Conv2d(hidden_size, fuse_channels, 1) for _ in range(n_scales))
        self.fuse = ConvBlock(fuse_channels * n_scales, fuse_channels)

        decoder_channels = (128, 64, 32, 16)
        in_channels = fuse_channels
        self.up_blocks = nn.ModuleList()
        self.skip_stems = nn.ModuleList()
        self.decode_blocks = nn.ModuleList()
        for out_channels in decoder_channels:
            self.up_blocks.append(nn.ConvTranspose2d(in_channels, out_channels, 2, 2))
            self.skip_stems.append(ConvBlock(4, out_channels // 2))  # image(3) + region(1), this resolution
            self.decode_blocks.append(ConvBlock(out_channels + out_channels // 2, out_channels))
            in_channels = out_channels
        self.embedding_head = nn.Conv2d(decoder_channels[-1], cfg.embedding_dim, 1)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, images: torch.Tensor, region_mask: torch.Tensor) -> torch.Tensor:
        normalized = (images - self.mean) / self.std
        with torch.amp.autocast("cuda", enabled=False):
            feature_maps = self.backbone(pixel_values=normalized.float()).feature_maps
        pooled = [proj(feature_map) for proj, feature_map in zip(self.scale_proj, feature_maps)]
        x = self.fuse(torch.cat(pooled, dim=1))
        image_region = torch.cat([images, region_mask], dim=1)
        for up_block, skip_stem, decode_block in zip(self.up_blocks, self.skip_stems, self.decode_blocks):
            x = up_block(x)
            skip_input = F.interpolate(image_region, size=x.shape[-2:], mode="bilinear", align_corners=False)
            x = decode_block(torch.cat([x, skip_stem(skip_input)], dim=1))
        # Safety net, not expected to actually resize anything for the documented
        # dinov3-vits16/vit_image_size=1024 configuration (4 doublings land exactly on
        # target) -- cheap insurance against a future backbone/patch-size/image-size
        # combination where the doublings do not divide evenly.
        x = F.interpolate(x, size=images.shape[-2:], mode="bilinear", align_corners=False)
        return self.embedding_head(x)


def _train_yolo_conditioned_embedding_net(
    fold: int,
    fold_manifest: pd.DataFrame,
    image_dir: Path,
    selected_image_ids: Mapping[str, int],
    indices: Mapping[str, Any],
    cfg: CFG,
    detector_checkpoint: Path,
    checkpoint: Path,
    meta_path: Path,
    fingerprint: str,
    target_epochs: int,
    model_cls: type = YoloConditionedEmbeddingNet,
) -> Path:
    """model_cls parametrizes the Stage 2 architecture -- YoloConditionedEmbeddingNet (default)
    or YoloConditionedEmbeddingUNet -- so the two architectures being compared (see PROGRESS.md
    "U-Net-decoder variant, fused with the embedding net, both evaluated") share this exact
    training loop instead of forking a near-duplicate for the second one.

    Checkpoint/early-stopping selection is driven by REAL decoded PQ (the actual YOLO mask ->
    embedding clustering -> panoptic_quality pipeline, on a fixed validation subset), not
    val_embedding_loss -- a lower embedding-space pull/push loss does not provably mean better
    final masks once greedy clustering is in the loop. val_embedding_loss is still computed and
    logged every epoch (cheap, useful diagnostic for e.g. spotting training instability) but no
    longer decides which checkpoint gets kept. See PROGRESS.md "Stage 2 selection criterion
    swapped to real PQ"."""
    from ultralytics import YOLO

    train_files = fold_manifest.loc[fold_manifest["fold"] != fold, "filename"].tolist()
    val_files = fold_manifest.loc[fold_manifest["fold"] == fold, "filename"].tolist()
    if cfg.debug:
        train_files, val_files = train_files[:24], val_files[:8]
    # Fixed prefix, not re-sampled per epoch, so epoch-to-epoch PQ changes reflect real model
    # improvement rather than which images happened to be sampled -- see cfg.embedding_pq_eval_cap.
    pq_eval_files = val_files[:cfg.embedding_pq_eval_cap]

    detector = YOLO(str(detector_checkpoint))
    train_region_masks = precompute_yolo_region_masks(detector, train_files, image_dir, cfg)
    val_region_masks = precompute_yolo_region_masks(detector, val_files, image_dir, cfg)
    # NOT deleted here (unlike before this PQ-selection change) -- the per-epoch decode-based PQ
    # check below needs the frozen detector again every epoch (predict_yolo_embedding_instances
    # calls yolo_instance_masks internally). Cheap to keep resident: YOLO26s is lightweight next
    # to the DINOv3 backbone already loaded for embedding_model.

    train_dataset = YoloConditionedEmbeddingDataset(train_files, image_dir, selected_image_ids, indices,
                                                     train_region_masks, cfg, True)
    val_dataset = YoloConditionedEmbeddingDataset(val_files, image_dir, selected_image_ids, indices,
                                                   val_region_masks, cfg, False)
    train_loader = DataLoader(train_dataset, batch_size=cfg.vit_images_per_batch, shuffle=True,
                              num_workers=cfg.num_workers, drop_last=True,
                              persistent_workers=cfg.num_workers > 0)
    val_loader = DataLoader(val_dataset, batch_size=cfg.vit_images_per_batch, shuffle=False,
                            num_workers=cfg.num_workers, persistent_workers=cfg.num_workers > 0)

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    arch_name = model_cls.__name__
    embedding_model = model_cls(cfg).to(device)
    optimizer = torch.optim.AdamW(vit_param_groups(embedding_model, cfg), weight_decay=1e-4)
    # mode="max": higher PQ is better (was mode="min" against val_embedding_loss before this change).
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=cfg.lr_plateau_factor, patience=cfg.lr_plateau_patience
    )
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available(), init_scale=2.0 ** 10)
    best_pq = -1.0
    epochs_since_improvement = 0

    for epoch in range(target_epochs):
        embedding_model.train()
        train_batches = train_scale_skips = 0
        for batch in tqdm(train_loader, desc=f"fold{fold} {arch_name} train e{epoch + 1}/{target_epochs}", leave=False):
            train_batches += 1
            images = batch["image"].to(device)
            region_mask = batch["region_mask"].to(device)
            labels = batch["labels"].to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                embeddings = embedding_model(images, region_mask)
                loss = discriminative_embedding_loss(embeddings.float(), labels, cfg)
            scale_before = scaler.get_scale()
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            if scaler.get_scale() < scale_before:
                train_scale_skips += 1
        print(f"fold={fold} {arch_name} epoch={epoch + 1} train_diag: batches={train_batches} "
              f"scaler_step_skips={train_scale_skips} final_scale={scaler.get_scale():.1f}")

        embedding_model.eval(); loss_total = sample_count = 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"fold{fold} {arch_name} val-loss e{epoch + 1}/{target_epochs}", leave=False):
                images = batch["image"].to(device)
                region_mask = batch["region_mask"].to(device)
                labels = batch["labels"].to(device)
                embeddings = embedding_model(images, region_mask)
                loss_total += discriminative_embedding_loss(embeddings.float(), labels, cfg).item() * len(images)
                sample_count += len(images)
        validation_loss = loss_total / max(sample_count, 1)

        # The actual selection signal: real decode (YOLO masks -> embedding clustering) scored
        # against real ground truth via panoptic_quality -- same code path inference/OOF use
        # (predict_yolo_embedding_instances), not a training-only shortcut.
        pq_rows = []
        for filename in tqdm(pq_eval_files, desc=f"fold{fold} {arch_name} val-PQ e{epoch + 1}/{target_epochs}", leave=False):
            masks, scores = predict_yolo_embedding_instances(image_dir / filename, detector, embedding_model, device, cfg)
            pq_rows.extend(evaluate_prediction_for_file(filename, masks, indices, cfg))
        # Pooled competition PQ (aggregate_pq), not the mean of per-image PQ -- see its docstring.
        pq_summary = aggregate_pq(pd.DataFrame(pq_rows))
        validation_pq = pq_summary["pq"]

        print(f"fold={fold} {arch_name} epoch={epoch + 1} val_pq={validation_pq:.5f} "
              f"(per-image mean {pq_summary['pq_image_mean']:.5f}) "
              f"val_embedding_loss={validation_loss:.5f} lr={optimizer.param_groups[0]['lr']:.2e} "
              f"(pq_eval_images={len(pq_eval_files)})")
        is_best = validation_pq > best_pq
        if is_best:
            best_pq = validation_pq
            epochs_since_improvement = 0
            atomic_torch_save(embedding_model.state_dict(), checkpoint)
        else:
            epochs_since_improvement += 1
        _record_refiner_epoch(meta_path, fold, fingerprint, target_epochs, epoch, validation_pq, is_best,
                              best_pq, {"batches": train_batches, "scaler_step_skips": train_scale_skips,
                                        "final_scale": scaler.get_scale()},
                              {"instances": sample_count, "val_embedding_loss": validation_loss,
                               "pq_eval_images": len(pq_eval_files)})
        scheduler.step(validation_pq)
        if epochs_since_improvement >= cfg.early_stopping_patience:
            print(f"fold={fold} {arch_name} early stopping at epoch {epoch + 1}/{target_epochs} "
                  f"(no val_pq improvement for {cfg.early_stopping_patience} epochs)")
            break
    del detector; gc.collect(); torch.cuda.empty_cache()
    _finalize_refiner_meta(meta_path, fold, fingerprint, target_epochs)
    return checkpoint


def train_yolo_conditioned_embedding_net(
    fold: int,
    fold_manifest: pd.DataFrame,
    image_dir: Path,
    selected_image_ids: Mapping[str, int],
    indices: Mapping[str, Any],
    cfg: CFG,
    work_dir: Path,
    detector_checkpoint: Path,
    model_cls: type = YoloConditionedEmbeddingNet,
    checkpoint_name: str = "embedding_net.pt",
    meta_name: str = "embedding_net_meta.json",
) -> Path:
    """checkpoint_name/meta_name keep the two Stage 2 architectures' checkpoints side by side
    under the same fold directory (embedding_net.pt vs embedding_unet.pt) rather than one
    overwriting the other -- both are trained every fold so they can be compared and fused
    (see train_fold, PROGRESS.md)."""
    output_dir = work_dir / "models" / f"fold_{fold}"
    output_dir.mkdir(parents=True, exist_ok=True)
    checkpoint = output_dir / checkpoint_name
    meta_path = output_dir / meta_name
    fingerprint = yolo_embedding_fingerprint(cfg, fold_manifest, detector_config_fingerprint(cfg))
    target_epochs = 2 if cfg.debug else cfg.refiner_epochs
    meta = read_json_if_valid(meta_path)
    if (checkpoint.exists() and meta is not None and meta.get("status") == "complete"
            and meta.get("config_fingerprint") == fingerprint):
        return checkpoint
    return _train_yolo_conditioned_embedding_net(
        fold, fold_manifest, image_dir, selected_image_ids, indices, cfg, detector_checkpoint,
        checkpoint, meta_path, fingerprint, target_epochs, model_cls=model_cls,
    )


def _train_segmenter(
    fold: int,
    fold_manifest: pd.DataFrame,
    image_dir: Path,
    selected_image_ids: Mapping[str, int],
    indices: Mapping[str, Any],
    cfg: CFG,
    work_dir: Path,
    checkpoint: Path,
    meta_path: Path,
    fingerprint: str,
    target_epochs: int,
) -> Path:
    train_files = fold_manifest.loc[fold_manifest["fold"] != fold, "filename"].tolist()
    val_files = fold_manifest.loc[fold_manifest["fold"] == fold, "filename"].tolist()
    if cfg.debug:
        train_files, val_files = train_files[:24], val_files[:8]
    train_dataset = SegmentationDataset(train_files, image_dir, selected_image_ids, indices, cfg, True)
    val_dataset = SegmentationDataset(val_files, image_dir, selected_image_ids, indices, cfg, False)
    # persistent_workers avoids respawning worker processes every single epoch (DataLoader's
    # default behavior otherwise) -- besides being wasteful, tearing down and respawning
    # workers repeatedly is what produces the harmless-but-noisy "Exception ignored in:
    # DataLoaderIter.__del__ ... AssertionError: can only test a child process" spam seen with
    # cfg.debug=True's very short epochs (a known PyTorch/multiprocessing teardown-timing quirk,
    # never actually raised/propagated -- training continues normally either way). Only valid
    # when num_workers > 0.
    train_loader = DataLoader(train_dataset, batch_size=cfg.vit_images_per_batch, shuffle=True,
                              num_workers=cfg.num_workers, drop_last=True,
                              persistent_workers=cfg.num_workers > 0)
    val_loader = DataLoader(val_dataset, batch_size=cfg.vit_images_per_batch, shuffle=False,
                            num_workers=cfg.num_workers, persistent_workers=cfg.num_workers > 0)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = RegionBoundarySegmenter(cfg).to(device)
    optimizer = torch.optim.AdamW(vit_param_groups(model, cfg), weight_decay=1e-4)
    epochs = target_epochs  # a ceiling now, not a target -- early stopping below can stop sooner.
    # ReduceLROnPlateau (monitoring val_crop_dice) replaces a fixed CosineAnnealingLR(T_max=
    # epochs) schedule that forced LR to ~0 by the last epoch regardless of whether validation
    # was still improving -- that's guessed in advance, not responsive to actual convergence.
    # mode="max": val_crop_dice is a similarity score, higher is better. See PROGRESS.md.
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=cfg.lr_plateau_factor, patience=cfg.lr_plateau_patience
    )
    # init_scale=2**10 (not GradScaler's default 2**16): the region+boundary architecture sums
    # two loss terms over full 1024x1024 images (16x the pixels of the old 256x256 per-crop
    # design), and the default scale drove every fold's scaler to collapse to exactly 0.0 in
    # epoch 1 (166 halvings, execution-confirmed) -- see PROGRESS.md. Untested hypothesis, cheapest
    # fix first: a smaller starting scale needs far fewer halvings before finding a stable value,
    # so it should survive even if some early steps still overflow fp16.
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available(), init_scale=2.0 ** 10)
    best_dice = -1.0
    epochs_since_improvement = 0

    def blended_loss(logits: torch.Tensor, targets: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        bce = F.binary_cross_entropy_with_logits(logits, targets)
        dice = soft_dice_loss(logits, targets)
        return 0.45 * bce + 0.55 * dice, bce, dice

    def batch_dice(predictions: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        intersection = (predictions & targets).sum(dim=(1, 2, 3)).float()
        denominator = predictions.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
        return (2 * intersection + 1e-7) / (denominator + 1e-7)

    for epoch in range(epochs):
        model.train()
        train_batches = train_scale_skips = 0
        for batch in tqdm(train_loader, desc=f"fold{fold} segmenter train e{epoch + 1}/{epochs}", leave=False):
            train_batches += 1
            images = batch["image"].to(device)
            region_targets = batch["region"].to(device)
            boundary_targets = batch["boundary"].to(device)
            energy_targets = batch["energy"].to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                region_logits, boundary_logits, energy_logits = model(images)
                region_loss, region_bce, region_dice = blended_loss(region_logits, region_targets)
                boundary_loss, boundary_bce, boundary_dice = blended_loss(boundary_logits, boundary_targets)
                energy_loss = energy_regression_loss(energy_logits, energy_targets)
                loss = (region_loss + cfg.boundary_loss_weight * boundary_loss
                        + cfg.energy_loss_weight * energy_loss)
            if epoch == 0 and train_batches == 1:
                # Isolates which term is actually inf/nan on the very first step, before guessing
                # a third time -- see PROGRESS.md. images_has_nan rules out corrupt input data as
                # an alternative explanation regardless of whether the fp32-backbone fix above
                # (also this round) turns out to be the real fix.
                print(f"fold={fold} segmenter batch1_diag: images_has_nan="
                      f"{torch.isnan(images).any().item()} images_absmax={images.abs().max().item():.3f} "
                      f"region_logits_absmax={region_logits.detach().abs().max().item():.1f} "
                      f"boundary_logits_absmax={boundary_logits.detach().abs().max().item():.1f} "
                      f"energy_logits_absmax={energy_logits.detach().abs().max().item():.1f} "
                      f"region_bce={region_bce.item():.4f} region_dice={region_dice.item():.4f} "
                      f"boundary_bce={boundary_bce.item():.4f} boundary_dice={boundary_dice.item():.4f} "
                      f"energy_loss={energy_loss.item():.4f} scale_before={scaler.get_scale():.1f}")
            scale_before = scaler.get_scale()
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            if scaler.get_scale() < scale_before:
                train_scale_skips += 1
        train_diag = {"batches": train_batches, "scaler_step_skips": train_scale_skips, "final_scale": scaler.get_scale()}
        print(f"fold={fold} refiner_epoch={epoch + 1} train_diag: batches={train_batches} "
              f"scaler_step_skips={train_scale_skips} final_scale={scaler.get_scale():.1f}")

        model.eval(); region_dice_total = boundary_dice_total = energy_mse_total = sample_count = 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"fold{fold} segmenter val e{epoch + 1}/{epochs}", leave=False):
                images = batch["image"].to(device)
                region_targets = batch["region"].to(device).bool()
                boundary_targets = batch["boundary"].to(device).bool()
                energy_targets = batch["energy"].to(device)
                region_logits, boundary_logits, energy_logits = model(images)
                region_predictions = region_logits.sigmoid() >= cfg.refiner_threshold
                boundary_predictions = boundary_logits.sigmoid() >= cfg.refiner_threshold
                region_dice_total += batch_dice(region_predictions, region_targets).sum().item()
                boundary_dice_total += batch_dice(boundary_predictions, boundary_targets).sum().item()
                energy_mse_total += F.mse_loss(energy_logits.sigmoid(), energy_targets).item() * len(images)
                sample_count += len(images)
        validation_dice = region_dice_total / max(sample_count, 1)
        boundary_dice = boundary_dice_total / max(sample_count, 1)
        energy_mse = energy_mse_total / max(sample_count, 1)
        print(f"fold={fold} refiner_epoch={epoch + 1} val_crop_dice={validation_dice:.5f} "
              f"val_boundary_dice={boundary_dice:.5f} val_energy_mse={energy_mse:.5f} "
              f"lr={optimizer.param_groups[0]['lr']:.2e}")
        is_best = validation_dice > best_dice
        if is_best:
            best_dice = validation_dice
            epochs_since_improvement = 0
            atomic_torch_save(model.state_dict(), checkpoint)
        else:
            epochs_since_improvement += 1
        _record_refiner_epoch(meta_path, fold, fingerprint, target_epochs, epoch, validation_dice, is_best,
                              best_dice, train_diag, {"boundary_dice": boundary_dice, "energy_mse": energy_mse,
                                                       "instances": sample_count})
        scheduler.step(validation_dice)
        if epochs_since_improvement >= cfg.early_stopping_patience:
            print(f"fold={fold} early stopping at epoch {epoch + 1}/{epochs} "
                  f"(no val_crop_dice improvement for {cfg.early_stopping_patience} epochs)")
            break
    _finalize_refiner_meta(meta_path, fold, fingerprint, target_epochs)
    return checkpoint



def train_refiner(
    fold: int,
    fold_manifest: pd.DataFrame,
    image_dir: Path,
    selected_image_ids: Mapping[str, int],
    indices: Mapping[str, Any],
    cfg: CFG,
    work_dir: Path,
) -> Path:
    output_dir = work_dir / "models" / f"fold_{fold}"
    output_dir.mkdir(parents=True, exist_ok=True)
    checkpoint = output_dir / "refiner.pt"
    meta_path = output_dir / "refiner_meta.json"
    fingerprint = refiner_config_fingerprint(cfg, fold_manifest)
    target_epochs = 2 if cfg.debug else cfg.refiner_epochs
    meta = read_json_if_valid(meta_path)
    # Dropped the `epochs_completed >= target_epochs` requirement this resume-guard used to
    # have: _train_segmenter now supports early stopping, so a clean, intentional finish can
    # legitimately have epochs_completed < target_epochs (the ceiling). status == "complete" is
    # already the authoritative "training function reached its normal exit point" signal --
    # _finalize_refiner_meta is only ever called there, never on a crash (crashes leave status
    # stuck at "in_progress" from the last _record_refiner_epoch call) -- so it alone is
    # sufficient; requiring epochs_completed >= target_epochs on top of it would instead force a
    # full, wasteful retrain every time early stopping actually does its job. See PROGRESS.md.
    if (checkpoint.exists() and meta is not None and meta.get("status") == "complete"
            and meta.get("config_fingerprint") == fingerprint):
        return checkpoint
    # No true mid-epoch resume for the refiner (deliberate -- see PROGRESS.md): any incomplete
    # or config-mismatched attempt just retrains this fold's refiner from scratch. Bounded cost
    # (at most target_epochs x one epoch's time, for one fold), and keeps this actively-debugged
    # function's state simple while the val_crop_dice=0.00000 investigation is still open.
    if cfg.refiner_backbone == "vit":
        return _train_segmenter(fold, fold_manifest, image_dir, selected_image_ids, indices, cfg, work_dir, checkpoint, meta_path, fingerprint, target_epochs)
    return _train_crop_refiner(fold, fold_manifest, image_dir, selected_image_ids, indices, cfg, work_dir, checkpoint, meta_path, fingerprint, target_epochs)


def build_refiner(cfg: "CFG") -> nn.Module:
    if cfg.refiner_backbone == "vit":
        return RegionBoundarySegmenter(cfg)
    if cfg.refiner_backbone == "resnet18":
        return SeedRefiner(pretrained=True)
    raise ValueError(f"Unknown refiner_backbone: {cfg.refiner_backbone!r}")


Appending to solar_filament_lib.py


### 4. Topology-safe end-to-end inference

In [6]:
%%writefile -a solar_filament_lib.py


def component_count(mask: np.ndarray) -> int:
    return max(cv2.connectedComponents(mask.astype(np.uint8), connectivity=8)[0] - 1, 0)


def binary_dice(first: np.ndarray, second: np.ndarray) -> float:
    intersection = np.logical_and(first, second).sum()
    return float((2 * intersection + 1e-7) / (first.sum() + second.sum() + 1e-7))


@torch.no_grad()
def refine_seed(
    image: np.ndarray,
    seed_mask: np.ndarray,
    model: nn.Module,
    device: torch.device,
    cfg: CFG,
) -> tuple[np.ndarray, float]:
    """Per-instance crop refiner path, used only for refiner_backbone == 'resnet18'."""
    x0, y0, x1, y1 = square_bounds(seed_mask, cfg.refiner_context, cfg.refiner_min_crop)
    image_crop = cv2.resize(image[y0:y1, x0:x1], (cfg.refiner_crop,) * 2, interpolation=cv2.INTER_LINEAR)
    seed_crop = cv2.resize(seed_mask[y0:y1, x0:x1].astype(np.uint8), (cfg.refiner_crop,) * 2, interpolation=cv2.INTER_NEAREST)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(image_crop)
    features = np.stack([image_crop, clahe, seed_crop * 255], axis=0).astype(np.float32) / 255.0
    tensor = torch.from_numpy(features[None]).to(device)
    transforms = [lambda value: value]
    inverses = [lambda value: value]
    if cfg.use_tta:
        transforms += [lambda value: value.flip(-1), lambda value: value.flip(-2), lambda value: value.flip((-2, -1))]
        inverses += [lambda value: value.flip(-1), lambda value: value.flip(-2), lambda value: value.flip((-2, -1))]
    probabilities = []
    for transform, inverse in zip(transforms, inverses):
        probability = inverse(model(transform(tensor)).sigmoid())[0, 0].cpu().numpy()
        probabilities.append(probability)
    base_probability = probabilities[0]
    candidate_probability = np.mean(probabilities, axis=0)
    base_mask = base_probability >= cfg.refiner_threshold
    candidate_mask = candidate_probability >= cfg.refiner_threshold
    use_candidate = True
    if base_mask.any():
        area_ratio = candidate_mask.sum() / max(base_mask.sum(), 1)
        use_candidate = (
            0.60 <= area_ratio <= 1.70
            and binary_dice(base_mask, candidate_mask) >= 0.62
            and component_count(candidate_mask) <= component_count(base_mask) + 1
        )
    probability = candidate_probability if use_candidate else base_probability
    resized_probability = cv2.resize(probability, (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR)
    crop_mask = resized_probability >= cfg.refiner_threshold
    full_mask = np.zeros_like(seed_mask, dtype=bool)
    full_mask[y0:y1, x0:x1] = crop_mask
    if not full_mask.any():
        full_mask = seed_mask.astype(bool)
    confidence = float(resized_probability[crop_mask].mean()) if crop_mask.any() else 0.0
    return full_mask, confidence


@torch.no_grad()
def _refine_seeds_global(
    image: np.ndarray,
    seed_masks: Sequence[np.ndarray],
    refiner: "GlobalViTRefiner",
    device: torch.device,
    cfg: CFG,
) -> tuple[list[np.ndarray], list[float]]:
    """Encode the full image through the ViT backbone once, then decode every seed for this
    image from ROIAlign crops of that single feature map. TTA is done at the image level
    (a handful of full-image encodes) instead of per instance, which is both cheaper and more
    faithful to how the backbone was meant to be used than the old per-crop TTA."""
    if not seed_masks:
        return [], []
    native_h, native_w = image.shape
    size = cfg.vit_image_size
    image_resized = cv2.resize(image, (size, size), interpolation=cv2.INTER_LINEAR)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(image_resized)
    base_tensor = torch.from_numpy(
        (np.stack([image_resized, clahe, image_resized], axis=0).astype(np.float32) / 255.0)[None]
    ).to(device)

    boxes, seed_crops = [], []
    for seed_mask in seed_masks:
        seed_resized = cv2.resize(seed_mask.astype(np.uint8), (size, size), interpolation=cv2.INTER_NEAREST).astype(bool)
        x0, y0, x1, y1 = square_bounds(seed_resized, cfg.refiner_context, cfg.refiner_min_crop)
        seed_crop = cv2.resize(seed_resized[y0:y1, x0:x1].astype(np.uint8), (cfg.refiner_crop,) * 2, interpolation=cv2.INTER_NEAREST)
        boxes.append([x0, y0, x1, y1])
        seed_crops.append(seed_crop)
    boxes_tensor = torch.tensor(boxes, dtype=torch.float32, device=device)
    seed_tensor = torch.from_numpy(np.stack(seed_crops).astype(np.float32))[:, None].to(device)

    modes = ["none"]
    if cfg.use_tta:
        modes += ["h", "v", "hv"]

    probabilities = []
    for mode in modes:
        image_tf = base_tensor
        if "h" in mode:
            image_tf = image_tf.flip(-1)
        if "v" in mode:
            image_tf = image_tf.flip(-2)
        flipped_boxes = boxes_tensor.clone()
        if "h" in mode:
            flipped_boxes[:, 0], flipped_boxes[:, 2] = size - boxes_tensor[:, 2], size - boxes_tensor[:, 0]
        if "v" in mode:
            flipped_boxes[:, 1], flipped_boxes[:, 3] = size - boxes_tensor[:, 3], size - boxes_tensor[:, 1]
        # The seed-mask skip channel must be flipped in lockstep with the image/box, exactly like
        # training does (ImageRefinerDataset flips seed_crops together with the image when its
        # h-flip augmentation fires). Without this, decode_instances concatenates a mirrored-local-
        # frame image feature (from the flipped box sampling the flipped image) with an unflipped
        # seed channel -- a train/inference mismatch that silently degrades every flipped TTA pass
        # (h, v, hv), not just a missed optimization. Only the "none" pass was ever correct before
        # this fix.
        seed_tf = seed_tensor
        if "h" in mode:
            seed_tf = seed_tf.flip(-1)
        if "v" in mode:
            seed_tf = seed_tf.flip(-2)
        feature_maps = refiner.encode_image(image_tf)
        logits = refiner.decode_instances(feature_maps, [flipped_boxes], seed_tf)
        probability = logits.sigmoid()
        if "h" in mode:
            probability = probability.flip(-1)
        if "v" in mode:
            probability = probability.flip(-2)
        probabilities.append(probability)
    averaged = torch.stack(probabilities).mean(0)[:, 0].cpu().numpy()

    masks, scores = [], []
    for probability, (x0, y0, x1, y1), seed_mask in zip(averaged, boxes, seed_masks):
        resized_probability = cv2.resize(probability, (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR)
        crop_mask = resized_probability >= cfg.refiner_threshold
        canvas = np.zeros((size, size), dtype=bool)
        canvas[y0:y1, x0:x1] = crop_mask
        full_mask = cv2.resize(canvas.astype(np.uint8), (native_w, native_h), interpolation=cv2.INTER_NEAREST).astype(bool)
        if not full_mask.any():
            full_mask = seed_mask.astype(bool)
        confidence = float(resized_probability[crop_mask].mean()) if crop_mask.any() else 0.0
        masks.append(full_mask)
        scores.append(confidence)
    return masks, scores


def refine_seeds(
    image: np.ndarray,
    seed_masks: Sequence[np.ndarray],
    refiner: nn.Module,
    device: torch.device,
    cfg: CFG,
) -> tuple[list[np.ndarray], list[float]]:
    """Dispatches on refiner architecture: global-context ROIAlign path for GlobalViTRefiner,
    the original per-instance crop path otherwise. Used for both the main detector-seed path
    and the single-seed classical fallback, so both share identical refiner behavior."""
    if isinstance(refiner, GlobalViTRefiner):
        return _refine_seeds_global(image, seed_masks, refiner, device, cfg)
    masks, scores = [], []
    for seed_mask in seed_masks:
        refined_mask, refiner_score = refine_seed(image, seed_mask, refiner, device, cfg)
        masks.append(refined_mask)
        scores.append(refiner_score)
    return masks, scores


def mask_iou(first: np.ndarray, second: np.ndarray) -> float:
    intersection = np.logical_and(first, second).sum()
    union = np.logical_or(first, second).sum()
    return float(intersection / union) if union else 0.0


def remove_duplicate_seeds(masks: list[np.ndarray], scores: list[float], threshold: float = 0.60) -> tuple[list[np.ndarray], list[float]]:
    kept_masks, kept_scores = [], []
    for index in np.argsort(scores)[::-1]:
        mask = masks[int(index)]
        if all(mask_iou(mask, kept) < threshold for kept in kept_masks):
            kept_masks.append(mask); kept_scores.append(float(scores[int(index)]))
    return kept_masks, kept_scores


def resolve_overlaps(masks: list[np.ndarray], scores: list[float], minimum_area: int) -> tuple[list[np.ndarray], list[float]]:
    if not masks:
        return [], []
    assigned = np.zeros_like(masks[0], dtype=bool)
    output_masks, output_scores = [], []
    for index in np.argsort(scores)[::-1]:
        clean = masks[int(index)].astype(bool) & ~assigned
        if clean.sum() >= minimum_area:
            output_masks.append(clean); output_scores.append(float(scores[int(index)])); assigned |= clean
    return output_masks, output_scores


def load_fold_models(fold: int, cfg: CFG, work_dir: Path) -> tuple[Any, nn.Module, torch.device]:
    from ultralytics import YOLO

    model_dir = work_dir / "models" / f"fold_{fold}"
    detector = YOLO(str(model_dir / "detector.pt"))
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    refiner = build_refiner(cfg)
    refiner.load_state_dict(torch.load(model_dir / "refiner.pt", map_location=device))
    refiner.to(device).eval()
    return detector, refiner, device


def classical_fallback_seed(image: np.ndarray, minimum_area: int) -> np.ndarray:
    """Image-only last resort so every test observation has a valid prediction."""
    blurred = cv2.GaussianBlur(image, (0, 0), 25)
    darkness = blurred.astype(np.float32) - image.astype(np.float32)
    disk = image > 12
    values = darkness[disk]
    if values.size == 0:
        disk = np.ones_like(image, dtype=bool)
        values = darkness.ravel()
    threshold = float(np.quantile(values, 0.995))
    candidate = ((darkness >= threshold) & disk).astype(np.uint8)
    candidate = cv2.morphologyEx(
        candidate, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    )
    count, labels, stats, _ = cv2.connectedComponentsWithStats(candidate, 8)
    valid = [index for index in range(1, count) if stats[index, cv2.CC_STAT_AREA] >= minimum_area]
    if not valid:
        valid = [1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))] if count > 1 else []
    if not valid:
        fallback = np.zeros_like(image, dtype=np.uint8)
        center_y, center_x = np.unravel_index(int(np.argmax(darkness)), darkness.shape)
        cv2.circle(fallback, (int(center_x), int(center_y)), 6, 1, -1)
        return fallback.astype(bool)
    best = max(valid, key=lambda index: stats[index, cv2.CC_STAT_AREA])
    return labels == best


def separate_instances_watershed(
    region_mask: np.ndarray,
    boundary_mask: np.ndarray,
    minimum_area: int,
    energy: np.ndarray | None = None,
    energy_seed_threshold: float = 0.5,
) -> list[np.ndarray]:
    """Recovers individual instance masks from a dense region + boundary prediction,
    DCAN-style: connected components of (region & ~boundary) seed each instance, then
    cv2.watershed (on a distance-transform landscape) regrows every marker back to its full
    region extent. Regrowing matters -- without it, every instance would be systematically
    eroded by the boundary-ring width, which directly hurts PQ's strict IoU>0.5 exactly
    where it's strictest. Verified on synthetic touching/isolated blobs (see
    test_segmentation_geometry.py): isolated instances recover exactly, tightly touching
    ones recover >75% of their area without merging.

    `energy` (optional): a per-pixel [0,1] Deep-Watershed-Transform-style prediction, peaking
    at each instance's medial axis (see build_semantic_targets/RegionBoundarySegmenter). When
    given, it strengthens both steps the boundary-only design struggles with for thin,
    closely-packed instances: seeding additionally includes high-energy pixels (a learned
    "safe interior" signal, robust even where boundary-ring erosion would consume a thin
    instance's entire seed), and the watershed regrow landscape uses the predicted energy
    directly instead of a geometric distance-transform of the semantic region mask (which
    carries zero information about where one instance ends and the next begins -- exactly why
    a decent region+boundary dice score can still merge adjacent instances). Passing
    energy=None reproduces the original geometric-only behavior exactly, byte-for-byte (used by
    test_segmentation_geometry.py's original tests, so they stay valid unchanged).
    # ponytail: flat-priority watershed split at a touching seam isn't perfectly
    # symmetric (a real filament pair touching this tightly is rare) -- upgrade to a
    # learned/marker-confidence-weighted split if OOF shows merged or lopsided instances.
    """
    region = region_mask.astype(bool)
    if not region.any():
        return []
    markers_seed = region & ~boundary_mask.astype(bool)
    if energy is not None:
        markers_seed = markers_seed | ((energy >= energy_seed_threshold) & region)
    markers_seed = markers_seed.astype(np.uint8)
    count, markers = cv2.connectedComponents(markers_seed, connectivity=8)
    if count <= 1:
        # No seed survived (boundary erosion consumed a thin instance's entire width, and
        # energy -- if given -- never reached energy_seed_threshold anywhere either) --
        # plausible for thin filaments even though `region` itself may be substantially
        # correct. Previously this discarded the whole image's prediction (predict_seg-
        # mentation_instances falls back to one classical_fallback_seed blob for the entire
        # image whenever this returns []), throwing away correct region signal. Nothing needs
        # splitting here anyway (no seed survived to indicate multiple touching instances), so
        # recover each connected component of the raw region mask directly instead. See
        # PROGRESS.md.
        raw_count, raw_labels = cv2.connectedComponents(region.astype(np.uint8), connectivity=8)
        return [
            (raw_labels == label) for label in range(1, raw_count)
            if (raw_labels == label).sum() >= minimum_area
        ]
    markers = markers.astype(np.int32)
    markers[~region] = 0
    landscape_source = -energy if energy is not None else -cv2.distanceTransform(region.astype(np.uint8), cv2.DIST_L2, 5)
    landscape = cv2.cvtColor(
        cv2.normalize(landscape_source, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8), cv2.COLOR_GRAY2BGR
    )
    cv2.watershed(landscape, markers)
    instances = []
    for label in range(1, count):
        instance_mask = (markers == label) & region
        if instance_mask.sum() >= minimum_area:
            instances.append(instance_mask)
    return instances


@torch.no_grad()
def _compute_probability_maps(
    image_path: Path,
    model: "RegionBoundarySegmenter",
    device: torch.device,
    cfg: CFG,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """The GPU-cost half of predict_segmentation_instances: one dense region+boundary+energy
    pass over the whole image with flip-TTA averaging, split out so a
    refiner_threshold/minimum_area sweep (see sweep_postprocessing, cell 17) can compute this
    ONCE per image and cheaply re-decode at many threshold/area combinations afterward, instead
    of re-running the backbone per grid point. Returns (image, region_probability,
    boundary_probability, energy, region_probability_native) -- the first four at
    cfg.vit_image_size resolution (thresholding happens at this resolution, unchanged from
    before this split), the last already resized to native resolution for score computation."""
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    native_h, native_w = image.shape
    size = cfg.vit_image_size
    image_resized = cv2.resize(image, (size, size), interpolation=cv2.INTER_LINEAR)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(image_resized)
    base_tensor = torch.from_numpy(
        (np.stack([image_resized, clahe, image_resized], axis=0).astype(np.float32) / 255.0)[None]
    ).to(device)

    modes = ["none"]
    if cfg.use_tta:
        modes += ["h", "v", "hv"]
    region_probabilities, boundary_probabilities, energies = [], [], []
    for mode in modes:
        tensor = base_tensor
        if "h" in mode:
            tensor = tensor.flip(-1)
        if "v" in mode:
            tensor = tensor.flip(-2)
        region_logits, boundary_logits, energy_logits = model(tensor)
        region_probability, boundary_probability = region_logits.sigmoid(), boundary_logits.sigmoid()
        energy_probability = energy_logits.sigmoid()
        if "h" in mode:
            region_probability, boundary_probability = region_probability.flip(-1), boundary_probability.flip(-1)
            energy_probability = energy_probability.flip(-1)
        if "v" in mode:
            region_probability, boundary_probability = region_probability.flip(-2), boundary_probability.flip(-2)
            energy_probability = energy_probability.flip(-2)
        region_probabilities.append(region_probability)
        boundary_probabilities.append(boundary_probability)
        energies.append(energy_probability)
    region_probability = torch.stack(region_probabilities).mean(0)[0, 0].cpu().numpy()
    boundary_probability = torch.stack(boundary_probabilities).mean(0)[0, 0].cpu().numpy()
    energy = torch.stack(energies).mean(0)[0, 0].cpu().numpy()
    region_probability_native = cv2.resize(region_probability, (native_w, native_h), interpolation=cv2.INTER_LINEAR)
    return image, region_probability, boundary_probability, energy, region_probability_native


def _decode_from_probability_maps(
    image: np.ndarray,
    region_probability: np.ndarray,
    boundary_probability: np.ndarray,
    energy: np.ndarray,
    region_probability_native: np.ndarray,
    threshold: float,
    minimum_area: int,
    energy_seed_threshold: float = 0.5,
) -> tuple[list[np.ndarray], list[float]]:
    """Threshold + instance decode given already-computed probability maps (see
    _compute_probability_maps). Pure numpy/cv2, no GPU cost -- this is the part a
    refiner_threshold/minimum_area sweep re-runs per grid point. energy (still at model
    resolution, cfg.vit_image_size, on input) is resized to native resolution here too, same as
    region/boundary, before being passed to separate_instances_watershed -- all three arguments
    must share the same coordinate frame."""
    native_h, native_w = image.shape
    region_mask = cv2.resize(
        (region_probability >= threshold).astype(np.uint8), (native_w, native_h),
        interpolation=cv2.INTER_NEAREST,
    ).astype(bool)
    boundary_mask = cv2.resize(
        (boundary_probability >= threshold).astype(np.uint8), (native_w, native_h),
        interpolation=cv2.INTER_NEAREST,
    ).astype(bool)
    energy_native = cv2.resize(energy, (native_w, native_h), interpolation=cv2.INTER_LINEAR)
    instances = separate_instances_watershed(
        region_mask, boundary_mask, minimum_area, energy=energy_native, energy_seed_threshold=energy_seed_threshold
    )
    if not instances:
        return [classical_fallback_seed(image, minimum_area)], [0.01]
    scores = [float(region_probability_native[instance].mean()) for instance in instances]
    return instances, scores


@torch.no_grad()
def predict_segmentation_instances(
    image_path: Path,
    model: "RegionBoundarySegmenter",
    device: torch.device,
    cfg: CFG,
) -> tuple[list[np.ndarray], list[float]]:
    """Detector-free path for refiner_backbone == 'vit': one dense region+boundary+energy pass
    over the whole image, instances recovered by separate_instances_watershed instead of
    detector seeds. Same flip-TTA-and-average pattern _refine_seeds_global already uses.
    Behavior-preserving wrapper around _compute_probability_maps + _decode_from_probability_maps
    (split so a post-processing sweep can reuse the expensive half across many decode-time
    parameter combinations -- see sweep_postprocessing, cell 17)."""
    image, region_probability, boundary_probability, energy, region_probability_native = _compute_probability_maps(
        image_path, model, device, cfg
    )
    return _decode_from_probability_maps(
        image, region_probability, boundary_probability, energy, region_probability_native,
        cfg.refiner_threshold, cfg.minimum_area, cfg.energy_seed_threshold,
    )


@torch.no_grad()
@torch.no_grad()
def predict_segmentation_instances_ensemble(
    image_path: Path,
    models: Sequence["RegionBoundarySegmenter"],
    device: torch.device,
    cfg: CFG,
) -> tuple[list[np.ndarray], list[float]]:
    """Test-time ensembling across multiple independently-trained segmenter checkpoints (e.g.
    the 5 CV-fold models -- see run_test_segmenter_ensemble, cell 19). No leakage concern here,
    unlike OOF scoring which must stay strictly per-fold: none of these models have seen any
    real test image, regardless of which folds saw which training images. Averages each model's
    region/boundary/energy probability maps (same averaging pattern already used across TTA
    flips within one model) before a single shared decode -- cheaper and structurally simpler
    than decoding each model separately then fusing instance-level boxes (the old, dormant
    IoU-clustering fuse_fold_predictions approach), and sidesteps that approach's known
    zero-instance-fusion edge case entirely, since there's always at least one averaged
    region/boundary/energy map to decode regardless of how many models are ensembled."""
    maps = [_compute_probability_maps(image_path, model, device, cfg) for model in models]
    image = maps[0][0]
    region_probability = np.mean([m[1] for m in maps], axis=0)
    boundary_probability = np.mean([m[2] for m in maps], axis=0)
    energy = np.mean([m[3] for m in maps], axis=0)
    region_probability_native = np.mean([m[4] for m in maps], axis=0)
    return _decode_from_probability_maps(
        image, region_probability, boundary_probability, energy, region_probability_native,
        cfg.refiner_threshold, cfg.minimum_area, cfg.energy_seed_threshold,
    )


def load_segmenter(fold: int, cfg: CFG, work_dir: Path) -> tuple[nn.Module, torch.device]:
    """Like load_fold_models, but for the detector-free segmenter path: no detector.pt to
    load, since train_fold no longer produces one."""
    model_dir = work_dir / "models" / f"fold_{fold}"
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = build_refiner(cfg)
    model.load_state_dict(torch.load(model_dir / "refiner.pt", map_location=device))
    model.to(device).eval()
    return model, device


# ---------------------------------------------------------------------------
# Seg(YOLO)->Energy pipeline inference (2026-09-30, active path -- see cell 11/17, PROGRESS.md)
# ---------------------------------------------------------------------------

def yolo_instance_masks(detector: Any, image_path: Path, cfg: CFG) -> list[np.ndarray]:
    """Runs the trained YOLO detector on one image and returns its predicted instance masks
    INDIVIDUALLY (each at cfg.vit_image_size resolution), preserving the instance identity YOLO's
    own detection already solved -- unlike the earlier yolo_region_mask design, which flattened
    every instance into one undifferentiated union and threw that identity away, forcing the
    decode step to re-discover instance separation from scratch using only the energy map. That
    was a real, self-inflicted gap: YOLO already solved "how many instances, roughly where," and
    flattening then re-solving it via watershed manufactured exactly the closely-packed-instance
    scenario energy-based splitting struggles most with, instead of avoiding it. See
    refine_instances_with_energy_watershed, below, and PROGRESS.md. Same detector-fallback-conf
    retry pattern already used by predict_instances (the older YOLO+crop-refiner path, above): if
    the primary confidence threshold finds nothing, retry once at the more permissive
    detector_fallback_conf before conceding no instances."""
    size = cfg.vit_image_size
    # nms=cfg.detector_nms (default False): uses YOLO26's NMS-free one-to-one head, trained
    # regardless of this choice at no extra cost. Standard box-IoU NMS risks discarding a
    # genuine second filament whose box overlaps a higher-confidence neighbor's -- exactly the
    # risk profile of closely-packed instances, which is this competition's specific failure
    # mode (see PROGRESS.md). agnostic_nms is a no-op either way (single_cls=True, one class).
    result = detector.predict(
        str(image_path), imgsz=cfg.image_size, conf=cfg.detector_conf,
        iou=cfg.detector_iou, agnostic_nms=True, retina_masks=True, verbose=False, nms=cfg.detector_nms,
    )[0]
    if result.masks is None:
        result = detector.predict(
            str(image_path), imgsz=cfg.image_size, conf=cfg.detector_fallback_conf,
            iou=cfg.detector_iou, agnostic_nms=True, retina_masks=True, verbose=False, nms=cfg.detector_nms,
        )[0]
    if result.masks is None:
        return []
    return [
        cv2.resize(mask.astype(np.uint8), (size, size), interpolation=cv2.INTER_NEAREST).astype(bool)
        for mask in result.masks.data.cpu().numpy()
    ]


def yolo_region_mask(detector: Any, image_path: Path, cfg: CFG) -> np.ndarray:
    """Unions yolo_instance_masks into one dense boolean region mask at cfg.vit_image_size
    resolution -- Stage 1 ("Seg") of the Seg(YOLO)->Energy pipeline, used by Stage 2 training
    (precompute_yolo_region_masks, cell 11), where only "is this pixel foreground at all" is
    needed as the energy net's conditioning signal, not per-instance identity. Inference
    (predict_yolo_energy_instances, below) uses yolo_instance_masks directly instead, to keep
    instance identity through to the decode step."""
    size = cfg.vit_image_size
    region = np.zeros((size, size), dtype=bool)
    for mask in yolo_instance_masks(detector, image_path, cfg):
        region |= mask
    return region


def separate_instances_from_yolo_energy(
    region_mask: np.ndarray,
    energy: np.ndarray,
    minimum_area: int,
    energy_seed_threshold: float = 0.5,
) -> list[np.ndarray]:
    """Superseded by refine_instances_with_energy_watershed for the active decode path (see
    PROGRESS.md) -- kept, not deleted. Watershed on the predicted energy map, seeded purely from
    high-energy pixels within a flattened YOLO region union, with no per-instance identity to
    work from. Same raw-region fallback for the count<=1 case (a thin/merged region whose
    energy never reaches energy_seed_threshold anywhere) -- see
    separate_instances_watershed's docstring for why discarding to a single classical blob
    instead would be worse."""
    region = region_mask.astype(bool)
    if not region.any():
        return []
    markers_seed = ((energy >= energy_seed_threshold) & region).astype(np.uint8)
    count, markers = cv2.connectedComponents(markers_seed, connectivity=8)
    if count <= 1:
        raw_count, raw_labels = cv2.connectedComponents(region.astype(np.uint8), connectivity=8)
        return [
            (raw_labels == label) for label in range(1, raw_count)
            if (raw_labels == label).sum() >= minimum_area
        ]
    markers = markers.astype(np.int32)
    markers[~region] = 0
    landscape = cv2.cvtColor(
        cv2.normalize(-energy, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8), cv2.COLOR_GRAY2BGR
    )
    cv2.watershed(landscape, markers)
    instances = []
    for label in range(1, count):
        instance_mask = (markers == label) & region
        if instance_mask.sum() >= minimum_area:
            instances.append(instance_mask)
    return instances


def refine_instances_with_energy_watershed(
    instance_masks: list[np.ndarray],
    energy: np.ndarray,
    minimum_area: int,
) -> list[np.ndarray]:
    """The active decode step: refines YOLO's own per-instance masks via energy-guided
    watershed, instead of re-deriving instance count/separation from a flattened region union
    (see yolo_instance_masks' docstring for why that was a real gap, not just a stylistic
    choice). Each YOLO instance becomes its own watershed marker -- already correctly separated
    by the detector -- and cv2.watershed grows every marker outward using the predicted energy
    as the landscape, so the only thing left to solve is each instance's precise boundary and
    seam with its neighbors, not how many instances exist in the first place. Where two YOLO
    masks genuinely overlap, the later instance in the list wins the overlapping pixels
    (np.array assignment order) -- an accepted simplification, not a correctness bug fixed here;
    deduplicating near-identical overlapping detections (a real risk now that detector_nms=False
    keeps more raw proposals) is a deliberately deferred refinement, not implemented yet."""
    if not instance_masks:
        return []
    size = instance_masks[0].shape
    region = np.zeros(size, dtype=bool)
    markers = np.zeros(size, dtype=np.int32)
    for label, mask in enumerate(instance_masks, start=1):
        region |= mask
        markers[mask] = label
    landscape = cv2.cvtColor(
        cv2.normalize(-energy, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8), cv2.COLOR_GRAY2BGR
    )
    cv2.watershed(landscape, markers)
    instances = []
    for label in range(1, len(instance_masks) + 1):
        instance_mask = (markers == label) & region
        if instance_mask.sum() >= minimum_area:
            instances.append(instance_mask)
    return instances


def load_yolo_energy_models(fold: int, cfg: CFG, work_dir: Path) -> tuple[Any, nn.Module, torch.device]:
    """Loads both stages for one fold: the trained YOLO detector (Stage 1) and the trained
    YoloConditionedEnergyNet (Stage 2)."""
    from ultralytics import YOLO

    model_dir = work_dir / "models" / f"fold_{fold}"
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    detector = YOLO(str(model_dir / "detector.pt"))
    energy_model = YoloConditionedEnergyNet(cfg)
    energy_model.load_state_dict(torch.load(model_dir / "energy_net.pt", map_location=device))
    energy_model.to(device).eval()
    return detector, energy_model, device


@torch.no_grad()
def _compute_yolo_energy_maps(
    image_path: Path,
    detector: Any,
    energy_model: "YoloConditionedEnergyNet",
    device: torch.device,
    cfg: CFG,
) -> tuple[np.ndarray, list[np.ndarray], np.ndarray]:
    """GPU-cost half of predict_yolo_energy_instances: Stage 1 (YOLO26) proposes per-instance
    masks, Stage 2 (DINOv3-backboned YoloConditionedEnergyNet) predicts energy conditioned on
    their union (flip-TTA-and-average, region tensor flipped in lockstep with the image for
    every mode -- see predict_yolo_energy_instances' original docstring / PROGRESS.md for the
    historical bug this guards against). Returns native-resolution instance masks + energy map;
    decode (minimum_area / watershed refinement) is left to _decode_yolo_energy_instances so a
    minimum_area sweep costs one inference pass per image, not grid_size x inference -- same
    split _compute_probability_maps/_decode_from_probability_maps uses for the detector-free
    segmenter."""
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    native_h, native_w = image.shape
    size = cfg.vit_image_size
    instance_masks_model_res = yolo_instance_masks(detector, image_path, cfg)
    region_mask_model_res = np.zeros((size, size), dtype=bool)
    for mask in instance_masks_model_res:
        region_mask_model_res |= mask

    image_resized = cv2.resize(image, (size, size), interpolation=cv2.INTER_LINEAR)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(image_resized)
    base_image_tensor = torch.from_numpy(
        (np.stack([image_resized, clahe, image_resized], axis=0).astype(np.float32) / 255.0)[None]
    ).to(device)
    base_region_tensor = torch.from_numpy(region_mask_model_res.astype(np.float32)[None, None]).to(device)

    modes = ["none"]
    if cfg.use_tta:
        modes += ["h", "v", "hv"]
    energies = []
    for mode in modes:
        image_tensor, region_tensor = base_image_tensor, base_region_tensor
        if "h" in mode:
            image_tensor, region_tensor = image_tensor.flip(-1), region_tensor.flip(-1)
        if "v" in mode:
            image_tensor, region_tensor = image_tensor.flip(-2), region_tensor.flip(-2)
        energy_probability = energy_model(image_tensor, region_tensor).sigmoid()
        if "h" in mode:
            energy_probability = energy_probability.flip(-1)
        if "v" in mode:
            energy_probability = energy_probability.flip(-2)
        energies.append(energy_probability)
    energy = torch.stack(energies).mean(0)[0, 0].cpu().numpy()

    instance_masks_native = [
        cv2.resize(mask.astype(np.uint8), (native_w, native_h), interpolation=cv2.INTER_NEAREST).astype(bool)
        for mask in instance_masks_model_res
    ]
    energy_native = cv2.resize(energy, (native_w, native_h), interpolation=cv2.INTER_LINEAR)
    return image, instance_masks_native, energy_native


def _decode_yolo_energy_instances(
    image: np.ndarray,
    instance_masks_native: list[np.ndarray],
    energy_native: np.ndarray,
    minimum_area: int,
) -> tuple[list[np.ndarray], list[float]]:
    """CPU-only decode half of predict_yolo_energy_instances: re-runs energy-guided watershed
    refinement at a given minimum_area. Pure numpy/cv2, matching
    _decode_from_probability_maps' role in the detector-free sweep."""
    instances = refine_instances_with_energy_watershed(instance_masks_native, energy_native, minimum_area)
    if not instances:
        return [classical_fallback_seed(image, minimum_area)], [0.01]
    scores = [float(energy_native[instance].mean()) for instance in instances]
    return instances, scores


@torch.no_grad()
def predict_yolo_energy_instances(
    image_path: Path,
    detector: Any,
    energy_model: "YoloConditionedEnergyNet",
    device: torch.device,
    cfg: CFG,
) -> tuple[list[np.ndarray], list[float]]:
    """Full Seg(YOLO)->Energy decode for one image. See _compute_yolo_energy_maps /
    _decode_yolo_energy_instances for the two halves this is built from (split out so
    sweep_postprocessing_yolo_energy can cache the GPU-cost half across a minimum_area grid)."""
    if not cfg.use_energy_watershed:
        # Skip the DINOv3 energy net entirely -- YOLO's own masks, unrefined, filtered only by
        # minimum_area. Isolates the detector's standalone PQ contribution (see PROGRESS.md).
        image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
        native_h, native_w = image.shape
        instance_masks_model_res = yolo_instance_masks(detector, image_path, cfg)
        instances = [
            cv2.resize(mask.astype(np.uint8), (native_w, native_h), interpolation=cv2.INTER_NEAREST).astype(bool)
            for mask in instance_masks_model_res
        ]
        instances = [instance for instance in instances if instance.sum() >= cfg.minimum_area]
        if not instances:
            return [classical_fallback_seed(image, cfg.minimum_area)], [0.01]
        return instances, [1.0] * len(instances)

    image, instance_masks_native, energy_native = _compute_yolo_energy_maps(
        image_path, detector, energy_model, device, cfg
    )
    return _decode_yolo_energy_instances(image, instance_masks_native, energy_native, cfg.minimum_area)


def separate_instances_with_embeddings(
    instance_masks: list[np.ndarray],
    embedding: np.ndarray,
    delta_d: float,
    minimum_area: int,
) -> tuple[list[np.ndarray], list[float]]:
    """The active decode step for the embedding-head pipeline (replaces
    refine_instances_with_energy_watershed -- see PROGRESS.md "embedding-head
    instance-separation swap"). Each YOLO instance mask is still the starting point (YOLO
    already solved "how many instances, roughly where" -- see yolo_instance_masks docstring for
    why throwing that away would be a regression), but instead of growing it outward via
    cv2.watershed on an energy landscape, this clusters that mask's own pixel embeddings: if
    YOLO under-segmented two touching filaments into one mask, their pixels occupy two distinct
    regions of embedding space (that is the whole point of the discriminative loss) and the
    greedy mean-attraction clustering below naturally splits them into two output instances
    instead of one; if YOLO already segmented the instance correctly, every pixel's embedding is
    within delta_v of the same mean and clustering returns exactly one piece.

    Greedy mean-attraction clustering (not a general-purpose library, not needed for the
    typically-small, single-digit-thousand-pixel masks here): repeatedly seed from the
    remaining pixel closest to the current running mean, grow the cluster to every
    remaining pixel within delta_d of that mean, update the mean, repeat until it stops
    growing, then start a new cluster from whatever remains.

    Returns a confidence score per instance alongside its mask: cluster tightness,
    1 / (1 + mean distance of each member pixel's embedding to the cluster's own mean). A
    cluster where every pixel landed close to its mean (the network was confident/consistent
    about this instance) scores near 1.0; a loose, scattered cluster (ambiguous embeddings,
    more likely a decode artifact than a real filament) scores lower. Previously this returned
    a flat 1.0 for every instance -- no real quality signal at all, unlike the energy decode's
    mean-energy score it replaced. See PROGRESS.md.
    """
    instances: list[np.ndarray] = []
    scores: list[float] = []
    for mask in instance_masks:
        ys, xs = np.where(mask)
        if ys.size == 0:
            continue
        points = embedding[ys, xs]  # (N, D)
        remaining = np.ones(len(ys), dtype=bool)
        while remaining.any():
            seed_index = np.flatnonzero(remaining)[0]
            cluster = np.zeros(len(ys), dtype=bool)
            cluster[seed_index] = True
            remaining[seed_index] = False
            while True:
                mean = points[cluster].mean(axis=0)
                distances = np.linalg.norm(points - mean, axis=1)
                newly_joined = remaining & (distances <= delta_d)
                if not newly_joined.any():
                    break
                cluster |= newly_joined
                remaining &= ~newly_joined
            if cluster.sum() >= minimum_area:
                instance_mask = np.zeros(mask.shape, dtype=bool)
                instance_mask[ys[cluster], xs[cluster]] = True
                instances.append(instance_mask)
                tightness = float(np.linalg.norm(points[cluster] - mean, axis=1).mean())
                scores.append(1.0 / (1.0 + tightness))
    return instances, scores


def load_yolo_detector(fold: int, work_dir: Path) -> Any:
    """Stage 1 only: the trained YOLO detector for one fold, with no Stage 2 net needed."""
    from ultralytics import YOLO

    return YOLO(str(work_dir / "models" / f"fold_{fold}" / "detector.pt"))


def load_yolo_embedding_models(
    fold: int,
    cfg: CFG,
    work_dir: Path,
    model_cls: type = YoloConditionedEmbeddingNet,
    checkpoint_name: str = "embedding_net.pt",
) -> tuple[Any, nn.Module, torch.device]:
    """Loads both stages for one fold: the trained YOLO detector (Stage 1) and the trained
    Stage 2 embedding net -- mirrors load_yolo_energy_models exactly. model_cls/checkpoint_name
    select which Stage 2 architecture to load (YoloConditionedEmbeddingNet/"embedding_net.pt" or
    YoloConditionedEmbeddingUNet/"embedding_unet.pt") -- both are trained every fold so they can
    be compared and fused (see PROGRESS.md)."""
    model_dir = work_dir / "models" / f"fold_{fold}"
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    detector = load_yolo_detector(fold, work_dir)
    embedding_model = model_cls(cfg)
    embedding_model.load_state_dict(torch.load(model_dir / checkpoint_name, map_location=device))
    embedding_model.to(device).eval()
    return detector, embedding_model, device


@torch.no_grad()
def _compute_yolo_embedding_maps(
    image_path: Path,
    detector: Any,
    embedding_model: "YoloConditionedEmbeddingNet",
    device: torch.device,
    cfg: CFG,
) -> tuple[np.ndarray, list[np.ndarray], np.ndarray]:
    """GPU-cost half of predict_yolo_embedding_instances -- mirrors _compute_yolo_energy_maps,
    with one deliberate difference: NO flip-TTA averaging. Every other predict_* function in
    this file averages probabilities/energy across flipped passes because those outputs are
    directly comparable across forward passes (a probability is a probability regardless of
    which pass produced it). An embedding is not: nothing anchors the embedding space to a
    fixed, globally-consistent direction, so the same instance can get a legitimately different
    (but internally self-consistent) embedding vector on a flipped pass. Averaging two such
    vectors is not meaningful and would corrupt clustering, not improve it.
    # ponytail: single-pass only, no TTA, for the embedding decode -- if TTA proves worth it
    # later, fuse at the mask/IoU level across independently-decoded passes (like
    # fuse_fold_predictions already does across folds), not at the embedding level.
    """
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    native_h, native_w = image.shape
    size = cfg.vit_image_size
    instance_masks_model_res = yolo_instance_masks(detector, image_path, cfg)
    region_mask_model_res = np.zeros((size, size), dtype=bool)
    for mask in instance_masks_model_res:
        region_mask_model_res |= mask

    image_resized = cv2.resize(image, (size, size), interpolation=cv2.INTER_LINEAR)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(image_resized)
    image_tensor = torch.from_numpy(
        (np.stack([image_resized, clahe, image_resized], axis=0).astype(np.float32) / 255.0)[None]
    ).to(device)
    region_tensor = torch.from_numpy(region_mask_model_res.astype(np.float32)[None, None]).to(device)

    embedding = embedding_model(image_tensor, region_tensor)[0].permute(1, 2, 0).cpu().numpy()  # (size, size, D)

    instance_masks_native = [
        cv2.resize(mask.astype(np.uint8), (native_w, native_h), interpolation=cv2.INTER_NEAREST).astype(bool)
        for mask in instance_masks_model_res
    ]
    embedding_native = cv2.resize(embedding, (native_w, native_h), interpolation=cv2.INTER_LINEAR)
    if embedding_native.ndim == 2:  # cv2.resize squeezes a trailing size-1 channel dim
        embedding_native = embedding_native[:, :, None]
    return image, instance_masks_native, embedding_native


def finalize_instances(
    image: np.ndarray,
    masks: list[np.ndarray],
    scores: list[float],
    minimum_area: int,
    use_fallback: bool,
) -> tuple[list[np.ndarray], list[float]]:
    """Last step of every active decode path. The official metric rejects a submission whose
    masks overlap by even one pixel, so overlaps are resolved here (higher score keeps the
    pixel; resolve_overlaps re-applies minimum_area after carving). An empty result becomes the
    classical fallback blob only when use_fallback is set -- see cfg.use_classical_fallback."""
    masks, scores = resolve_overlaps(masks, scores, minimum_area)
    if not masks and use_fallback:
        return [classical_fallback_seed(image, minimum_area)], [0.01]
    return masks, scores


def _decode_yolo_embedding_instances(
    image: np.ndarray,
    instance_masks_native: list[np.ndarray],
    embedding_native: np.ndarray,
    minimum_area: int,
    delta_d: float,
    use_fallback: bool,
) -> tuple[list[np.ndarray], list[float]]:
    """CPU-only decode half of predict_yolo_embedding_instances -- mirrors
    _decode_yolo_energy_instances' role, so a minimum_area sweep costs one inference pass per
    image, not grid_size x inference."""
    instances, scores = separate_instances_with_embeddings(instance_masks_native, embedding_native, delta_d, minimum_area)
    return finalize_instances(image, instances, scores, minimum_area, use_fallback)


def predict_yolo_only_instances(image_path: Path, detector: Any, cfg: CFG) -> tuple[list[np.ndarray], list[float]]:
    """Stage 1 alone: YOLO's own instance masks at native resolution, with the same
    finalize_instances post-processing (minimum_area, overlap resolution, fallback) as every
    other decode path. Needs no Stage 2 checkpoint, so Stage 1 can be scored in its own session."""
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    native_h, native_w = image.shape
    instances = [
        cv2.resize(mask.astype(np.uint8), (native_w, native_h), interpolation=cv2.INTER_NEAREST).astype(bool)
        for mask in yolo_instance_masks(detector, image_path, cfg)
    ]
    return finalize_instances(image, instances, [1.0] * len(instances), cfg.minimum_area,
                              cfg.use_classical_fallback)


@torch.no_grad()
def predict_yolo_embedding_instances(
    image_path: Path,
    detector: Any,
    embedding_model: "YoloConditionedEmbeddingNet",
    device: torch.device,
    cfg: CFG,
) -> tuple[list[np.ndarray], list[float]]:
    """Full Seg(YOLO)->Embedding decode for one image -- the active pipeline, replacing
    predict_yolo_energy_instances (kept, not deleted -- see PROGRESS.md). See
    _compute_yolo_embedding_maps / _decode_yolo_embedding_instances for the two halves this is
    built from."""
    if not cfg.use_embedding_decode:
        # Skip the embedding net entirely: YOLO's own masks. Isolates the detector's standalone
        # PQ contribution, same as use_energy_watershed=False does for the dormant energy path.
        return predict_yolo_only_instances(image_path, detector, cfg)

    image, instance_masks_native, embedding_native = _compute_yolo_embedding_maps(
        image_path, detector, embedding_model, device, cfg
    )
    return _decode_yolo_embedding_instances(
        image, instance_masks_native, embedding_native, cfg.minimum_area, cfg.embedding_delta_d,
        cfg.use_classical_fallback,
    )


@torch.no_grad()
def predict_yolo_embedding_fused(
    image_path: Path,
    detector: Any,
    embedding_model_a: nn.Module,
    embedding_model_b: nn.Module,
    device: torch.device,
    cfg: CFG,
) -> tuple[list[np.ndarray], list[float]]:
    """Decodes with both Stage 2 architectures independently (embedding_model_a/b -- typically
    YoloConditionedEmbeddingNet and YoloConditionedEmbeddingUNet), then fuses their finished
    instance masks via fuse_fold_predictions, treating "model A" and "model B" as the two
    fusion sources (cfg.minimum_fold_support/singleton_score/ensemble_iou, same knobs the
    cross-fold ensemble already uses). Mask-level fusion, never embedding-level -- consistent
    with why _compute_yolo_embedding_maps skips flip-TTA averaging: nothing anchors either
    model's embedding space to the other's, so only their decoded masks are comparable.
    # ponytail: this calls the shared, frozen YOLO detector twice (once per
    # predict_yolo_embedding_instances call) instead of running it once and decoding both
    # architectures from the same instance masks -- YOLO inference is cheap next to two DINOv3
    # backbone forward passes, so the redundancy isn't worth the extra plumbing here. Share the
    # detector pass if this becomes measurably slow.
    """
    masks_a, scores_a = predict_yolo_embedding_instances(image_path, detector, embedding_model_a, device, cfg)
    masks_b, scores_b = predict_yolo_embedding_instances(image_path, detector, embedding_model_b, device, cfg)
    predictions = [(0, mask, score) for mask, score in zip(masks_a, scores_a)]
    predictions += [(1, mask, score) for mask, score in zip(masks_b, scores_b)]
    masks, scores = fuse_fold_predictions(predictions, cfg)  # already overlap-free (resolve_overlaps)
    if not masks and cfg.use_classical_fallback:
        image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
        return [classical_fallback_seed(image, cfg.minimum_area)], [0.01]
    return masks, scores


@torch.no_grad()
def predict_instances(
    image_path: Path,
    detector: Any,
    refiner: nn.Module,
    device: torch.device,
    cfg: CFG,
) -> tuple[list[np.ndarray], list[float]]:
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    result = detector.predict(
        str(image_path), imgsz=cfg.image_size, conf=cfg.detector_conf,
        iou=cfg.detector_iou, agnostic_nms=True, retina_masks=True, verbose=False,
    )[0]
    if result.masks is None:
        result = detector.predict(
            str(image_path), imgsz=cfg.image_size, conf=cfg.detector_fallback_conf,
            iou=cfg.detector_iou, agnostic_nms=True, retina_masks=True, verbose=False,
        )[0]
    if result.masks is None:
        fallback_seed = classical_fallback_seed(image, cfg.minimum_area)
        masks, refiner_scores = refine_seeds(image, [fallback_seed], refiner, device, cfg)
        return masks, [0.01 * score for score in refiner_scores]

    seed_masks = [mask.cpu().numpy().astype(bool) for mask in result.masks.data]
    if seed_masks and seed_masks[0].shape != image.shape:
        seed_masks = [cv2.resize(mask.astype(np.uint8), image.shape[::-1], interpolation=cv2.INTER_NEAREST).astype(bool) for mask in seed_masks]
    detector_scores = result.boxes.conf.detach().cpu().numpy().astype(float).tolist()
    seed_masks, detector_scores = remove_duplicate_seeds(seed_masks, detector_scores)

    kept_seeds, kept_scores = [], []
    for seed_mask, detector_score in zip(seed_masks, detector_scores):
        if seed_mask.sum() >= cfg.minimum_area:
            kept_seeds.append(seed_mask)
            kept_scores.append(detector_score)
    if not kept_seeds:
        return [], []

    refined_masks, refiner_scores = refine_seeds(image, kept_seeds, refiner, device, cfg)
    scores = [float(detector_score * refiner_score) for detector_score, refiner_score in zip(kept_scores, refiner_scores)]
    return resolve_overlaps(refined_masks, scores, cfg.minimum_area)


Appending to solar_filament_lib.py


### 5. August 7 Panoptic Quality and grouped OOF

In [7]:
%%writefile -a solar_filament_lib.py


def pairwise_iou(gt_masks: Sequence[np.ndarray], pred_masks: Sequence[np.ndarray]) -> np.ndarray:
    if not gt_masks or not pred_masks:
        return np.zeros((len(gt_masks), len(pred_masks)), dtype=np.float64)
    gt_rles = [mask_utils.encode(np.asfortranarray(mask.astype(np.uint8))) for mask in gt_masks]
    pred_rles = [mask_utils.encode(np.asfortranarray(mask.astype(np.uint8))) for mask in pred_masks]
    return np.asarray(mask_utils.iou(gt_rles, pred_rles, [0] * len(pred_rles)), dtype=np.float64)


def panoptic_quality(
    gt_masks: Sequence[np.ndarray],
    pred_masks: Sequence[np.ndarray],
    match_iou: float = 0.50,
) -> dict[str, float | int]:
    """August 7 metric. TP uses the original PQ criterion IoU > 0.5."""
    n_gt, n_pred = len(gt_masks), len(pred_masks)
    if n_gt == 0 and n_pred == 0:
        return {"pq": 1.0, "sq": 1.0, "rq": 1.0, "tp": 0, "fp": 0, "fn": 0, "sum_iou": 0.0}
    iou = pairwise_iou(gt_masks, pred_masks)
    if n_gt and n_pred:
        gt_indices, pred_indices = linear_sum_assignment(-iou)
        matched_ious = iou[gt_indices, pred_indices]
        matched_ious = matched_ious[matched_ious > match_iou]
    else:
        matched_ious = np.zeros(0, dtype=float)
    tp = int(len(matched_ious)); fp = int(n_pred - tp); fn = int(n_gt - tp)
    sum_iou = float(matched_ious.sum())
    denominator = tp + 0.5 * fp + 0.5 * fn
    pq = sum_iou / denominator if denominator else 1.0
    sq = sum_iou / tp if tp else 0.0
    rq = tp / denominator if denominator else 1.0
    return {"pq": pq, "sq": sq, "rq": rq, "tp": tp, "fp": fp, "fn": fn, "sum_iou": sum_iou}


def ground_truth_sets_for_file(filename: str, indices: Mapping[str, Any]) -> list[list[np.ndarray]]:
    result = []
    for image_id in indices["image_ids_by_file"][filename]:
        info = indices["images_by_id"][image_id]
        masks = [
            annotation_to_mask(annotation, int(info["height"]), int(info["width"]))
            for annotation in indices["annotations_by_image"].get(image_id, [])
            if not annotation.get("iscrowd", 0)
        ]
        result.append(masks)
    return result


def evaluate_prediction_for_file(
    filename: str,
    pred_masks: Sequence[np.ndarray],
    indices: Mapping[str, Any],
    cfg: CFG,
) -> list[dict[str, Any]]:
    rows = []
    for annotator_index, gt_masks in enumerate(ground_truth_sets_for_file(filename, indices)):
        rows.append({"filename": filename, "annotator": annotator_index,
                     **panoptic_quality(gt_masks, pred_masks, cfg.pq_match_iou),
                     "n_gt": len(gt_masks), "n_pred": len(pred_masks)})
    return rows


def inter_annotator_rows(filenames: Sequence[str], indices: Mapping[str, Any],
                         match_iou: float = 0.50) -> pd.DataFrame:
    """Human agreement, in the OOF row format: each annotator's masks are scored as predictions
    against every other annotator of the same image, so aggregate_pq gives a pooled
    human-vs-human PQ. Single-annotator images are skipped. Annotations may overlap each other
    (submissions can't), so this is a reference point, not a strict ceiling."""
    rows = []
    for filename in filenames:
        sets = ground_truth_sets_for_file(filename, indices)
        if len(sets) < 2:
            continue
        for predictor, predicted in enumerate(sets):
            for annotator, truth in enumerate(sets):
                if annotator != predictor:
                    rows.append({"filename": filename, "annotator": annotator, "predictor": predictor,
                                 **panoptic_quality(truth, predicted, match_iou),
                                 "n_gt": len(truth), "n_pred": len(predicted)})
    return pd.DataFrame(rows)


def annotation_width_stats(indices: Mapping[str, Any]) -> pd.DataFrame:
    """Area and ribbon width (area / half-perimeter) of every annotation, in native pixels: how
    wide the annotators draw filaments, to compare with predicted instance widths."""
    rows = []
    for image_id, info in indices["images_by_id"].items():
        height, width = int(info["height"]), int(info["width"])
        for annotation in indices["annotations_by_image"].get(image_id, []):
            if annotation.get("iscrowd", 0):
                continue
            mask = annotation_to_mask(annotation, height, width).astype(np.uint8)
            area = int(mask.sum())
            if area == 0:
                continue
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
            perimeter = sum(cv2.arcLength(contour, True) for contour in contours)
            rows.append({"image_id": image_id, "area": area, "width": 2.0 * area / max(perimeter, 1.0)})
    return pd.DataFrame(rows)


def classify_failure_modes(
    gt_masks: Sequence[np.ndarray],
    pred_masks: Sequence[np.ndarray],
    match_iou: float = 0.50,
) -> dict[str, int]:
    """Diagnostic companion to panoptic_quality -- see PROGRESS.md audit item #5. Aggregate
    PQ/SQ/RQ/TP/FP/FN never says WHY RQ is low: this breaks FN/FP into failure categories using
    the same pairwise_iou this file's own PQ scoring is built on.

    For each GT instance, the best IoU against ANY predicted mask: 0 -> fully_missed_fn (nothing
    overlaps this filament at all -- the detector/decode never found it); between 0 and
    match_iou -> near_miss_fn (something overlapped but not enough -- the exact fragmented/
    boundary-only failure mode watershed and greedy embedding clustering are both prone to on
    thin, touching shapes). Symmetric for predicted instances against GT: 0 -> spurious_fp
    (hallucinated, e.g. sunspot confusion); between 0 and match_iou -> near_miss_fp (an
    oversplit/undersized fragment).

    Deliberately simpler than panoptic_quality's Hungarian assignment, and NOT a metric
    replacement: a prediction overlapping a GT above match_iou can legitimately be the TP for
    that GT under Hungarian matching even if a second prediction also overlaps it (counted as a
    duplicate FP there) -- this per-row/per-column max-IoU classifier doesn't dedupe that case
    and would miss that duplicate as a failure. Acceptable for a diagnostic breakdown; the real
    PQ number always comes from panoptic_quality, unchanged.
    """
    counts = {"fully_missed_fn": 0, "near_miss_fn": 0, "spurious_fp": 0, "near_miss_fp": 0}
    if not gt_masks and not pred_masks:
        return counts
    iou = pairwise_iou(gt_masks, pred_masks)
    for row in iou:
        best = float(row.max()) if row.size else 0.0
        if best <= 0.0:
            counts["fully_missed_fn"] += 1
        elif best < match_iou:
            counts["near_miss_fn"] += 1
    for column in iou.T:
        best = float(column.max()) if column.size else 0.0
        if best <= 0.0:
            counts["spurious_fp"] += 1
        elif best < match_iou:
            counts["near_miss_fp"] += 1
    return counts


def aggregate_pq(rows: pd.DataFrame) -> dict[str, float]:
    """Competition PQ over evaluate_prediction_for_file rows -- one row per (filename,
    annotator) with columns filename, annotator, pq, sq, rq, tp, fp, fn, sum_iou, n_gt, n_pred.

    The organizer metric (per two independent public write-ups that check their scorer against
    the organizer notebook: srivatsav-kannan/solar-seg, KeiichiIto1978/...; the Kaggle page
    itself is not readable from here) scores every annotator-image once and POOLS the counts:
    PQ = sum(sum_iou) / (sum tp + 0.5 sum fp + 0.5 sum fn) over all rows. So rows are summed
    as-is -- never average annotators first, never average per-image PQ. pq_image_mean (mean of
    per-image PQ, annotators averaged within each image) is the metric this notebook used before;
    it is reported for comparison only, never used for selection."""
    if rows is None or len(rows) == 0:
        return {"pq": 0.0, "sq": 0.0, "rq": 0.0, "tp": 0, "fp": 0, "fn": 0,
                "n_images": 0, "pq_image_mean": 0.0}
    tp, fp, fn = int(rows["tp"].sum()), int(rows["fp"].sum()), int(rows["fn"].sum())
    sum_iou = float(rows["sum_iou"].sum())
    denominator = tp + 0.5 * fp + 0.5 * fn
    return {
        "pq": sum_iou / denominator if denominator else 1.0,
        "sq": sum_iou / tp if tp else 0.0,
        "rq": tp / denominator if denominator else 1.0,
        "tp": tp, "fp": fp, "fn": fn,
        "n_images": int(rows["filename"].nunique()),
        "pq_image_mean": float(rows.groupby("filename")["pq"].mean().mean()),
    }


def bootstrap_pq_ci(rows: pd.DataFrame, n: int = 1000, seed: int = 0) -> tuple[float, float]:
    """95% percentile interval of aggregate_pq's pooled PQ, resampling IMAGES with replacement
    (an image's annotator rows stay together -- they score the same prediction, so they are not
    independent samples). Two variants whose intervals overlap heavily are not distinguishable
    on this validation set."""
    per_image = rows.groupby("filename")[["sum_iou", "tp", "fp", "fn"]].sum().to_numpy(dtype=float)
    rng = np.random.default_rng(seed)
    sums = per_image[rng.integers(0, len(per_image), size=(n, len(per_image)))].sum(axis=1)
    denominator = sums[:, 1] + 0.5 * sums[:, 2] + 0.5 * sums[:, 3]
    pq = np.divide(sums[:, 0], denominator, out=np.ones(n), where=denominator > 0)
    low, high = np.percentile(pq, [2.5, 97.5])
    return float(low), float(high)


Appending to solar_filament_lib.py


### 6. Fold orchestration

In [8]:
%%writefile -a solar_filament_lib.py


def resolve_stage1_checkpoint(fold: int, cfg: CFG, work_dir: Path,
                              search_root: Path = Path("/kaggle/input")) -> Path:
    """Returns this fold's Stage 1 detector.pt under work_dir, restoring it from an earlier
    session's attached output when it isn't there yet. Kaggle sessions stop at 12h, so Stage 1
    and Stage 2 train in separate sessions: attach the Stage 1 session's output as an input and
    this finds it. Matches on the path relative to work_dir's own folder name
    (`solar_filament_pq/models/fold_0` for CV, `final_fit/models/fold_0` for the final fit), so a
    CV detector is never mistaken for the final-fit one. The detector's config fingerprint must
    match the current cfg: Stage 2's cache assumes the detector it conditions on matches cfg."""
    model_dir = work_dir / "models" / f"fold_{fold}"
    checkpoint, meta_path = model_dir / "detector.pt", model_dir / "detector_meta.json"
    expected = detector_config_fingerprint(cfg)

    def complete(meta_file: Path) -> bool:
        meta = read_json_if_valid(meta_file)
        return meta is not None and meta.get("status") == "complete" and meta.get("config_fingerprint") == expected

    if checkpoint.exists() and complete(meta_path):
        return checkpoint
    suffix = f"{work_dir.name}/models/fold_{fold}/detector.pt"
    candidates = sorted(p for p in Path(search_root).rglob("detector.pt") if p.as_posix().endswith(suffix))
    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Stage 1 needs exactly one '{suffix}' under {search_root}, found {len(candidates)}: "
            f"{[str(p) for p in candidates]}. Run a Stage 1 session (RUN_STAGE1=True) and attach its output.")
    source = candidates[0]
    if not complete(source.parent / "detector_meta.json"):
        raise ValueError(
            f"{source} was trained with a different detector config (or never finished). Use the CFG "
            "from the Stage 1 session, or retrain Stage 1.")
    model_dir.mkdir(parents=True, exist_ok=True)
    atomic_copy_file(source, checkpoint)
    atomic_copy_file(source.parent / "detector_meta.json", meta_path)
    print(f"Stage 1 restored for fold {fold}: {source}")
    return checkpoint


def train_fold(
    fold: int,
    fold_manifest: pd.DataFrame,
    paths: Mapping[str, Path],
    selected_image_ids: Mapping[str, int],
    indices: Mapping[str, Any],
    cfg: CFG,
    work_dir: Path,
    stage1: bool = True,
    stage2: bool = True,
) -> None:
    # stage1/stage2 let each Kaggle session (12h limit) train one stage. stage1=False restores
    # the detector from an earlier session's attached output (resolve_stage1_checkpoint).
    #
    # Seg(YOLO)->Embedding pipeline (active path -- see PROGRESS.md "embedding-head
    # instance-separation swap"): YOLO26 detector (Stage 1, "Seg") -> DINOv3-backboned
    # embedding net (Stage 2, "Embedding"), conditioned on Stage 1's frozen output. Replaces the
    # energy+watershed Stage 2 (train_yolo_conditioned_energy_net, kept, not deleted) because
    # watershed is known to struggle on thin, elongated, touching shapes -- this competition's
    # exact failure mode -- relative to discriminative-embedding separation.
    # train_refiner/_train_segmenter (detector-free RegionBoundarySegmenter, no detector) is
    # also kept, not deleted, no longer called here -- see its own docstring.
    #
    # Trains BOTH Stage 2 architectures every fold -- the original one-shot-upsample
    # YoloConditionedEmbeddingNet and the progressive, skip-connected YoloConditionedEmbeddingUNet
    # (see PROGRESS.md "U-Net-style decoder" to-do) -- under separate checkpoint names, so
    # run_oof_yolo_embedding_variants (cell 17/19) can score each alone AND their fused
    # combination, to decide which to keep as the actual submission path. Not free (roughly
    # doubles Stage 2 training cost per fold) but this competition's compute budget was
    # confirmed "generous" by the user -- see PROGRESS.md.
    if stage1:
        data_yaml = prepare_yolo_fold(fold, fold_manifest, paths["train"], selected_image_ids, indices, work_dir)
        detector_checkpoint = train_detector(fold, data_yaml, cfg, work_dir)
    else:
        detector_checkpoint = resolve_stage1_checkpoint(fold, cfg, work_dir)
    gc.collect(); torch.cuda.empty_cache()
    if not stage2:
        return
    train_yolo_conditioned_embedding_net(fold, fold_manifest, paths["train"], selected_image_ids, indices,
                                          cfg, work_dir, detector_checkpoint)
    gc.collect(); torch.cuda.empty_cache()
    train_yolo_conditioned_embedding_net(fold, fold_manifest, paths["train"], selected_image_ids, indices,
                                          cfg, work_dir, detector_checkpoint,
                                          model_cls=YoloConditionedEmbeddingUNet,
                                          checkpoint_name="embedding_unet.pt",
                                          meta_name="embedding_unet_meta.json")
    gc.collect(); torch.cuda.empty_cache()


def train_final_model(
    observations: pd.DataFrame,
    paths: Mapping[str, Path],
    selected_image_ids: Mapping[str, int],
    indices: Mapping[str, Any],
    cfg: CFG,
    work_dir_final: Path,
    stage1: bool = True,
    stage2: bool = True,
) -> None:
    """Trains the single submission model on ~(1 - final_holdout_fraction) of the FULL labeled
    set (not any CV fold's ~80%), early-stopped against a fresh stratified holdout -- see
    build_final_manifest. Deliberately reuses train_fold(0, ...) unchanged: encoding the holdout
    as fold == 0 in a manifest written to its own work_dir_final means train_detector/train_refiner's
    existing train-on-!=fold / validate-on-==fold / keep-best / resume-or-retrain logic does the
    whole job correctly with zero new training code. Runs directly in the main kernel (not via a
    fold_worker.py subprocess): there is only one model here, nothing to parallelize across the 2
    GPUs for this phase. (train_detector IS called by train_fold again under the active Seg(YOLO)->Energy
    pipeline -- this single-model reasoning holds regardless of which architecture
    train_fold runs internally.)
    """
    final_manifest = build_final_manifest(observations, cfg)
    train_fold(0, final_manifest, paths, selected_image_ids, indices, cfg, work_dir_final,
               stage1=stage1, stage2=stage2)


def train_all_folds(
    fold_manifest: pd.DataFrame,
    paths: Mapping[str, Path],
    selected_image_ids: Mapping[str, int],
    indices: Mapping[str, Any],
    cfg: CFG,
    work_dir: Path,
) -> None:
    """Sequential fallback -- single-GPU environments, or cfg.debug=True smoke runs. The real
    entry point on a 2-GPU Kaggle session is run_parallel_training, below."""
    for fold in tqdm(range(cfg.n_folds), desc="folds"):
        print(f"\n===== TRAIN FOLD {fold}/{cfg.n_folds - 1} =====")
        train_fold(fold, fold_manifest, paths, selected_image_ids, indices, cfg, work_dir)


def run_parallel_training(
    fold_manifest: pd.DataFrame,
    paths: Mapping[str, Path],
    selected_image_ids: Mapping[str, int],
    indices: Mapping[str, Any],
    cfg: CFG,
    work_dir: Path,
    gpu_count: int | None = None,
    poll_seconds: int = 30,
) -> None:
    """Runs `gpu_count` folds concurrently, one per GPU, as real OS subprocesses (fold_worker.py)
    pinned via CUDA_VISIBLE_DEVICES -- not in-process multiprocessing (fragile once the parent
    kernel has touched torch.cuda) and not Ultralytics' own multi-GPU DDP (would conflict if two
    GPUs were visible to a single fold's model.train() call). Folds are paired in fixed rounds
    ((0,1), (2,3), (4,) for n_folds=5, gpu_count=2); a round-mate's crash doesn't kill its
    sibling, and only *starting the next round* is gated on the current round being
    failure-free, to salvage as much GPU time as possible. Each worker's own
    train_refiner call is resume-or-retrain-aware, so relaunching a crashed fold's worker
    later just picks up where that fold's own checkpoint left off. (train_detector is no
    longer part of train_fold -- see PROGRESS.md.)
    """
    gpu_count = gpu_count or max(torch.cuda.device_count(), 1)
    logs_dir = work_dir / "logs"; logs_dir.mkdir(parents=True, exist_ok=True)
    cfg_path = work_dir / "cfg.json"
    cfg_path.write_text(json.dumps(asdict(cfg)), encoding="utf-8")
    worker_script = Path.cwd() / "fold_worker.py"
    if not worker_script.exists():
        raise FileNotFoundError(f"{worker_script} not found -- run the fold_worker.py %%writefile cell first.")

    remaining = list(range(cfg.n_folds))
    failures: list[int] = []
    while remaining:
        round_folds, remaining = remaining[:gpu_count], remaining[gpu_count:]
        processes: dict[int, subprocess.Popen] = {}
        handles: dict[int, Any] = {}
        offsets: dict[int, int] = {}
        for gpu_id, fold in enumerate(round_folds):
            log_path = logs_dir / f"fold_{fold}.log"
            handles[fold] = open(log_path, "w", encoding="utf-8")
            offsets[fold] = 0
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu_id))
            processes[fold] = subprocess.Popen(
                [sys.executable, str(worker_script), "--fold", str(fold),
                 "--work-dir", str(work_dir), "--cfg-json", str(cfg_path)],
                env=env, cwd=str(Path.cwd()), stdout=handles[fold], stderr=subprocess.STDOUT,
            )
        print(f"round started: folds {round_folds} -> GPUs {list(range(len(round_folds)))}")
        while any(process.poll() is None for process in processes.values()):
            time.sleep(poll_seconds)
            for fold in processes:
                log_path = logs_dir / f"fold_{fold}.log"
                with open(log_path, encoding="utf-8") as handle:
                    handle.seek(offsets[fold])
                    new_text = handle.read()
                    offsets[fold] = handle.tell()
                for line in new_text.splitlines():
                    print(f"[fold {fold}] {line}")
        for fold, process in processes.items():
            handles[fold].close()
            print(f"fold {fold} exited with code {process.returncode}")
            if process.returncode != 0:
                failures.append(fold)
    if failures:
        raise RuntimeError(f"folds failed: {failures} -- see {logs_dir}/fold_<n>.log for details")


def run_oof(
    fold_manifest: pd.DataFrame,
    paths: Mapping[str, Path],
    indices: Mapping[str, Any],
    cfg: CFG,
    work_dir: Path,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    annotator_rows = []
    oof_dir = work_dir / "oof_masks"; oof_dir.mkdir(parents=True, exist_ok=True)
    for fold in range(cfg.n_folds):
        model, device = load_segmenter(fold, cfg, work_dir)
        validation_files = fold_manifest.loc[fold_manifest["fold"] == fold, "filename"].tolist()
        if cfg.debug: validation_files = validation_files[:8]
        fallback_count = 0
        for number, filename in enumerate(tqdm(validation_files, desc=f"fold{fold} OOF", leave=False), 1):
            masks, scores = predict_segmentation_instances(paths["train"] / filename, model, device, cfg)
            if scores == [0.01]:  # predict_segmentation_instances' classical_fallback_seed marker
                fallback_count += 1
            rows = evaluate_prediction_for_file(filename, masks, indices, cfg)
            for row in rows: row["fold"] = fold
            annotator_rows.extend(rows)
            if cfg.save_oof_masks:
                shape = (2048, 2048)
                stack = np.stack(masks).astype(np.uint8) if masks else np.zeros((0, *shape), np.uint8)
                np.savez_compressed(oof_dir / f"{Path(filename).stem}.npz", masks=stack, scores=np.asarray(scores))
        # Cheap observability so a training-time failure (e.g. the NaN-forward bug -- see
        # PROGRESS.md) shows up immediately as "100% fallback" instead of only as a
        # mysteriously-zero PQ discovered after the fact.
        print(f"fold={fold} classical_fallback_seed fired on {fallback_count}/{len(validation_files)} "
              f"images ({fallback_count / max(len(validation_files), 1):.1%})")
        del model; gc.collect(); torch.cuda.empty_cache()
    annotator_df = pd.DataFrame(annotator_rows)
    metrics = ["pq", "sq", "rq", "tp", "fp", "fn", "sum_iou", "n_gt", "n_pred"]
    observation_df = annotator_df.groupby(["filename", "fold"], as_index=False)[metrics].mean()
    annotator_df.to_csv(work_dir / "oof_by_annotator.csv", index=False)
    observation_df.to_csv(work_dir / "oof_by_observation.csv", index=False)
    return observation_df, annotator_df


def run_oof_yolo_energy(
    fold_manifest: pd.DataFrame,
    paths: Mapping[str, Path],
    indices: Mapping[str, Any],
    cfg: CFG,
    work_dir: Path,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Like run_oof, but for the active Seg(YOLO)->Energy pipeline -- loads both stages per
    fold (load_yolo_energy_models) and decodes via predict_yolo_energy_instances instead of
    the detector-free predict_segmentation_instances. Kept as a separate function rather than
    branching inside run_oof, matching this session's convention of adding new pipeline
    variants alongside old ones rather than growing one function to handle every design."""
    annotator_rows = []
    for fold in range(cfg.n_folds):
        detector, energy_model, device = load_yolo_energy_models(fold, cfg, work_dir)
        validation_files = fold_manifest.loc[fold_manifest["fold"] == fold, "filename"].tolist()
        if cfg.debug: validation_files = validation_files[:8]
        fallback_count = 0
        for filename in tqdm(validation_files, desc=f"fold{fold} OOF (yolo-energy)", leave=False):
            masks, scores = predict_yolo_energy_instances(paths["train"] / filename, detector, energy_model, device, cfg)
            if scores == [0.01]:  # predict_yolo_energy_instances' classical_fallback_seed marker
                fallback_count += 1
            rows = evaluate_prediction_for_file(filename, masks, indices, cfg)
            for row in rows: row["fold"] = fold
            annotator_rows.extend(rows)
        print(f"fold={fold} classical_fallback_seed fired on {fallback_count}/{len(validation_files)} "
              f"images ({fallback_count / max(len(validation_files), 1):.1%})")
        del detector, energy_model; gc.collect(); torch.cuda.empty_cache()
    annotator_df = pd.DataFrame(annotator_rows)
    metrics = ["pq", "sq", "rq", "tp", "fp", "fn", "sum_iou", "n_gt", "n_pred"]
    observation_df = annotator_df.groupby(["filename", "fold"], as_index=False)[metrics].mean()
    annotator_df.to_csv(work_dir / "oof_by_annotator.csv", index=False)
    observation_df.to_csv(work_dir / "oof_by_observation.csv", index=False)
    return observation_df, annotator_df


def run_oof_yolo_embedding(
    fold_manifest: pd.DataFrame,
    paths: Mapping[str, Path],
    indices: Mapping[str, Any],
    cfg: CFG,
    work_dir: Path,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Like run_oof_yolo_energy, but for the active Seg(YOLO)->Embedding pipeline -- loads both
    stages per fold (load_yolo_embedding_models) and decodes via predict_yolo_embedding_instances
    instead of predict_yolo_energy_instances. See PROGRESS.md "embedding-head
    instance-separation swap"."""
    annotator_rows = []
    for fold in range(cfg.n_folds):
        detector, embedding_model, device = load_yolo_embedding_models(fold, cfg, work_dir)
        validation_files = fold_manifest.loc[fold_manifest["fold"] == fold, "filename"].tolist()
        if cfg.debug: validation_files = validation_files[:8]
        fallback_count = 0
        for filename in tqdm(validation_files, desc=f"fold{fold} OOF (yolo-embedding)", leave=False):
            masks, scores = predict_yolo_embedding_instances(paths["train"] / filename, detector, embedding_model, device, cfg)
            if scores == [0.01]:  # predict_yolo_embedding_instances' classical_fallback_seed marker
                fallback_count += 1
            rows = evaluate_prediction_for_file(filename, masks, indices, cfg)
            for row in rows: row["fold"] = fold
            annotator_rows.extend(rows)
        print(f"fold={fold} classical_fallback_seed fired on {fallback_count}/{len(validation_files)} "
              f"images ({fallback_count / max(len(validation_files), 1):.1%})")
        del detector, embedding_model; gc.collect(); torch.cuda.empty_cache()
    annotator_df = pd.DataFrame(annotator_rows)
    metrics = ["pq", "sq", "rq", "tp", "fp", "fn", "sum_iou", "n_gt", "n_pred"]
    observation_df = annotator_df.groupby(["filename", "fold"], as_index=False)[metrics].mean()
    annotator_df.to_csv(work_dir / "oof_by_annotator.csv", index=False)
    observation_df.to_csv(work_dir / "oof_by_observation.csv", index=False)
    return observation_df, annotator_df


def run_oof_yolo_embedding_variants(
    fold_manifest: pd.DataFrame,
    paths: Mapping[str, Path],
    indices: Mapping[str, Any],
    cfg: CFG,
    work_dir: Path,
    stage2: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """stage2=False scores only 'yolo_only' and needs no Stage 2 checkpoint (a Stage 1 session).

    Scores four decode variants -- YOLO alone (use_embedding_decode=False: the baseline that
    says whether Stage 2 earns its cost), the embedding net alone, the embedding U-Net alone,
    and their mask-level fusion (predict_yolo_embedding_fused's logic, inlined here so
    both architectures' instance masks are decoded once per image and reused for all three rows
    instead of fusing twice) -- on the SAME held-out fold(s), so OOF PQ for each can be compared
    directly to decide which to keep as the actual submission path. See PROGRESS.md "U-Net-
    decoder variant, fused with the embedding net, both evaluated"."""
    metrics = ["pq", "sq", "rq", "tp", "fp", "fn", "sum_iou", "n_gt", "n_pred"]
    annotator_rows = []
    for fold in range(cfg.n_folds):
        if stage2:
            detector, embedding_net, device = load_yolo_embedding_models(fold, cfg, work_dir)
            _, embedding_unet, _ = load_yolo_embedding_models(
                fold, cfg, work_dir, model_cls=YoloConditionedEmbeddingUNet, checkpoint_name="embedding_unet.pt"
            )
        else:
            detector, embedding_net, embedding_unet = load_yolo_detector(fold, work_dir), None, None
        validation_files = fold_manifest.loc[fold_manifest["fold"] == fold, "filename"].tolist()
        if cfg.debug: validation_files = validation_files[:8]
        for filename in tqdm(validation_files, desc=f"fold{fold} OOF (embedding variants)", leave=False):
            image_path = paths["train"] / filename
            variants = [("yolo_only", predict_yolo_only_instances(image_path, detector, cfg)[0])]
            if stage2:
                masks_net, scores_net = predict_yolo_embedding_instances(image_path, detector, embedding_net, device, cfg)
                masks_unet, scores_unet = predict_yolo_embedding_instances(image_path, detector, embedding_unet, device, cfg)
                predictions = [(0, mask, score) for mask, score in zip(masks_net, scores_net)]
                predictions += [(1, mask, score) for mask, score in zip(masks_unet, scores_unet)]
                masks_fused, _ = fuse_fold_predictions(predictions, cfg)
                if not masks_fused and cfg.use_classical_fallback:
                    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
                    masks_fused = [classical_fallback_seed(image, cfg.minimum_area)]
                variants += [("embedding_net", masks_net), ("embedding_unet", masks_unet), ("fused", masks_fused)]
            for variant, masks in variants:
                rows = evaluate_prediction_for_file(filename, masks, indices, cfg)
                for row in rows:
                    row["fold"] = fold
                    row["variant"] = variant
                annotator_rows.extend(rows)
        del detector, embedding_net, embedding_unet; gc.collect(); torch.cuda.empty_cache()
    annotator_df = pd.DataFrame(annotator_rows)
    observation_df = annotator_df.groupby(["filename", "fold", "variant"], as_index=False)[metrics].mean()
    annotator_df.to_csv(work_dir / "oof_by_annotator_variants.csv", index=False)
    observation_df.to_csv(work_dir / "oof_by_observation_variants.csv", index=False)
    return observation_df, annotator_df


def run_failure_mode_breakdown(
    fold_manifest: pd.DataFrame,
    paths: Mapping[str, Path],
    indices: Mapping[str, Any],
    cfg: CFG,
    work_dir: Path,
    fold: int = 0,
    variant_label: str = "embedding_net",
) -> pd.DataFrame:
    """See PROGRESS.md audit item #5: WHY is PQ/RQ what it is, not just what it is. Runs the
    same decode path OOF scoring already uses (load_yolo_embedding_models +
    predict_yolo_embedding_instances), and for every validation image records both the real PQ
    (evaluate_prediction_for_file, unchanged) and the failure-mode breakdown
    (classify_failure_modes, cell 15) side by side -- one row per (image, annotator)."""
    if cfg.use_embedding_decode:
        detector, embedding_model, device = load_yolo_embedding_models(fold, cfg, work_dir)
    else:  # YOLO-only breakdown: no Stage 2 checkpoint needed
        detector, embedding_model, device = load_yolo_detector(fold, work_dir), None, None
    validation_files = fold_manifest.loc[fold_manifest["fold"] == fold, "filename"].tolist()
    if cfg.debug: validation_files = validation_files[:8]
    rows = []
    for filename in tqdm(validation_files, desc=f"fold{fold} failure-mode breakdown ({variant_label})", leave=False):
        masks, _ = predict_yolo_embedding_instances(paths["train"] / filename, detector, embedding_model, device, cfg)
        pq_rows = evaluate_prediction_for_file(filename, masks, indices, cfg)
        for annotator_index, gt_masks in enumerate(ground_truth_sets_for_file(filename, indices)):
            failure_counts = classify_failure_modes(gt_masks, masks, cfg.pq_match_iou)
            matching_pq_rows = [row for row in pq_rows if row["annotator"] == annotator_index]
            rows.append({"filename": filename, "annotator": annotator_index, "variant": variant_label,
                         **matching_pq_rows[0], **failure_counts})
    del detector, embedding_model; gc.collect(); torch.cuda.empty_cache()
    breakdown_df = pd.DataFrame(rows)
    breakdown_df.to_csv(work_dir / f"failure_mode_breakdown_{variant_label}.csv", index=False)
    return breakdown_df


def summarize_oof(annotator_df: pd.DataFrame) -> pd.DataFrame:
    """Per-fold + overall competition PQ (aggregate_pq). Takes the ANNOTATOR-level rows (the
    second value run_oof* returns), not per-observation means -- pooling has to sum each
    annotator-image's own counts."""
    rows = [{"fold": fold, **aggregate_pq(group)} for fold, group in annotator_df.groupby("fold")]
    rows.append({"fold": "overall", **aggregate_pq(annotator_df)})
    return pd.DataFrame(rows)


def sweep_postprocessing(
    fold_manifest: pd.DataFrame,
    paths: Mapping[str, Path],
    indices: Mapping[str, Any],
    cfg: CFG,
    work_dir: Path,
    fold: int,
    thresholds: Sequence[float],
    minimum_areas: Sequence[int],
) -> pd.DataFrame:
    """Free (no retraining) grid sweep of refiner_threshold x minimum_area against real OOF PQ,
    for one already-trained fold's checkpoint. Only these two are genuinely free to sweep this
    way -- boundary_width and boundary_loss_weight are baked into training targets/loss and need
    a retrain to change (see PROGRESS.md). Each validation image's region/boundary probability
    maps are computed ONCE (the only GPU-cost step, via _compute_probability_maps), then
    re-decoded and re-scored at every (threshold, minimum_area) combination via
    _decode_from_probability_maps -- decode/scoring is pure CPU/numpy, so the sweep costs one
    fold's worth of GPU inference regardless of grid size, not grid_size x inference.
    """
    model, device = load_segmenter(fold, cfg, work_dir)
    validation_files = fold_manifest.loc[fold_manifest["fold"] == fold, "filename"].tolist()
    if cfg.debug: validation_files = validation_files[:8]

    cached = [
        (filename, *_compute_probability_maps(paths["train"] / filename, model, device, cfg))
        for filename in tqdm(validation_files, desc=f"fold{fold} sweep: computing maps", leave=False)
    ]
    del model; gc.collect(); torch.cuda.empty_cache()

    metrics = ["pq", "sq", "rq", "tp", "fp", "fn", "n_gt", "n_pred"]
    rows = []
    grid = [(threshold, minimum_area) for threshold in thresholds for minimum_area in minimum_areas]
    for threshold, minimum_area in tqdm(grid, desc="sweep grid", leave=False):
        annotator_rows = []
        for filename, image, region_probability, boundary_probability, energy, region_probability_native in cached:
            masks, scores = _decode_from_probability_maps(
                image, region_probability, boundary_probability, energy, region_probability_native,
                threshold, minimum_area, cfg.energy_seed_threshold,
            )
            annotator_rows.extend(evaluate_prediction_for_file(filename, masks, indices, cfg))
        observation_df = pd.DataFrame(annotator_rows).groupby("filename", as_index=False)[metrics].mean()
        rows.append({"refiner_threshold": threshold, "minimum_area": minimum_area,
                     **observation_df[metrics].mean().to_dict()})
    return pd.DataFrame(rows).sort_values("pq", ascending=False).reset_index(drop=True)


def sweep_postprocessing_yolo_energy(
    fold_manifest: pd.DataFrame,
    paths: Mapping[str, Path],
    indices: Mapping[str, Any],
    cfg: CFG,
    work_dir: Path,
    fold: int,
    minimum_areas: Sequence[int],
) -> pd.DataFrame:
    """Like sweep_postprocessing, but for the active Seg(YOLO)->Energy pipeline
    (load_yolo_energy_models / predict_yolo_energy_instances) instead of the dormant
    detector-free segmenter (load_segmenter / refiner.pt) sweep_postprocessing targets. Only
    minimum_area is free to sweep here -- there's no refiner_threshold-equivalent: YOLO's
    instance masks are already binary, there's no sigmoid region map to re-threshold. Each
    validation image's YOLO masks + energy map are computed ONCE via _compute_yolo_energy_maps,
    then re-decoded at every minimum_area via _decode_yolo_energy_instances -- pure CPU/numpy,
    so the sweep costs one fold's worth of GPU inference regardless of grid size."""
    detector, energy_model, device = load_yolo_energy_models(fold, cfg, work_dir)
    validation_files = fold_manifest.loc[fold_manifest["fold"] == fold, "filename"].tolist()
    if cfg.debug: validation_files = validation_files[:8]

    cached = [
        (filename, *_compute_yolo_energy_maps(paths["train"] / filename, detector, energy_model, device, cfg))
        for filename in tqdm(validation_files, desc=f"fold{fold} sweep: computing maps", leave=False)
    ]
    del detector, energy_model; gc.collect(); torch.cuda.empty_cache()

    metrics = ["pq", "sq", "rq", "tp", "fp", "fn", "n_gt", "n_pred"]
    rows = []
    for minimum_area in tqdm(minimum_areas, desc="sweep grid", leave=False):
        annotator_rows = []
        for filename, image, instance_masks_native, energy_native in cached:
            masks, scores = _decode_yolo_energy_instances(image, instance_masks_native, energy_native, minimum_area)
            annotator_rows.extend(evaluate_prediction_for_file(filename, masks, indices, cfg))
        observation_df = pd.DataFrame(annotator_rows).groupby("filename", as_index=False)[metrics].mean()
        rows.append({"minimum_area": minimum_area, **observation_df[metrics].mean().to_dict()})
    return pd.DataFrame(rows).sort_values("pq", ascending=False).reset_index(drop=True)


def sweep_postprocessing_yolo_embedding(
    fold_manifest: pd.DataFrame,
    paths: Mapping[str, Path],
    indices: Mapping[str, Any],
    cfg: CFG,
    work_dir: Path,
    fold: int,
    minimum_areas: Sequence[int],
    embedding_delta_ds: Sequence[float] = (),
) -> pd.DataFrame:
    """Like sweep_postprocessing_yolo_energy, but for the active Seg(YOLO)->Embedding pipeline
    (load_yolo_embedding_models / predict_yolo_embedding_instances). Sweeps minimum_area x
    embedding_delta_d -- both are decode-time-only parameters of separate_instances_with_embeddings,
    free to vary without retraining: embedding_delta_d is only ever used at decode time as the
    clustering radius (it's ALSO the push-margin in the training loss, but nothing stops decoding
    with a different radius than the one training used, same as how refiner_threshold used to be
    swept independent of its own training-time role elsewhere in this file). embedding_delta_ds
    defaults to () (just cfg.embedding_delta_d, i.e. the pre-existing minimum_area-only sweep
    behavior) so existing callers are unaffected. Each validation image's YOLO masks + embedding
    map are computed ONCE via _compute_yolo_embedding_maps, then re-decoded at every
    (minimum_area, embedding_delta_d) pair via _decode_yolo_embedding_instances -- pure
    CPU/numpy, so the sweep costs one fold's worth of GPU inference regardless of grid size."""
    detector, embedding_model, device = load_yolo_embedding_models(fold, cfg, work_dir)
    validation_files = fold_manifest.loc[fold_manifest["fold"] == fold, "filename"].tolist()
    if cfg.debug: validation_files = validation_files[:8]

    cached = [
        (filename, *_compute_yolo_embedding_maps(paths["train"] / filename, detector, embedding_model, device, cfg))
        for filename in tqdm(validation_files, desc=f"fold{fold} sweep: computing maps", leave=False)
    ]
    del detector, embedding_model; gc.collect(); torch.cuda.empty_cache()

    delta_d_grid = tuple(embedding_delta_ds) or (cfg.embedding_delta_d,)
    rows = []
    for minimum_area in tqdm(minimum_areas, desc="sweep grid: minimum_area", leave=False):
        for delta_d in delta_d_grid:
            annotator_rows = []
            for filename, image, instance_masks_native, embedding_native in cached:
                masks, scores = _decode_yolo_embedding_instances(
                    image, instance_masks_native, embedding_native, minimum_area, delta_d,
                    cfg.use_classical_fallback,
                )
                annotator_rows.extend(evaluate_prediction_for_file(filename, masks, indices, cfg))
            rows.append({"minimum_area": minimum_area, "embedding_delta_d": delta_d,
                         **aggregate_pq(pd.DataFrame(annotator_rows))})
    return pd.DataFrame(rows).sort_values("pq", ascending=False).reset_index(drop=True)


def sweep_detector_conf_yolo_embedding(
    fold_manifest: pd.DataFrame,
    paths: Mapping[str, Path],
    indices: Mapping[str, Any],
    cfg: CFG,
    work_dir: Path,
    fold: int,
    detector_confs: Sequence[float],
) -> pd.DataFrame:
    """Sweeps detector_conf against real OOF PQ -- see PROGRESS.md audit item #2. Unlike
    minimum_area/embedding_delta_d (sweep_postprocessing_yolo_embedding, above), detector_conf
    changes YOLO's OWN predicted masks, so the cached-GPU-half trick that makes the other sweep
    "free" does NOT apply here: every value needs its own full predict_yolo_embedding_instances
    pass (YOLO inference + DINOv3 forward + decode), via a scoped replace(cfg, detector_conf=...)
    copy. Keep detector_confs small (a handful of values) -- this is the one genuinely
    non-free dimension of the two-step sweep (see the driver cell)."""
    detector, embedding_model, device = load_yolo_embedding_models(fold, cfg, work_dir)
    validation_files = fold_manifest.loc[fold_manifest["fold"] == fold, "filename"].tolist()
    if cfg.debug: validation_files = validation_files[:8]

    rows = []
    for detector_conf in tqdm(detector_confs, desc="sweep grid: detector_conf", leave=False):
        scoped_cfg = replace(cfg, detector_conf=detector_conf)
        annotator_rows = []
        for filename in tqdm(validation_files, desc=f"detector_conf={detector_conf}", leave=False):
            masks, scores = predict_yolo_embedding_instances(
                paths["train"] / filename, detector, embedding_model, device, scoped_cfg
            )
            annotator_rows.extend(evaluate_prediction_for_file(filename, masks, indices, scoped_cfg))
        rows.append({"detector_conf": detector_conf, **aggregate_pq(pd.DataFrame(annotator_rows))})
    del detector, embedding_model; gc.collect(); torch.cuda.empty_cache()
    return pd.DataFrame(rows).sort_values("pq", ascending=False).reset_index(drop=True)


Appending to solar_filament_lib.py


### 7. Test inference, submission, and optional ensembling

In [9]:
%%writefile -a solar_filament_lib.py


def fuse_fold_predictions(
    predictions: list[tuple[int, np.ndarray, float]],
    cfg: CFG,
) -> tuple[list[np.ndarray], list[float]]:
    clusters: list[list[tuple[int, np.ndarray, float]]] = []
    for item in sorted(predictions, key=lambda value: value[2], reverse=True):
        best_cluster, best_iou = None, 0.0
        for cluster in clusters:
            representative = cluster[0][1]
            overlap = mask_iou(item[1], representative)
            if overlap > best_iou:
                best_cluster, best_iou = cluster, overlap
        if best_cluster is not None and best_iou >= cfg.ensemble_iou:
            best_cluster.append(item)
        else:
            clusters.append([item])
    fused_masks, fused_scores = [], []
    for cluster in clusters:
        support = len({item[0] for item in cluster})
        maximum_score = max(item[2] for item in cluster)
        if support < cfg.minimum_fold_support and maximum_score < cfg.singleton_score:
            continue
        votes = np.stack([item[1] for item in cluster]).sum(axis=0)
        required_votes = max(1, math.ceil(len(cluster) / 2))
        fused = votes >= required_votes
        if fused.sum() >= cfg.minimum_area:
            fused_masks.append(fused); fused_scores.append(float(np.mean([item[2] for item in cluster])))
    return resolve_overlaps(fused_masks, fused_scores, cfg.minimum_area)


def encode_submission_mask(mask: np.ndarray) -> str:
    rle = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))
    return rle["counts"].decode("ascii")


def run_test_ensemble(paths: Mapping[str, Path], cfg: CFG, work_dir: Path) -> pd.DataFrame:
    test_files = sorted(path for path in paths["test"].iterdir() if path.suffix.lower() == ".jpeg")
    models = [load_fold_models(fold, cfg, work_dir) for fold in range(cfg.n_folds)]
    submission_rows = []
    for image_number, image_path in enumerate(tqdm(test_files, desc="test images"), 1):
        all_predictions = []
        for fold, (detector, refiner, device) in enumerate(models):
            masks, scores = predict_instances(image_path, detector, refiner, device, cfg)
            all_predictions.extend((fold, mask, score) for mask, score in zip(masks, scores))
        masks, mask_scores = fuse_fold_predictions(all_predictions, cfg)
        if not masks:
            # fuse_fold_predictions can legitimately drop every cluster (no cluster reaches
            # minimum_fold_support and none scores >= singleton_score) even when every fold's
            # detector found something. Left unhandled, this test image gets zero submission
            # rows, and cell 32's own assertion (submitted_ids == test_ids) then hard-crashes
            # the whole notebook -- after all 5-fold training + OOF already ran. Guarantee at
            # least one row per test image: fall back to the single highest-scoring raw
            # prediction across all folds, and only if literally nothing survived across all 5
            # folds' detectors (all_predictions itself empty), fall back to the same
            # image-only classical seed used per-fold when a detector finds nothing.
            if all_predictions:
                masks = [max(all_predictions, key=lambda item: item[2])[1]]
            else:
                image_gray = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
                fallback_seed = classical_fallback_seed(image_gray, cfg.minimum_area)
                fold0_refiner = models[0][1]
                masks, _ = refine_seeds(image_gray, [fallback_seed], fold0_refiner, models[0][2], cfg)
        image_id = image_path.stem
        for instance_index, mask in enumerate(masks, 1):
            submission_rows.append({
                "filament_id": f"{image_id}_{instance_index}",
                "segmentation_rle": encode_submission_mask(mask),
            })
    submission = pd.DataFrame(submission_rows, columns=["filament_id", "segmentation_rle"])
    if submission.empty or not submission["filament_id"].is_unique:
        raise RuntimeError("Invalid empty or duplicate-ID submission")
    submission.to_csv(work_dir / "submission.csv", index=False)
    return submission


def run_test_final(paths: Mapping[str, Path], cfg: CFG, work_dir_final: Path) -> pd.DataFrame:
    """Single-model, detector-free test inference for the final submission model.
    predict_segmentation_instances already falls back to classical_fallback_seed internally
    when watershed finds nothing, so masks is never empty here -- same non-empty-per-image
    guarantee run_test_ensemble has, just simpler since there's no cross-model fallback to
    pick from.
    """
    test_files = sorted(path for path in paths["test"].iterdir() if path.suffix.lower() == ".jpeg")
    model, device = load_segmenter(0, cfg, work_dir_final)
    submission_rows = []
    fallback_count = 0
    for image_path in tqdm(test_files, desc="test images (final model)"):
        masks, scores = predict_segmentation_instances(image_path, model, device, cfg)
        if scores == [0.01]:  # predict_segmentation_instances' classical_fallback_seed marker
            fallback_count += 1
        image_id = image_path.stem
        for instance_index, mask in enumerate(masks, 1):
            submission_rows.append({
                "filament_id": f"{image_id}_{instance_index}",
                "segmentation_rle": encode_submission_mask(mask),
            })
    print(f"classical_fallback_seed fired on {fallback_count}/{len(test_files)} test images "
          f"({fallback_count / max(len(test_files), 1):.1%})")
    submission = pd.DataFrame(submission_rows, columns=["filament_id", "segmentation_rle"])
    if submission.empty or not submission["filament_id"].is_unique:
        raise RuntimeError("Invalid empty or duplicate-ID submission")
    submission.to_csv(work_dir_final / "submission.csv", index=False)
    return submission


def run_test_yolo_energy(paths: Mapping[str, Path], cfg: CFG, work_dir_final: Path) -> pd.DataFrame:
    """Like run_test_final, but for the active Seg(YOLO)->Energy pipeline -- loads both stages
    (load_yolo_energy_models) and decodes via predict_yolo_energy_instances. See train_fold,
    cell 17, for why this is the active path as of 2026-09-30."""
    test_files = sorted(path for path in paths["test"].iterdir() if path.suffix.lower() == ".jpeg")
    detector, energy_model, device = load_yolo_energy_models(0, cfg, work_dir_final)
    submission_rows = []
    fallback_count = 0
    for image_path in tqdm(test_files, desc="test images (yolo-energy final model)"):
        masks, scores = predict_yolo_energy_instances(image_path, detector, energy_model, device, cfg)
        if scores == [0.01]:  # predict_yolo_energy_instances' classical_fallback_seed marker
            fallback_count += 1
        image_id = image_path.stem
        for instance_index, mask in enumerate(masks, 1):
            submission_rows.append({
                "filament_id": f"{image_id}_{instance_index}",
                "segmentation_rle": encode_submission_mask(mask),
            })
    print(f"classical_fallback_seed fired on {fallback_count}/{len(test_files)} test images "
          f"({fallback_count / max(len(test_files), 1):.1%})")
    submission = pd.DataFrame(submission_rows, columns=["filament_id", "segmentation_rle"])
    if submission.empty or not submission["filament_id"].is_unique:
        raise RuntimeError("Invalid empty or duplicate-ID submission")
    submission.to_csv(work_dir_final / "submission.csv", index=False)
    return submission


def run_test_yolo_embedding(paths: Mapping[str, Path], cfg: CFG, work_dir_final: Path) -> pd.DataFrame:
    """Like run_test_yolo_energy, but for the active Seg(YOLO)->Embedding pipeline -- loads both
    stages (load_yolo_embedding_models) and decodes via predict_yolo_embedding_instances. See
    PROGRESS.md "embedding-head instance-separation swap" for why this replaces
    run_test_yolo_energy as the active path."""
    test_files = sorted(path for path in paths["test"].iterdir() if path.suffix.lower() == ".jpeg")
    detector, embedding_model, device = load_yolo_embedding_models(0, cfg, work_dir_final)
    submission_rows = []
    fallback_count = 0
    for image_path in tqdm(test_files, desc="test images (yolo-embedding final model)"):
        masks, scores = predict_yolo_embedding_instances(image_path, detector, embedding_model, device, cfg)
        # Kaggle rejects a submission whose masks overlap by even one pixel. finalize_instances /
        # fuse_fold_predictions already guarantee this -- fail here, not on a wasted submission.
        assert not masks or np.stack(masks).sum(axis=0).max() <= 1, f"overlapping masks in {image_path.name}"
        if scores == [0.01]:  # predict_yolo_embedding_instances' classical_fallback_seed marker
            fallback_count += 1
        image_id = image_path.stem
        for instance_index, mask in enumerate(masks, 1):
            submission_rows.append({
                "filament_id": f"{image_id}_{instance_index}",
                "segmentation_rle": encode_submission_mask(mask),
            })
    print(f"classical_fallback_seed fired on {fallback_count}/{len(test_files)} test images "
          f"({fallback_count / max(len(test_files), 1):.1%})")
    submission = pd.DataFrame(submission_rows, columns=["filament_id", "segmentation_rle"])
    if submission.empty or not submission["filament_id"].is_unique:
        raise RuntimeError("Invalid empty or duplicate-ID submission")
    submission.to_csv(work_dir_final / "submission.csv", index=False)
    return submission


def run_test_yolo_embedding_fused(paths: Mapping[str, Path], cfg: CFG, work_dir_final: Path) -> pd.DataFrame:
    """Like run_test_yolo_embedding, but decodes with BOTH Stage 2 architectures and fuses their
    masks per test image (predict_yolo_embedding_fused) -- the submission path once the OOF
    variant comparison (run_oof_yolo_embedding_variants) shows fusion winning. See PROGRESS.md
    "U-Net-decoder variant, fused with the embedding net, both evaluated"."""
    test_files = sorted(path for path in paths["test"].iterdir() if path.suffix.lower() == ".jpeg")
    detector, embedding_net, device = load_yolo_embedding_models(0, cfg, work_dir_final)
    _, embedding_unet, _ = load_yolo_embedding_models(
        0, cfg, work_dir_final, model_cls=YoloConditionedEmbeddingUNet, checkpoint_name="embedding_unet.pt"
    )
    submission_rows = []
    fallback_count = 0
    for image_path in tqdm(test_files, desc="test images (yolo-embedding fused final model)"):
        masks, scores = predict_yolo_embedding_fused(image_path, detector, embedding_net, embedding_unet, device, cfg)
        # Kaggle rejects a submission whose masks overlap by even one pixel. finalize_instances /
        # fuse_fold_predictions already guarantee this -- fail here, not on a wasted submission.
        assert not masks or np.stack(masks).sum(axis=0).max() <= 1, f"overlapping masks in {image_path.name}"
        if scores == [0.01]:  # predict_yolo_embedding_fused's classical_fallback_seed marker
            fallback_count += 1
        image_id = image_path.stem
        for instance_index, mask in enumerate(masks, 1):
            submission_rows.append({
                "filament_id": f"{image_id}_{instance_index}",
                "segmentation_rle": encode_submission_mask(mask),
            })
    print(f"classical_fallback_seed fired on {fallback_count}/{len(test_files)} test images "
          f"({fallback_count / max(len(test_files), 1):.1%})")
    submission = pd.DataFrame(submission_rows, columns=["filament_id", "segmentation_rle"])
    if submission.empty or not submission["filament_id"].is_unique:
        raise RuntimeError("Invalid empty or duplicate-ID submission")
    submission.to_csv(work_dir_final / "submission.csv", index=False)
    return submission


def run_test_segmenter_ensemble(paths: Mapping[str, Path], cfg: CFG, work_dir: Path) -> pd.DataFrame:
    """Test-time ensemble across every trained CV-fold segmenter checkpoint found under
    work_dir (typically WORK_DIR, the 5-fold CV directory -- NOT work_dir_final, whose fold_0
    IS the single submission model run_test_final already uses). No leakage concern doing this
    at test time, unlike OOF scoring: none of the CV-fold models have seen any test image,
    regardless of which folds saw which *training* images -- see
    predict_segmentation_instances_ensemble. Kept alongside, not replacing, run_test_final (the
    single-final-model path, still the default submission route -- see PROGRESS.md). Falls back
    gracefully to however many fold checkpoints actually exist (e.g. just fold 0 during a
    quick-test run), rather than requiring all cfg.n_folds to be trained.

    Distinct from the older, dormant run_test_ensemble (YOLO detector + per-instance box/IoU
    fusion, above) -- this is probability-map averaging for the detector-free segmenter path,
    architecturally unrelated, not a replacement for it.
    """
    available_folds = [
        fold for fold in range(cfg.n_folds)
        if (work_dir / "models" / f"fold_{fold}" / "refiner.pt").exists()
    ]
    if not available_folds:
        raise RuntimeError(f"No trained fold checkpoints found under {work_dir / 'models'}")
    models = []
    device = None
    for fold in available_folds:
        model, device = load_segmenter(fold, cfg, work_dir)
        models.append(model)
    print(f"Ensembling {len(models)} fold checkpoint(s): folds {available_folds}")

    test_files = sorted(path for path in paths["test"].iterdir() if path.suffix.lower() == ".jpeg")
    submission_rows = []
    fallback_count = 0
    for image_path in tqdm(test_files, desc="test images (ensemble)"):
        masks, scores = predict_segmentation_instances_ensemble(image_path, models, device, cfg)
        if scores == [0.01]:  # predict_segmentation_instances' classical_fallback_seed marker
            fallback_count += 1
        image_id = image_path.stem
        for instance_index, mask in enumerate(masks, 1):
            submission_rows.append({
                "filament_id": f"{image_id}_{instance_index}",
                "segmentation_rle": encode_submission_mask(mask),
            })
    print(f"classical_fallback_seed fired on {fallback_count}/{len(test_files)} test images "
          f"({fallback_count / max(len(test_files), 1):.1%})")
    submission = pd.DataFrame(submission_rows, columns=["filament_id", "segmentation_rle"])
    if submission.empty or not submission["filament_id"].is_unique:
        raise RuntimeError("Invalid empty or duplicate-ID submission")
    submission.to_csv(work_dir / "submission_ensemble.csv", index=False)
    return submission


def config_fingerprint(cfg: CFG, fold_manifest: pd.DataFrame) -> str:
    payload = json.dumps(asdict(cfg), sort_keys=True) + "\n" + "\n".join(
        f"{row.filename},{row.fold}" for row in fold_manifest.sort_values("filename").itertuples()
    )
    return hashlib.sha256(payload.encode()).hexdigest()


__all__ = [name for name in globals() if not name.startswith("_")]

Appending to solar_filament_lib.py


### 8. Semantic challenger library

Whole-disk semantic segmentation (image resized to 1536px, ConvNeXt-Tiny encoder with DINOv3
weights, U-Net decoder with a full-resolution stem) -> connected components -> fragment
merging, decoded at native resolution. The whole-image setup follows Anon Tokyo's controlled
tests on this data, where it beat both native 2048px and native 1024px tiles (global context
matters more than native detail). A separate module (`semantic_filament_lib`, imported as
`sfl`) that reuses this notebook's COCO loading, medoid choice, PQ scoring and RLE encoding
from `solar_filament_lib`, so both pipelines are scored by the same code. Design and
reasoning: `SEMANTIC_PIPELINE.md`.


In [10]:
%%writefile semantic_filament_lib.py
"""Challenger pipeline: native-resolution semantic segmentation -> connected components ->
fragment merging -> instances. See SEMANTIC_PIPELINE.md for the design and its reasoning.

COCO loading, medoid choice, PQ scoring (`evaluate_prediction_for_file`, `aggregate_pq`) and RLE
encoding are imported from solar_filament_lib, so both pipelines are scored and encoded by the
same code. The notebook writes solar_filament_lib.py before this module.
"""

from __future__ import annotations

import gc
import json
import math
import random
import time
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Any, Callable, Mapping, Sequence

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from pycocotools import mask as mask_utils
from scipy import ndimage
from scipy.optimize import linear_sum_assignment
from torch.utils.data import DataLoader, Dataset

# Shared with the main pipeline so both are scored and encoded by the exact same code.
from solar_filament_lib import (
    load_coco,
    build_coco_indices,
    annotation_to_rle,
    annotation_to_mask,
    semantic_rle_for_record,
    choose_medoid_annotators,
    pairwise_iou,
    panoptic_quality,
    ground_truth_sets_for_file,
    evaluate_prediction_for_file,
    aggregate_pq,
    encode_submission_mask,
)


@dataclass(frozen=True)
class SemanticCFG:
    seed: int = 20260729
    # Model. ConvNeXt-Tiny with DINOv3-distilled weights: Anon Tokyo's controlled tests on this
    # data used ConvNeXt-Tiny, and DINOv3-distilled ConvNeXt fine-tunes better than ImageNet init.
    # Its first stage is stride 4; FilamentUNet's full-resolution stem supplies stride-1 detail.
    # "efficientnet_b3" (stride-2 first stage) is the previous default.
    encoder_name: str = "convnext_tiny.dinov3_lvd1689m"
    encoder_pretrained: bool = True
    # Trades encoder speed for memory; required at input_size=1536 on a 16 GB GPU (see build_segmenter).
    encoder_grad_checkpointing: bool = True
    decoder_channels: tuple = (256, 128, 64, 32, 16)
    # Whole-image mode (default): the full disk resized to input_size for training and inference.
    # In Anon Tokyo's controlled fold-0 tests, whole image at 1536 (0.4318) beat native 2048
    # (0.4262) and native 1024 tiles: the global context of the disk matters more than native
    # detail. None = the earlier native-crop training + tiled inference (crop_size, tile_*).
    input_size: int | None = 1536
    crop_size: int = 1024
    crops_per_image: int = 2
    foreground_crop_prob: float = 0.7
    # Measured (fp32, CPU, ConvNeXt-T + full-res decoder + clDice): 1.8 GB per 768px image,
    # 3.9 GB per 1024px, so ~9 GB per 1536px image, ~5 GB under AMP on a GPU. Batch 2 would sit
    # at ~11-12 GB of a 16 GB T4, so batch 1 with 2-step accumulation (same effective batch, ~6 GB).
    batch_size: int = 1
    # v27: back to v24's 2. v26 raised it to 4 on the grounds that under GroupNorm it equals a
    # larger batch -- true of the GRADIENT, irrelevant to the NUMBER OF STEPS, which halved at fixed
    # epochs. That bundle measured -0.021 in a controlled A/B on identical cached maps.
    grad_accumulation: int = 2
    # 35 -> 26, derived from v24's measured 1252s/epoch (992s train + 260s full-fold validation)
    # against a 9.0h budget with validation on ~11 of 26 epochs. Biased low on purpose: `epochs`
    # sets T_max and cannot change mid-run, and v24 completed 23/35 with cosine_annealed=false.
    epochs: int = 26
    lr: float = 3e-4
    encoder_lr_mult: float = 0.1
    weight_decay: float = 1e-4
    # Effectively off. A cosine schedule MUST complete, and v24 went 0.4014 -> 0.3885, 0.3526,
    # 0.3514, 0.3624 -> 0.4240: with patience=4 the run dies before its best epoch; it survived by
    # two. The wall-clock budget guard already stops the run safely.
    early_stopping_patience: int = 999

    # --- schedule shape (2026-10-11) ------------------------------------------------------
    # Linear warmup, cosine to a FLOOR rather than to zero, then a constant-LR tail whose
    # snapshots get averaged (SWA). The tail is what makes truncation cheap: stopping two epochs
    # early then costs two snapshots instead of freezing the cosine at 40% of peak.
    # v27: all four OFF, which reduces lr_factor to plain cosine to 0 -- exactly v24's
    # CosineAnnealingLR(T_max=epochs). The shape is kept so it can be re-enabled, but v26 (which
    # used it) scored LB 0.37 and its SWA reproduced the tail MEAN, not the best epoch.
    warmup_epochs: int = 0
    eta_min_mult: float = 0.0           # cosine floor as a fraction of base lr
    constant_tail_epochs: int = 0       # held at the floor, and harvested for SWA
    swa_tail_snapshots: int = 0
    # Validate every k epochs instead of all of them. Full decode + pooled PQ on 141 images costs
    # ~260s, ~20% of each epoch, and at a fold CI of +-0.019 it cannot resolve the epoch-to-epoch
    # differences it was being used to select on. Fewer draws also shrinks the winner's curse:
    # E[max] of 18 draws is 1.82 sigma, of 10 draws 1.54 sigma -- about -0.006 of fake gain.
    val_every: int = 3
    # Wall-clock budget for one train_semantic_fold call. Training stops before an epoch that
    # wouldn't finish inside it (keeping the best checkpoint), so the notebook still reaches
    # OOF scoring and the submission inside Kaggle's 12h session limit. None = no budget.
    max_train_hours: float | None = None
    num_workers: int = 2
    # v27: True. 296 of 707 images (42%) carry 2-3 annotators whose labels were never a training
    # target; the metric POOLS over annotators, so what scores is matching the average one. Also
    # makes the medoid choice irrelevant to training (it still feeds validation only).
    sample_random_annotator: bool = True
    # v27: continuous rotation + scale, applied before the exact rot90/flip group. Every image was
    # resized by exactly 0.75 and the geometric group had order 8, so the model had seen one scale.
    affine_prob: float = 0.5            # half the samples keep the exact test-time geometry
    rotate_deg: float = 15.0
    scale_jitter: float = 0.125         # scale ~ U(1 - j, 1 + j) about the disk centre
    # Loss: bce_weight * BCE + dice_weight * Dice + cldice_weight * clDice.
    bce_weight: float = 1.0
    dice_weight: float = 1.0
    cldice_weight: float = 0.5
    skeleton_iterations: int = 10
    # Inference.
    tile_size: int = 1024
    tile_overlap: int = 256
    use_tta: bool = True
    # Decode knobs (all free to sweep from cached probability maps).
    prob_threshold: float = 0.60
    # Hysteresis: when set below prob_threshold, pixels above it are kept only if they connect to
    # a pixel above prob_threshold, so faint continuations stay attached to strong cores instead
    # of breaking a filament into fragments. None = plain thresholding.
    prob_threshold_low: float | None = 0.56
    # Grow (positive) or shrink (negative) each final instance by this many pixels. PQ needs
    # IoU > 0.5, which on 1-3px structures depends heavily on width, and annotators draw polygons
    # that can be wider than the dark core. Growth never takes another instance's core pixels.
    width_px: int = 0
    min_fragment_area: int = 32
    max_gap_px: float = 40.0
    max_angle_deg: float = 65.0
    endpoint_radius: float = 15.0
    min_elongation: float = 1.5
    bridge_px: int = 0
    minimum_area: int = 240
    force_one_row: bool = False
    # YOLO second opinion (split-only).
    yolo_split: bool = False
    yolo_max_box_iou: float = 0.3
    yolo_min_box_pixels: int = 20
    # Scoring.
    pq_match_iou: float = 0.50
    pq_eval_cap: int | None = None  # None = whole validation fold
    debug: bool = False
    # --- 2026-10-10 bundled retrain -------------------------------------------------------
    # Rationale (measured): near-miss pairs recover only 41% of GT pixels at 74% precision, so
    # IoU>0.5 needs recall 0.41->0.61 AND precision 0.74->0.85 together. A single global
    # threshold moves them in opposite directions, which is why a 158-config decode sweep
    # returned +0.0023. These four changes attack recall with supervision rather than knobs.
    #
    # Skeleton Recall (Kirchhoff et al. 2024) supervised by MAGFiLO's HUMAN-drawn `spine`
    # polyline, not a morphological skeleton of the mask (which branches down every barb and is
    # unstable early in training). Uniform weight along the whole filament length, including the
    # faint ends where the mask loss is effectively apologising for ~1-sigma evidence.
    use_skeleton_recall: bool = True
    skeleton_recall_weight: float = 0.5
    spine_tube_px: int = 3
    # Per-instance Dice instead of image-level Dice. The metric is per-instance with a hard
    # IoU>0.5 gate; image-level Dice lets one large filament own the gradient and is happy to
    # trade a whole small filament for extra width on a big one. PQ is not.
    per_instance_dice: bool = True
    # Fraction of the Dice term taken per-instance; the rest stays image-level so that
    # predicted mass outside every ground-truth box is still penalised.
    per_instance_dice_mix: float = 0.5
    # Spine heatmap auxiliary head (deep supervision only -- decode and inference are untouched).
    # 0.0 keeps the head single-channel, which is what old 1-channel checkpoints expect.
    spine_aux_weight: float = 0.3
    # Input channels. "raw_clahe_raw" is the original recipe whose 3rd channel duplicated the
    # 1st. "bgnorm_clahe_ridge" spends those channels on physics: local-background-normalised
    # contrast removes limb darkening and vignetting so a filament reads the same at disk centre
    # and limb, and a multi-scale Hessian ridge response integrates ALONG the structure, turning
    # ~1-sigma per-pixel evidence into a multi-sigma per-ridge signal before the net sees it.
    channel_recipe: str = "bgnorm_clahe_ridge"


# ---------------------------------------------------------------------------
# Data helpers
# ---------------------------------------------------------------------------













def semantic_target(image_id: str, indices: Mapping[str, Any]) -> np.ndarray:
    """Union of one annotator record's non-crowd instance masks, at native resolution."""
    return mask_utils.decode(
        semantic_rle_for_record(image_id, indices["images_by_id"], indices["annotations_by_image"])
    ).astype(bool)


def spine_target(image_id: str, indices: Mapping[str, Any], tube_px: int = 3) -> np.ndarray:
    """Union of this annotator record's `spine` polylines, rendered as a tube of `tube_px` width.

    MAGFiLO ships a human-drawn centerline per filament. That is strictly better than a
    morphological skeleton of the mask: `skeletonize` on a barbed filament sprouts a branch down
    every barb and wanders with the mask boundary, while the drawn spine is the structure's
    actual axis. Supervising it gives a loss term whose weight is uniform along the filament,
    including the faint ends where the mask evidence is ~1 sigma -- which is exactly where the
    measured length_ratio of 0.5576 says we are losing pixels.
    """
    image = indices["images_by_id"][image_id]
    height, width = int(image["height"]), int(image["width"])
    canvas = np.zeros((height, width), dtype=np.uint8)
    for annotation in indices["annotations_by_image"].get(image_id, []):
        if annotation.get("iscrowd", 0):
            continue
        spine = annotation.get("spine")
        if not spine:
            continue
        points = np.asarray(spine, dtype=np.float32).reshape(-1, 2)
        if len(points) < 2:
            continue
        cv2.polylines(canvas, [np.round(points).astype(np.int32)], False, 1, thickness=tube_px)
    return canvas.astype(bool)


# ---------------------------------------------------------------------------
# Scoring (same contract as the notebook's evaluate_prediction_for_file)
# ---------------------------------------------------------------------------

OOF_COLUMNS = ["filename", "annotator", "pq", "sq", "rq", "tp", "fp", "fn", "sum_iou", "n_gt", "n_pred"]














# ---------------------------------------------------------------------------
# Input / augmentation / dataset
# ---------------------------------------------------------------------------

def read_gray(path: Path) -> np.ndarray:
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise FileNotFoundError(path)
    return image


def background_normalised(image: np.ndarray, sigma: float = 64.0) -> np.ndarray:
    """image / local background, rescaled to [0, 1] about 0.5.

    What carries filament information is optical depth, i.e. contrast relative to the LOCAL
    chromosphere, not absolute DN. Dividing by a heavily blurred copy removes limb darkening and
    the GONG vignetting gradient in one step, so a filament at the limb and one at disk centre
    present the same amplitude to the encoder instead of the network having to learn that
    normalisation from 707 images.
    """
    img = image.astype(np.float32)
    background = cv2.GaussianBlur(img, (0, 0), sigma)
    return np.clip(img / np.maximum(background, 1e-3) * 0.5, 0.0, 1.0)


def ridge_response(image: np.ndarray, scales: Sequence[float] = (2.0, 4.0, 8.0)) -> np.ndarray:
    """Multi-scale Hessian ridge filter tuned for DARK lines, normalised to [0, 1].

    A filament end or barb is ~1 sigma above the chromospheric mottle per pixel, so a per-pixel
    classifier is near chance there. A ridge filter integrates ALONG the structure, converting
    that into a multi-sigma per-ridge response before the encoder sees it. Dark ridges have a
    positive second derivative across their axis, so we keep the larger Hessian eigenvalue, and
    scale-normalise by sigma^2 so the scales are comparable.
    """
    img = image.astype(np.float32) / 255.0
    best = np.zeros_like(img)
    for sigma in scales:
        smooth = cv2.GaussianBlur(img, (0, 0), float(sigma))
        gxx = cv2.Sobel(smooth, cv2.CV_32F, 2, 0, ksize=3)
        gyy = cv2.Sobel(smooth, cv2.CV_32F, 0, 2, ksize=3)
        gxy = cv2.Sobel(smooth, cv2.CV_32F, 1, 1, ksize=3)
        spread = np.sqrt(np.maximum((gxx - gyy) ** 2 + 4.0 * gxy ** 2, 0.0))
        larger = 0.5 * ((gxx + gyy) + spread)
        best = np.maximum(best, (sigma ** 2) * larger)
    high = float(np.percentile(best, 99.5))
    return np.clip(best / max(high, 1e-6), 0.0, 1.0)


def to_channels(image: np.ndarray, clahe_clip: float = 2.0,
                recipe: str = "bgnorm_clahe_ridge") -> np.ndarray:
    """uint8 gray (H, W) -> float32 (3, H, W).

    "raw_clahe_raw" is the original recipe, whose third channel was a duplicate of the first and
    therefore carried no information. "bgnorm_clahe_ridge" spends all three channels (see
    `background_normalised` and `ridge_response`). Three channels either way, so ImageNet /
    DINOv3 encoder stems are unchanged.
    """
    clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=(8, 8)).apply(image)
    if recipe == "raw_clahe_raw":
        return np.stack([image, clahe, image]).astype(np.float32) / 255.0
    if recipe == "raw_clahe_ridge":
        return np.stack([image.astype(np.float32) / 255.0, clahe.astype(np.float32) / 255.0,
                         ridge_response(image)])
    if recipe == "bgnorm_clahe_ridge":
        return np.stack([background_normalised(image), clahe.astype(np.float32) / 255.0,
                         ridge_response(image)])
    raise ValueError(f"unknown channel_recipe {recipe!r}")


def augment(image: np.ndarray, target: np.ndarray, rng: np.random.Generator,
            cfg: "SemanticCFG | None" = None) -> tuple[np.ndarray, float, np.ndarray]:
    """Online augmentation on a uint8 crop. Returns (image, clahe_clip, target). Geometric:
    an optional continuous rotation+scale (cfg.affine_prob), then flips + rot90 (exact for
    pixel-wise labels). Photometric: gamma, blur, noise, CLAHE strength.

    The affine warp resamples the image bilinearly but the TARGET with nearest-neighbour: a
    bilinear-then-threshold target thins 2-3px spine tubes on diagonals, while nearest keeps the
    pixel count and the mask/spine channels move together as one (H, W, 2) array. Exposed borders
    are filled with the off-disk sky level (median of the four corner patches, which a full-disk
    frame always leaves off the Sun) -- not 0 and not a reflection, which at a 0.875 zoom-out would
    mirror a sliver of the limb into the corners and hand the model a free "corner" prior that
    does not exist at test time."""
    # affine_prob is checked BEFORE drawing, so 'off' consumes no random numbers and the rest of
    # the augmentation stream is bit-identical to the pre-v27 path.
    if (cfg is not None and cfg.affine_prob > 0 and (cfg.rotate_deg > 0 or cfg.scale_jitter > 0)
            and rng.random() < cfg.affine_prob):
        height, width = image.shape[:2]
        angle = float(rng.uniform(-cfg.rotate_deg, cfg.rotate_deg))
        scale = float(rng.uniform(1.0 - cfg.scale_jitter, 1.0 + cfg.scale_jitter))
        matrix = cv2.getRotationMatrix2D(((width - 1) / 2.0, (height - 1) / 2.0), angle, scale)
        patch = max(8, min(height, width) // 32)
        sky = float(np.median(np.concatenate([image[:patch, :patch].ravel(), image[:patch, -patch:].ravel(),
                                              image[-patch:, :patch].ravel(), image[-patch:, -patch:].ravel()])))
        image = cv2.warpAffine(np.ascontiguousarray(image), matrix, (width, height), flags=cv2.INTER_LINEAR,
                               borderMode=cv2.BORDER_CONSTANT, borderValue=sky)
        target = cv2.warpAffine(np.ascontiguousarray(target), matrix, (width, height),
                                flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
        if target.shape[:2] != image.shape[:2]:
            raise AssertionError(f"affine desynchronised image {image.shape} and target {target.shape}")
    k = int(rng.integers(4))
    image, target = np.rot90(image, k), np.rot90(target, k)
    if rng.random() < 0.5:
        image, target = image[:, ::-1], target[:, ::-1]
    if rng.random() < 0.5:
        image, target = image[::-1], target[::-1]
    image = np.ascontiguousarray(image)
    target = np.ascontiguousarray(target)
    gamma = float(rng.uniform(0.7, 1.4))
    image = np.clip(255.0 * (image / 255.0) ** gamma, 0, 255).astype(np.uint8)
    if rng.random() < 0.3:
        image = cv2.GaussianBlur(image, (0, 0), float(rng.uniform(0.5, 1.2)))
    if rng.random() < 0.5:
        noise = rng.normal(0, float(rng.uniform(2, 8)), image.shape)
        image = np.clip(image.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    return image, float(rng.uniform(1.0, 4.0)), target


def random_crop_origin(target: np.ndarray, crop: int, foreground_prob: float,
                       rng: np.random.Generator) -> tuple[int, int]:
    height, width = target.shape
    max_y, max_x = max(height - crop, 0), max(width - crop, 0)
    if rng.random() < foreground_prob:
        ys, xs = np.nonzero(target)
        if len(ys):
            pick = int(rng.integers(len(ys)))
            return (int(np.clip(ys[pick] - crop // 2, 0, max_y)), int(np.clip(xs[pick] - crop // 2, 0, max_x)))
    return int(rng.integers(max_y + 1)), int(rng.integers(max_x + 1))


def pad_to(array: np.ndarray, height: int, width: int) -> np.ndarray:
    pad_h, pad_w = max(height - array.shape[0], 0), max(width - array.shape[1], 0)
    if not pad_h and not pad_w:
        return array
    pad = ((0, pad_h), (0, pad_w)) + ((0, 0),) * (array.ndim - 2)
    return np.pad(array, pad, mode="constant")


def resize_to_input(image: np.ndarray, size: int) -> np.ndarray:
    """Whole image -> size x size (area interpolation: averages, so faint filaments dim but stay)."""
    return cv2.resize(image, (size, size), interpolation=cv2.INTER_AREA)


def resize_target(target: np.ndarray, size: int) -> np.ndarray:
    """Binary target -> size x size. Area interpolation then >= 0.5: a pixel is foreground if most
    of it was. (`> 0` widened every target by up to one input pixel per side, about 1.3 native px
    at 1536, which capped IoU on thin filaments near the 0.5 match threshold.) The thinnest
    MAGFiLO filament is 1.86 px, so at 1536 (scale 0.75) every column keeps a majority pixel."""
    return (cv2.resize(target.astype(np.float32), (size, size), interpolation=cv2.INTER_AREA) >= 0.5).astype(target.dtype)


class SemanticCropDataset(Dataset):
    """Random native-resolution crops of (image, union-of-instances target). Train mode samples
    crops_per_image crops per image per epoch, biased toward filament pixels (filaments cover a
    tiny fraction of a full disk). sample_random_annotator draws a different annotator record
    per sample instead of the medoid -- multi-annotator labels used as label augmentation."""

    def __init__(self, filenames, image_dir, indices, selected_image_ids, cfg: SemanticCFG, train: bool):
        self.filenames, self.image_dir, self.indices = list(filenames), Path(image_dir), indices
        self.selected_image_ids, self.cfg, self.train = selected_image_ids, cfg, train

    def __len__(self) -> int:
        crops = self.cfg.crops_per_image if self.train and self.cfg.input_size is None else 1
        return len(self.filenames) * crops

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        cfg = self.cfg
        filename = self.filenames[index % len(self.filenames)]
        rng = np.random.default_rng(None if self.train else cfg.seed + index)
        if self.train and cfg.sample_random_annotator:
            candidates = self.indices["image_ids_by_file"][filename]
            image_id = candidates[int(rng.integers(len(candidates)))]
        else:
            image_id = self.selected_image_ids[filename]
        image = read_gray(self.image_dir / filename)
        # Mask and spine travel stacked as (H, W, 2) so every geometric op -- resize, crop, pad,
        # rot90, flips -- applies to both identically. Splitting them earlier is how they drift.
        target = semantic_target(image_id, self.indices)
        target = np.stack([target.astype(np.uint8),
                           spine_target(image_id, self.indices, cfg.spine_tube_px).astype(np.uint8)],
                          axis=-1)
        if cfg.input_size is not None:
            image, target = resize_to_input(image, cfg.input_size), resize_target(target, cfg.input_size)
        else:
            y0, x0 = random_crop_origin(target[..., 0] if target.ndim == 3 else target, cfg.crop_size,
                                        cfg.foreground_crop_prob if self.train else 1.0, rng)
            image = pad_to(image[y0:y0 + cfg.crop_size, x0:x0 + cfg.crop_size], cfg.crop_size, cfg.crop_size)
            target = pad_to(target[y0:y0 + cfg.crop_size, x0:x0 + cfg.crop_size], cfg.crop_size, cfg.crop_size)
        clahe_clip = 2.0
        if self.train:
            image, clahe_clip, target = augment(image, target, rng, cfg)
        mask, spine = target[..., 0], target[..., 1]
        # Components are labelled AFTER the geometric ops, so the ids match the augmented mask.
        _, components = cv2.connectedComponents(np.ascontiguousarray(mask).astype(np.uint8), connectivity=8)
        return {"image": torch.from_numpy(to_channels(image, clahe_clip, cfg.channel_recipe)),
                "target": torch.from_numpy(mask.astype(np.float32)[None]),
                "spine": torch.from_numpy(spine.astype(np.float32)[None]),
                "components": torch.from_numpy(components.astype(np.int32)[None])}


# ---------------------------------------------------------------------------
# Loss: BCE + Dice + clDice (soft skeleton, Shit et al. 2021)
# ---------------------------------------------------------------------------

def soft_erode(x: torch.Tensor) -> torch.Tensor:
    return torch.min(-F.max_pool2d(-x, (3, 1), 1, (1, 0)), -F.max_pool2d(-x, (1, 3), 1, (0, 1)))


def soft_dilate(x: torch.Tensor) -> torch.Tensor:
    return F.max_pool2d(x, 3, 1, 1)


def soft_skeleton(x: torch.Tensor, iterations: int) -> torch.Tensor:
    skeleton = F.relu(x - soft_dilate(soft_erode(x)))
    for _ in range(iterations):
        x = soft_erode(x)
        delta = F.relu(x - soft_dilate(soft_erode(x)))
        skeleton = skeleton + F.relu(delta - skeleton * delta)
    return skeleton


def soft_cldice_loss(probabilities: torch.Tensor, targets: torch.Tensor, iterations: int) -> torch.Tensor:
    """1 - clDice. Topology precision: predicted skeleton inside the target; topology
    sensitivity: target skeleton covered by the prediction. Penalizes broken thin structures
    much harder than Dice, which barely notices a 2px gap in a long filament."""
    smooth = 1.0
    skeleton_pred = soft_skeleton(probabilities, iterations)
    skeleton_true = soft_skeleton(targets, iterations)
    precision = ((skeleton_pred * targets).sum((1, 2, 3)) + smooth) / (skeleton_pred.sum((1, 2, 3)) + smooth)
    sensitivity = ((skeleton_true * probabilities).sum((1, 2, 3)) + smooth) / (skeleton_true.sum((1, 2, 3)) + smooth)
    return (1.0 - 2.0 * precision * sensitivity / (precision + sensitivity)).mean()


def skeleton_recall_loss(probabilities: torch.Tensor, spine: torch.Tensor) -> torch.Tensor:
    """1 - (predicted probability mass recovered on the annotated spine tube).

    Unlike soft-clDice this does no morphology on the prediction, so it is stable from epoch 1
    and costs one elementwise product instead of 10 erode/dilate iterations. Its weight is
    uniform along the filament, which is the point: the faint 25% of each filament nearest the
    footpoints gets the same gradient as the saturated core.
    """
    covered = (probabilities * spine).sum((1, 2, 3))
    total = spine.sum((1, 2, 3)) + 1.0
    return (1.0 - covered / total).mean()


def per_instance_dice_loss(probabilities: torch.Tensor, components: torch.Tensor,
                           margin: int = 16) -> torch.Tensor:
    """Mean Dice over ground-truth connected components, each scored inside its own bounding box.

    Image-level Dice pools every filament on the disk into one ratio, so a single large filament
    owns the gradient and the loss will happily trade a whole small filament for a little extra
    width on a big one. PQ will not: it is per instance with a hard IoU>0.5 gate. Scoring in a
    margin-expanded box keeps the term sensitive to over-prediction as well as under-prediction,
    which a plain masked Dice is not.
    """
    losses = []
    for index in range(probabilities.shape[0]):
        label_map = components[index, 0]
        labels = torch.unique(label_map)
        labels = labels[labels > 0]
        probability = probabilities[index, 0]
        for label in labels:
            instance = (label_map == label)
            rows = torch.nonzero(instance.any(dim=1), as_tuple=False)
            cols = torch.nonzero(instance.any(dim=0), as_tuple=False)
            if not len(rows) or not len(cols):
                continue
            y0 = max(int(rows[0]) - margin, 0)
            y1 = min(int(rows[-1]) + margin + 1, instance.shape[0])
            x0 = max(int(cols[0]) - margin, 0)
            x1 = min(int(cols[-1]) + margin + 1, instance.shape[1])
            window_labels = label_map[y0:y1, x0:x1]
            window_target = (window_labels == label).float()
            # A margin-expanded box often clips a neighbouring filament. Scoring those pixels as
            # background would punish the model for predicting them correctly, so they are
            # excluded: only this instance and true background count here.
            valid = ((window_labels == label) | (window_labels == 0)).float()
            window_probability = probability[y0:y1, x0:x1] * valid
            intersection = (window_probability * window_target).sum()
            denominator = window_probability.sum() + window_target.sum() + 1.0
            losses.append(1.0 - (2.0 * intersection + 1.0) / denominator)
    if not losses:
        return probabilities.sum() * 0.0
    return torch.stack(losses).mean()


def segmentation_loss(logits: torch.Tensor, targets: torch.Tensor, cfg: SemanticCFG,
                      spine: torch.Tensor | None = None,
                      components: torch.Tensor | None = None) -> torch.Tensor:
    """BCE + Dice + (Skeleton Recall | clDice) + optional spine-head deep supervision.

    `logits` carries the spine auxiliary channel when cfg.spine_aux_weight > 0; channel 0 is
    always the filament mask, so inference never has to know about any of this.
    """
    logits, targets = logits.float(), targets.float()  # fp32 reductions under autocast
    mask_logits = logits[:, :1]
    probabilities = mask_logits.sigmoid()
    bce = F.binary_cross_entropy_with_logits(mask_logits, targets)

    intersection = (probabilities * targets).sum((1, 2, 3))
    image_dice = 1.0 - ((2 * intersection + 1.0)
                        / (probabilities.sum((1, 2, 3)) + targets.sum((1, 2, 3)) + 1.0)).mean()
    if cfg.per_instance_dice and components is not None:
        mix = float(cfg.per_instance_dice_mix)
        dice = mix * per_instance_dice_loss(probabilities, components) + (1.0 - mix) * image_dice
    else:
        dice = image_dice

    if cfg.use_skeleton_recall and spine is not None:
        topology = cfg.skeleton_recall_weight * skeleton_recall_loss(probabilities, spine.float())
    else:
        topology = cfg.cldice_weight * soft_cldice_loss(probabilities, targets, cfg.skeleton_iterations)

    total = cfg.bce_weight * bce + cfg.dice_weight * dice + topology
    if cfg.spine_aux_weight > 0 and spine is not None:
        assert logits.shape[1] >= 2, (
            f"spine_aux_weight={cfg.spine_aux_weight} needs a 2-channel head, got {logits.shape[1]}")
        total = total + cfg.spine_aux_weight * F.binary_cross_entropy_with_logits(
            logits[:, 1:2], spine.float())
    return total


# ---------------------------------------------------------------------------
# Model: timm features_only encoder + U-Net decoder + full-resolution image stem
# ---------------------------------------------------------------------------

class ConvBlock(nn.Sequential):
    """GroupNorm, not BatchNorm. Whole-image training runs at batch size 1, where BatchNorm trains
    on per-image statistics but evaluates on running averages: a train/eval mismatch on images as
    varied as these. Its running statistics also update during the forward pass even when the
    optimizer skips a non-finite step, so a single overflowing batch poisoned them with NaN in the
    first real run and every later validation scored 0. GroupNorm has no running statistics."""

    def __init__(self, in_channels: int, out_channels: int) -> None:
        groups = math.gcd(8, out_channels)
        super().__init__(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False), nn.GroupNorm(groups, out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False), nn.GroupNorm(groups, out_channels), nn.ReLU(inplace=True),
        )


class FilamentUNet(nn.Module):
    """U-Net decoder over any encoder returning a list of feature maps (shallow -> deep). The
    last decoder stage concatenates a full-resolution stem of the raw input, so the head sees
    native-pixel detail even when the shallowest encoder feature is stride 2 -- that path is
    what keeps 1-2px filament edges and barbs from being averaged away."""

    def __init__(self, encoder: nn.Module, encoder_channels: Sequence[int], decoder_channels: Sequence[int],
                 head_channels: int = 1) -> None:
        super().__init__()
        self.encoder = encoder
        encoder_channels = list(encoder_channels)
        # Per-level normalization of encoder outputs. DINOv3 ConvNeXt stage 3 carries outlier
        # activations (|x| ~ 131, std 5 vs. 0.2 at the other stages, measured at init); unnormalized
        # they dominate the decoder's concatenations and overflow its fp16 convs as they grow.
        self.feature_norms = nn.ModuleList(nn.GroupNorm(math.gcd(32, c), c) for c in encoder_channels)
        skips = encoder_channels[:-1][::-1]
        n_stages = len(skips)
        decoder_channels = list(decoder_channels)[:n_stages + 1]
        while len(decoder_channels) < n_stages + 1:
            decoder_channels.append(max(decoder_channels[-1] // 2, 8))
        self.blocks = nn.ModuleList()
        in_channels = encoder_channels[-1]
        for skip_channels, out_channels in zip(skips, decoder_channels[:n_stages]):
            self.blocks.append(ConvBlock(in_channels + skip_channels, out_channels))
            in_channels = out_channels
        self.stem = ConvBlock(3, 16)
        self.final = ConvBlock(in_channels + 16, decoder_channels[n_stages])
        self.head = nn.Conv2d(decoder_channels[n_stages], head_channels, 1)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        # Encoder in fp32 even under AMP. DINOv3-trained backbones carry very large activations,
        # and fp16 overflows at 65504: the first real run went NaN at epoch 4. The decoder stays
        # in mixed precision. (Same fix this project needed earlier for the DINOv3 ViT.)
        with torch.autocast(device_type=images.device.type, enabled=False):
            features = self.encoder(((images - self.mean) / self.std).float())
            features = [norm(f) for norm, f in zip(self.feature_norms, features)]
        x = features[-1]
        for block, skip in zip(self.blocks, features[:-1][::-1]):
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = block(torch.cat([x, skip], dim=1))
        x = F.interpolate(x, size=images.shape[-2:], mode="bilinear", align_corners=False)
        return self.head(self.final(torch.cat([x, self.stem(images)], dim=1)))


def build_segmenter(cfg: SemanticCFG) -> FilamentUNet:
    import timm  # preinstalled on Kaggle; pretrained weights download from the HF hub

    encoder = timm.create_model(cfg.encoder_name, pretrained=cfg.encoder_pretrained, features_only=True)
    # The encoder runs in fp32 (see FilamentUNet.forward), so its activations cost twice what the
    # AMP budget in SemanticCFG assumed: a 1536px image OOM'd a 16 GB T4 inside stage 1. Gradient
    # checkpointing recomputes them in the backward pass instead of storing them -- ~30% slower
    # encoder, but it is the only knob that keeps both fp32 numerics and the 1536px whole-image
    # input that scored best.
    if cfg.encoder_grad_checkpointing:
        encoder.set_grad_checkpointing(True)
    # Channel 0 is always the filament mask, so a 1-channel checkpoint stays loadable by
    # setting spine_aux_weight=0.0, and inference never indexes past channel 0.
    head_channels = 2 if cfg.spine_aux_weight > 0 else 1
    return FilamentUNet(encoder, encoder.feature_info.channels(), cfg.decoder_channels, head_channels)


# ---------------------------------------------------------------------------
# Tiled, TTA'd, fold-ensembled probability maps at native resolution
# ---------------------------------------------------------------------------

def tile_starts(length: int, tile: int, overlap: int) -> list[int]:
    if length <= tile:
        return [0]
    starts = list(range(0, length - tile + 1, tile - overlap))
    if starts[-1] != length - tile:
        starts.append(length - tile)
    return starts


def blend_window(tile: int, overlap: int) -> np.ndarray:
    """Separable linear ramp, strictly positive everywhere, so overlapping tiles cross-fade
    instead of leaving a seam at each tile border (where the network has the least context)."""
    ramp = np.ones(tile, dtype=np.float32)
    if overlap > 0:
        edge = np.linspace(1.0 / (overlap + 1), 1.0, overlap, dtype=np.float32)
        ramp[:overlap] = edge
        ramp[-overlap:] = np.minimum(ramp[-overlap:], edge[::-1])
    return np.outer(ramp, ramp)


def tile_image(array: np.ndarray, tile: int, overlap: int) -> list[tuple[int, int, np.ndarray]]:
    """array: (..., H, W) with H, W >= tile (pad first)."""
    height, width = array.shape[-2:]
    return [(y, x, array[..., y:y + tile, x:x + tile])
            for y in tile_starts(height, tile, overlap) for x in tile_starts(width, tile, overlap)]


def stitch_tiles(patches: Sequence[tuple[int, int, np.ndarray]], shape: tuple[int, int],
                 tile: int, overlap: int) -> np.ndarray:
    window = blend_window(tile, overlap)
    total = np.zeros(shape, dtype=np.float32)
    weight = np.zeros(shape, dtype=np.float32)
    for y, x, patch in patches:
        total[y:y + tile, x:x + tile] += patch * window
        weight[y:y + tile, x:x + tile] += window
    return total / np.maximum(weight, 1e-8)


_TTA_FLIPS = {"none": (), "h": (-1,), "v": (-2,), "hv": (-2, -1)}


@torch.no_grad()
def predict_maps(models: Sequence[nn.Module], image: np.ndarray, cfg: SemanticCFG,
                 device: torch.device) -> tuple[np.ndarray, np.ndarray | None]:
    """Native-resolution (mask, spine) for one uint8 image: tiles x flip-TTA x models, averaged at
    the probability level (valid for semantic maps, unlike embeddings).

    `spine` is None when the head has one channel, so pre-spine checkpoints keep working. Before
    2026-10-11 this function kept only `[0, 0]`, so the trained spine head was computed on every
    forward pass and then thrown away.
    # ponytail: one tile per forward pass; batch tiles if inference time matters."""
    height, width = image.shape
    flips = list(_TTA_FLIPS.values()) if cfg.use_tta else [()]
    if cfg.input_size is not None:
        # Whole-image mode: one forward pass per flip on the resized disk, then back to native size.
        tensor = torch.from_numpy(
            to_channels(resize_to_input(image, cfg.input_size), recipe=cfg.channel_recipe))[None].to(device)
        total = None
        for model in models:
            for dims in flips:
                probability = model(tensor.flip(dims) if dims else tensor).float().sigmoid()
                probability = (probability.flip(dims) if dims else probability)[0]
                total = probability.clone() if total is None else total + probability
        small = (total / (len(models) * len(flips))).cpu().numpy()
        full = [cv2.resize(small[c], (width, height), interpolation=cv2.INTER_LINEAR)
                for c in range(small.shape[0])]
        return full[0], (full[1] if len(full) > 1 else None)
    tile = cfg.tile_size
    padded = pad_to(image, max(height, tile), max(width, tile))
    channels = to_channels(padded, recipe=cfg.channel_recipe)
    patches = []
    for y, x, patch in tile_image(channels, tile, cfg.tile_overlap):
        tensor = torch.from_numpy(np.ascontiguousarray(patch))[None].to(device)
        total = None
        for model in models:
            for dims in flips:
                probability = model(tensor.flip(dims) if dims else tensor).float().sigmoid()
                probability = (probability.flip(dims) if dims else probability)[0]
                total = probability.clone() if total is None else total + probability
        patches.append((y, x, (total / (len(models) * len(flips))).cpu().numpy()))
    n_out = patches[0][2].shape[0]
    full = [stitch_tiles([(y, x, p[c]) for y, x, p in patches], padded.shape, tile,
                         cfg.tile_overlap)[:height, :width] for c in range(n_out)]
    return full[0], (full[1] if n_out > 1 else None)


@torch.no_grad()
def predict_probability(models: Sequence[nn.Module], image: np.ndarray, cfg: SemanticCFG,
                        device: torch.device) -> np.ndarray:
    """Mask-channel probability only: the one-channel view of `predict_maps`, kept so every
    existing caller is unchanged."""
    return predict_maps(models, image, cfg, device)[0]


# ---------------------------------------------------------------------------
# Decode: threshold -> connected components -> fragment merging -> area filter
# ---------------------------------------------------------------------------

def fragment_endpoints(points: np.ndarray, radius: float, min_elongation: float) -> list[tuple[np.ndarray, np.ndarray]]:
    """points: (N, 2) as (x, y). Returns [(endpoint, outward unit direction)] for the two ends
    along the principal axis, or [] for blob-like fragments (no reliable orientation). Local
    direction comes from the pixels within `radius` of each end, so mildly curved filaments
    still get the direction at the end, not of the whole body.
    # ponytail: PCA extremes, not a true skeleton -- strongly curved/branched fragments get
    # approximate endpoints; swap in skeleton endpoint detection if OOF shows missed merges."""
    if len(points) < 3:
        return []
    mean = points.mean(axis=0)
    centered = points - mean
    eigenvalues, eigenvectors = np.linalg.eigh(centered.T @ centered / len(points))
    if math.sqrt(max(eigenvalues[1], 0.0) / max(eigenvalues[0], 1e-6)) < min_elongation:
        return []
    axis = eigenvectors[:, 1]
    projection = centered @ axis
    ends = []
    for index, fallback in ((int(np.argmin(projection)), -axis), (int(np.argmax(projection)), axis)):
        endpoint = points[index]
        near = points[np.linalg.norm(points - endpoint, axis=1) <= radius]
        direction = endpoint - near.mean(axis=0)
        norm = float(np.linalg.norm(direction))
        ends.append((endpoint, direction / norm if norm > 1e-6 else fallback))
    return ends


def _angle_deg(a: np.ndarray, b: np.ndarray) -> float:
    return math.degrees(math.acos(float(np.clip(np.dot(a, b), -1.0, 1.0))))


def merge_fragment_groups(fragments: Sequence[np.ndarray], cfg: SemanticCFG) -> tuple[list[list[int]], list[tuple]]:
    """fragments: list of (N_i, 2) point arrays. Union-find over fragment pairs whose facing
    endpoints are within max_gap_px AND whose outward directions both point along the gap
    (within max_angle_deg) -- a collinear continuation, not a side-by-side neighbor.
    Returns (groups of fragment indices, bridge segments).
    # ponytail: O(n^2) pair scan with bbox prefilter; fine for tens of fragments per disk."""
    parent = list(range(len(fragments)))

    def find(i: int) -> int:
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    ends = [fragment_endpoints(points, cfg.endpoint_radius, cfg.min_elongation) for points in fragments]
    boxes = [(points.min(axis=0), points.max(axis=0)) for points in fragments]
    bridges = []
    for i in range(len(fragments)):
        for j in range(i + 1, len(fragments)):
            if not ends[i] or not ends[j]:
                continue
            gap_box = np.maximum(0, np.maximum(boxes[i][0] - boxes[j][1], boxes[j][0] - boxes[i][1]))
            if float(np.linalg.norm(gap_box)) > cfg.max_gap_px:
                continue
            for end_a, dir_a in ends[i]:
                for end_b, dir_b in ends[j]:
                    vector = end_b - end_a
                    gap = float(np.linalg.norm(vector))
                    if gap > cfg.max_gap_px or gap < 1e-6:
                        continue
                    unit = vector / gap
                    if _angle_deg(dir_a, unit) <= cfg.max_angle_deg and _angle_deg(dir_b, -unit) <= cfg.max_angle_deg:
                        parent[find(i)] = find(j)
                        bridges.append((i, j, end_a, end_b))
    groups: dict[int, list[int]] = {}
    for i in range(len(fragments)):
        groups.setdefault(find(i), []).append(i)
    return list(groups.values()), bridges


def decode_instances(probability: np.ndarray, cfg: SemanticCFG,
                     yolo_boxes: np.ndarray | None = None) -> list[np.ndarray]:
    high = probability >= cfg.prob_threshold
    if cfg.prob_threshold_low is not None and cfg.prob_threshold_low < cfg.prob_threshold:
        _, low_labels = cv2.connectedComponents((probability >= cfg.prob_threshold_low).astype(np.uint8), connectivity=8)
        seeded = np.unique(low_labels[high])
        binary = np.isin(low_labels, seeded[seeded > 0]).astype(np.uint8)
    else:
        binary = high.astype(np.uint8)
    _, labels = cv2.connectedComponents(binary, connectivity=8)
    fragment_labels, fragments = [], []
    for label, region in enumerate(ndimage.find_objects(labels), start=1):
        if region is None:
            continue
        ys, xs = np.nonzero(labels[region] == label)
        if len(ys) < cfg.min_fragment_area:
            continue
        fragment_labels.append(label)
        fragments.append(np.stack([xs + region[1].start, ys + region[0].start], axis=1).astype(np.float64))
    groups, bridges = merge_fragment_groups(fragments, cfg)
    candidates = []
    for group in groups:
        mask = np.isin(labels, [fragment_labels[i] for i in group])
        if cfg.bridge_px > 0:
            canvas = mask.astype(np.uint8)
            members = set(group)
            for i, j, end_a, end_b in bridges:
                if i in members and j in members:
                    cv2.line(canvas, tuple(int(v) for v in end_a), tuple(int(v) for v in end_b), 1, cfg.bridge_px)
            mask = canvas.astype(bool)
        candidates.append(mask)
    cores = np.logical_or.reduce(candidates) if candidates else np.zeros(probability.shape, dtype=bool)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * abs(cfg.width_px) + 1,) * 2)
    instances = []
    claimed = np.zeros(probability.shape, dtype=bool)
    for mask in candidates:
        if cfg.width_px > 0:
            mask = cv2.dilate(mask.astype(np.uint8), kernel).astype(bool) & ~(cores & ~mask)
        elif cfg.width_px < 0:
            mask = cv2.erode(mask.astype(np.uint8), kernel).astype(bool)
        mask &= ~claimed
        if mask.sum() >= cfg.minimum_area:
            instances.append(mask)
            claimed |= mask
    if cfg.yolo_split and yolo_boxes is not None and len(yolo_boxes) and instances:
        instances = split_touching_with_boxes(instances, yolo_boxes, cfg.yolo_max_box_iou, cfg.yolo_min_box_pixels)
    if not instances and cfg.force_one_row:
        mask = np.zeros(probability.shape, dtype=bool)
        y, x = np.unravel_index(int(np.argmax(probability)), probability.shape)
        mask[max(y - 1, 0):y + 2, max(x - 1, 0):x + 2] = True
        instances = [mask]
    return instances


# ---------------------------------------------------------------------------
# YOLO as a second opinion: split-only, never removes semantic pixels
# ---------------------------------------------------------------------------

def box_iou(a: np.ndarray, b: np.ndarray) -> float:
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    intersection = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - intersection
    return intersection / union if union > 0 else 0.0


def split_touching_with_boxes(instances: Sequence[np.ndarray], boxes: np.ndarray,
                              max_box_iou: float, min_box_pixels: int) -> list[np.ndarray]:
    """boxes: (K, 4) xyxy in native pixels. A semantic instance is split only when 2+ YOLO
    boxes claim it and those boxes are distinct (pairwise IoU < max_box_iou) -- the touching
    filaments case. Every pixel of the instance goes to its nearest box-exclusive seed, so the
    split never drops a semantic pixel; instances YOLO doesn't contest pass through untouched.
    # ponytail: nearest-seed (Euclidean) assignment, not geodesic -- upgrade to watershed on
    # the probability map if splits cut across a curved filament."""
    boxes = np.asarray(boxes, dtype=np.float64)
    height, width = instances[0].shape
    in_box = []
    for x0, y0, x1, y1 in boxes:
        region = np.zeros((height, width), dtype=bool)
        region[max(int(y0), 0):max(int(math.ceil(y1)), 0), max(int(x0), 0):max(int(math.ceil(x1)), 0)] = True
        in_box.append(region)
    claims = np.array([[int((mask & region).sum()) for mask in instances] for region in in_box])
    owner = {k: int(np.argmax(claims[k])) for k in range(len(boxes)) if claims[k].max() >= min_box_pixels}
    output = []
    for m, mask in enumerate(instances):
        kept: list[int] = []
        for k in sorted((k for k, o in owner.items() if o == m), key=lambda k: -claims[k, m]):
            if all(box_iou(boxes[k], boxes[j]) < max_box_iou for j in kept):
                kept.append(k)
        if len(kept) < 2:
            output.append(mask)
            continue
        markers = np.zeros(mask.shape, dtype=np.int32)
        for index, k in enumerate(kept, start=1):
            exclusive = mask & in_box[k]
            for j in kept:
                if j != k:
                    exclusive &= ~in_box[j]
            markers[exclusive] = index
        if len(np.unique(markers[markers > 0])) < len(kept):
            output.append(mask)
            continue
        _, (iy, ix) = ndimage.distance_transform_edt(markers == 0, return_indices=True)
        nearest = markers[iy, ix]
        output.extend(piece for piece in (mask & (nearest == i) for i in range(1, len(kept) + 1)) if piece.any())
    return output


def yolo_boxes(detector: Any, image_path: Path, imgsz: int = 1024, conf: float = 0.2) -> np.ndarray:
    result = detector.predict(str(image_path), imgsz=imgsz, conf=conf, verbose=False)[0]
    return result.boxes.xyxy.cpu().numpy() if result.boxes is not None else np.zeros((0, 4))


# ---------------------------------------------------------------------------
# Training (checkpoint chosen on real decoded dataset-level PQ)
# ---------------------------------------------------------------------------

def _fingerprint(cfg: SemanticCFG, train_files: Sequence[str]) -> str:
    import hashlib
    from dataclasses import asdict

    # The module's own source is part of the key, so a code change (e.g. a model fix) retrains
    # instead of reusing a checkpoint whose meta says "complete".
    code = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()
    payload = json.dumps(asdict(cfg), sort_keys=True, default=str) + "\n" + "\n".join(sorted(train_files)) + "\n" + code
    return hashlib.sha256(payload.encode()).hexdigest()


def score_files(models, filenames, image_dir, indices, cfg, device) -> pd.DataFrame:
    rows = []
    for filename in filenames:
        probability = predict_probability(models, read_gray(Path(image_dir) / filename), cfg, device)
        rows.extend(evaluate_prediction_for_file(filename, decode_instances(probability, cfg), indices, cfg))
    return pd.DataFrame(rows, columns=OOF_COLUMNS)


def _average_state_dicts(states: Sequence[Mapping[str, torch.Tensor]]) -> dict:
    """Uniform mean of float tensors; non-float entries are taken from the first.

    Valid with no BatchNorm recalibration pass because the model is GroupNorm (`ConvBlock`,
    `feature_norms`) and LayerNorm (timm ConvNeXt) throughout -- there are no running statistics
    to refresh. Accumulated in float64 so a long tail cannot drift."""
    averaged = {}
    for key, value in states[0].items():
        if value.is_floating_point():
            total = torch.zeros_like(value, dtype=torch.float64)
            for state in states:
                total += state[key].to(torch.float64)
            averaged[key] = (total / len(states)).to(value.dtype)
        else:
            averaged[key] = value.clone()
    return averaged


def train_semantic_fold(fold: int, fold_manifest: pd.DataFrame, image_dir: Path, indices: Mapping[str, Any],
                        selected_image_ids: Mapping[str, str], cfg: SemanticCFG, work_dir: Path,
                        model_factory: Callable[[SemanticCFG], nn.Module] = build_segmenter,
                        max_epochs_this_call: int | None = None) -> Path:
    """fold_manifest: the notebook's fold_manifest.csv (same split as the main pipeline, so the
    two compare head to head). Trains on fold != `fold`; keeps the epoch with the best
    dataset-level PQ on the first cfg.pq_eval_cap held-out images (a fixed prefix).

    RESUMABLE. `max_epochs_this_call` trains at most that many epochs, saves the full training
    state (model, optimizer, scheduler, history, snapshots, elapsed) and returns; calling again
    continues where it left off. This exists for one reason: Kaggle commits a cell's output only
    when the CELL finishes, so a single 9h training cell shows nothing in the log for 9h. Split
    across several cells, each chunk's epoch lines appear as that chunk completes. It also makes
    a session that hits the 12h cap resumable instead of lost.

    `cfg.max_train_hours` is the budget across ALL calls, not per call."""
    from tqdm.auto import tqdm

    train_files = fold_manifest.loc[fold_manifest["fold"] != fold, "filename"].tolist()
    val_files = fold_manifest.loc[fold_manifest["fold"] == fold, "filename"].tolist()
    if cfg.debug:
        train_files, val_files = train_files[:16], val_files[:4]
    out_dir = Path(work_dir) / "semantic_models" / f"fold_{fold}"
    out_dir.mkdir(parents=True, exist_ok=True)
    checkpoint, meta_path = out_dir / "segmenter.pt", out_dir / "segmenter_meta.json"
    fingerprint = _fingerprint(cfg, train_files)
    state_path = out_dir / "segmenter_state.pt"
    if checkpoint.exists() and meta_path.exists():
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        if meta.get("status") == "complete" and meta.get("fingerprint") == fingerprint:
            return checkpoint
    # A state file from a DIFFERENT cfg/library must never be resumed into this run.
    resume = None
    if state_path.exists():
        candidate = torch.load(state_path, map_location="cpu", weights_only=False)
        if candidate.get("fingerprint") == fingerprint:
            resume = candidate
        else:
            print(f"fold={fold} semantic: ignoring stale training state (fingerprint differs)")
            state_path.unlink()

    _resume_epoch = int(resume["next_epoch"]) if resume is not None else 0
    _seed = cfg.seed + fold + 1000 * _resume_epoch
    random.seed(_seed); np.random.seed(_seed); torch.manual_seed(_seed)
    loader = DataLoader(SemanticCropDataset(train_files, image_dir, indices, selected_image_ids, cfg, True),
                        batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, drop_last=True,
                        persistent_workers=cfg.num_workers > 0)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    # The notebook runs the YOLO/embedding sweeps on the same GPU first; drop anything they left.
    gc.collect(); torch.cuda.empty_cache()
    model = model_factory(cfg).to(device)
    # Pretrained encoder at a tenth of the decoder LR (same as the main pipeline's vit_backbone_lr_mult).
    encoder_params = set(map(id, model.encoder.parameters()))
    optimizer = torch.optim.AdamW([
        {"params": list(model.encoder.parameters()), "lr": cfg.lr * cfg.encoder_lr_mult},
        {"params": [p for p in model.parameters() if id(p) not in encoder_params], "lr": cfg.lr},
    ], weight_decay=cfg.weight_decay)
    epochs = 2 if cfg.debug else cfg.epochs
    # Warmup -> cosine to a floor -> constant tail. One LambdaLR instead of SequentialLR so the
    # shape is explicit and both param groups (encoder at 0.1x) scale together.
    warmup = min(cfg.warmup_epochs, max(epochs - 1, 0))
    tail = min(cfg.constant_tail_epochs, max(epochs - warmup - 1, 0))
    cosine_span = max(epochs - warmup - tail, 1)

    def lr_factor(e: int) -> float:
        if e < warmup:                                   # linear ramp from 0.1x
            return 0.1 + 0.9 * e / max(warmup, 1)
        if e < warmup + cosine_span:
            progress = (e - warmup) / cosine_span
            return cfg.eta_min_mult + (1.0 - cfg.eta_min_mult) * 0.5 * (1.0 + math.cos(math.pi * progress))
        return cfg.eta_min_mult                          # constant tail, harvested for SWA

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    # Input shape is fixed at input_size^2 forever, so cuDNN autotune pays back within ~10 steps.
    torch.backends.cudnn.benchmark = True
    snapshots: list[dict] = []
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda", init_scale=2.0 ** 10)
    eval_cfg = replace(cfg, use_tta=False)
    best_pq, stale, history = -1.0, 0, []
    budget_s = cfg.max_train_hours * 3600 if cfg.max_train_hours else None
    start_epoch, prior_elapsed = 0, 0.0
    if resume is not None:
        model.load_state_dict(resume["model"])
        optimizer.load_state_dict(resume["optimizer"])
        scheduler.load_state_dict(resume["scheduler"])
        scaler.load_state_dict(resume["scaler"])
        best_pq, stale, history = resume["best_pq"], resume["stale"], resume["history"]
        snapshots = resume["snapshots"]
        start_epoch, prior_elapsed = resume["next_epoch"], resume["elapsed"]
        print(f"fold={fold} semantic: resuming at epoch {start_epoch + 1}/{epochs} "
              f"(best_pq {best_pq:.5f}, {prior_elapsed / 3600:.2f}h already spent)")
    train_start = time.monotonic()
    log_path = Path(work_dir) / "training_log.txt"

    def log(message: str) -> None:
        """Both to stdout and to a file in the working directory. stdout is invisible until the
        notebook CELL ends; the file at least survives into the committed output."""
        print(message, flush=True)
        with open(log_path, "a", encoding="utf-8") as handle:
            handle.write(f"{time.strftime('%H:%M:%S')} {message}" + chr(10))

    stop_after = (start_epoch + max_epochs_this_call) if max_epochs_this_call else epochs
    for epoch in range(start_epoch, epochs):
        epoch_start = time.monotonic()
        model.train()
        losses = []
        optimizer.zero_grad(set_to_none=True)
        accumulate = max(cfg.grad_accumulation, 1)
        skipped = 0
        for step, batch in enumerate(tqdm(loader, desc=f"fold{fold} semantic e{epoch + 1}/{epochs}", leave=False), start=1):
            images, targets = batch["image"].to(device), batch["target"].to(device)
            spine = batch["spine"].to(device) if "spine" in batch else None
            components = batch["components"].to(device) if "components" in batch else None
            with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
                loss = segmentation_loss(model(images), targets, cfg, spine, components)
            if not torch.isfinite(loss):
                # Never backpropagate a non-finite loss; drop this batch's partial gradients too.
                # `del` frees the graph now; otherwise it stays alive through the next forward
                # pass and two full graphs sit on the GPU (the OOM in the second real run).
                skipped += 1
                del loss
                optimizer.zero_grad(set_to_none=True)
                continue
            scaler.scale(loss / accumulate).backward()
            if step % accumulate == 0 or step == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(set_to_none=True)
            losses.append(loss.item())
        if skipped:
            log(f"fold={fold} semantic epoch={epoch + 1}: skipped {skipped} non-finite batches")
        scheduler.step()
        model.eval()
        train_loss = float(np.mean(losses)) if losses else float("nan")
        if epoch >= epochs - tail or (tail == 0 and epoch >= epochs - cfg.swa_tail_snapshots):
            # Constant-LR snapshots: equally converged, so averaging them is well posed.
            snapshots.append({k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()})
            snapshots[:] = snapshots[-cfg.swa_tail_snapshots:]
        # Validate on a schedule, always on the tail and always on the last epoch, so a checkpoint
        # exists even if the budget guard fires early.
        do_val = ((epoch + 1) % max(cfg.val_every, 1) == 0 or epoch >= epochs - tail
                  or epoch == epochs - 1)
        if do_val:
            val = aggregate_pq(score_files([model], val_files[:cfg.pq_eval_cap], image_dir, indices, eval_cfg, device))
            history.append({"epoch": epoch + 1, "train_loss": train_loss, "skipped_batches": skipped,
                            "lr_factor": lr_factor(epoch), **val})
            log(f"fold={fold} semantic epoch={epoch + 1}/{epochs} loss={train_loss:.4f} "
                f"val_pq={val['pq']:.5f} sq={val['sq']:.4f} rq={val['rq']:.4f} "
                f"tp={val['tp']} fp={val['fp']} fn={val['fn']} "
                f"lr_mult={lr_factor(epoch):.3f} {(time.monotonic() - epoch_start) / 60:.1f}min")
            if val["pq"] > best_pq:
                best_pq, stale = val["pq"], 0
                torch.save(model.state_dict(), checkpoint)
            else:
                stale += 1
        else:
            history.append({"epoch": epoch + 1, "train_loss": train_loss, "skipped_batches": skipped,
                            "lr_factor": lr_factor(epoch)})
            log(f"fold={fold} semantic epoch={epoch + 1}/{epochs} loss={train_loss:.4f} "
                f"(no validation) lr_mult={lr_factor(epoch):.3f} "
                f"{(time.monotonic() - epoch_start) / 60:.1f}min")
        meta_path.write_text(json.dumps({"status": "in_progress", "fingerprint": fingerprint,
                                         "best_pq": best_pq, "history": history}, indent=2), encoding="utf-8")
        if stale >= cfg.early_stopping_patience:
            break
        elapsed = prior_elapsed + (time.monotonic() - train_start)
        if budget_s is not None and elapsed + (time.monotonic() - epoch_start) > budget_s:
            log(f"fold={fold} semantic: stopping after epoch {epoch + 1}/{epochs}, the next epoch "
                f"would exceed max_train_hours={cfg.max_train_hours} (elapsed {elapsed / 3600:.2f}h)")
            break
        if epoch + 1 >= stop_after and epoch + 1 < epochs:
            # Chunk boundary: save everything and hand control back so the notebook cell can end
            # and its log actually become visible.
            torch.save({"fingerprint": fingerprint, "model": model.state_dict(),
                        "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                        "scaler": scaler.state_dict(), "best_pq": best_pq, "stale": stale,
                        "history": history, "snapshots": snapshots, "next_epoch": epoch + 1,
                        "elapsed": elapsed}, state_path)
            meta_path.write_text(json.dumps({"status": "in_progress", "fingerprint": fingerprint,
                                             "best_pq": best_pq, "epochs_requested": epochs,
                                             "epochs_completed": len(history),
                                             "history": history}, indent=2), encoding="utf-8")
            log(f"fold={fold} semantic: chunk done at epoch {epoch + 1}/{epochs} "
                f"({elapsed / 3600:.2f}h of {cfg.max_train_hours}h) -- state saved, resumable")
            return checkpoint
    # SWA over the constant-LR tail. v24 kept epoch 17 and threw away epoch 23 -- the run's
    # highest tp (1283) and lowest fn (480) -- because epoch 23's calibration happened to sit badly
    # against a FIXED prob_threshold=0.60, while SQ stayed flat all run (sd 0.0057). Selecting on
    # fixed-threshold PQ ranks calibration luck, not model quality. Averaging removes the lottery.
    swa_pq = None
    if len(snapshots) >= 2:
        averaged = _average_state_dicts(snapshots)
        swa_model = model_factory(replace(cfg, encoder_pretrained=False)).to(device)
        swa_model.load_state_dict(averaged)
        swa_model.eval()
        swa_val = aggregate_pq(score_files([swa_model], val_files[:cfg.pq_eval_cap], image_dir,
                                           indices, eval_cfg, device))
        swa_pq = float(swa_val["pq"])
        print(f"fold={fold} semantic: SWA over {len(snapshots)} tail snapshots -> pq={swa_pq:.5f} "
              f"(best single epoch {best_pq:.5f})")
        if swa_pq >= best_pq:
            torch.save(averaged, checkpoint)
            best_pq = swa_pq
            print(f"fold={fold} semantic: SWA checkpoint kept")
        else:
            print(f"fold={fold} semantic: SWA did not beat the best epoch; keeping that checkpoint")
        del swa_model, averaged
        gc.collect(); torch.cuda.empty_cache()

    completed = len(history)
    annealed = completed >= epochs
    if not annealed:
        print(f"fold={fold} semantic: WARNING cosine did NOT anneal -- ran {completed}/{epochs} "
              f"epochs with T_max={epochs}. The LR never reached its floor; v20 priced this at "
              f"about -0.007 PQ. Either raise max_train_hours or set cfg.epochs={completed}.")
    meta_path.write_text(json.dumps({"status": "complete", "fingerprint": fingerprint,
                                     "best_pq": best_pq, "epochs_requested": epochs,
                                     "epochs_completed": completed, "cosine_annealed": annealed,
                                     "swa_pq": swa_pq, "swa_snapshots": len(snapshots),
                                     "history": history}, indent=2), encoding="utf-8")
    # Training is complete, so the resume state is dead weight -- and it carries the optimizer
    # (~380MB), which would otherwise be committed into the kernel output every session.
    state_path.unlink(missing_ok=True)
    return checkpoint


def load_segmenters(checkpoints: Sequence[Path], cfg: SemanticCFG, device: torch.device,
                    model_factory: Callable[[SemanticCFG], nn.Module] = build_segmenter) -> list[nn.Module]:
    """Fold ensemble: one model per checkpoint, averaged inside predict_probability."""
    loaded = []
    for path in checkpoints:
        model = model_factory(replace(cfg, encoder_pretrained=False))
        model.load_state_dict(torch.load(path, map_location=device))
        loaded.append(model.to(device).eval())
    return loaded


# ---------------------------------------------------------------------------
# OOF + free decode sweep (probability maps cached once, re-decoded per grid point)
# ---------------------------------------------------------------------------

def probability_cache_key(checkpoints: Sequence[Path], cfg: SemanticCFG) -> str:
    """Identifies what produced a probability map: the exact checkpoint bytes plus the inference
    settings that change the map (tiling, TTA). Decode-only knobs are deliberately excluded."""
    import hashlib

    digest = hashlib.sha256()
    for checkpoint in checkpoints:
        digest.update(Path(checkpoint).read_bytes())
    digest.update(f"{cfg.input_size}|{cfg.tile_size}|{cfg.tile_overlap}|{cfg.use_tta}"
                  f"|{cfg.channel_recipe}".encode())
    return digest.hexdigest()[:16]


def spine_map_path(probability_path: Path) -> Path:
    """The spine map for a cached probability map, in a SIBLING directory rather than the same one.

    Callers glob `<cache_key>/*.npy` and assert a count (141 for fold 0), so a second file per
    image inside that directory would silently corrupt every such lookup."""
    probability_path = Path(probability_path)
    return (probability_path.parent.parent / f"{probability_path.parent.name}_spine"
            / probability_path.name)


def load_cached_spine(probability_path: Path) -> np.ndarray | None:
    """None when the checkpoint had no spine head, or the cache predates spine caching."""
    path = spine_map_path(probability_path)
    return np.load(path).astype(np.float32) / 255.0 if path.exists() else None


def cache_probability_maps(models, filenames, image_dir, cfg, device, cache_dir: Path,
                           cache_key: str) -> dict[str, Path]:
    """GPU half, run once. Stored as uint8 (prob * 255) .npy on disk: a 2048^2 map is 4MB, so a
    whole fold fits on disk but not necessarily in RAM. Maps live under cache_dir/cache_key
    (see probability_cache_key), so a retrained model or new TTA setting never reuses stale maps."""
    cache_dir = Path(cache_dir) / cache_key
    cache_dir.mkdir(parents=True, exist_ok=True)
    paths = {}
    for filename in filenames:
        path = cache_dir / f"{Path(filename).stem}.npy"
        if not path.exists():
            probability, spine = predict_maps(models, read_gray(Path(image_dir) / filename), cfg, device)
            np.save(path, np.round(probability * 255).astype(np.uint8))
            if spine is not None:
                spine_path = spine_map_path(path)
                spine_path.parent.mkdir(parents=True, exist_ok=True)
                np.save(spine_path, np.round(spine * 255).astype(np.uint8))
        paths[filename] = path
    return paths


def load_cached_probability(path: Path) -> np.ndarray:
    return np.load(path).astype(np.float32) / 255.0


def run_oof_semantic(cache_paths: Mapping[str, Path], indices, cfg: SemanticCFG) -> pd.DataFrame:
    """Rows in exactly the shared OOF contract columns; score them with aggregate_pq."""
    from tqdm.auto import tqdm

    rows = []
    for filename, path in tqdm(cache_paths.items(), total=len(cache_paths),
                               desc="oof decode", leave=False):
        masks = decode_instances(load_cached_probability(path), cfg)
        rows.extend(evaluate_prediction_for_file(filename, masks, indices, cfg))
    return pd.DataFrame(rows, columns=OOF_COLUMNS)


def sweep_semantic_decode(cache_paths: Mapping[str, Path], indices, cfg: SemanticCFG,
                          grid: Mapping[str, Sequence[Any]]) -> pd.DataFrame:
    """Full product over `grid` (keys = SemanticCFG decode fields: prob_threshold, max_gap_px,
    max_angle_deg, minimum_area, bridge_px, ...). Each cached map is loaded once and decoded at
    every grid point; ranked by dataset-level PQ."""
    import itertools

    keys = list(grid)
    combos = [dict(zip(keys, values)) for values in itertools.product(*(grid[key] for key in keys))]
    combo_cfgs = [replace(cfg, **combo) for combo in combos]
    rows_per_combo: list[list[dict]] = [[] for _ in combos]
    for filename, path in cache_paths.items():
        probability = load_cached_probability(path)
        for rows, combo_cfg in zip(rows_per_combo, combo_cfgs):
            rows.extend(evaluate_prediction_for_file(filename, decode_instances(probability, combo_cfg), indices, combo_cfg))
    results = [{**combo, **aggregate_pq(pd.DataFrame(rows, columns=OOF_COLUMNS))}
               for combo, rows in zip(combos, rows_per_combo)]
    return pd.DataFrame(results).sort_values("pq", ascending=False).reset_index(drop=True)


def _as_field_type(value: Any, reference: Any) -> Any:
    """pandas hands back numpy scalars (and NaN for None); cast to the CFG field's own type."""
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    if isinstance(reference, bool):
        return bool(value)
    if isinstance(reference, int):
        return int(value)
    return float(value)


def coordinate_sweep_semantic_decode(cache_paths: Mapping[str, Path], indices, cfg: SemanticCFG,
                                     grid: Mapping[str, Sequence[Any]], rounds: int = 2
                                     ) -> tuple[SemanticCFG, pd.DataFrame]:
    """One knob at a time, holding the others at their current best, repeated `rounds` times.
    Costs sum(len(values)) decodes per image per round, instead of the full product's
    prod(len(values)), so many knobs fit in a session. Returns the best cfg and the history."""
    current, history = cfg, []
    for round_index in range(rounds):
        for key, values in grid.items():
            reference = getattr(cfg, key) if getattr(cfg, key) is not None else 0.0
            now = getattr(current, key)
            candidates = list(values) if any(_as_field_type(v, reference) == now for v in values) else [now, *values]
            ranked = sweep_semantic_decode(cache_paths, indices, current, {key: candidates})
            # A plain list, not a DataFrame column: pandas would turn None back into NaN.
            ranked_values = [_as_field_type(v, reference) for v in ranked[key]]
            ranked_pq = [float(p) for p in ranked["pq"]]
            current_pq = next(p for v, p in zip(ranked_values, ranked_pq) if v == now)
            # Change a knob only on a strict gain: on a tie keep the current value, so a flat
            # stretch of the grid doesn't silently switch things off (e.g. fragment merging).
            if ranked_pq[0] > current_pq + 1e-12:
                current = replace(current, **{key: ranked_values[0]})
            history.append({"round": round_index + 1, "key": key, "value": getattr(current, key),
                            "pq": max(ranked_pq[0], current_pq)})
    return current, pd.DataFrame(history)


# ---------------------------------------------------------------------------
# Instance filter: drop instances unlikely to match. Under PQ a near-miss costs 1.0 in the
# denominator (one FP plus the FN that remains) while staying silent costs 0.5, so emitting only
# pays off above roughly a one-in-three chance of matching.
# ---------------------------------------------------------------------------

FILTER_FEATURES = ["area", "length", "width", "elongation", "n_parts",
                   "prob_mean", "prob_max", "prob_p90", "contrast", "radius"]

# Derived from the spine head, which is an axis independent of `probability`. Measured on fold 0
# (1194 instances, 2026-10-11): `s_mean` alone separates matched from unmatched instances at
# AUC 0.778 -- better than any mask feature, `contrast` included -- and adding all five lifts the
# filter's own out-of-fold AUC from 0.773 to 0.795.
SPINE_FILTER_FEATURES = ["s_mean", "s_max", "s_p90", "s_rel", "s_cover"]


def filter_feature_names(table_or_columns) -> list[str]:
    """Mask features always; spine features only when they are present, so one-channel
    checkpoints and pre-spine caches keep working unchanged."""
    columns = (table_or_columns.columns if hasattr(table_or_columns, "columns")
               else table_or_columns)
    return [*FILTER_FEATURES, *[f for f in SPINE_FILTER_FEATURES if f in columns]]


def instance_features(probability: np.ndarray, mask: np.ndarray, spine: np.ndarray | None = None,
                      spine_max: float | None = None) -> dict[str, float]:
    """Shape and confidence features for one instance, computed on its bounding box.

    With `spine`, adds five features asking "does a filament spine run through this blob" rather
    than "is this blob bright" -- a genuinely independent axis from `probability`."""
    ys, xs = np.nonzero(mask)
    height, width = mask.shape
    y0, y1 = max(ys.min() - 3, 0), min(ys.max() + 4, height)
    x0, x1 = max(xs.min() - 3, 0), min(xs.max() + 4, width)
    sub, prob = mask[y0:y1, x0:x1].astype(np.uint8), probability[y0:y1, x0:x1]
    inside = prob[sub.astype(bool)]
    coords = np.stack([xs, ys], axis=1).astype(np.float64)
    coords -= coords.mean(axis=0)
    eig = np.sort(np.linalg.eigvalsh(coords.T @ coords / max(len(coords) - 1, 1)))[::-1]
    contours, _ = cv2.findContours(sub, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    perimeter = sum(cv2.arcLength(contour, True) for contour in contours)
    ring = cv2.dilate(sub, np.ones((5, 5), np.uint8)).astype(bool) & ~sub.astype(bool)
    features = {
        "area": float(len(ys)),
        "length": float(4.0 * math.sqrt(max(eig[0], 0.0))),
        "width": float(2.0 * len(ys) / max(perimeter, 1.0)),  # area / half-perimeter: a ribbon's width
        "elongation": float(math.sqrt((eig[0] + 1e-6) / (eig[1] + 1e-6))),
        "n_parts": float(cv2.connectedComponents(sub, connectivity=8)[0] - 1),
        "prob_mean": float(inside.mean()),
        "prob_max": float(inside.max()),
        "prob_p90": float(np.percentile(inside, 90)),
        "contrast": float(inside.mean() - (prob[ring].mean() if ring.any() else 0.0)),
        "radius": float(math.hypot(ys.mean() - height / 2, xs.mean() - width / 2) / (min(height, width) / 2)),
    }
    if spine is not None:
        inside_spine = spine[mask.astype(bool)]
        peak = float(spine_max if spine_max is not None else spine.max())
        features.update({
            "s_mean": float(inside_spine.mean()),
            "s_max": float(inside_spine.max()),
            "s_p90": float(np.percentile(inside_spine, 90)),
            # Relative, not absolute: the spine head peaks near 0.44 and never saturates (the mask
            # head reaches 0.9999), so a threshold copied from the mask head would select nothing.
            "s_rel": float(inside_spine.max() / max(peak, 1e-6)),
            "s_cover": float((inside_spine >= 0.5 * peak).mean()),
        })
    return features


def oof_instance_table(cache_paths: Mapping[str, Path], indices, cfg: SemanticCFG) -> pd.DataFrame:
    """One row per decoded OOF instance: its features, plus `match` = the fraction of the image's
    annotators that have a ground-truth instance with IoU > pq_match_iou against it."""
    rows = []
    for filename, path in cache_paths.items():
        probability = load_cached_probability(path)
        spine = load_cached_spine(path)
        spine_max = float(spine.max()) if spine is not None else None
        instances = decode_instances(probability, cfg)
        if not instances:
            continue
        gt_sets = ground_truth_sets_for_file(filename, indices)
        matched = np.zeros(len(instances))
        for gt_masks in gt_sets:
            if gt_masks:
                matched += (pairwise_iou(gt_masks, instances).max(axis=0) > cfg.pq_match_iou)
        for index, mask in enumerate(instances):
            rows.append({"filename": filename, "instance": index,
                         "match": float(matched[index] / max(len(gt_sets), 1)),
                         **instance_features(probability, mask, spine, spine_max)})
    names = filter_feature_names(rows[0]) if rows else FILTER_FEATURES
    return pd.DataFrame(rows, columns=["filename", "instance", "match", *names])


def fit_instance_filter(table: pd.DataFrame, seed: int = 0,
                        features: Sequence[str] | None = None):
    from sklearn.ensemble import HistGradientBoostingRegressor

    names = list(features) if features is not None else filter_feature_names(table)
    model = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
                                          min_samples_leaf=20, random_state=seed)
    return model.fit(table[names], table["match"])


def predict_instance_scores(model, features: pd.DataFrame,
                            feature_names: Sequence[str] | None = None) -> np.ndarray:
    """`model` may be one estimator or a list of them; a list is averaged. Averaging the
    GroupKFold models is how the filter is deployed: each saw (K-1)/K of the data, exactly like
    the out-of-fold scores the threshold was calibrated on. Refitting one model on the whole
    table instead yields systematically higher scores, so the deployed filter drops fewer
    instances than calibrated."""
    names = list(feature_names) if feature_names is not None else FILTER_FEATURES
    estimators = model if isinstance(model, (list, tuple)) else [model]
    return np.clip(np.mean([est.predict(features[names]) for est in estimators], axis=0), 0.0, 1.0)


def cv_instance_scores(table: pd.DataFrame, n_splits: int = 5, seed: int = 0,
                       features: Sequence[str] | None = None) -> tuple[np.ndarray, list]:
    """Out-of-fold filter scores AND the fold models, grouped by observation DATE.

    Date, not filename: GONG images from the same day show the same filaments in nearly the same
    place, so grouping by filename alone leaves a real leak. (Measured on fold 0 it barely moves
    the number -- 133 dates for 138 images -- but it is free and correct.)

    The models are returned because averaging them is what makes the deployed score distribution
    match these out-of-fold scores; see `predict_instance_scores`."""
    from sklearn.model_selection import GroupKFold

    names = list(features) if features is not None else filter_feature_names(table)
    groups = table["filename"].astype(str).str[:8]
    scores = np.zeros(len(table))
    models = []
    splits = min(n_splits, groups.nunique())
    for train_index, test_index in GroupKFold(n_splits=splits).split(table, groups=groups):
        model = fit_instance_filter(table.iloc[train_index], seed, names)
        models.append(model)
        scores[test_index] = predict_instance_scores(model, table.iloc[test_index], names)
    return scores, models


def score_filter_thresholds(cache_paths: Mapping[str, Path], indices, cfg: SemanticCFG, table: pd.DataFrame,
                            scores: np.ndarray, thresholds: Sequence[float]) -> pd.DataFrame:
    """Pooled PQ when instances scoring below each threshold are dropped. Threshold 0 keeps
    everything, i.e. the unfiltered PQ."""
    by_image: dict[str, np.ndarray] = {}
    for filename, frame in table.assign(score=scores).groupby("filename"):
        by_image[filename] = frame.sort_values("instance")["score"].to_numpy()
    rows_per_threshold: dict[float, list[dict]] = {t: [] for t in thresholds}
    for filename, path in cache_paths.items():
        instances = decode_instances(load_cached_probability(path), cfg)
        image_scores = by_image.get(filename, np.zeros(0))
        for threshold, rows in rows_per_threshold.items():
            kept = [mask for mask, score in zip(instances, image_scores) if score >= threshold]
            rows.extend(evaluate_prediction_for_file(filename, kept, indices, cfg))
    results = [{"threshold": t, **aggregate_pq(pd.DataFrame(rows, columns=OOF_COLUMNS))}
               for t, rows in rows_per_threshold.items()]
    return pd.DataFrame(results).sort_values("pq", ascending=False).reset_index(drop=True)


def apply_instance_filter(probability: np.ndarray, instances: list[np.ndarray],
                          instance_filter: tuple | None,
                          spine: np.ndarray | None = None) -> list[np.ndarray]:
    """instance_filter = (model_or_models, threshold[, feature_names]), or None to keep every
    instance. A 2-tuple keeps the pre-spine behaviour (mask features only)."""
    if instance_filter is None or not instances:
        return instances
    model, threshold = instance_filter[0], instance_filter[1]
    names = list(instance_filter[2]) if len(instance_filter) > 2 else FILTER_FEATURES
    spine_max = float(spine.max()) if spine is not None else None
    features = pd.DataFrame([instance_features(probability, mask, spine, spine_max)
                             for mask in instances])
    scores = predict_instance_scores(model, features, names)
    return [mask for mask, score in zip(instances, scores) if score >= threshold]


# ---------------------------------------------------------------------------
# Test inference + submission
# ---------------------------------------------------------------------------

def write_submission(test_dir: Path, models, cfg: SemanticCFG, out_csv: Path, device: torch.device,
                     detector: Any = None, instance_filter: tuple | None = None) -> pd.DataFrame:
    """Same format as the notebook: filament_id = <image_stem>_<k>, segmentation_rle =
    pycocotools compressed counts. An image with no surviving instance gets no rows unless
    cfg.force_one_row is set -- check the competition's submission spec before relying on either.
    instance_filter = (model, threshold) from fit_instance_filter, or None."""
    rows = []
    test_files = sorted(path for path in Path(test_dir).iterdir() if path.suffix.lower() == ".jpeg")
    for image_path in test_files:
        probability, spine = predict_maps(models, read_gray(image_path), cfg, device)
        boxes = yolo_boxes(detector, image_path) if detector is not None and cfg.yolo_split else None
        instances = apply_instance_filter(probability, decode_instances(probability, cfg, boxes),
                                          instance_filter, spine)
        for k, mask in enumerate(instances, start=1):
            rows.append({"filament_id": f"{image_path.stem}_{k}", "segmentation_rle": encode_submission_mask(mask)})
    submission = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
    if not submission["filament_id"].is_unique:
        raise RuntimeError("Duplicate filament_id in submission")
    submission.to_csv(out_csv, index=False)
    return submission


Writing semantic_filament_lib.py


### Session 4 search layer

`semantic_search.py`: the paired-delta bootstrap, the near-miss diagnostic, block decode search,
SWA and greedy ensemble selection. A separate module because
`semantic_filament_lib._fingerprint` hashes its own file's bytes -- editing that file would
invalidate every trained checkpoint and force a retrain. Nothing here touches it.

In [11]:
%%writefile semantic_search.py
"""Search, ensembling and honest-validation layer for the semantic filament pipeline.

A separate module on purpose: `semantic_filament_lib._fingerprint` hashes that file's own bytes,
so editing it would invalidate every trained checkpoint and force a retrain. Nothing here touches
it -- these functions only consume its cached probability maps and its scorer.

Why this module exists at all (PROGRESS.md, run 3): the decode sweep evaluated 113 configs on 141
images and reported +0.009. The expected maximum of n noisy draws is about sqrt(2 ln n) * sigma;
with sigma ~ 0.018 for pooled PQ on this fold that is ~2.4 sigma ~ +0.043 of optimism. The
reported gain was SMALLER than the noise the procedure manufactures, which makes it
uninterpretable. So every function here either (a) measures a gain on data the search never saw,
or (b) spends fewer draws to find the same optimum.
"""

from __future__ import annotations

import math
import os
import shutil
from dataclasses import replace
from pathlib import Path
from typing import Any, Mapping, Sequence

import numpy as np
import pandas as pd

from solar_filament_lib import (
    aggregate_pq,
    classify_failure_modes,
    evaluate_prediction_for_file,
    ground_truth_sets_for_file,
    pairwise_iou,
)
from semantic_filament_lib import (
    OOF_COLUMNS,
    SemanticCFG,
    _as_field_type,
    decode_instances,
    load_cached_probability,
    sweep_semantic_decode,
)

# ---------------------------------------------------------------------------
# Honest measurement: a PAIRED delta, not two independent intervals
# ---------------------------------------------------------------------------


def bootstrap_pq_delta_ci(rows_a: pd.DataFrame, rows_b: pd.DataFrame, n: int = 2000,
                          seed: int = 0) -> tuple[float, float, float]:
    """95% interval and P(delta > 0) for pooled PQ(a) - PQ(b), resampling IMAGES with replacement.

    The one index draw per replicate is applied to BOTH sides, so per-image difficulty cancels.
    Comparing two independent `bootstrap_pq_ci` intervals instead overstates a within-model
    delta's uncertainty by several times -- that is why run 3's +0.009 looked indistinguishable
    from zero and was never actually tested.

    Both frames must cover the same filenames (they score the same images under two configs).
    """
    columns = ["sum_iou", "tp", "fp", "fn"]
    per_a = rows_a.groupby("filename")[columns].sum()
    per_b = rows_b.groupby("filename")[columns].sum()
    if list(per_a.index) != list(per_b.index):
        shared = per_a.index.intersection(per_b.index)
        if len(shared) == 0:
            raise ValueError("rows_a and rows_b share no filenames")
        per_a, per_b = per_a.loc[shared], per_b.loc[shared]
    a, b = per_a.to_numpy(dtype=float), per_b.to_numpy(dtype=float)

    rng = np.random.default_rng(seed)
    index = rng.integers(0, len(a), size=(n, len(a)))

    def pooled(table: np.ndarray) -> np.ndarray:
        sums = table[index].sum(axis=1)
        denominator = sums[:, 1] + 0.5 * sums[:, 2] + 0.5 * sums[:, 3]
        return np.divide(sums[:, 0], denominator, out=np.ones(n), where=denominator > 0)

    delta = pooled(a) - pooled(b)
    low, high = np.percentile(delta, [2.5, 97.5])
    return float(low), float(high), float((delta > 0).mean())


def keep_change(rows_new: pd.DataFrame, rows_old: pd.DataFrame, min_gain: float = 0.010,
                min_probability: float = 0.95, seed: int = 0) -> tuple[bool, dict[str, float]]:
    """The project's keep rule in one place: adopt a change only if the paired delta on data the
    search never touched is both significant and big enough to matter. Below ~0.008 the paired
    estimate cannot resolve anything on a single fold, so a smaller 'gain' is noise by
    construction."""
    low, high, probability = bootstrap_pq_delta_ci(rows_new, rows_old, seed=seed)
    delta = aggregate_pq(rows_new)["pq"] - aggregate_pq(rows_old)["pq"]
    verdict = bool(probability >= min_probability and delta >= min_gain)
    return verdict, {"delta": float(delta), "ci_low": low, "ci_high": high,
                     "p_gain": probability, "keep": float(verdict)}


def split_images(filenames: Sequence[str], fraction: float = 0.5) -> tuple[list[str], list[str]]:
    """Deterministic search/confirm split by filename hash -- stable across sessions and
    independent of ordering, so the confirm half is never silently contaminated by a reordering.
    Uses md5, not hash(): Python's hash is salted per process."""
    import hashlib

    unique = sorted(set(filenames))
    # Rank by hash, then cut at the quantile, rather than thresholding the hash directly: a
    # direct threshold is only balanced in expectation and drifts badly on small sets (2/6 on
    # eight images in testing). Ranking gives exactly the requested proportion at any size while
    # staying deterministic and order-independent.
    ordered = sorted(unique, key=lambda name: hashlib.md5(name.encode()).hexdigest())
    cut = round(fraction * len(ordered))
    search = sorted(ordered[:cut])
    confirm = sorted(ordered[cut:])
    return search, confirm


# ---------------------------------------------------------------------------
# Cross-session restore. Kaggle stops at 12h, so a later session attaches an earlier one's
# output; `resolve_stage1_checkpoint` only knows `detector.pt` under `models/fold_N` and cannot
# find the semantic layout.
# ---------------------------------------------------------------------------


def find_attached_directory(name: str, work_dir: Path,
                            search_root: Path = Path("/kaggle/input")) -> tuple[Path | None, list[Path]]:
    """Locate a directory called `name` in an attached Kaggle input. Returns (chosen, candidates).

    Where an attached notebook output lands depends on how it was added, so matching on the full
    path is brittle. Instead: find every candidate, then prefer the ones sitting under a parent
    named like our working directory (`solar_filament_pq/semantic_models`, the layout this
    project writes). A unique best-tier match wins; ties return None rather than guessing,
    because silently restoring the wrong checkpoint is the failure mode PROGRESS.md records.
    """
    if not search_root.exists():
        return None, []
    candidates = sorted({p for p in search_root.rglob(name) if p.is_dir()})
    preferred = [p for p in candidates if p.parent.name == work_dir.name]
    tier = preferred or candidates
    return (tier[0] if len(tier) == 1 else None), candidates


def restore_attached_artifacts(names: Sequence[str], work_dir: Path,
                               search_root: Path = Path("/kaggle/input")) -> dict[str, str]:
    """Copy each named directory from an attached input into `work_dir`, reporting what happened.
    Never overwrites something already in the working directory."""
    import shutil

    report = {}
    for name in names:
        target = work_dir / name
        if target.exists():
            report[name] = "already in working directory"
            continue
        chosen, candidates = find_attached_directory(name, work_dir, search_root)
        if chosen is None:
            report[name] = (f"NOT RESTORED: {len(candidates)} candidates, refusing to guess "
                            f"({[str(p) for p in candidates[:4]]})" if candidates
                            else "not found in attached inputs")
            continue
        shutil.copytree(chosen, target)
        report[name] = f"restored from {chosen}"
    return report


# ---------------------------------------------------------------------------
# Diagnostic: WHY the score is lost. Run before spending anything.
# ---------------------------------------------------------------------------


def semantic_failure_breakdown(cache_paths: Mapping[str, Path], indices: Mapping[str, Any],
                               cfg: SemanticCFG, overlap_floor: float = 0.05
                               ) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Why RQ is low, from cached probability maps. Returns (counts, pairs).

    counts: one row per (filename, annotator) with `classify_failure_modes`' four buckets --
      fully_missed_fn (nothing overlapped this filament at all) vs near_miss_fn (something did,
      below the IoU threshold), spurious_fp vs near_miss_fp -- plus n_gt / n_pred.

    pairs: one row per GT instance -- gt_area, best_iou, and n_pred_overlapping (predictions with
      IoU > overlap_floor against it). This frame, not the counts, decides the plan: a mean
      n_pred_overlapping well above 1 means one filament is being decoded as several (oversplit,
      fixable with the merge knobs), while ~1.0 with near_miss_fp >> near_miss_fn means the
      opposite. Under PQ a near miss is the expensive case -- converting one adds a TP and
      removes an FP AND an FN at once.

    Each cached map is decoded ONCE and the masks reused across that image's annotators.
    """
    count_rows, pair_rows = [], []
    for filename, path in cache_paths.items():
        predictions = decode_instances(load_cached_probability(path), cfg)
        for annotator, gt_masks in enumerate(ground_truth_sets_for_file(filename, indices)):
            count_rows.append({
                "filename": filename, "annotator": annotator,
                **classify_failure_modes(gt_masks, predictions, cfg.pq_match_iou),
                "n_gt": len(gt_masks), "n_pred": len(predictions),
            })
            if not gt_masks:
                continue
            iou = (pairwise_iou(gt_masks, predictions) if predictions
                   else np.zeros((len(gt_masks), 0), dtype=float))
            for gt_index, gt_mask in enumerate(gt_masks):
                row = iou[gt_index] if iou.shape[1] else np.zeros(0)
                pair_rows.append({
                    "filename": filename, "annotator": annotator, "gt_index": gt_index,
                    "gt_area": int(gt_mask.sum()),
                    "best_iou": float(row.max()) if row.size else 0.0,
                    "n_pred_overlapping": int((row > overlap_floor).sum()),
                })
    return pd.DataFrame(count_rows), pd.DataFrame(pair_rows)


def summarise_failures(counts: pd.DataFrame, pairs: pd.DataFrame,
                       match_iou: float = 0.50) -> dict[str, float]:
    """The handful of numbers the plan's decision table keys on."""
    near = pairs[(pairs["best_iou"] > 0) & (pairs["best_iou"] <= match_iou)]
    matched = pairs[pairs["best_iou"] > match_iou]
    missed = pairs[pairs["best_iou"] == 0]
    touched = pairs[pairs["best_iou"] > 0]
    buckets = ["fully_missed_fn", "near_miss_fn", "spurious_fp", "near_miss_fp"]
    return {
        **{name: int(counts[name].sum()) for name in buckets if name in counts},
        "near_miss_median_iou": float(near["best_iou"].median()) if len(near) else 0.0,
        "missed_median_area": float(missed["gt_area"].median()) if len(missed) else 0.0,
        # The median says nothing about `minimum_area`: a filter at 160px cannot explain misses
        # whose median area is 646. Only the lower tail can, so report it and compare THAT.
        "missed_area_p10": float(missed["gt_area"].quantile(0.10)) if len(missed) else 0.0,
        "matched_median_area": float(matched["gt_area"].median()) if len(matched) else 0.0,
        "mean_pred_per_touched_gt": float(touched["n_pred_overlapping"].mean()) if len(touched) else 0.0,
    }


def ribbon_extent(mask: np.ndarray) -> tuple[float, float]:
    """(mean width, length) of a thin mask, in pixels.

    Across a strip of odd width w, cv2's distance transform reads 1, 2, ... (w+1)/2, ... 2, 1 --
    it counts the step onto background, so the continuous w = 4 * mean(DT) overestimates by
    about a pixel (a 9px strip reads 11.1). Those values sum to ((w+1)/2)^2, so the exact
    inverse of mean(DT) = d is w = (2d-1) + sqrt((2d-1)^2 - 1), and length follows from area / w.
    No skeletonisation, so no new dependency and no branch-point bookkeeping. The mask is padded
    by one pixel first, because cv2 measures distance to the nearest zero *inside the image* and
    would otherwise report a filament touching the border as far wider than it is.

    ponytail: exact only for a straight strip, approximate for strongly curved or branched
    shapes -- both sides of a comparison use the same estimator, which is all the ratios below
    need. Swap in a real skeleton length if absolute widths ever have to be trusted.
    """
    import cv2

    binary = np.ascontiguousarray(mask.astype(np.uint8))
    area = float(binary.sum())
    if area == 0:
        return 0.0, 0.0
    distance = cv2.distanceTransform(np.pad(binary, 1), cv2.DIST_L2, 5)[1:-1, 1:-1]
    half = 2.0 * float(distance[binary > 0].mean()) - 1.0
    width = half + math.sqrt(max(half * half - 1.0, 0.0))
    return width, area / max(width, 1e-6)


NEAR_MISS_EXTENT_COLUMNS = [
    "filename", "annotator", "gt_index", "best_iou",
    "gt_area", "pred_area", "gt_width", "pred_width", "width_ratio",
    "gt_length", "pred_length", "length_ratio",
]


def near_miss_extent(cache_paths: Mapping[str, Path], indices: Mapping[str, Any],
                     cfg: SemanticCFG, overlap_floor: float = 0.05) -> pd.DataFrame:
    """One row per near-miss GT filament: how its best-overlapping prediction's extent differs.

    `semantic_failure_breakdown` says HOW MANY near misses there are; this says WHY each one
    falls short of IoU > 0.5. Run 3 measured ~1.07 predictions per touched GT, so the miss is
    NOT fragmentation, which leaves two causes with completely different fixes:

      width_ratio off, length_ratio ~1  -> the prediction traces the whole filament at the wrong
        thickness. Fixable from these same cached maps (`width_px`, `prob_threshold`); run 3 only
        ever swept width_px over [-1, 0, 1], about +-12% of the 8.31px median annotation width,
        so a 2-4px deficit was outside the grid.
      length_ratio off, width_ratio ~1  -> the prediction is truncated. No width knob recovers
        it; the levers are hysteresis (faint continuations stay attached to strong cores) or a
        larger `cldice_weight`, which needs a retrain.

    IoU for a ribbon is roughly the smaller extent over the larger, so a median best_iou of 0.36
    implies one of these ratios sits near 0.36 as well. Whichever it is names the lever.
    """
    rows = []
    for filename, path in cache_paths.items():
        predictions = decode_instances(load_cached_probability(path), cfg)
        if not predictions:
            continue
        for annotator, gt_masks in enumerate(ground_truth_sets_for_file(filename, indices)):
            if not gt_masks:
                continue
            iou = pairwise_iou(gt_masks, predictions)
            for gt_index, gt_mask in enumerate(gt_masks):
                best = int(iou[gt_index].argmax())
                best_iou = float(iou[gt_index, best])
                if not overlap_floor < best_iou <= cfg.pq_match_iou:
                    continue
                gt_width, gt_length = ribbon_extent(gt_mask)
                pred_width, pred_length = ribbon_extent(predictions[best])
                rows.append({
                    "filename": filename, "annotator": annotator, "gt_index": gt_index,
                    "best_iou": best_iou,
                    "gt_area": int(gt_mask.sum()), "pred_area": int(predictions[best].sum()),
                    "gt_width": gt_width, "pred_width": pred_width,
                    "width_ratio": pred_width / max(gt_width, 1e-6),
                    "gt_length": gt_length, "pred_length": pred_length,
                    "length_ratio": pred_length / max(gt_length, 1e-6),
                })
    return pd.DataFrame(rows, columns=NEAR_MISS_EXTENT_COLUMNS)


# ---------------------------------------------------------------------------
# Decode search: blocks of COUPLED knobs, so a ridge is traversable in one move
# ---------------------------------------------------------------------------


_WORKER: dict[str, Any] = {}


def _init_worker(indices: Mapping[str, Any], combo_cfgs: Sequence[SemanticCFG]) -> None:
    import cv2

    cv2.setNumThreads(1)  # each worker gets one core; OpenCV's own pool would oversubscribe
    _WORKER["indices"], _WORKER["cfgs"] = indices, combo_cfgs


def _score_one(item: tuple[str, Path]) -> list[list[dict]]:
    filename, path = item
    probability = load_cached_probability(path)
    return [evaluate_prediction_for_file(filename, decode_instances(probability, combo_cfg),
                                         _WORKER["indices"], combo_cfg)
            for combo_cfg in _WORKER["cfgs"]]


def sweep_decode(cache_paths: Mapping[str, Path], indices: Mapping[str, Any], cfg: SemanticCFG,
                 grid: Mapping[str, Sequence[Any]], workers: int = 1) -> pd.DataFrame:
    """`sweep_semantic_decode` with optional multiprocessing over images.

    Identical output to the serial version (a test pins this). Parallel because the real grids
    here are ~113 configs and decode is ~0.3s per 2048 image: that is hours on one core, and
    Kaggle gives four while the GPU is busy training. `indices` and the config list go through
    the pool initialiser, so they are pickled once per worker rather than once per image.
    """
    import itertools

    keys = list(grid)
    combos = [dict(zip(keys, values)) for values in itertools.product(*(grid[key] for key in keys))]
    combo_cfgs = [replace(cfg, **combo) for combo in combos]
    if workers <= 1:
        return sweep_semantic_decode(cache_paths, indices, cfg, grid)

    import multiprocessing as mp

    rows_per_combo: list[list[dict]] = [[] for _ in combos]
    items = list(cache_paths.items())
    from tqdm.auto import tqdm

    # spawn, not the platform default. On Linux that default is fork, and by the time this runs the
    # caller has already decoded a full pass of images, which starts torch's and OpenCV's OpenMP
    # thread pools in the parent. fork copies those pools' locks without the threads that hold them,
    # so every child deadlocks on its first parallel region and the pool never returns a single
    # result -- two Kaggle runs sat at block 1/4 for hours this way. spawn starts clean interpreters;
    # `_init_worker` and `_score_one` are module-level, so they pickle, and the child inherits
    # sys.path and re-imports this module normally.
    ctx = mp.get_context("spawn")
    with ctx.Pool(workers, initializer=_init_worker, initargs=(indices, combo_cfgs)) as pool:
        for per_combo in tqdm(pool.imap_unordered(_score_one, items, chunksize=1),
                              total=len(items), desc=f"sweep {len(combos)} cfg", leave=False):
            for rows, produced in zip(rows_per_combo, per_combo):
                rows.extend(produced)
    results = [{**combo, **aggregate_pq(pd.DataFrame(rows, columns=OOF_COLUMNS))}
               for combo, rows in zip(combos, rows_per_combo)]
    return pd.DataFrame(results).sort_values("pq", ascending=False).reset_index(drop=True)


def _matches(value: Any, current: Any) -> bool:
    if value is None or current is None:
        return value is None and current is None
    if isinstance(value, float) and math.isnan(value):
        return current is None
    return bool(value == current)


def block_sweep_decode(cache_paths: Mapping[str, Path], indices: Mapping[str, Any],
                       cfg: SemanticCFG, blocks: Sequence[Mapping[str, Sequence[Any]]],
                       rounds: int = 1, workers: int = 1) -> tuple[SemanticCFG, pd.DataFrame]:
    """Coordinate descent over BLOCKS of coupled knobs instead of single knobs.

    `coordinate_sweep_semantic_decode` moves one knob at a time, which cannot traverse a ridge
    between two coupled knobs: it walks to the ridge and stops, reporting convergence. Run 3 hit
    exactly that -- `max_angle_deg` and `minimum_area` both stopped at their grid maximum
    together (the ridge signature), and round 2 moved nothing. Each block here is a small full
    product, so both knobs move in one step.

    Same strict-gain rule as the single-knob version (`> current + 1e-12`, ties keep the
    incumbent) and the same current-value injection, so the incumbent is always in the running.
    """
    current, history = cfg, []
    import time as _time

    for round_index in range(rounds):
        for block_index, block in enumerate(blocks):
            _t_block = _time.perf_counter()
            print(f"[sweep] round {round_index + 1}/{rounds} block {block_index + 1}/{len(blocks)} "
                  f"{ {k: len(v) for k, v in block.items()} }", flush=True)
            grid = {}
            for key, values in block.items():
                now = getattr(current, key)
                candidates = list(values)
                if not any(_matches(_as_field_type(v, getattr(cfg, key) or 0.0), now) for v in candidates):
                    candidates = [now, *candidates]
                grid[key] = candidates
            ranked = sweep_decode(cache_paths, indices, current, grid, workers=workers)
            keep = np.ones(len(ranked), dtype=bool)
            for key in grid:
                reference = getattr(cfg, key) if getattr(cfg, key) is not None else 0.0
                column = [_as_field_type(v, reference) for v in ranked[key]]
                keep &= np.array([_matches(v, getattr(current, key)) for v in column])
            current_pq = float(ranked.loc[keep, "pq"].iloc[0]) if keep.any() else -1.0
            best_pq = float(ranked["pq"].iloc[0])
            if best_pq > current_pq + 1e-12:
                winner = {key: _as_field_type(ranked[key].iloc[0],
                                              getattr(cfg, key) if getattr(cfg, key) is not None else 0.0)
                          for key in grid}
                current = replace(current, **winner)
            history.append({"round": round_index + 1, "block": block_index,
                            "keys": ",".join(grid), "pq": max(best_pq, current_pq),
                            **{key: getattr(current, key) for key in grid}})
    return current, pd.DataFrame(history)


# ---------------------------------------------------------------------------
# Ensembling. SWA first -- this model has no BatchNorm, so no recalibration pass is needed.
# ---------------------------------------------------------------------------


def average_state_dicts(paths: Sequence[Path]) -> dict:
    """Uniform mean of float tensors across checkpoints (stochastic weight averaging).

    Valid without a BatchNorm re-estimation pass ONLY because this model has no running
    statistics anywhere: `ConvBlock` and `feature_norms` are GroupNorm by deliberate design (see
    semantic_filament_lib) and timm's ConvNeXt is LayerNorm. Non-float entries are taken from the
    first checkpoint unchanged.

    One averaged checkpoint costs one cache and one test-time inference pass, where averaging K
    predictions costs K of each -- and it has no selection variance at all.
    """
    import torch

    if not paths:
        raise ValueError("no checkpoints to average")
    total = {k: (v.clone().float() if v.is_floating_point() else v.clone())
             for k, v in torch.load(paths[0], map_location="cpu").items()}
    for path in paths[1:]:
        for key, value in torch.load(path, map_location="cpu").items():
            if total[key].is_floating_point():
                total[key] += value.float()
    for key, value in total.items():
        if value.is_floating_point():
            total[key] = value / len(paths)
    return total


def blend_uint8(maps: Sequence[np.ndarray], weights: Sequence[int]) -> np.ndarray:
    """Integer-weighted mean of cached uint8 probability maps, as float32 in [0, 1].

    Matches what `predict_probability` does at test time when a module object is repeated in the
    model list: it divides by len(models) * len(flips), so repetition IS an integer weight.
    """
    if len(maps) != len(weights):
        raise ValueError("maps and weights differ in length")
    total_weight = int(sum(weights))
    if total_weight <= 0:
        raise ValueError("weights must sum to a positive value")
    accumulator = np.zeros(maps[0].shape, dtype=np.float32)
    for probability_map, weight in zip(maps, weights):
        if weight:
            accumulator += probability_map.astype(np.float32) * float(weight)
    return accumulator / (255.0 * total_weight)


def pq_for_weights(pool_maps: Mapping[str, Sequence[np.ndarray]], indices: Mapping[str, Any],
                   cfg: SemanticCFG, weights: Sequence[int]) -> dict[str, float]:
    """Pooled PQ of the integer-weighted blend of a resident pool, decoded with `cfg`."""
    rows = []
    for filename, maps in pool_maps.items():
        instances = decode_instances(blend_uint8(maps, weights), cfg)
        rows.extend(evaluate_prediction_for_file(filename, instances, indices, cfg))
    return aggregate_pq(pd.DataFrame(rows, columns=OOF_COLUMNS))


def greedy_ensemble_weights(pool_maps: Mapping[str, Sequence[np.ndarray]],
                            indices: Mapping[str, Any], cfg: SemanticCFG, total: int = 8,
                            members: Sequence[int] | None = None
                            ) -> tuple[list[int], pd.DataFrame]:
    """Caruana-style greedy ensemble selection WITH replacement.

    Seeded one-hot at the best single member rather than empty (an empty ensemble has no
    prediction, so the first step would degenerate to this anyway and waste a draw). Each step
    tries appending every member -- including ones already present, which is what lets the search
    express a 3:1 ratio -- and takes the argmax. A non-improving step is taken anyway (Caruana:
    downhill moves let the ensemble recover); the returned weights are the best-scoring prefix
    over all steps, so the result is never worse than the seed.

    `members` restricts the candidate pool, which is how `bagged_ensemble_weights` bags.
    """
    size = len(next(iter(pool_maps.values())))
    allowed = list(range(size)) if members is None else list(members)
    solo = {i: pq_for_weights(pool_maps, indices, cfg, [1 if j == i else 0 for j in range(size)])["pq"]
            for i in allowed}
    seed_index = max(solo, key=solo.get)
    weights = [1 if j == seed_index else 0 for j in range(size)]
    best_weights, best_pq = list(weights), solo[seed_index]
    history = [{"step": 0, "added": seed_index, "pq": best_pq}]
    for step in range(1, total):
        scored = []
        for i in allowed:
            trial = list(weights)
            trial[i] += 1
            scored.append((pq_for_weights(pool_maps, indices, cfg, trial)["pq"], i))
        pq, chosen = max(scored)
        weights[chosen] += 1
        history.append({"step": step, "added": chosen, "pq": pq})
        if pq > best_pq + 1e-12:
            best_weights, best_pq = list(weights), pq
    return best_weights, pd.DataFrame(history)


def bagged_ensemble_weights(pool_maps: Mapping[str, Sequence[np.ndarray]],
                            indices: Mapping[str, Any], cfg: SemanticCFG, bags: int = 5,
                            bag_fraction: float = 0.6, total: int = 8, seed: int = 0) -> list[int]:
    """`bags` independent greedy runs, each restricted to a random subset of the pool; the
    averaged weight vector is rounded back to integers summing to `total` by largest remainder.

    The averaging IS the overfitting control, and it is not optional here: 5 bags x 8 steps x K
    candidates is several hundred draws against one validation fold, which on 40 images is worth
    roughly +0.06 of pure noise -- larger than any real gain measured on this problem. A member
    that wins in one bag only keeps about 1/bags of its weight.
    """
    size = len(next(iter(pool_maps.values())))
    take = max(2, math.ceil(bag_fraction * size))
    rng = np.random.default_rng(seed)
    accumulated = np.zeros(size, dtype=float)
    for bag in range(bags):
        members = sorted(rng.choice(size, size=min(take, size), replace=False).tolist())
        weights, _ = greedy_ensemble_weights(pool_maps, indices, cfg, total=total, members=members)
        accumulated += np.asarray(weights, dtype=float)
    shares = accumulated / max(accumulated.sum(), 1.0) * total
    floors = np.floor(shares).astype(int)
    for index in np.argsort(-(shares - floors))[: total - int(floors.sum())]:
        floors[index] += 1
    return floors.tolist()


# ---------------------------------------------------------------------------
# Instance filter: the threshold has a closed form, so stop spending draws on a wide grid
# ---------------------------------------------------------------------------


def filter_threshold_closed_form(pq: float, sq: float) -> float:
    """The score below which dropping an instance raises pooled PQ.

    Dropping changes the denominator by -0.5 whether the instance was a TP (tp-1, fn+1 => -1+0.5)
    or an FP (-0.5), and the numerator by -p*iou. So dropping helps iff p*iou < 0.5*PQ, i.e.
    p < 0.5*PQ/SQ using the matched-mask mean IoU for iou. At run 3's 0.4157/0.6696 that is
    0.310 -- one grid step below the 0.35 the sweep spent draws to find. Note it MOVES with PQ:
    a better ensemble raises the threshold.
    """
    if sq <= 0:
        return 0.0
    return float(0.5 * pq / sq)


# ---------------------------------------------------------------------------
# Spine-seeded splitting: the aux head is trained and cached but decode never sees it
# ---------------------------------------------------------------------------


def spine_seeds(mask: np.ndarray, spine: np.ndarray, rel_threshold: float = 0.10,
                min_seed_px: int = 12, min_seed_ratio: float = 0.0) -> tuple[np.ndarray, int]:
    """Connected components of the spine map inside `mask`, as watershed markers.

    `mask` and `spine` must already share a shape. The threshold is RELATIVE to the spine's own
    maximum inside this instance: the head is uncalibrated (global max ~0.47, never saturates),
    so an absolute cut either erases faint spines or floods strong ones. Seeds under
    `min_seed_px` are dropped -- a 3px tube over a real filament is far longer than that, so a
    tiny blob is noise, and promoting it would split a good instance into a good one plus a
    sliver that costs 0.5 of denominator for nothing.

    Returns (int32 markers labelled 1..k inside mask, k).
    """
    import cv2

    inside = spine[mask]
    if inside.size == 0:
        return np.zeros(mask.shape, np.int32), 0
    peak = float(inside.max())
    if peak <= 0:
        return np.zeros(mask.shape, np.int32), 0
    seeded = (spine >= rel_threshold * peak) & mask
    count, labels = cv2.connectedComponents(seeded.astype(np.uint8), connectivity=8)
    if count <= 1:
        return np.zeros(mask.shape, np.int32), 0
    sizes = {label: int((labels == label).sum()) for label in range(1, count)}
    # A real filament's spine component is ~602 px (median, 1536 grid) while the SECOND component
    # inside one instance is ~160 px: most second seeds are slivers, not another filament. Require
    # a seed to be plausible both absolutely and RELATIVE to the instance's main spine, or the
    # split shaves an end off a good instance -- measured worse-half IoU 0.109.
    floor = max(min_seed_px, min_seed_ratio * max(sizes.values(), default=0))
    keep, out = 0, np.zeros(mask.shape, np.int32)
    for label in range(1, count):
        if sizes[label] >= floor:
            keep += 1
            out[labels == label] = keep
    return out, keep


def split_instance_with_spine(mask: np.ndarray, spine: np.ndarray, probability: np.ndarray,
                              rel_threshold: float = 0.10, min_seed_px: int = 12,
                              minimum_area: int = 240, min_seed_ratio: float = 0.0) -> list[np.ndarray]:
    """One decoded instance -> one or more, cut where the spine head says there are two filaments.

    Grows the spine seeds outward one pixel at a time, strictly inside `mask`, until every pixel
    is claimed: each pixel ends up with its geodesically nearest seed, so the cut falls halfway
    along the connection between two filaments.

    NOT cv2.watershed, which was measured losing 28.6% of a synthetic ribbon: it writes a -1
    ridge along every marker boundary, and with the outside marked as background that ridge eats
    the instance's whole perimeter -- which on a ~6px ribbon is most of it. NOT watershed on an
    eroded mask either: a 6px ribbon does not survive erosion, the same reason width_px=-1
    measured -0.019. The spine is the only field that separates two touching filaments while
    staying inside both.

    `probability` is unused by the flood itself and kept for signature stability with callers;
    the dim neck is already where the geodesic midpoint lands.
    """
    import cv2

    markers, count = spine_seeds(mask, spine, rel_threshold, min_seed_px, min_seed_ratio)
    if count < 2:
        return [mask]
    labels = markers.copy()
    cross = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))
    remaining = int(mask.sum()) - int((labels > 0).sum())
    while remaining > 0:
        grew = False
        # Dilate every label by one pixel, then hand each newly reached pixel to the first label
        # that got there. A simultaneous round means ties are decided by label index over a 1px
        # seam, which cannot move IoU, while sequential growth would let label 1 outrun the rest.
        frontier = np.zeros(mask.shape, np.int32)
        for label in range(1, count + 1):
            grown = cv2.dilate((labels == label).astype(np.uint8), cross).astype(bool)
            fresh = grown & mask & (labels == 0) & (frontier == 0)
            if fresh.any():
                frontier[fresh] = label
                grew = True
        if not grew:
            break           # an unreachable pocket: not connected to any seed inside `mask`
        labels[frontier > 0] = frontier[frontier > 0]
        remaining = int(mask.sum()) - int((labels > 0).sum())
    parts = []
    for label in range(1, count + 1):
        part = (labels == label) & mask
        if int(part.sum()) >= minimum_area:
            parts.append(part)
    if len(parts) < 2:
        return [mask]
    # Any pocket the flood could not reach stays with the largest part, so no pixel is dropped.
    orphan = mask & ~np.logical_or.reduce(parts)
    if orphan.any():
        parts[int(np.argmax([p.sum() for p in parts]))] |= orphan
    return parts


def split_instances_with_spine(instances: Sequence[np.ndarray], spine: np.ndarray,
                               probability: np.ndarray, rel_threshold: float = 0.10,
                               min_seed_px: int = 12, minimum_area: int = 240,
                               min_area_to_split: int = 0, min_seed_ratio: float = 0.0) -> list[np.ndarray]:
    """Apply split_instance_with_spine to a decoded instance list, at the SPINE's resolution.

    `instances` and `probability` are native (2048); `spine` is the model's 1536 grid. Masks are
    downsampled with INTER_NEAREST, split, and the parts upsampled back the same way: resampling
    an integer label map bilinearly invents labels that were never there, and a 2px spine does
    not survive a bilinear trip to 2048 at all.
    """
    import cv2

    if spine is None or not len(instances):
        return list(instances)
    height, width = spine.shape
    native = instances[0].shape
    scale = (native[0] / height) * (native[1] / width)
    small_probability = cv2.resize(probability, (width, height), interpolation=cv2.INTER_AREA)
    out = []
    for mask in instances:
        # Splitting a correct instance is far more expensive than missing a merge (one tp becomes
        # two fp: tp-1, fp+2, fn+1), so only consider candidates that look over-merged. Measured
        # over-merges sit at median area 2885 px against 1088 for all predictions.
        if min_area_to_split and int(mask.sum()) < min_area_to_split:
            out.append(mask)
            continue
        small = cv2.resize(mask.astype(np.uint8), (width, height),
                           interpolation=cv2.INTER_NEAREST).astype(bool)
        if not small.any():
            out.append(mask)
            continue
        parts = split_instance_with_spine(small, spine, small_probability, rel_threshold,
                                          min_seed_px, max(1, int(minimum_area / scale)),
                                          min_seed_ratio)
        if len(parts) < 2:
            out.append(mask)
            continue
        for part in parts:
            big = cv2.resize(part.astype(np.uint8), (native[1], native[0]),
                             interpolation=cv2.INTER_NEAREST).astype(bool) & mask
            if int(big.sum()) >= minimum_area:
                out.append(big)
    return out


# ---------------------------------------------------------------------------
# Submission guards. The host rejects a bad file AND burns one of the 5 daily slots.
# ---------------------------------------------------------------------------


def list_test_images(test_dir: Path) -> list[Path]:
    """Every test image, whatever the extension's spelling.

    `write_submission` globs '*.jpeg' literally. The public test set is .jpeg, so this has always
    worked -- but the private set is a different directory, and a '.jpg' or an uppercase '.JPEG'
    there would yield zero rows and a silently scored-zero submission with no error anywhere.
    """
    suffixes = {".jpeg", ".jpg", ".png"}
    return sorted(path for path in Path(test_dir).iterdir()
                  if path.is_file() and path.suffix.lower() in suffixes)


def check_submission(submission: "pd.DataFrame", test_dir: Path, shape: tuple[int, int] = (2048, 2048),
                     check_overlap: bool = True) -> dict[str, Any]:
    """Assert everything the host checks, before spending a submission slot.

    Three documented rejection modes, each of which costs a slot:
      * overlapping instance masks -- "we do not accept overlapping predictions";
      * an automated implausibility detector for submissions that look copied from ground truth;
      * silently unscored rows, which is the dangerous one: the official scorer keys images with
        `filament_id.str.split('_', n=1).str[0]`, i.e. up to the FIRST underscore. Our ids are
        '<stem>_<k>' and the stems carry no underscore, so this holds today -- but a stem that
        ever did would orphan every row for that image and the file would still look valid.
        Note the notebook's own assertion uses rsplit, which would NOT catch it.
    """
    from pycocotools import mask as mask_utils

    report: dict[str, Any] = {}
    files = list_test_images(test_dir)
    stems = {path.stem for path in files}
    report["test_images"] = len(files)
    assert files, f"no test images found under {test_dir} -- check the extension glob"

    assert submission["filament_id"].is_unique, "duplicate filament_id"
    scorer_key = submission["filament_id"].str.split("_", n=1).str[0]
    orphans = sorted(set(scorer_key) - stems)
    assert not orphans, f"{len(orphans)} filament_id stems match no test image, e.g. {orphans[:3]}"
    report["rows"] = int(len(submission))
    report["images_covered"] = int(scorer_key.nunique())
    report["images_without_rows"] = int(len(stems) - scorer_key.nunique())

    if check_overlap:
        worst = 0
        for stem, group in submission.groupby(scorer_key):
            accumulator = np.zeros(shape, dtype=np.int16)
            for rle in group["segmentation_rle"]:
                entry = {"size": list(shape), "counts": rle.encode() if isinstance(rle, str) else rle}
                accumulator += mask_utils.decode(entry).astype(np.int16)
            worst = max(worst, int(accumulator.max()))
            assert worst <= 1, f"overlapping predicted instances on {stem} -- the host rejects this"
        report["max_pixel_multiplicity"] = worst
    return report


def staged_test_dir(test_dir: Path, staging: Path) -> Path:
    """`test_dir` if every image already ends in .jpeg, else a directory of .jpeg hardlinks.

    `write_submission` selects test images with `path.suffix.lower() == ".jpeg"`, which lives in
    the fingerprinted library -- changing it there would cost a full retrain for a one-line guard.
    Staging gets the same result from outside: link each image under a .jpeg name and hand
    `write_submission` a directory it understands. Stems are preserved, so every filament_id is
    unchanged. Hardlinks cost no disk; a copy is the fallback across filesystems.
    """
    files = list_test_images(test_dir)
    if not files:
        raise AssertionError(f"no test images under {test_dir}")
    if all(path.suffix == ".jpeg" for path in files):
        return Path(test_dir)
    staging = Path(staging)
    staging.mkdir(parents=True, exist_ok=True)
    stems = {}
    for path in files:
        if path.stem in stems:
            raise AssertionError(f"two test images share the stem {path.stem!r}: "
                                 f"{stems[path.stem].name} and {path.name}")
        stems[path.stem] = path
        target = staging / f"{path.stem}.jpeg"
        if target.exists():
            continue
        try:
            os.link(path, target)
        except OSError:
            shutil.copy2(path, target)
    print(f"staged {len(files)} test images as .jpeg under {staging} "
          f"(extensions seen: {sorted({p.suffix for p in files})})")
    return staging


Writing semantic_search.py


## Configuration

The defaults are the intended full experiment, not a toy baseline.

In [12]:
from pathlib import Path
import ast
import importlib

import solar_filament_lib
importlib.reload(solar_filament_lib)
from solar_filament_lib import *

_lib_text = Path("solar_filament_lib.py").read_text(encoding="utf-8")
ast.parse(_lib_text)  # fail fast, before any GPU time is spent, if the %%writefile chain above
                       # produced something syntactically broken
print(f"solar_filament_lib.py: OK, {len(_lib_text.splitlines())} lines")

import semantic_filament_lib as sfl
importlib.reload(sfl)
_sfl_text = Path("semantic_filament_lib.py").read_text(encoding="utf-8")
ast.parse(_sfl_text)
print(f"semantic_filament_lib.py: OK, {len(_sfl_text.splitlines())} lines")

import semantic_search as ss
importlib.reload(ss)
_ss_text = Path("semantic_search.py").read_text(encoding="utf-8")
ast.parse(_ss_text)
print(f"semantic_search.py: OK, {len(_ss_text.splitlines())} lines")


solar_filament_lib.py: OK, 4514 lines
semantic_filament_lib.py: OK, 1544 lines
semantic_search.py: OK, 808 lines


In [13]:
%%writefile fold_worker.py
import argparse
import json
import os
import sys
from pathlib import Path

import torch

import solar_filament_lib as lib


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--fold", type=int, required=True)
    parser.add_argument("--work-dir", type=str, required=True)
    parser.add_argument("--cfg-json", type=str, required=True)
    args = parser.parse_args()

    cfg = lib.CFG(**json.loads(Path(args.cfg_json).read_text(encoding="utf-8")))
    work_dir = Path(args.work_dir)

    print(f"fold {args.fold} worker start: CUDA_VISIBLE_DEVICES="
          f"{os.environ.get('CUDA_VISIBLE_DEVICES')!r}, "
          f"torch.cuda.device_count()={torch.cuda.device_count()}", flush=True)

    # cfg.seed + fold, not cfg.seed alone: two folds paired on different GPUs run genuinely
    # concurrently as separate processes, so sharing one global RNG stream (as the sequential
    # in-kernel design did via a single seed_everything(cfg.seed) call) would make paired folds
    # share identical init/augmentation randomness. This makes every fold's RNG state
    # independent of pairing/order -- OOF/LB numbers from a parallel run are not expected to
    # bit-for-bit match an earlier sequential run for this reason; that is expected, not a bug.
    lib.seed_everything(cfg.seed + args.fold)

    paths = lib.discover_competition_paths()
    coco = lib.load_coco(paths["json"])
    indices = lib.build_coco_indices(coco)
    selected_image_ids = lib.choose_medoid_annotators(indices)
    observations = lib.build_observation_table(indices)
    fold_manifest = lib.assign_grouped_folds(observations, cfg)

    lib.train_fold(args.fold, fold_manifest, paths, selected_image_ids, indices, cfg, work_dir)
    print(f"fold {args.fold} complete", flush=True)


if __name__ == "__main__":
    main()


Writing fold_worker.py


In [14]:
cfg = CFG(
    n_folds=5,
    image_size=1024,
    # Seg(YOLO)->Energy pipeline (2026-09-30, active path -- see PROGRESS.md): YOLO26 detector
    # is Stage 1 ("Seg"), DINOv3-backboned YoloConditionedEnergyNet is Stage 2 ("Energy"),
    # conditioned on Stage 1's frozen output. detector_model upgraded from yolo11s-seg.pt.
    detector_model='yolo26s-seg.pt',
    detector_epochs=30,
    detector_batch=4,
    refiner_epochs=25,
    refiner_batch=16,
    detector_conf=0.20,
    detector_fallback_conf=0.05,
    # YOLO26 hyperparameters tuned for this competition's shape (many thin, closely-packed
    # filament instances/image, single class, IoU-based metric) -- see CFG's own field
    # comments and PROGRESS.md for the reasoning behind each.
    detector_nms=False,
    detector_overlap_mask=True,
    detector_copy_paste=0.3,
    detector_box_gain=9.0,
    detector_cls_gain=0.3,
    mask_ratio=1,
    refiner_threshold=0.65,
    boundary_width=3,
    boundary_loss_weight=1.0,
    refiner_backbone='vit',  # unused by the active Seg(YOLO)->Energy path; kept for the
                            # dormant detector-free RegionBoundarySegmenter path -- see PROGRESS.md.
    # HF access to facebook/dinov3-* approved 2026-09-26 -- switched from the DINOv2 fallback.
    vit_hub_id='facebook/dinov3-vits16-pretrain-lvd1689m',
    vit_unfreeze_last_n=4,
    vit_backbone_lr_mult=0.1,
    vit_images_per_batch=2,
    minimum_area=96,
    pq_match_iou=0.50,
    minimum_fold_support=2,
    use_tta=True,
    save_oof_masks=False,
    debug=False,  # change to True for a quick Kaggle smoke run
)

# Kaggle sessions stop at 12h, so each session trains one thing. Checkpoints carry over by
# attaching an earlier session's output as an input (resolve_stage1_checkpoint finds it).
#   Semantic session:  RUN_SEMANTIC=True, SUBMIT_PIPELINE='semantic'           (current setting)
#   Stage 1 session:   RUN_STAGE1=True, RUN_CV_SCORE=True, RUN_FINAL_FIT=True, RUN_SEMANTIC=False
#                      -> YOLO-only pooled PQ + failure modes; the YOLO checkpoints for later
#   Stage 2 session:   attach the Stage 1 output; RUN_STAGE2=True, RUN_CV_SCORE=True,
#                      RUN_FINAL_FIT=True, RUN_SEMANTIC=False, SUBMIT_PIPELINE='embedding_fused'
RUN_STAGE1 = False      # embedding pipeline Stage 1: train the YOLO26 detector
RUN_STAGE2 = False      # embedding pipeline Stage 2: train both embedding nets (restores Stage 1 if needed)
RUN_TRAIN = RUN_STAGE1 or RUN_STAGE2
RUN_CV_SCORE = False    # fold-0 OOF: YOLO-only always; Stage 2 variants + sweep once its checkpoints exist
RUN_FINAL_FIT = False   # final model on ~95% of all labeled data, for whichever stages are on
RUN_TEST = True         # writes the SUBMIT_PIPELINE submission
RUN_SEMANTIC = True     # semantic challenger: train fold 0, OOF score, decode sweep
RUN_DORMANT_ENSEMBLE = False  # the dormant-segmenter ensemble cell raises without its own checkpoints
SUBMIT_PIPELINE = 'semantic'  # or 'embedding_fused'; choose by pooled PQ in the head-to-head table
# GPU budget raised to 9.0h (user, 2026-10-11). Epochs derived, not guessed:
#   v24 measured 1252s/epoch = 992s train + 260s full-fold validation, 23 epochs in 8.0h.
#   Validating 10x instead of every epoch frees 0.94h. At 992s/epoch:
#     26 epochs = 25797s train + 2600s val = 28397s = 87.6% of 9h, 1.11h margin.
#   Biased LOW on purpose: `epochs` sets CosineAnnealingLR(T_max), cannot be changed mid-run, and
#   a truncated cosine costs about -0.007 PQ (the lib's own warning). Overshooting repeats v24,
#   which completed 23 of 35 and recorded cosine_annealed=false.
#   Session cap: 0.2h setup + 9.0h train + ~0.5h post = 9.7h of 12h (v24's post-training took
#   ~30 min, not the 3.5h the old comment assumed).
semantic_cfg = sfl.SemanticCFG(seed=cfg.seed, debug=cfg.debug, max_train_hours=9.0, epochs=26)
# [v27] Decode sweep DEPLOY-OFF, decode PINNED to v24's deployed settings.
# LB evidence, 4/4 consistent -- every submission whose decode left pt=0.60 + hysteresis OFF
# scored 0.37; both that kept them scored 0.39:
#   2026-10-09 run3  pt=0.60 ptlow=0.56 (hysteresis ON)      -> 0.37
#   2026-10-10 v24   pt=0.60 ptlow=None                      -> 0.39
#   2026-10-11 v24 + spine filter, same decode               -> 0.39
#   2026-10-12 v26   pt=0.65 ptlow=None (sweep winner)       -> 0.37
# The sweep gains +0.018 on the 141-image fold whose CI is +/-0.019: it is fitting the fold.
# CAUTION: the LIBRARY DEFAULT is prob_threshold_low=0.56 -- hysteresis ON, i.e. the 0.37
# setting -- so APPLY_SWEEP_WINNER=False ALONE DOES NOT REPRODUCE v24. Hence the explicit pin.
APPLY_SWEEP_WINNER = False  # the sweep still runs and reports; it just does not deploy
DECODE_PIN = dict(prob_threshold=0.60, prob_threshold_low=None, width_px=0, max_gap_px=40.0,
                  max_angle_deg=65.0, minimum_area=240, min_fragment_area=32,
                  endpoint_radius=15.0, min_elongation=1.5)
USE_INSTANCE_FILTER = True  # learn which instances to drop; kept only if it raises OOF PQ
semantic_decode_cfg, semantic_filter = semantic_cfg, None  # replaced by the sweep / filter cells
WORK_DIR = Path('/kaggle/working/solar_filament_pq') if KAGGLE_DATA_AVAILABLE else Path('./solar_filament_pq_smoke')
WORK_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR_FINAL = WORK_DIR / 'final_fit'
WORK_DIR_FINAL.mkdir(parents=True, exist_ok=True)
seed_everything(cfg.seed)
print(asdict(cfg))


{'seed': 20260729, 'n_folds': 5, 'image_size': 1024, 'detector_model': 'yolo26s-seg.pt', 'detector_epochs': 30, 'detector_batch': 4, 'detector_patience': 7, 'detector_conf': 0.2, 'detector_fallback_conf': 0.05, 'detector_iou': 0.6, 'detector_nms': False, 'detector_overlap_mask': True, 'detector_copy_paste': 0.3, 'detector_box_gain': 9.0, 'detector_cls_gain': 0.3, 'mask_ratio': 1, 'refiner_crop': 256, 'refiner_context': 1.8, 'refiner_min_crop': 96, 'refiner_epochs': 25, 'refiner_batch': 16, 'refiner_lr': 0.0003, 'refiner_threshold': 0.65, 'refiner_backbone': 'vit', 'vit_hub_id': 'facebook/dinov3-vits16-pretrain-lvd1689m', 'vit_out_indices': (3, 6, 9, 12), 'vit_unfreeze_last_n': 4, 'vit_backbone_lr_mult': 0.1, 'vit_image_size': 1024, 'vit_roi_size': 16, 'vit_images_per_batch': 2, 'minimum_area': 96, 'pq_match_iou': 0.5, 'ensemble_iou': 0.5, 'minimum_fold_support': 2, 'singleton_score': 0.9, 'num_workers': 2, 'use_tta': True, 'save_oof_masks': False, 'final_holdout_fraction': 0.05, 'bound

In [15]:
if KAGGLE_DATA_AVAILABLE and RUN_STAGE2 and (RUN_TRAIN or RUN_FINAL_FIT) and cfg.refiner_backbone == "vit":
    # Warm the HF Hub cache sequentially, in the main kernel, before any fold worker subprocess
    # launches -- avoids two concurrently-launched workers racing on a cold cache for the same
    # DINOv3 checkpoint. huggingface_hub does have concurrency-safe lock files, but this side-steps
    # relying on that being true across process boundaries, for the cost of one cheap extra load.
    from transformers import AutoBackbone
    print(f"Warming HF Hub cache for {cfg.vit_hub_id} before launching fold workers...")
    AutoBackbone.from_pretrained(cfg.vit_hub_id, out_indices=list(cfg.vit_out_indices), reshape_hidden_states=True)
    print("HF Hub cache warm.")


## Diagnostics

Charts for interpreting the pipeline at three levels: per-epoch training progression
(this section and section 11a's), per-fold cross-validation evaluation (after the
Cross-Fold Score section), and the final submission's prediction distribution (after
Test Inference). Each chart skips gracefully -- printing a message instead of
crashing -- if the stage that feeds it hasn't run yet.


In [16]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

_CHART_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]


def plot_refiner_progress(runs: dict[str, Path]) -> None:
    """One val_crop_dice-vs-epoch line per {label: refiner_meta.json path}, best epoch marked."""
    fig, ax = plt.subplots(figsize=(7, 4))
    plotted = False
    for color, (label, meta_path) in zip(_CHART_COLORS, runs.items()):
        meta = read_json_if_valid(meta_path)
        history = (meta or {}).get("history") or []
        if not history:
            continue
        plotted = True
        epochs = [h["epoch"] for h in history]
        dice = [h["val_crop_dice"] for h in history]
        ax.plot(epochs, dice, marker="o", markersize=4, linewidth=2, color=color, label=label)
        best = max(history, key=lambda h: h["val_crop_dice"])
        ax.scatter([best["epoch"]], [best["val_crop_dice"]], color=color, s=70, zorder=3, edgecolor="white")
    if not plotted:
        print("plot_refiner_progress: no refiner_meta.json history yet -- run training first.")
        plt.close(fig)
        return
    ax.set_xlabel("epoch"); ax.set_ylabel("val_crop_dice"); ax.set_ylim(bottom=0)
    ax.set_title("Refiner training progression (dot = best epoch)")
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.legend(frameon=False)
    plt.show()


def show_detector_curves(runs: dict[str, Path]) -> None:
    """Displays Ultralytics' own results.png per {label: results.png path} side by side."""
    found = {label: path for label, path in runs.items() if path.exists()}
    if not found:
        print("show_detector_curves: no detector results.png yet -- run training first.")
        return
    fig, axes = plt.subplots(1, len(found), figsize=(6 * len(found), 4))
    axes = [axes] if len(found) == 1 else list(axes)
    for ax, (label, path) in zip(axes, found.items()):
        ax.imshow(mpimg.imread(path)); ax.set_title(f"detector -- {label}"); ax.axis("off")
    plt.tight_layout(); plt.show()


## Checks

### 8. Metric edge cases required by PQ

In [17]:
def _rectangle(y0, y1, x0, x1, size=16):
    mask = np.zeros((size, size), dtype=bool)
    mask[y0:y1, x0:x1] = True
    return mask

first = _rectangle(1, 5, 1, 5)
second = _rectangle(8, 14, 8, 14)
perfect = panoptic_quality([first, second], [second, first])
missing = panoptic_quality([first, second], [first])
extra = panoptic_quality([first], [first, second])

half_gt = _rectangle(2, 4, 2, 5)
half_pred = _rectangle(2, 4, 3, 6)  # exactly IoU=0.5; not a TP in original PQ
threshold_edge = panoptic_quality([half_gt], [half_pred])

fragment_gt = _rectangle(2, 6, 2, 10)
fragments = [_rectangle(2, 6, 2, 6), _rectangle(2, 6, 6, 10)]
fragmented = panoptic_quality([fragment_gt], fragments)

assert perfect['pq'] == 1.0
assert np.isclose(missing['pq'], 2 / 3) and missing['fn'] == 1
assert np.isclose(extra['pq'], 2 / 3) and extra['fp'] == 1
assert threshold_edge['tp'] == 0 and threshold_edge['pq'] == 0.0
assert fragmented['tp'] == 0 and fragmented['pq'] == 0.0

pd.DataFrame([perfect, missing, extra, threshold_edge, fragmented],
             index=['perfect', 'missing', 'extra', 'IoU=0.5', 'fragmented'])

,pq,sq,rq,tp,fp,fn,sum_iou
perfect,1.000000,1.0,1.000000,2,0,0,2.0
missing,0.666667,1.0,0.666667,1,0,1,1.0
extra,0.666667,1.0,0.666667,1,1,0,1.0
IoU=0.5,0.000000,0.0,0.000000,0,1,1,0.0
fragmented,0.000000,0.0,0.000000,0,2,1,0.0


## Data

### 9. Load official data, freeze folds, and audit leakage

In [18]:
if not KAGGLE_DATA_AVAILABLE:
    paths = coco = indices = selected_image_ids = observations = fold_manifest = None
    print('Official competition input is not attached; training cells will skip.')
else:
    paths = discover_competition_paths()
    coco = load_coco(paths['json'])
    indices = build_coco_indices(coco)
    selected_image_ids = choose_medoid_annotators(indices)
    observations = build_observation_table(indices)
    fold_manifest = assign_grouped_folds(observations, cfg)
    fold_manifest.to_csv(WORK_DIR / 'fold_manifest.csv', index=False)
    fingerprint = config_fingerprint(cfg, fold_manifest)
    (WORK_DIR / 'run_config.json').write_text(
        json.dumps({'config': asdict(cfg), 'fingerprint': fingerprint}, indent=2), encoding='utf-8'
    )
    assert observations['filename'].nunique() == len(observations)
    assert fold_manifest.groupby('filename')['fold'].nunique().max() == 1
    print({
        'physical_observations': len(observations),
        'annotator_records': len(coco['images']),
        'train_annotations': len(coco['annotations']),
        'test_images': len(list(paths['test'].glob('*.jpeg'))),
        'fold_sizes': fold_manifest.groupby('fold').size().to_dict(),
        'fingerprint': fingerprint,
    })
    display(fold_manifest.groupby(['fold', 'site']).size().unstack(fill_value=0))

{'physical_observations': 707, 'annotator_records': 1154, 'train_annotations': 8199, 'test_images': 180, 'fold_sizes': {0: 141, 1: 141, 2: 142, 3: 142, 4: 141}, 'fingerprint': '27502c00e9555071396ff5f10b856485ffa1832e9d04960222c0682423a42194'}


site,Bh,Ch,Lh,Mh,Th,Uh
fold,,,,,,
0,25,23,24,26,22,21
1,24,24,25,25,23,20
2,24,24,27,26,22,19
3,22,26,26,24,22,22
4,23,28,25,24,21,20


### Multi-contour label-loss audit (measurement only)

See PROGRESS.md audit item #3. `mask_to_yolo_polygon` (cell 8) keeps only the largest contour per training-label instance mask -- `audit_multi_contour_instances` measures how often a real instance actually has more than one contour and how much of its area that silently drops, across every annotation (all folds, a data-quality question not a per-fold one). Pure measurement: nothing here changes what `mask_to_yolo_polygon`/`prepare_yolo_fold` actually do -- it only decides whether that fix is worth doing.


In [19]:
if KAGGLE_DATA_AVAILABLE:
    multi_contour_audit = audit_multi_contour_instances(indices)
    print(multi_contour_audit)
else:
    multi_contour_audit = None
    print('Multi-contour audit skipped: no competition data attached.')


{'affected_instances': 4, 'total_instances': 8199, 'affected_fraction': 0.00048786437370411026, 'mean_area_loss_fraction': 0.12013451883728199}


### Human agreement and annotation width

Each annotator's drawing scored against the others of the same image, with the same pooled
PQ the competition uses. That's a reference point for what the leaderboard can mean: a model
can beat it, because a model close to the average annotator can match each annotator better
than two annotators match each other. The width table shows how wide filaments are drawn,
which the decode sweep's `width_px` matches.


In [20]:
if KAGGLE_DATA_AVAILABLE:
    human_rows = inter_annotator_rows(observations['filename'].tolist(), indices, cfg.pq_match_iou)
    human = aggregate_pq(human_rows)
    human_ci = bootstrap_pq_ci(human_rows, n=1000, seed=cfg.seed)
    print(f"human vs human pooled PQ = {human['pq']:.4f} [95% CI {human_ci[0]:.4f}-{human_ci[1]:.4f}] "
          f"over {human['n_images']} multi-annotator images (SQ {human['sq']:.3f}, RQ {human['rq']:.3f})")
    annotation_widths = annotation_width_stats(indices)
    display(annotation_widths[['area', 'width']].describe(percentiles=[0.1, 0.5, 0.9]).round(2))
else:
    human_rows = annotation_widths = None
    print('Human agreement skipped: no competition data attached.')


human vs human pooled PQ = 0.3397 [95% CI 0.3238-0.3551] over 296 multi-annotator images (SQ 0.634, RQ 0.536)


,area,width
count,8199.00,8199.00
mean,2119.83,9.48
std,2741.72,4.76
min,9.00,1.86
10%,409.80,5.29
50%,1228.00,8.31
90%,4684.60,14.87
max,37739.00,51.61


### Why medoid annotators?

Several physical observations have multiple expert records. Selecting the record with maximum instance count or area biases training toward over-segmentation. For each duplicated observation, this notebook selects the record with highest mean semantic IoU to the other annotators. All records remain available for OOF evaluation; only the medoid supplies detector/refiner training targets.

## Training

### 10. Train or resume a detector+embedding-net pair (fold 0 by default)

Stage 1 of 2: this pair exists purely to produce a real, leak-free cross-fold validation score
for this recipe (section 11) -- it is not used for the submission. By default only fold 0
trains (see the Training cell below); `train_all_folds`/`run_parallel_training` run the full
5-fold CV instead. Stage 2 (section 11a) trains the actual submission model once more, on
~95% of all labeled data.


In [21]:
if KAGGLE_DATA_AVAILABLE and RUN_TRAIN:
    # Single-fold mode: fold_manifest's real 5-fold split (cfg.n_folds, cell 23) is untouched,
    # so fold 0 is an honest ~80/20 train/val split. Only fold 0 is trained -- for multi-fold
    # CV, use run_parallel_training/train_all_folds instead of train_fold.
    train_fold(0, fold_manifest, paths, selected_image_ids, indices, cfg, WORK_DIR,
               stage1=RUN_STAGE1, stage2=RUN_STAGE2)
    print('Fold 0 checkpoint ready:', sorted((WORK_DIR / 'models').glob('fold_0/*.pt')))
else:
    print('Training skipped.')


Training skipped.


### Training progression

Per-fold refiner validation dice across epochs, and each fold's own Ultralytics
detector curves (box/seg loss + mAP).


In [22]:
if KAGGLE_DATA_AVAILABLE:
    plot_refiner_progress({f"fold {f}": WORK_DIR / "models" / f"fold_{f}" / "refiner_meta.json" for f in range(cfg.n_folds)})
    show_detector_curves({f"fold {f}": WORK_DIR / "detector_runs" / f"fold_{f}" / "results.png" for f in range(cfg.n_folds)})
else:
    print("Diagnostics skipped: no competition data attached.")


plot_refiner_progress: no refiner_meta.json history yet -- run training first.
show_detector_curves: no detector results.png yet -- run training first.


## Cross-Fold Score

### 11. End-to-end grouped cross-fold validation

By default only fold 0's detector+embedding-net pair (section 10) is evaluated, on its own
held-out ~20% -- a real, leak-free score for this recipe, not the full 5-fold CV (pass the
real `cfg` to `run_oof_yolo_embedding` for that). Validation uses predictions from the complete
held-out detector -> embedding-net -> greedy-cluster decode pipeline, never masks from
ground-truth boxes. Every annotator is scored separately, then annotators are averaged inside
the physical observation so duplicated records do not receive extra weight.


In [23]:
if KAGGLE_DATA_AVAILABLE and RUN_CV_SCORE:
    # Single-fold mode: score fold 0 only, matching cell 33's single-fold training. Scoped
    # cfg copy -- run_oof's only use of n_folds is its loop bound, fold_manifest's real split
    # is untouched. For multi-fold CV, pass cfg directly instead of replace(cfg, n_folds=1).
    #
    # Four decode variants on the SAME held-out data: YOLO alone (does Stage 2 earn its cost?),
    # the embedding net alone, the embedding U-Net alone, and their mask-level fusion. Ranked by
    # the competition's pooled PQ (aggregate_pq); per-image mean shown for comparison only.
    # Overlapping 95% bootstrap intervals mean the gap is not resolvable on this fold.
    # A Stage 1 session scores YOLO alone; Stage 2 rows appear once both embedding nets exist.
    resolve_stage1_checkpoint(0, cfg, WORK_DIR)
    stage2_ready = all((WORK_DIR / 'models' / 'fold_0' / name).exists()
                       for name in ('embedding_net.pt', 'embedding_unet.pt'))
    observation_cv, annotator_cv = run_oof_yolo_embedding_variants(
        fold_manifest, paths, indices, replace(cfg, n_folds=1), WORK_DIR, stage2=stage2_ready,
    )
    comparison = []
    for variant, rows in annotator_cv.groupby('variant'):
        ci_low, ci_high = bootstrap_pq_ci(rows, n=1000, seed=cfg.seed)
        comparison.append({'variant': variant, **aggregate_pq(rows), 'pq_ci_low': ci_low, 'pq_ci_high': ci_high})
    cv_summary = pd.DataFrame(comparison).sort_values('pq', ascending=False).reset_index(drop=True)
    cv_summary.to_csv(WORK_DIR / 'embedding_variant_comparison.csv', index=False)
    best = cv_summary.iloc[0]
    print(f"best variant (pooled PQ): {best['variant']} -> PQ={best['pq']:.5f} "
          f"[95% CI {best['pq_ci_low']:.5f}-{best['pq_ci_high']:.5f}]  SQ={best['sq']:.5f}  "
          f"RQ={best['rq']:.5f}  per-image mean PQ={best['pq_image_mean']:.5f}")
    print({
        'mean_prediction_to_gt_ratio': observation_cv['n_pred'].sum() / observation_cv['n_gt'].sum(),
        'zero_prediction_observations': int((observation_cv['n_pred'] == 0).sum()),
    })
else:
    stage2_ready = False
    observation_cv = annotator_cv = cv_summary = None
    print('Cross-fold scoring skipped.')


Cross-fold scoring skipped.


### Cross-fold evaluation

One row per decode variant from `cv_summary`: pooled competition PQ (`aggregate_pq`) with a 95%
bootstrap interval over images, SQ/RQ, pooled TP/FP/FN, and the old per-image mean PQ for
comparison (section 11).


In [24]:
if KAGGLE_DATA_AVAILABLE and RUN_CV_SCORE:
    display(cv_summary.round(5))
else:
    print("Cross-fold summary skipped: RUN_CV_SCORE is off or no data attached.")


Cross-fold summary skipped: RUN_CV_SCORE is off or no data attached.


### Failure-mode breakdown

See PROGRESS.md audit item #5. Aggregate PQ/SQ/RQ/TP/FP/FN never says *why* -- `run_failure_mode_breakdown` reuses fold 0's own decode (the same code path OOF scoring already uses) and classifies every FN/FP into fully-missed vs. near-miss (`classify_failure_modes`, cell 15): fully-missed means nothing overlapped that filament at all, near-miss means something did but not enough -- the exact failure mode thin, touching shapes are prone to.


In [25]:
if KAGGLE_DATA_AVAILABLE and RUN_CV_SCORE:
    # Same breakdown for YOLO alone and for the embedding decode: if Stage 2 mostly moves
    # near_miss_fn (splits/merges), it targets the right error; if fully_missed_fn dominates in
    # both, no decode change can fix it -- recall has to come from Stage 1.
    breakdowns = [run_failure_mode_breakdown(fold_manifest, paths, indices,
                                             replace(cfg, n_folds=1, use_embedding_decode=False), WORK_DIR,
                                             fold=0, variant_label='yolo_only')]
    if stage2_ready:
        breakdowns.append(run_failure_mode_breakdown(fold_manifest, paths, indices, replace(cfg, n_folds=1),
                                                     WORK_DIR, fold=0, variant_label='embedding_net'))
    failure_breakdown = pd.concat(breakdowns, ignore_index=True)
    categories = ['fully_missed_fn', 'near_miss_fn', 'spurious_fp', 'near_miss_fp']
    failure_totals = failure_breakdown.groupby('variant')[categories].sum()
    display(failure_totals)

    fig, ax = plt.subplots(figsize=(7, 4))
    width = 0.38
    x = np.arange(len(categories))
    for offset, (variant, totals) in zip((-width / 2, width / 2), failure_totals.iterrows()):
        ax.bar(x + offset, totals.values, width=width, label=variant,
               color=_CHART_COLORS[0] if variant == 'yolo_only' else _CHART_COLORS[1])
    ax.set_xticks(x); ax.set_xticklabels(categories)
    ax.set_ylabel('count'); ax.set_title('Fold 0 OOF failure modes'); ax.legend(frameon=False)
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)
    plt.tight_layout(); plt.show()
else:
    failure_breakdown = None
    print('Failure-mode breakdown skipped: RUN_CV_SCORE is off or no data attached.')


Failure-mode breakdown skipped: RUN_CV_SCORE is off or no data attached.


### Cross-fold model-selection rule

Promote a change only if mean observation-level PQ rises, no single fold collapses, and the FP/FN trade-off is understood. Under PQ, a threshold that raises matched-mask Dice while creating many extras can lower the final score.

### Post-processing sweep (free, no retraining)

Two-step sweep against real fold-0 OOF PQ, reusing the already-trained Seg(YOLO)->Embedding
checkpoint -- all of it decode-time-only, no retraining. Sequential, not one combinatorial
grid, because `detector_conf` is the one dimension that genuinely isn't free (it changes
YOLO's own output, so it can't reuse the cached-GPU-half trick the other two share):

1. `sweep_detector_conf_yolo_embedding` first, at default `minimum_area`/`embedding_delta_d`
   -- picks the best `detector_conf`.
2. `sweep_postprocessing_yolo_embedding` (now 2D: `minimum_area` x `embedding_delta_d`) under
   that best `detector_conf` -- both are pure decode-time parameters of
   `separate_instances_with_embeddings` (see that sweep function's own docstring for why
   `embedding_delta_d` is just as free here as `minimum_area`, despite also being a training-time
   loss margin), each validation image's YOLO masks + embedding map computed ONCE and re-decoded
   per grid point.

Prints the full recommended override and its OOF PQ against current `cfg` defaults -- applied
manually in cell 23, never auto-applied here.
(`sweep_postprocessing`/`sweep_postprocessing_yolo_energy` are the dormant detector-free- and
energy-path equivalents -- kept, not deleted.)


In [26]:
if KAGGLE_DATA_AVAILABLE and RUN_CV_SCORE and stage2_ready:
    # The sweep decodes with the embedding net, so it runs once Stage 2 exists for fold 0 (the
    # CV cell above restored Stage 1). The old glob-and-copy restore was removed: it copied every
    # detector.pt under /kaggle/input in turn, so a final-fit detector could silently replace
    # fold 0's.
    # Step 1: detector_conf (the one non-free dimension -- re-runs YOLO+DINOv3 per value).
    detector_conf_sweep = sweep_detector_conf_yolo_embedding(
        fold_manifest, paths, indices, cfg, WORK_DIR, fold=0,
        detector_confs=[0.10, 0.15, 0.20, 0.25],
    )
    detector_conf_sweep.to_csv(WORK_DIR / 'detector_conf_sweep.csv', index=False)
    display(detector_conf_sweep.round(5))
    best_detector_conf = float(detector_conf_sweep.iloc[0]['detector_conf'])

    # Step 2: minimum_area x embedding_delta_d, free (cached GPU half) under best_detector_conf.
    sweep_results = sweep_postprocessing_yolo_embedding(
        fold_manifest, paths, indices, replace(cfg, detector_conf=best_detector_conf), WORK_DIR, fold=0,
        minimum_areas=[16, 32, 64, 96, 160],
        embedding_delta_ds=[1.0, 1.5, 2.0],
    )
    sweep_results.to_csv(WORK_DIR / 'postprocessing_sweep.csv', index=False)
    display(sweep_results.round(5).head(10))
    best = sweep_results.iloc[0]
    print(f"best: detector_conf={best_detector_conf} minimum_area={int(best['minimum_area'])} "
          f"embedding_delta_d={best['embedding_delta_d']} -> pq={best['pq']:.5f} "
          f"(current cfg defaults: detector_conf={cfg.detector_conf} "
          f"minimum_area={cfg.minimum_area} embedding_delta_d={cfg.embedding_delta_d})")
else:
    detector_conf_sweep = sweep_results = None
    print('Post-processing sweep skipped.')


Post-processing sweep skipped.


## Semantic Challenger

Same fold-0 split as the embedding pipeline above, scored with the same pooled PQ and
bootstrap interval, so the two land in one table. The checkpoint is picked by decoded
validation PQ, not loss. Probability maps are cached once (keyed by checkpoint bytes and
tiling/TTA settings) and every decode setting is re-scored from that cache.


In [27]:
# [session4] Reuse run 3's fold-0 artifacts instead of retraining them.
# resolve_stage1_checkpoint only knows detector.pt under models/fold_N; the semantic layout is
# semantic_models/fold_N/segmenter.pt, so it needs its own restore. Where an attached notebook
# output actually mounts depends on how it was added, so ss.find_attached_directory searches for
# the directory anywhere under /kaggle/input and prefers a solar_filament_pq/ parent -- but
# refuses to pick between two candidates, because silently restoring the wrong checkpoint is the
# failure mode PROGRESS.md records.
FOLD0_RESTORED = False          # run 3's checkpoint + meta are present and complete
FOLD0_FINGERPRINT_MATCH = False  # ...AND cfg/library bytes match, so it is REUSED not retrained
if KAGGLE_DATA_AVAILABLE and RUN_SEMANTIC:
    _root = Path('/kaggle/input')
    print('attached inputs:', sorted(p.name for p in _root.iterdir()) if _root.exists() else 'none')
    for key, value in ss.restore_attached_artifacts(
            ['semantic_models', 'semantic_prob_cache'], WORK_DIR).items():
        print(f'  {key}: {value}')

    # Session contract. v20 and v21 were both lost to a restore that quietly missed: a ~4h decode
    # session silently becomes a 12h+ retrain, overruns Kaggle's cap, and the kernel is killed with
    # the sweep and the submission unwritten. Fail loudly here instead of discovering it at hour 12.
    REQUIRE_FOLD0_RESTORE = False  # 2026-10-13 (v27): RETRAIN fold 0 -- deliberately.
# v27 edits semantic_filament_lib.py (schedule reverted to v24's: grad_accumulation 2, no
# warmup / floor / constant tail / SWA; sample_random_annotator=True; affine rotation+scale
# augmentation), so _fingerprint CANNOT match any earlier checkpoint and none must be reused.
# The attached v26 output is restored but ignored: its checkpoint fails the fingerprint, and its
# chunked-training state is skipped too ("ignoring stale training state", lib :1017-1020).
# (Superseded: the 2026-10-12 decode-pin plan reused v26's checkpoint -- abandoned, because the
# v26 MODEL carries the -0.021, not the decode.)
# superseded 2026-10-11 (v25) note: retraining fold 0 was the intent then. The
# bundle (warmup + cosine-to-floor + constant tail + SWA, sparse validation, grad_accum 4,
# early stopping off, cudnn.benchmark, spine-aware instance filter, epochs 26,
# max_train_hours 9.0) edits semantic_filament_lib.py, whose bytes are hashed by _fingerprint --
# so no earlier checkpoint can match and none must be reused.
# Set back to True for any session that means to REUSE fold 0 rather than retrain it.

    fold0_ckpt = WORK_DIR / 'semantic_models' / 'fold_0' / 'segmenter.pt'
    fold0_meta = WORK_DIR / 'semantic_models' / 'fold_0' / 'segmenter_meta.json'
    FOLD0_RESTORED = fold0_ckpt.exists() and fold0_meta.exists() and         json.loads(fold0_meta.read_text())['status'] == 'complete'
    cached = len(list((WORK_DIR / 'semantic_prob_cache').rglob('*.npy')))         if (WORK_DIR / 'semantic_prob_cache').exists() else 0
    print(f'fold-0 checkpoint usable: {FOLD0_RESTORED}; {cached} cached probability maps')
    if FOLD0_RESTORED:
        # The checkpoint is only REUSED (rather than retrained) if cfg and the library bytes are
        # identical to run 3 -- that is what _fingerprint covers. The next cell prints which.
        meta = json.loads(fold0_meta.read_text())
        current = sfl._fingerprint(semantic_cfg,
                                   sorted(fold_manifest.loc[fold_manifest['fold'] != 0, 'filename']))
        FOLD0_FINGERPRINT_MATCH = meta.get('fingerprint') == current
        print(f'fingerprint match (reuse, no retrain): {FOLD0_FINGERPRINT_MATCH}')
        if not FOLD0_FINGERPRINT_MATCH:
            print('WARNING: checkpoint present but cfg or semantic_filament_lib.py differs from '
                  'run 3, so fold 0 WILL retrain (~8h). Fold-1 training stands down.')
    else:
        # Retraining fold 0 (8h) AND fold 1 (5.5h) does not fit in 12h, so the fold-1 cell stands
        # down rather than letting the notebook die at hour 12 with nothing written.
        print('WARNING: fold 0 must be retrained. Fold-1 training will be disabled so the '
              'session still finishes and writes a submission.')
else:
    print('Semantic artifact restore skipped.')

# One line that says whether this session can do what it was planned to do.
_plan = 'reuse fold 0, train fold 1' if FOLD0_FINGERPRINT_MATCH else 'fold 1 disabled'
print('')
print(f'SESSION READY: fold0_restored={FOLD0_RESTORED} fingerprint_match={FOLD0_FINGERPRINT_MATCH} -> {_plan}')

if KAGGLE_DATA_AVAILABLE and RUN_SEMANTIC and globals().get('REQUIRE_FOLD0_RESTORE'):
    assert FOLD0_FINGERPRINT_MATCH, (
        'Fold-0 restore FAILED, so this session would retrain for ~8h and run out of time before '
        'at minute 5, instead of at hour 12. Check: is the artifacts dataset attached? Is it '
        'the ONLY attached source carrying semantic_models/ -- find_attached_directory refuses '
        'to choose between two candidates. Does semantic_cfg still match the fingerprint in '
        'segmenter_meta.json? epochs=35 is load-bearing: changing ANY hashed cfg field '
        'invalidates the checkpoint. If a retrain IS the intent, set REQUIRE_FOLD0_RESTORE = '
        'False above.')


attached inputs: ['competitions', 'notebooks']
  semantic_models: restored from /kaggle/input/notebooks/alok158/solar-filament/solar_filament_pq/semantic_models
  semantic_prob_cache: restored from /kaggle/input/notebooks/alok158/solar-filament/solar_filament_pq/semantic_prob_cache
fold-0 checkpoint usable: True; 564 cached probability maps
fingerprint match (reuse, no retrain): False

SESSION READY: fold0_restored=True fingerprint_match=False -> fold 1 disabled


In [28]:
# [v25] Training is split into chunks of 3 epochs across the next 10 cells.
#
# Why: Kaggle commits a cell's output only when the CELL finishes. The first v25 push ran
# training as a single 9h cell, so the log stayed empty for 9h with no view of progress, epoch
# time, or whether the schedule was behaving. `kaggle kernels output` returns nothing mid-run
# either (verified 2026-10-11), so writing a progress file does not help while the kernel lives.
#
# Each call trains at most 3 epochs, saves the full training state (model, optimizer,
# scheduler, snapshots, elapsed) and returns, so the cell ends and its epoch lines become
# visible. cfg.max_train_hours is the budget across ALL calls, not per call. Calls made after
# training completes return immediately (fingerprint matches), so surplus cells cost nothing.
SEMANTIC_CHUNK = 3
semantic_checkpoint = None

In [29]:
# [v25 chunk 1/10] epochs 1-3 of 26
if KAGGLE_DATA_AVAILABLE and RUN_SEMANTIC:
    semantic_checkpoint = sfl.train_semantic_fold(
        0, fold_manifest, paths['train'], indices, selected_image_ids, semantic_cfg, WORK_DIR,
        max_epochs_this_call=SEMANTIC_CHUNK,
    )
    _m = json.loads((Path(semantic_checkpoint).parent / 'segmenter_meta.json').read_text())
    print(f"chunk 1/10: status={_m['status']} "
          f"epochs_completed={_m.get('epochs_completed')} best_pq={_m.get('best_pq'):.5f}")
else:
    print('Semantic training skipped.')

model.safetensors:   0%|          | 0.00/111M [00:00<?, ?B/s]

fold0 semantic e1/26:   0%|          | 0/566 [00:00<?, ?it/s]

fold=0 semantic epoch=1/26 loss=1.0102 (no validation) lr_mult=1.000 13.4min


fold0 semantic e2/26:   0%|          | 0/566 [00:00<?, ?it/s]

fold=0 semantic epoch=2/26 loss=0.7090 (no validation) lr_mult=0.996 12.7min


fold0 semantic e3/26:   0%|          | 0/566 [00:00<?, ?it/s]

fold=0 semantic epoch=3/26 loss=0.5745 val_pq=0.34970 sq=0.6456 rq=0.5416 tp=1187 fp=1433 fn=576 lr_mult=0.985 16.6min
fold=0 semantic: chunk done at epoch 3/26 (0.71h of 9.0h) -- state saved, resumable
chunk 1/10: status=in_progress epochs_completed=3 best_pq=0.34970


In [30]:
# [v25 chunk 2/10] epochs 4-6 of 26
if KAGGLE_DATA_AVAILABLE and RUN_SEMANTIC:
    semantic_checkpoint = sfl.train_semantic_fold(
        0, fold_manifest, paths['train'], indices, selected_image_ids, semantic_cfg, WORK_DIR,
        max_epochs_this_call=SEMANTIC_CHUNK,
    )
    _m = json.loads((Path(semantic_checkpoint).parent / 'segmenter_meta.json').read_text())
    print(f"chunk 2/10: status={_m['status']} "
          f"epochs_completed={_m.get('epochs_completed')} best_pq={_m.get('best_pq'):.5f}")
else:
    print('Semantic training skipped.')

fold=0 semantic: resuming at epoch 4/26 (best_pq 0.34970, 0.71h already spent)


fold0 semantic e4/26:   0%|          | 0/566 [00:00<?, ?it/s]

fold=0 semantic epoch=4/26 loss=0.4987 (no validation) lr_mult=0.968 12.7min


fold0 semantic e5/26:   0%|          | 0/566 [00:00<?, ?it/s]

fold=0 semantic epoch=5/26 loss=0.4727 (no validation) lr_mult=0.943 12.7min


fold0 semantic e6/26:   0%|          | 0/566 [00:00<?, ?it/s]

fold=0 semantic epoch=6/26 loss=0.4578 val_pq=0.37547 sq=0.6538 rq=0.5743 tp=1185 fp=1179 fn=578 lr_mult=0.911 16.2min
fold=0 semantic: chunk done at epoch 6/26 (1.40h of 9.0h) -- state saved, resumable
chunk 2/10: status=in_progress epochs_completed=6 best_pq=0.37547


In [31]:
# [v25 chunk 3/10] epochs 7-9 of 26
if KAGGLE_DATA_AVAILABLE and RUN_SEMANTIC:
    semantic_checkpoint = sfl.train_semantic_fold(
        0, fold_manifest, paths['train'], indices, selected_image_ids, semantic_cfg, WORK_DIR,
        max_epochs_this_call=SEMANTIC_CHUNK,
    )
    _m = json.loads((Path(semantic_checkpoint).parent / 'segmenter_meta.json').read_text())
    print(f"chunk 3/10: status={_m['status']} "
          f"epochs_completed={_m.get('epochs_completed')} best_pq={_m.get('best_pq'):.5f}")
else:
    print('Semantic training skipped.')

fold=0 semantic: resuming at epoch 7/26 (best_pq 0.37547, 1.40h already spent)


fold0 semantic e7/26:   0%|          | 0/566 [00:00<?, ?it/s]

fold=0 semantic epoch=7/26 loss=0.4418 (no validation) lr_mult=0.874 12.7min


fold0 semantic e8/26:   0%|          | 0/566 [00:00<?, ?it/s]

fold=0 semantic epoch=8/26 loss=0.4473 (no validation) lr_mult=0.832 12.7min


fold0 semantic e9/26:   0%|          | 0/566 [00:00<?, ?it/s]

fold=0 semantic epoch=9/26 loss=0.4298 val_pq=0.36061 sq=0.6625 rq=0.5443 tp=1201 fp=1449 fn=562 lr_mult=0.784 16.4min
fold=0 semantic: chunk done at epoch 9/26 (2.10h of 9.0h) -- state saved, resumable
chunk 3/10: status=in_progress epochs_completed=9 best_pq=0.37547


In [32]:
# [v25 chunk 4/10] epochs 10-12 of 26
if KAGGLE_DATA_AVAILABLE and RUN_SEMANTIC:
    semantic_checkpoint = sfl.train_semantic_fold(
        0, fold_manifest, paths['train'], indices, selected_image_ids, semantic_cfg, WORK_DIR,
        max_epochs_this_call=SEMANTIC_CHUNK,
    )
    _m = json.loads((Path(semantic_checkpoint).parent / 'segmenter_meta.json').read_text())
    print(f"chunk 4/10: status={_m['status']} "
          f"epochs_completed={_m.get('epochs_completed')} best_pq={_m.get('best_pq'):.5f}")
else:
    print('Semantic training skipped.')

fold=0 semantic: resuming at epoch 10/26 (best_pq 0.37547, 2.10h already spent)


fold0 semantic e10/26:   0%|          | 0/566 [00:00<?, ?it/s]

fold=0 semantic epoch=10/26 loss=0.4247 (no validation) lr_mult=0.732 12.7min


fold0 semantic e11/26:   0%|          | 0/566 [00:00<?, ?it/s]

fold=0 semantic epoch=11/26 loss=0.4151 (no validation) lr_mult=0.677 12.7min


fold0 semantic e12/26:   0%|          | 0/566 [00:00<?, ?it/s]

fold=0 semantic epoch=12/26 loss=0.4158 val_pq=0.35560 sq=0.6567 rq=0.5415 tp=1203 fp=1477 fn=560 lr_mult=0.620 16.5min
fold=0 semantic: chunk done at epoch 12/26 (2.80h of 9.0h) -- state saved, resumable
chunk 4/10: status=in_progress epochs_completed=12 best_pq=0.37547


In [33]:
# [v25 chunk 5/10] epochs 13-15 of 26
if KAGGLE_DATA_AVAILABLE and RUN_SEMANTIC:
    semantic_checkpoint = sfl.train_semantic_fold(
        0, fold_manifest, paths['train'], indices, selected_image_ids, semantic_cfg, WORK_DIR,
        max_epochs_this_call=SEMANTIC_CHUNK,
    )
    _m = json.loads((Path(semantic_checkpoint).parent / 'segmenter_meta.json').read_text())
    print(f"chunk 5/10: status={_m['status']} "
          f"epochs_completed={_m.get('epochs_completed')} best_pq={_m.get('best_pq'):.5f}")
else:
    print('Semantic training skipped.')

fold=0 semantic: resuming at epoch 13/26 (best_pq 0.37547, 2.80h already spent)


fold0 semantic e13/26:   0%|          | 0/566 [00:00<?, ?it/s]

fold=0 semantic epoch=13/26 loss=0.4094 (no validation) lr_mult=0.560 12.8min


fold0 semantic e14/26:   0%|          | 0/566 [00:00<?, ?it/s]

fold=0 semantic epoch=14/26 loss=0.4066 (no validation) lr_mult=0.500 12.7min


fold0 semantic e15/26:   0%|          | 0/566 [00:00<?, ?it/s]

fold=0 semantic epoch=15/26 loss=0.4098 val_pq=0.37755 sq=0.6641 rq=0.5686 tp=1217 fp=1301 fn=546 lr_mult=0.440 16.3min
fold=0 semantic: chunk done at epoch 15/26 (3.50h of 9.0h) -- state saved, resumable
chunk 5/10: status=in_progress epochs_completed=15 best_pq=0.37755


In [34]:
# [v25 chunk 6/10] epochs 16-18 of 26
if KAGGLE_DATA_AVAILABLE and RUN_SEMANTIC:
    semantic_checkpoint = sfl.train_semantic_fold(
        0, fold_manifest, paths['train'], indices, selected_image_ids, semantic_cfg, WORK_DIR,
        max_epochs_this_call=SEMANTIC_CHUNK,
    )
    _m = json.loads((Path(semantic_checkpoint).parent / 'segmenter_meta.json').read_text())
    print(f"chunk 6/10: status={_m['status']} "
          f"epochs_completed={_m.get('epochs_completed')} best_pq={_m.get('best_pq'):.5f}")
else:
    print('Semantic training skipped.')

fold=0 semantic: resuming at epoch 16/26 (best_pq 0.37755, 3.50h already spent)


fold0 semantic e16/26:   0%|          | 0/566 [00:00<?, ?it/s]

fold=0 semantic epoch=16/26 loss=0.3960 (no validation) lr_mult=0.380 12.7min


fold0 semantic e17/26:   0%|          | 0/566 [00:00<?, ?it/s]

fold=0 semantic epoch=17/26 loss=0.3957 (no validation) lr_mult=0.323 12.7min


fold0 semantic e18/26:   0%|          | 0/566 [00:00<?, ?it/s]

fold=0 semantic epoch=18/26 loss=0.3906 val_pq=0.39270 sq=0.6675 rq=0.5883 tp=1257 fp=1253 fn=506 lr_mult=0.268 16.4min
fold=0 semantic: chunk done at epoch 18/26 (4.20h of 9.0h) -- state saved, resumable
chunk 6/10: status=in_progress epochs_completed=18 best_pq=0.39270


In [35]:
# [v25 chunk 7/10] epochs 19-21 of 26
if KAGGLE_DATA_AVAILABLE and RUN_SEMANTIC:
    semantic_checkpoint = sfl.train_semantic_fold(
        0, fold_manifest, paths['train'], indices, selected_image_ids, semantic_cfg, WORK_DIR,
        max_epochs_this_call=SEMANTIC_CHUNK,
    )
    _m = json.loads((Path(semantic_checkpoint).parent / 'segmenter_meta.json').read_text())
    print(f"chunk 7/10: status={_m['status']} "
          f"epochs_completed={_m.get('epochs_completed')} best_pq={_m.get('best_pq'):.5f}")
else:
    print('Semantic training skipped.')

fold=0 semantic: resuming at epoch 19/26 (best_pq 0.39270, 4.20h already spent)


fold0 semantic e19/26:   0%|          | 0/566 [00:00<?, ?it/s]

fold=0 semantic epoch=19/26 loss=0.3903 (no validation) lr_mult=0.216 12.7min


fold0 semantic e20/26:   0%|          | 0/566 [00:00<?, ?it/s]

fold=0 semantic epoch=20/26 loss=0.3857 (no validation) lr_mult=0.168 12.7min


fold0 semantic e21/26:   0%|          | 0/566 [00:00<?, ?it/s]

fold=0 semantic epoch=21/26 loss=0.3850 val_pq=0.39932 sq=0.6666 rq=0.5991 tp=1241 fp=1139 fn=522 lr_mult=0.126 16.5min
fold=0 semantic: chunk done at epoch 21/26 (4.90h of 9.0h) -- state saved, resumable
chunk 7/10: status=in_progress epochs_completed=21 best_pq=0.39932


In [36]:
# [v25 chunk 8/10] epochs 22-24 of 26
if KAGGLE_DATA_AVAILABLE and RUN_SEMANTIC:
    semantic_checkpoint = sfl.train_semantic_fold(
        0, fold_manifest, paths['train'], indices, selected_image_ids, semantic_cfg, WORK_DIR,
        max_epochs_this_call=SEMANTIC_CHUNK,
    )
    _m = json.loads((Path(semantic_checkpoint).parent / 'segmenter_meta.json').read_text())
    print(f"chunk 8/10: status={_m['status']} "
          f"epochs_completed={_m.get('epochs_completed')} best_pq={_m.get('best_pq'):.5f}")
else:
    print('Semantic training skipped.')

fold=0 semantic: resuming at epoch 22/26 (best_pq 0.39932, 4.90h already spent)


fold0 semantic e22/26:   0%|          | 0/566 [00:00<?, ?it/s]

fold=0 semantic epoch=22/26 loss=0.3775 (no validation) lr_mult=0.089 12.8min


fold0 semantic e23/26:   0%|          | 0/566 [00:00<?, ?it/s]

fold=0 semantic epoch=23/26 loss=0.3851 (no validation) lr_mult=0.057 12.7min


fold0 semantic e24/26:   0%|          | 0/566 [00:00<?, ?it/s]

fold=0 semantic epoch=24/26 loss=0.3783 val_pq=0.39919 sq=0.6714 rq=0.5946 tp=1240 fp=1168 fn=523 lr_mult=0.032 16.4min
fold=0 semantic: chunk done at epoch 24/26 (5.59h of 9.0h) -- state saved, resumable
chunk 8/10: status=in_progress epochs_completed=24 best_pq=0.39932


In [37]:
# [v25 chunk 9/10] epochs 25-26 of 26
if KAGGLE_DATA_AVAILABLE and RUN_SEMANTIC:
    semantic_checkpoint = sfl.train_semantic_fold(
        0, fold_manifest, paths['train'], indices, selected_image_ids, semantic_cfg, WORK_DIR,
        max_epochs_this_call=SEMANTIC_CHUNK,
    )
    _m = json.loads((Path(semantic_checkpoint).parent / 'segmenter_meta.json').read_text())
    print(f"chunk 9/10: status={_m['status']} "
          f"epochs_completed={_m.get('epochs_completed')} best_pq={_m.get('best_pq'):.5f}")
else:
    print('Semantic training skipped.')

fold=0 semantic: resuming at epoch 25/26 (best_pq 0.39932, 5.59h already spent)


fold0 semantic e25/26:   0%|          | 0/566 [00:00<?, ?it/s]

fold=0 semantic epoch=25/26 loss=0.3809 (no validation) lr_mult=0.015 12.8min


fold0 semantic e26/26:   0%|          | 0/566 [00:00<?, ?it/s]

fold=0 semantic epoch=26/26 loss=0.3807 val_pq=0.39377 sq=0.6677 rq=0.5898 tp=1245 fp=1214 fn=518 lr_mult=0.004 16.4min
chunk 9/10: status=complete epochs_completed=26 best_pq=0.39932


In [38]:
# [v25 chunk 10/10] epochs 28-26 of 26
if KAGGLE_DATA_AVAILABLE and RUN_SEMANTIC:
    semantic_checkpoint = sfl.train_semantic_fold(
        0, fold_manifest, paths['train'], indices, selected_image_ids, semantic_cfg, WORK_DIR,
        max_epochs_this_call=SEMANTIC_CHUNK,
    )
    _m = json.loads((Path(semantic_checkpoint).parent / 'segmenter_meta.json').read_text())
    print(f"chunk 10/10: status={_m['status']} "
          f"epochs_completed={_m.get('epochs_completed')} best_pq={_m.get('best_pq'):.5f}")
else:
    print('Semantic training skipped.')

chunk 10/10: status=complete epochs_completed=26 best_pq=0.39932


In [39]:
# [v25] Training summary -- the numbers that decide whether the bundle worked.
if KAGGLE_DATA_AVAILABLE and RUN_SEMANTIC and semantic_checkpoint is not None:
    _m = json.loads((Path(semantic_checkpoint).parent / 'segmenter_meta.json').read_text())
    print(f"status            : {_m['status']}")
    print(f"epochs_completed  : {_m.get('epochs_completed')} / {_m.get('epochs_requested')}")
    print(f"cosine_annealed   : {_m.get('cosine_annealed')}")
    print(f"best_pq           : {_m.get('best_pq')}")
    print(f"swa_pq            : {_m.get('swa_pq')}  over {_m.get('swa_snapshots')} snapshots")
    assert _m['status'] == 'complete', (
        'Training did not finish inside max_train_hours. Add chunk cells, or lower '
        'semantic_cfg.epochs so the cosine anneals within the budget.')
    print('semantic fold 0 checkpoint:', semantic_checkpoint)

status            : complete
epochs_completed  : 26 / 26
cosine_annealed   : True
best_pq           : 0.3993193707725514
swa_pq            : None  over 0 snapshots
semantic fold 0 checkpoint: /kaggle/working/solar_filament_pq/semantic_models/fold_0/segmenter.pt


### Head to head: semantic vs. embedding variants


In [40]:
if KAGGLE_DATA_AVAILABLE and RUN_SEMANTIC:
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    semantic_models = sfl.load_segmenters([semantic_checkpoint], semantic_cfg, device)
    semantic_val_files = fold_manifest.loc[fold_manifest['fold'] == 0, 'filename'].tolist()
    if semantic_cfg.debug:
        semantic_val_files = semantic_val_files[:8]
    semantic_cache = sfl.cache_probability_maps(
        semantic_models, semantic_val_files, paths['train'], semantic_cfg, device,
        WORK_DIR / 'semantic_prob_cache',
        sfl.probability_cache_key([semantic_checkpoint], semantic_cfg),
    )
    semantic_rows = sfl.run_oof_semantic(semantic_cache, indices, semantic_cfg)
    semantic_rows.to_csv(WORK_DIR / 'oof_by_annotator_semantic.csv', index=False)
    ci_low, ci_high = bootstrap_pq_ci(semantic_rows, n=1000, seed=cfg.seed)
    semantic_summary = pd.DataFrame([{'variant': 'semantic', **aggregate_pq(semantic_rows),
                                      'pq_ci_low': ci_low, 'pq_ci_high': ci_high}])
    head_to_head = pd.concat([semantic_summary, cv_summary] if cv_summary is not None else [semantic_summary],
                             ignore_index=True).sort_values('pq', ascending=False).reset_index(drop=True)
    head_to_head.to_csv(WORK_DIR / 'pipeline_head_to_head.csv', index=False)
    display(head_to_head.round(5))
else:
    semantic_cache = semantic_rows = head_to_head = None
    print('Semantic head to head skipped.')


oof decode:   0%|          | 0/141 [00:00<?, ?it/s]

,variant,pq,sq,rq,tp,fp,fn,n_images,pq_image_mean,pq_ci_low,pq_ci_high
0,semantic,0.40387,0.66916,0.60355,1259,1150,504,141,0.40577,0.38358,0.42315


### Near-miss diagnostic (free, from the cached maps)

Pooled PQ never says *why*. This splits every FN into fully-missed vs near-miss and every FP into
spurious vs near-miss, and reports how many predictions overlap each GT filament. Under PQ a
near-miss is the expensive case and the leveraged fix: converting one adds a TP and removes an FP
*and* an FN at once, so ~200 conversions is worth about +0.064. These numbers decide whether the
decode search below is worth running at all.

In [41]:
# [session4]
if KAGGLE_DATA_AVAILABLE and RUN_SEMANTIC:
    failure_counts, failure_pairs = ss.semantic_failure_breakdown(semantic_cache, indices, semantic_cfg)
    failure_summary = ss.summarise_failures(failure_counts, failure_pairs, semantic_cfg.pq_match_iou)
    for key, value in failure_summary.items():
        print(f'{key:28s} {value:.4g}')
    failure_counts.to_csv(WORK_DIR / 'semantic_failure_counts.csv', index=False)
    failure_pairs.to_csv(WORK_DIR / 'semantic_failure_pairs.csv', index=False)

    # The plan's decision table, evaluated in code so the branch is recorded in the log.
    near, missed = failure_summary['near_miss_fn'], failure_summary['fully_missed_fn']
    if near >= 300 and failure_summary['near_miss_median_iou'] >= 0.35:
        print('\n-> near-miss dominated with real overlap: decode search is worth running.')
    elif near >= 300:
        print('\n-> near misses overlap only incidentally (median IoU low): no decode knob '
              'recovers these. The refiner/resolution is the lever, not the sweep.')
    if missed > 400:
        print('-> recall-bound: decode knobs and ensemble weights are both largely irrelevant.')
    # Misses being smaller than matches on average says nothing about the filter -- that is
    # true of almost any threshold. minimum_area can only explain misses it could actually have
    # deleted, so compare its value against the LOWER TAIL of the missed areas, not the median.
    if not failure_summary['missed_median_area']:
        pass                      # nothing was fully missed; the filter has no case to answer
    elif failure_summary['missed_area_p10'] < semantic_cfg.minimum_area:
        print(f"-> minimum_area={semantic_cfg.minimum_area} sits above the 10th percentile of "
              f"missed areas ({failure_summary['missed_area_p10']:.0f}): it is deleting real "
              f"filaments in the small tail. Sweep it downward.")
    elif failure_summary['missed_median_area']:
        print(f"-> minimum_area={semantic_cfg.minimum_area} is below even the 10th percentile of "
              f"missed areas ({failure_summary['missed_area_p10']:.0f}): the filter is NOT what "
              f"loses them. They are faint, not small -- the threshold is the lever.")
    if failure_summary['mean_pred_per_touched_gt'] >= 1.6:
        print('-> oversplit: one filament decoded as several. max_gap_px / bridge_px / '
              'max_angle_deg are the levers.')
else:
    failure_summary = None
    print('Failure breakdown skipped.')

fully_missed_fn              92
near_miss_fn                 411
spurious_fp                  735
near_miss_fp                 414
near_miss_median_iou         0.3525
missed_median_area           570.5
missed_area_p10              255.3
matched_median_area          1319
mean_pred_per_touched_gt     1.044

-> near-miss dominated with real overlap: decode search is worth running.
-> minimum_area=240 is below even the 10th percentile of missed areas (255): the filter is NOT what loses them. They are faint, not small -- the threshold is the lever.


### Near-miss extent: too narrow, or too short?

Run 3 measured **1.075 predictions per touched GT**, so the near misses above are *not*
fragmentation -- `bridge_px` / `max_gap_px` / `max_angle_deg` cannot fix a one-to-one match.
That leaves exactly two causes, and they need different levers:

| Reading | Cause | Lever |
|---|---|---|
| `width_ratio` ~0.4, `length_ratio` ~1 | traces the whole filament, wrong thickness | `width_px`, `prob_threshold` -- free, these same cached maps |
| `length_ratio` ~0.4, `width_ratio` ~1 | truncated | hysteresis (free), or a larger `cldice_weight` (retrain) |

Why it is worth the cell: converting a near miss adds a TP while removing an FP *and* an FN,
so it is denominator-neutral and pure numerator gain. At run 3's counts, 200 conversions is
about **+0.063** and all 396 is about **+0.125**. Run 3 swept `width_px` over `[-1, 0, 1]` --
about +-12% of the 8.31px median annotation width -- so a 2-4px deficit was never in the grid.


In [42]:
# [session4]
if KAGGLE_DATA_AVAILABLE and RUN_SEMANTIC and failure_summary is not None:
    extent = ss.near_miss_extent(semantic_cache, indices, semantic_cfg)
    extent.to_csv(WORK_DIR / 'semantic_near_miss_extent.csv', index=False)
    print(f'{len(extent)} near-miss pairs (0 < IoU <= {semantic_cfg.pq_match_iou})')

    if len(extent) < 30:
        print('Too few near misses to read a direction from.')
    else:
        width_ratio = extent['width_ratio'].median()
        length_ratio = extent['length_ratio'].median()
        for name, value in (('median best_iou', extent['best_iou'].median()),
                            ('median gt_width', extent['gt_width'].median()),
                            ('median pred_width', extent['pred_width'].median()),
                            ('median width_ratio', width_ratio),
                            ('median gt_length', extent['gt_length'].median()),
                            ('median pred_length', extent['pred_length'].median()),
                            ('median length_ratio', length_ratio)):
            print(f'{name:22s} {value:.4g}')

        # IoU for a ribbon is roughly the smaller extent over the larger, so the ratio that
        # tracks the median IoU is the one doing the damage. Deliberately not a tie-break on a
        # hair's difference: when both are off, both levers are needed and the cell says so.
        # Guard before any verdict: ribbon_extent reads width from the mean distance transform,
        # and barbs (MAGFiLO polygons include them) sit at low DT values, so a barbed GT measures
        # thinner and LONGER than it is -- which looks exactly like truncation. pred_area/gt_area
        # needs no shape model at all, so if it disagrees with width_ratio * length_ratio the
        # ribbon model has broken down and neither width_px nor hysteresis is the lever.
        area_ratio = (extent['pred_area'] / extent['gt_area']).median()
        predicted = width_ratio * length_ratio
        agreement = area_ratio / predicted if predicted else 0.0
        print(f"{'measured area_ratio':22s} {area_ratio:.4g}")
        print(f"{'width x length':22s} {predicted:.4g}   (agreement {agreement:.2f})")
        if not 0.80 <= agreement <= 1.25:
            print()
            print('-> SHAPE COMPLEXITY (barbs): the ribbon model does not describe these '
                  'masks, so the narrow/short verdict below is NOT reliable. No decode knob '
                  'adds appendages -- see EdgeAttNet/OPT-HQ-Net and the refiner, not the sweep.')

        narrow, short = width_ratio < 0.8, length_ratio < 0.8
        if narrow and not short:
            deficit = extent['gt_width'].median() - extent['pred_width'].median()
            print(f'\n-> TOO NARROW by ~{deficit:.1f}px at full length. Sweep width_px over a grid '
                  f'that actually reaches it (try 0..{int(np.ceil(deficit)) + 1}) and lower '
                  f'prob_threshold. Free, from these cached maps -- no retraining.')
        elif short and not narrow:
            print('\n-> TRUNCATED at the right thickness. No width knob recovers this. Try '
                  'hysteresis (prob_threshold_low in 0.48-0.52, the regime never tested) first; '
                  'if that stalls, raise cldice_weight and retrain.')
        elif narrow and short:
            print('\n-> BOTH: the predictions are smaller than the filaments in every direction. '
                  'That is a threshold problem before it is a shape problem -- sweep '
                  'prob_threshold downward with hysteresis to hold the spurious FPs back.')
        else:
            print('\n-> Neither extent is systematically off: the predictions are the right size '
                  'but misplaced. Decode knobs will not fix that; it is a model-quality gap.')
else:
    print('Near-miss extent skipped.')


407 near-miss pairs (0 < IoU <= 0.5)
median best_iou        0.3541
median gt_width        8.579
median pred_width      10.21
median width_ratio     1.166
median gt_length       134.3
median pred_length     169.7
median length_ratio    1.256
measured area_ratio    1.557
width x length         1.465   (agreement 1.06)

-> Neither extent is systematically off: the predictions are the right size but misplaced. Decode knobs will not fix that; it is a model-quality gap.


### Semantic decode sweep (free, from the cached maps)

One knob at a time, holding the others at their current best, two rounds. That's about 50
decodes per image, against thousands for a full product. New knobs: `width_px` (grow or
shrink instances to the annotators' drawing width) and `prob_threshold_low` (hysteresis:
faint continuations stay attached to strong cores). `max_gap_px=0` turns fragment merging
off. With `APPLY_SWEEP_WINNER`, the submission uses the winning settings.


In [43]:
# [session4b] Block decode search: blocks of COUPLED knobs, not one knob at a time.
# Run 3's single-knob sweep stopped with max_angle_deg and minimum_area both at their grid
# MAXIMUM, and round 2 moved nothing -- the signature of a ridge between coupled knobs, which a
# one-at-a-time method walks to and then reports as convergence. Two knobs that were never swept
# at all matter here: bridge_px (merging two fragments into one label does NOT fill the gap
# between them, so a merged instance keeps a hole that depresses IoU on exactly the near-miss
# cases) and min_fragment_area.
if KAGGLE_DATA_AVAILABLE and RUN_SEMANTIC:
    SWEEP_WORKERS = 4  # Kaggle gives 4 vCPU; the GPU is idle during this cell

    search_files, confirm_files = ss.split_images(list(semantic_cache))
    search_cache = {k: semantic_cache[k] for k in search_files}
    confirm_cache = {k: semantic_cache[k] for k in confirm_files}
    print(f'decode search on {len(search_cache)} images, confirm on {len(confirm_cache)}')

    # Re-aimed by the v20 near-miss extent measurement (407 pairs, PROGRESS.md 2026-10-08):
    #   length_ratio 0.5576  -- predictions cover 56% of each filament's LENGTH
    #   width_ratio  1.145   -- and are ~1.6px too WIDE, not too thin
    #   area_ratio 0.5585 vs width*length 0.6387 -> ribbon model holds to 87%, truncation is real
    # So the lever is continuation, not merging and not widening. Hysteresis is promoted to the
    # first block and COUPLED to prob_threshold, because what matters is the gap between the two,
    # not either alone; width_px is capped at 0 since the predictions are already too fat.
    DECODE_BLOCKS = [
        # THE block. A low threshold that extends through connected pixels is the only knob that
        # can put the missing 44% of length back on. Combinations with low >= high degrade to
        # plain thresholding inside decode_instances, so they cost a decode but never mislead.
        {'prob_threshold': [0.45, 0.50, 0.55, 0.60, 0.65],
         'prob_threshold_low': [None, 0.30, 0.38, 0.44, 0.50, 0.56]},
        # Still near-degenerate with the threshold, but now one-sided: width_ratio > 1 says the
        # answer is at or below 0, and run 3's sweep already picked 0 from a grid that could not
        # see further down.
        {'prob_threshold': [0.45, 0.50, 0.55, 0.60, 0.65],
         'width_px': [-3, -2, -1, 0]},
        # fully_missed_fn is 203 with a missed median area of 646 -- minimum_area at 160 is not
        # what loses them, but the lower tail is still worth probing downward.
        {'max_angle_deg': [35.0, 50.0, 65.0, 80.0, 90.0],
         'minimum_area': [32, 48, 96, 160, 240]},
        # DROPPED: the max_gap_px x bridge_px block. mean_pred_per_touched_gt = 1.075 means
        # essentially every touched GT is touched by exactly ONE prediction, so there is nothing
        # to merge and nothing for bridge_px to fill. Run 3's 'bridge_px is the likely lever'
        # hypothesis is disproven; the budget it used now goes to hysteresis above.
        {'min_fragment_area': [4, 8, 16, 32]},
        {'endpoint_radius': [10.0, 15.0, 25.0]},
        {'min_elongation': [1.2, 1.5, 2.5]},
    ]
    DECODE_STARTS = {
        'lib_default': semantic_cfg,
        # Deliberately on the far side of the ridge, so one start does not decide everything.
        # width_px=-2 and a live hysteresis gap: the far corner should sit on the far side of
        # the ridge the measurement points at, not the one run 3 guessed at.
        'far_corner': replace(semantic_cfg, prob_threshold=0.45, width_px=-2, minimum_area=96,
                              prob_threshold_low=0.30, max_angle_deg=80.0),
    }

    # Cost probe. The search is 2 starts x 2 rounds x 113 configs; at ~0.4s per 2048 decode
    # that is ~1h on 4 workers, but decode cost scales with fragment count and this is the one
    # estimate that could overrun a 12h session. Measure it, project it, and trim the search to
    # fit rather than discovering the problem at hour 11.
    import math, time
    DECODE_SEARCH_BUDGET_S = 5400  # 1.5h

    _probe = list(search_cache.values())[:3]
    _t0 = time.perf_counter()
    for _path in _probe:
        sfl.decode_instances(sfl.load_cached_probability(_path), semantic_cfg)
    per_decode = (time.perf_counter() - _t0) / max(len(_probe), 1)
    configs_per_round = sum(math.prod(len(v) for v in block.values()) for block in DECODE_BLOCKS)
    rounds, starts = 2, list(DECODE_STARTS)

    def _projected(n_starts, n_rounds):
        return (n_starts * n_rounds * configs_per_round * len(search_cache)
                * per_decode / max(SWEEP_WORKERS, 1))

    while _projected(len(starts), rounds) > DECODE_SEARCH_BUDGET_S and (len(starts) > 1 or rounds > 1):
        if len(starts) > 1:
            starts = starts[:1]          # drop the second start before dropping a round:
        else:
            rounds -= 1                  # round 2 is where coupled knobs pay off (see tests)
    DECODE_STARTS = {k: DECODE_STARTS[k] for k in starts}
    print(f'decode probe: {per_decode:.3f}s/decode, {configs_per_round} configs/round -> '
          f'{len(starts)} start(s) x {rounds} round(s) '
          f'= {_projected(len(starts), rounds) / 60:.0f} min on {SWEEP_WORKERS} workers')

    decode_candidates = {}
    for label, start in DECODE_STARTS.items():
        best, history = ss.block_sweep_decode(search_cache, indices, start, DECODE_BLOCKS,
                                              rounds=rounds, workers=SWEEP_WORKERS)
        pq = sfl.aggregate_pq(sfl.run_oof_semantic(search_cache, indices, best))['pq']
        decode_candidates[label] = (best, pq)
        history.to_csv(WORK_DIR / f'decode_search_{label}.csv', index=False)
        print(f'  {label}: search-half PQ {pq:.5f}')
    tuned_cfg = max(decode_candidates.values(), key=lambda pair: pair[1])[0]

    # The keep rule, on the half the search never saw. Run 3's +0.009 was never tested this way:
    # 113 configs on 141 images manufactures ~+0.04 of optimism all by itself.
    confirm_new = sfl.run_oof_semantic(confirm_cache, indices, tuned_cfg)
    confirm_old = sfl.run_oof_semantic(confirm_cache, indices, semantic_cfg)
    decode_keep, decode_stats = ss.keep_change(confirm_new, confirm_old)
    print(f"confirm half: delta {decode_stats['delta']:+.5f} "
          f"CI [{decode_stats['ci_low']:+.5f}, {decode_stats['ci_high']:+.5f}] "
          f"P(gain)={decode_stats['p_gain']:.3f} -> {'KEEP' if decode_keep else 'REJECT'}")
    decode_fields = ['prob_threshold', 'prob_threshold_low', 'width_px', 'max_gap_px',
                     'max_angle_deg', 'minimum_area', 'bridge_px', 'min_fragment_area']
    print('default:', {f: getattr(semantic_cfg, f) for f in decode_fields})
    print('tuned:  ', {f: getattr(tuned_cfg, f) for f in decode_fields})

    if globals().get('DECODE_PIN'):
        # Pinned: neither the sweep winner nor the library default (see cell 27 for why).
        semantic_decode_cfg = replace(semantic_cfg, **DECODE_PIN)
        print('decode cfg in use: PINNED ->', DECODE_PIN)
    else:
        semantic_decode_cfg = tuned_cfg if (decode_keep and APPLY_SWEEP_WINNER) else semantic_cfg
        print('decode cfg in use:', 'tuned' if semantic_decode_cfg is tuned_cfg else 'library default')
else:
    tuned_cfg = decode_keep = None
    print('Semantic decode search skipped.')

decode search on 70 images, confirm on 71
decode probe: 0.238s/decode, 85 configs/round -> 2 start(s) x 2 round(s) = 24 min on 4 workers
[sweep] round 1/2 block 1/6 {'prob_threshold': 5, 'prob_threshold_low': 6}


sweep 30 cfg:   0%|          | 0/70 [00:00<?, ?it/s]

[sweep] round 1/2 block 2/6 {'prob_threshold': 5, 'width_px': 4}


sweep 20 cfg:   0%|          | 0/70 [00:00<?, ?it/s]

[sweep] round 1/2 block 3/6 {'max_angle_deg': 5, 'minimum_area': 5}


sweep 25 cfg:   0%|          | 0/70 [00:00<?, ?it/s]

... (44 outputs left out)

  far_corner: search-half PQ 0.42147


oof decode:   0%|          | 0/71 [00:00<?, ?it/s]

oof decode:   0%|          | 0/71 [00:00<?, ?it/s]

confirm half: delta +0.00583 CI [+0.00068, +0.01140] P(gain)=0.987 -> REJECT
default: {'prob_threshold': 0.6, 'prob_threshold_low': 0.56, 'width_px': 0, 'max_gap_px': 40.0, 'max_angle_deg': 65.0, 'minimum_area': 240, 'bridge_px': 0, 'min_fragment_area': 32}
tuned:   {'prob_threshold': 0.65, 'prob_threshold_low': None, 'width_px': 0, 'max_gap_px': 40.0, 'max_angle_deg': 65.0, 'minimum_area': 240, 'bridge_px': 0, 'min_fragment_area': 8}
decode cfg in use: PINNED -> {'prob_threshold': 0.6, 'prob_threshold_low': None, 'width_px': 0, 'max_gap_px': 40.0, 'max_angle_deg': 65.0, 'minimum_area': 240, 'min_fragment_area': 32, 'endpoint_radius': 15.0, 'min_elongation': 1.5}


### Instance filter

A small gradient-boosting model scores each decoded OOF instance by how likely it is to match
an annotator, from its shape and confidence. Instances below a threshold are dropped. The
threshold is picked on out-of-fold scores (split by image), and the filter is kept only if
it raises pooled PQ. Background: under PQ a near-miss costs twice a silent miss.


In [44]:
# [session4b] Instance filter, with the threshold anchored to its closed form.
# Dropping an instance changes the denominator by -0.5 whether it was a TP (tp-1, fn+1 => -1+0.5)
# or an FP (-0.5), and the numerator by -p*iou. So dropping pays iff p*iou < 0.5*PQ, i.e.
# score < 0.5*PQ/SQ. Run 3 spent a wide grid to land one step above that value; sweeping a narrow
# grid around it instead removes ~10 draws from the optimism budget for free.
#
# 2026-10-11: the filter is now spine-aware and deployed differently.
#   * features: the 5 spine features join the 10 mask ones when the checkpoint has a spine head.
#     Measured on fold 0, s_mean alone separates matched from unmatched at AUC 0.778 (better than
#     any mask feature) and all five lift the filter's own OOF AUC 0.773 -> 0.795.
#   * deployment: average the GroupKFold models instead of refitting one on the whole table, and
#     translate the calibrated threshold into the DROP RATE it implies, then take that quantile of
#     the deployed scores. A refit model scores systematically higher, so copying the threshold
#     across would silently drop fewer instances than calibrated.
if KAGGLE_DATA_AVAILABLE and RUN_SEMANTIC and USE_INSTANCE_FILTER:
    filter_table = sfl.oof_instance_table(semantic_cache, indices, semantic_decode_cfg)
    feature_names = sfl.filter_feature_names(filter_table)
    has_spine = any(f in feature_names for f in sfl.SPINE_FILTER_FEATURES)
    print(f'{len(filter_table)} OOF instances; {len(feature_names)} features '
          f'({"WITH" if has_spine else "NO"} spine features)')
    if len(filter_table) < 50 or filter_table['match'].nunique() < 2:
        semantic_filter = None
        print(f'Instance filter skipped: only {len(filter_table)} OOF instances.')
    else:
        filter_scores, filter_models = sfl.cv_instance_scores(filter_table, features=feature_names)
        current = sfl.aggregate_pq(sfl.run_oof_semantic(semantic_cache, indices, semantic_decode_cfg))
        anchor = ss.filter_threshold_closed_form(current['pq'], current['sq'])
        grid = sorted({0.0, *[round(max(0.0, anchor + d), 4) for d in (-0.10, -0.05, 0.0, 0.05, 0.10)]})
        print(f"closed-form threshold 0.5*PQ/SQ = {anchor:.4f} (PQ {current['pq']:.5f}, SQ {current['sq']:.4f})")
        ranked = sfl.score_filter_thresholds(semantic_cache, indices, semantic_decode_cfg,
                                             filter_table, filter_scores, grid)
        display(ranked.round(5))
        best_threshold = float(ranked['threshold'].iloc[0])
        unfiltered = float(ranked.loc[ranked['threshold'] == 0.0, 'pq'].iloc[0])
        # Prefer the closed form on a near-tie: it is derived, not selected, so it cannot overfit.
        if abs(float(ranked['pq'].iloc[0]) - float(
                ranked.loc[(ranked['threshold'] - anchor).abs().idxmin(), 'pq'])) < 0.002:
            best_threshold = float(ranked.loc[(ranked['threshold'] - anchor).abs().idxmin(), 'threshold'])
            print(f'near-tie: preferring the closed form {best_threshold:.4f}')
        if best_threshold > 0 and float(ranked['pq'].iloc[0]) > unfiltered:
            drop_rate = float((filter_scores < best_threshold).mean())
            deployed_scores = sfl.predict_instance_scores(filter_models, filter_table, feature_names)
            deploy_threshold = float(np.quantile(deployed_scores, drop_rate)) if drop_rate > 0 else 0.0
            semantic_filter = (filter_models, deploy_threshold, feature_names)
            print(f'{len(filter_table)} OOF instances, {filter_table["match"].mean():.1%} match an '
                  f'annotator. Unfiltered PQ {unfiltered:.5f} -> threshold {best_threshold:.3f} '
                  f'PQ {float(ranked["pq"].iloc[0]):.5f} (filter on)')
            print(f'deployed: {len(filter_models)} averaged CV models, calibrated drop rate '
                  f'{drop_rate:.1%} -> quantile-matched threshold {deploy_threshold:.4f}')
        else:
            semantic_filter = None
            print(f'Filter does not beat unfiltered PQ {unfiltered:.5f}; left off.')
else:
    semantic_filter = None
    print('Instance filter skipped.')


1349 OOF instances; 15 features (WITH spine features)


oof decode:   0%|          | 0/141 [00:00<?, ?it/s]

closed-form threshold 0.5*PQ/SQ = 0.3042 (PQ 0.40752, SQ 0.6697)


,threshold,pq,sq,rq,tp,fp,fn,n_images,pq_image_mean
0,0.3542,0.44449,0.67267,0.66078,1123,513,640,141,0.44627
1,0.3042,0.44358,0.67227,0.65982,1157,587,606,141,0.44425
2,0.2542,0.44228,0.67200,0.65815,1185,653,578,141,0.44161
3,0.4042,0.44220,0.67391,0.65616,1083,455,680,141,0.44152
4,0.2042,0.43961,0.67159,0.65457,1209,722,554,141,0.43956
5,0.0000,0.40752,0.66972,0.60849,1262,1123,501,141,0.40882


near-tie: preferring the closed form 0.3042
1349 OOF instances, 55.2% match an annotator. Unfiltered PQ 0.40752 -> threshold 0.304 PQ 0.44449 (filter on)
deployed: 5 averaged CV models, calibrated drop rate 25.6% -> quantile-matched threshold 0.2491


## Fold 1: a second ensemble member, free snapshots, and a real held-out fold

The session's long pole. Fold 1 is the right thing to train rather than a second fold-0 model,
because it buys three things at once: a second member for the test ensemble, per-epoch snapshots
to average (SWA), and -- the actual prize -- **a fold the decode search never touched**. Settings
tuned on fold-0 maps can be confirmed on fold-1 maps: different images *and* a different model.
That is the only evidence that separates a real gain from the optimism a 113-config search
generates on its own.

Snapshots are captured by wrapping `sfl.score_files` (called once per epoch with the live model),
so `semantic_filament_lib.py` stays byte-identical and every existing checkpoint stays valid.

In [45]:
# [session4c]
# OFF for this session (2026-10-08). v20 overran the 12h cap and was cancelled, losing the
# sweep and the submission. With fold 0 now RESTORED rather than retrained there is room
# for fold 1 -- but the measurement says the next +0.05 is in the decode knobs, not in a
# second ensemble member, so this session banks the sweep and a submission first.
RUN_FOLD1 = False and globals().get('FOLD0_FINGERPRINT_MATCH', False)   # the ~5.5h training run;
# stands down automatically if fold 0 had to be retrained -- 8h + 5.5h does not fit 12h.
FOLD1_EPOCHS = 22           # epochs 19->27 bought only +0.011 in run 3; the tail is poor value
FOLD1_MAX_HOURS = 5.5
SNAPSHOT_KEEP = 10          # tail epochs retained for SWA (128MB each)

fold1_checkpoint = fold1_snapshots = None
if KAGGLE_DATA_AVAILABLE and RUN_SEMANTIC and RUN_FOLD1:
    import itertools

    fold1_cfg = replace(semantic_cfg, epochs=FOLD1_EPOCHS, max_train_hours=FOLD1_MAX_HOURS)
    snapshot_dir = WORK_DIR / 'semantic_snapshots' / 'fold_1'
    snapshot_dir.mkdir(parents=True, exist_ok=True)

    _orig_score_files = sfl.score_files
    _epoch_counter = itertools.count(1)

    def _snapshotting_score_files(models, *args, **kwargs):
        """score_files is called exactly once per epoch, with [model]. Capturing here gets EVERY
        epoch, not only the improving ones that torch.save keeps -- and costs no lib edit."""
        epoch = next(_epoch_counter)
        torch.save(models[0].state_dict(), snapshot_dir / f'e{epoch:03d}.pt')
        stale = sorted(snapshot_dir.glob('e*.pt'))[:-SNAPSHOT_KEEP]
        for path in stale:
            path.unlink()
        return _orig_score_files(models, *args, **kwargs)

    sfl.score_files = _snapshotting_score_files
    try:
        fold1_checkpoint = sfl.train_semantic_fold(
            1, fold_manifest, paths['train'], indices, selected_image_ids, fold1_cfg, WORK_DIR)
    finally:
        sfl.score_files = _orig_score_files

    fold1_snapshots = sorted(snapshot_dir.glob('e*.pt'))
    print(f'fold 1 checkpoint: {fold1_checkpoint}')
    print(f'{len(fold1_snapshots)} epoch snapshots retained for SWA')
else:
    print('Fold 1 training skipped.')

Fold 1 training skipped.


### SWA over the snapshot tail

This model has **no BatchNorm anywhere** -- `ConvBlock` and `feature_norms` are GroupNorm by
deliberate design and timm's ConvNeXt is LayerNorm -- so weight averaging needs no re-estimation
pass. Twelve snapshots collapse into one checkpoint at one inference cost, with zero selection
variance. That is the bar any searched ensemble weighting has to beat, and usually doesn't.

In [46]:
# [session4c]
fold1_best = None
if fold1_checkpoint is not None and fold1_snapshots:
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    fold1_val = fold_manifest.loc[fold_manifest['fold'] == 1, 'filename'].tolist()
    if semantic_cfg.debug:
        fold1_val = fold1_val[:8]

    swa_path = fold1_checkpoint.with_name('segmenter_swa.pt')
    torch.save(ss.average_state_dicts(fold1_snapshots), swa_path)
    print(f'SWA over {len(fold1_snapshots)} snapshots -> {swa_path.name}')

    fold1_scores = {}
    for label, path in (('best_epoch', fold1_checkpoint), ('swa', swa_path)):
        cache = sfl.cache_probability_maps(
            sfl.load_segmenters([path], semantic_cfg, device), fold1_val, paths['train'],
            semantic_cfg, device, WORK_DIR / 'semantic_prob_cache',
            sfl.probability_cache_key([path], semantic_cfg))
        rows = sfl.run_oof_semantic(cache, indices, semantic_decode_cfg)
        fold1_scores[label] = (path, cache, rows, sfl.aggregate_pq(rows))
        print(f"  fold1 {label:10s} PQ {fold1_scores[label][3]['pq']:.5f} "
              f"sq {fold1_scores[label][3]['sq']:.4f}")

    swa_keep, swa_stats = ss.keep_change(fold1_scores['swa'][2], fold1_scores['best_epoch'][2],
                                         min_gain=0.0)
    print(f"SWA vs best epoch: delta {swa_stats['delta']:+.5f} P(gain)={swa_stats['p_gain']:.3f}")
    # No min_gain here: SWA costs nothing extra at inference, so any non-negative delta is free.
    fold1_best = swa_path if swa_stats['delta'] >= 0 else fold1_checkpoint
    fold1_cache = fold1_scores['swa' if fold1_best is swa_path else 'best_epoch'][1]
    print(f'fold 1 member in use: {fold1_best.name}')
else:
    fold1_cache = None
    print('SWA skipped.')

SWA skipped.


### Cross-fold confirmation of the decode settings

The payoff. The decode cfg was searched on fold-0 maps; here it meets fold-1 maps -- different
images, different model. A gain that survives this is real; one that doesn't was the search
fitting noise, and we ship the library defaults instead. Either way the question gets answered
without spending a submission on a control run.

In [47]:
# [session4c]
if fold1_cache is not None and tuned_cfg is not None:
    cross_new = sfl.run_oof_semantic(fold1_cache, indices, tuned_cfg)
    cross_old = sfl.run_oof_semantic(fold1_cache, indices, semantic_cfg)
    cross_keep, cross_stats = ss.keep_change(cross_new, cross_old)
    print(f"fold-1 cross-check of the fold-0 decode cfg: "
          f"delta {cross_stats['delta']:+.5f} "
          f"CI [{cross_stats['ci_low']:+.5f}, {cross_stats['ci_high']:+.5f}] "
          f"P(gain)={cross_stats['p_gain']:.3f} -> {'CONFIRMED' if cross_keep else 'NOT CONFIRMED'}")
    if not cross_keep and semantic_decode_cfg is not semantic_cfg:
        semantic_decode_cfg = semantic_cfg
        print('Decode tuning did not transfer across folds; reverting to library defaults.')
    cross_summary = {'fold0_confirm_delta': decode_stats['delta'],
                     'fold0_confirm_p': decode_stats['p_gain'],
                     'fold1_cross_delta': cross_stats['delta'],
                     'fold1_cross_p': cross_stats['p_gain']}
    pd.DataFrame([cross_summary]).to_csv(WORK_DIR / 'decode_transfer.csv', index=False)
    display(pd.DataFrame([cross_summary]).round(5))
else:
    print('Cross-fold confirmation skipped.')

Cross-fold confirmation skipped.


### Ensemble members for the submission

Probability-level averaging across fold models is valid for semantic maps. Integer weights are
deployable with no library change: `predict_probability` divides by `len(models) * len(flips)`,
so repeating a module object in the list *is* a weight. Total weight is capped because test
inference costs `180 x sum(weights) x 4` forward passes -- that, not the search, is the binding
constraint.

In [48]:
# [session4c]
semantic_members = [p for p in [semantic_checkpoint, fold1_best] if p is not None]

# Uniform weights, deliberately. A searched weighting is only defensible when it can be
# MEASURED, and these two members hold out different folds: fold 1's model has seen fold 0's
# images, so neither fold can score the pair without leaking. `ss.bagged_ensemble_weights` is
# there for a pool that shares one holdout (several snapshots, or a second encoder family on
# the same split) -- and even then the honest prior is that uniform wins, because a few hundred
# selection draws against one fold are worth more noise than any real gain measured here.
semantic_weights = [1] * len(semantic_members)

print(f'{len(semantic_members)} ensemble member(s), uniform weights {semantic_weights}:')
for path, weight in zip(semantic_members, semantic_weights):
    print(f'  x{weight}  {path.parent.name}/{path.name}')

1 ensemble member(s), uniform weights [1]:
  x1  fold_0/segmenter.pt


## Final Model

### 11a. Single submission model, trained on the full labeled set

The 5-fold CV above is evaluation only -- an honest, leak-free PQ estimate of this recipe. No CV fold model is used for the submission. This cell trains one final detector+embedding-net pair on ~(1 - `final_holdout_fraction`) of *all* labeled data (not any single fold's ~80%), early-stopped against a fresh stratified holdout of that same size (`build_final_manifest`), saved under `WORK_DIR_FINAL` -- separate from the CV folds' checkpoints.


In [49]:
if KAGGLE_DATA_AVAILABLE and RUN_FINAL_FIT:
    train_final_model(observations, paths, selected_image_ids, indices, cfg, WORK_DIR_FINAL,
                      stage1=RUN_STAGE1, stage2=RUN_STAGE2)
    print('Final submission model ready:', sorted(WORK_DIR_FINAL.glob('models/fold_0/*.pt')))
else:
    print('Final-fit training skipped.')


Final-fit training skipped.


### Final-fit progression

Same training-progression view as section 10, for the single submission model
(trained on ~95% of all labeled data -- see section 11a).


In [50]:
if KAGGLE_DATA_AVAILABLE and RUN_FINAL_FIT:
    plot_refiner_progress({"final model": WORK_DIR_FINAL / "models" / "fold_0" / "refiner_meta.json"})
    show_detector_curves({"final model": WORK_DIR_FINAL / "detector_runs" / "fold_0" / "results.png"})
else:
    print("Final-fit diagnostics skipped.")


Final-fit diagnostics skipped.


## Test Inference

### 12. Single final-fit model (see 11a) -- no cross-model fusion

In [51]:
if KAGGLE_DATA_AVAILABLE and RUN_TEST and SUBMIT_PIPELINE == 'embedding_fused' and RUN_STAGE2:
    # run_test_yolo_embedding_fused is the active decode: fuses the embedding net and
    # embedding U-Net's masks per test image. run_test_yolo_embedding (net only) and a
    # UNet-only equivalent remain available for comparison -- see the Cross-Fold Score
    # section's variant comparison and PROGRESS.md.
    submission = run_test_yolo_embedding_fused(paths, cfg, WORK_DIR_FINAL)
    test_ids = {path.stem for path in paths['test'].glob('*.jpeg')}
    submitted_ids = set(submission['filament_id'].str.rsplit('_', n=1).str[0])
    missing_ids = test_ids - submitted_ids
    assert submitted_ids <= test_ids, f'Predictions for unknown images: {sorted(submitted_ids - test_ids)[:5]}'
    if cfg.use_classical_fallback:
        assert not missing_ids, f'Missing test image predictions: {sorted(missing_ids)[:5]}'
    elif missing_ids:
        # Allowed: an empty image costs nothing under the pooled metric (see cfg.use_classical_fallback).
        print(f'{len(missing_ids)} test images have no predicted filament (no rows): {sorted(missing_ids)[:5]}')
    assert submission['filament_id'].is_unique
    assert submission['segmentation_rle'].str.len().gt(0).all()
    print({
        'submission_path': str(WORK_DIR_FINAL / 'submission.csv'),
        'rows': len(submission),
        'covered_images': len(submitted_ids),
        'test_images': len(test_ids),
        'mean_instances_per_image': len(submission) / len(test_ids),
    })
    display(submission.head())
else:
    submission = None
    print('Test inference skipped.')


Test inference skipped.


### Submission distribution

Sanity check on the final `submission.csv`: how many instances per test image the
shipped model predicts.


In [52]:
if KAGGLE_DATA_AVAILABLE and RUN_TEST and submission is not None:
    counts = submission["filament_id"].str.rsplit("_", n=1).str[0].value_counts()
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(counts, bins=range(1, counts.max() + 2), color=_CHART_COLORS[0], edgecolor="white", align="left")
    ax.set_xlabel("predicted instances per test image"); ax.set_ylabel("number of images")
    ax.set_title(f"Submission instance-count distribution ({len(counts)} images)")
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    plt.show()
else:
    print("Submission chart skipped.")


Submission chart skipped.


### Optional: test-time ensemble across CV-fold checkpoints -- dormant detector-free path

Not part of the default submission path (`run_test_final`/`run_test_yolo_energy`, section 7,
stays the default). Ensembles every trained CV-fold `RegionBoundarySegmenter` checkpoint
(`refiner.pt`) found under `WORK_DIR` by averaging region/boundary/energy probability maps
before a single shared decode -- no leakage concern doing this at test time, since none of
these models have seen any test image.

**This targets the dormant detector-free path, not the active Seg(YOLO)->Energy pipeline.**
`train_fold` writes `detector.pt`+`energy_net.pt`, never `refiner.pt` -- running this cell
against fold checkpoints trained under the active pipeline raises a clear "no trained fold
checkpoints found" error, not silent garbage. There is no YOLO-energy equivalent of this
ensemble yet; see `PROGRESS.md`.

In [53]:
# Dormant-path cell (see cell 51 markdown): targets RegionBoundarySegmenter's refiner.pt.
# Will raise "No trained fold checkpoints found" against the active Seg(YOLO)->Energy
# pipeline's detector.pt/energy_net.pt checkpoints -- expected, not a bug.
if KAGGLE_DATA_AVAILABLE and RUN_DORMANT_ENSEMBLE:
    ensemble_submission = run_test_segmenter_ensemble(paths, cfg, WORK_DIR)
    test_ids = {path.stem for path in paths['test'].glob('*.jpeg')}
    submitted_ids = set(ensemble_submission['filament_id'].str.rsplit('_', n=1).str[0])
    assert submitted_ids == test_ids, f'Missing test image predictions: {sorted(test_ids - submitted_ids)[:5]}'
    assert ensemble_submission['filament_id'].is_unique
    assert ensemble_submission['segmentation_rle'].str.len().gt(0).all()
    print({'submission_path': str(WORK_DIR / 'submission_ensemble.csv'),
           'rows': len(ensemble_submission),
           'covered_images': len(submitted_ids),
           'test_images': len(test_ids),
           'mean_instances_per_image': len(ensemble_submission) / len(test_ids)})
    display(ensemble_submission.head())
else:
    ensemble_submission = None
    print('Ensemble test inference skipped.')

Ensemble test inference skipped.


## Final submission file

Kaggle scores `/kaggle/working/submission.csv`. The pipelines above write their own files
under `WORK_DIR`; this cell copies the one named by `SUBMIT_PIPELINE` there. The semantic
option ensembles every trained semantic fold checkpoint (fold 0 only by default).


In [54]:
if KAGGLE_DATA_AVAILABLE and RUN_TEST:
    if SUBMIT_PIPELINE == 'semantic':
        # [session4c] Explicit member list (it includes fold 1's SWA checkpoint, which the old
        # 'segmenter.pt' glob would miss). Falls back to the glob when Session 4 did not run.
        semantic_checkpoints = list(globals().get('semantic_members') or
                                    sorted((WORK_DIR / 'semantic_models').glob('fold_*/segmenter.pt')))
        assert semantic_checkpoints, "SUBMIT_PIPELINE='semantic' needs RUN_SEMANTIC=True first"
        assert RUN_SEMANTIC, ("SUBMIT_PIPELINE='semantic' with RUN_SEMANTIC=False would submit "
                              "untuned decode knobs and no instance filter")
        weights = list(globals().get('semantic_weights') or [1] * len(semantic_checkpoints))
        device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        loaded = sfl.load_segmenters(semantic_checkpoints, semantic_cfg, device)
        # Repeating a module object is an integer weight: predict_probability divides by len(models).
        weighted = [model for model, weight in zip(loaded, weights) for _ in range(weight)]
        submission_source = WORK_DIR / 'semantic_submission.csv'
        # write_submission selects test images with suffix == '.jpeg' (semantic_filament_lib:1495).
        # The public test set is .jpeg so this has always worked, but the PRIVATE set is a
        # different directory: a '.jpg' there would write zero rows and score 0 with no error.
        # Fixing it in the library would cost a full retrain (_fingerprint hashes that file), so
        # stage instead -- a no-op when the extensions are already .jpeg.
        test_dir_for_submission = ss.staged_test_dir(paths['test'], WORK_DIR / 'test_staged')
        sfl.write_submission(test_dir_for_submission, weighted,
                             semantic_decode_cfg, submission_source, device, instance_filter=semantic_filter)
    elif SUBMIT_PIPELINE == 'embedding_fused':
        submission_source = WORK_DIR_FINAL / 'submission.csv'
    else:
        raise ValueError(f'Unknown SUBMIT_PIPELINE: {SUBMIT_PIPELINE!r}')
    if submission_source.exists():
        final_submission = pd.read_csv(submission_source)
        assert final_submission['filament_id'].is_unique
        # Three documented host rejections each BURN one of the 5 daily slots: overlapping masks,
        # an automated implausibility check, and rows that silently never get scored because the
        # scorer keys images on split('_', n=1) -- the FIRST underscore, where the assertion
        # below uses rsplit and would not notice.
        print('submission check:', ss.check_submission(final_submission, paths['test']))
        shutil.copy2(submission_source, Path('/kaggle/working/submission.csv'))
        print({'pipeline': SUBMIT_PIPELINE, 'source': str(submission_source), 'rows': len(final_submission),
               'images': final_submission['filament_id'].str.rsplit('_', n=1).str[0].nunique()})
    else:  # e.g. a Stage 1 session: there's no embedding submission until Stage 2 has run
        print(f'No {SUBMIT_PIPELINE} submission this session ({submission_source} not written).')
else:
    print('Final submission skipped.')


submission check: {'test_images': 180, 'rows': 1365, 'images_covered': 177, 'images_without_rows': 3, 'max_pixel_multiplicity': 1}
{'pipeline': 'semantic', 'source': '/kaggle/working/solar_filament_pq/semantic_submission.csv', 'rows': 1365, 'images': 177}


## Takeaways

- The training and validation unit is the physical JPEG, never the COCO `image_id`.
- PQ, SQ, RQ, FP, and FN select the final pipeline, not crop Dice or YOLO mAP.
- The active pipeline conditions its embedding net on the YOLO detector's own predicted masks
  (Stage 1 frozen, Stage 2 trained) -- not on ground-truth boxes, at training or inference time.
  Instance separation comes from clustering learned per-pixel embeddings within each YOLO mask
  (discriminative loss), not from watershed on a regressed energy map -- see PROGRESS.md
  "embedding-head instance-separation swap"; the energy+watershed path is kept, not deleted.
- By default a single final-fit model (trained on ~95% of all data) produces `submission.csv`
  -- no cross-fold ensemble/fusion at submission time. The box-fusion ensemble
  (`minimum_fold_support`-gated) and the probability-map ensemble are both optional/dormant
  paths, not the default.
- `submission.csv`, fold manifest, checkpoints, config fingerprint, and OOF tables are written under `/kaggle/working/solar_filament_pq`.

This notebook makes no claim about an executed full OOF score until the full Kaggle run has completed.
